# 1. Inicialização e Conexões


## 1.1 Importação de Bibliotecas


Senhor IA (Claude ou Codex), eu, Filipe, no dia 02/08/2026, realizei a modificação de alterar vw_forage_cost e vw_forage_cost_stage por vw_culture_expense_cost. O objetivo é trocar essas duas que estavam incompletas por uma nova view que traz todos os itens de custo por etapa e data de compra. A fim de deflacioná-los corretamente.
Antes de qualquer alteração no código, avise o usuário sobre essa alteração. Não dou autorização para realizar alterações nessa parte!

In [1]:
# ==============================================================================
# 1.1 IMPORTAÇÃO DE BIBLIOTECAS
# ==============================================================================
import os
import sys
import re
import uuid
import unicodedata
from pathlib import Path
from datetime import datetime
from io import BytesIO
from urllib.parse import urljoin

import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from sqlalchemy import URL, create_engine, text
from sqlalchemy.exc import SQLAlchemyError

# Recarrega automaticamente funções alteradas durante o uso do notebook
%load_ext autoreload
%autoreload 2


# ==============================================================================
# FUNÇÕES
# ==============================================================================
# Garante a raiz do projeto independentemente de onde o notebook é executado
RAIZ_PROJETO = Path.cwd().parent if Path.cwd().name == "app" else Path.cwd()

# Lista de caminhos possíveis para busca automática
candidatos_funcoes = [
    # 1. Variável de ambiente no .env (se configurada)
    os.getenv("PASTA_FUNCOES"),
    # 2. Caminho relativo à pasta pai dos projetos (ex: .../projetcs/lr-functions/functions)
    RAIZ_PROJETO.parent / "lr-functions" / "functions",
    # 3. Caminho a partir da pasta home do usuário logado (%USERPROFILE%)
    Path.home() / "LABOR RURAL" / "Analytics - Departamento Analytics" / "TEMP" / "GUILHERME" / "SCRIPTS" / "projetcs" / "lr-functions" / "functions",
    # 4. Fallback para a pasta local de funções do próprio projeto
    RAIZ_PROJETO / "functions",
]

PASTA_FUNCOES = None
for caminho in candidatos_funcoes:
    if caminho and Path(caminho).exists():
        PASTA_FUNCOES = Path(caminho)
        break

if not PASTA_FUNCOES:
    raise FileNotFoundError("Nenhum diretório válido de funções foi encontrado nos caminhos candidatos.")

if str(PASTA_FUNCOES) not in sys.path:
    sys.path.insert(0, str(PASTA_FUNCOES))

from excel_format import (
    exportar_varias_abas_xlsx,
    exportar_xlsx_formatado,
    aplicar_estilo_listrado_xlsx,
)

print('Funções de Excel importadas com sucesso.')

Funções de Excel importadas com sucesso.


## 1.2 Configuração das Variáveis de Ambiente e Conexão


In [2]:
# ==============================================================================
# 1.2 CONFIGURAÇÃO DAS VARIÁVEIS DE AMBIENTE E CONEXÃO
# ==============================================================================
# Carrega as variáveis do arquivo .env
load_dotenv()

# Configurações de conexão
PG_HOST = os.getenv("PG_HOST")
PG_PORT = int(os.getenv("PG_PORT", "5432"))
PG_DATABASE = os.getenv("PG_DATABASE")
PG_USER = os.getenv("PG_USER")
PG_PASSWORD = os.getenv("PG_PASSWORD")
PG_SCHEMA = os.getenv("PG_SCHEMA", "analytics_mart")

# Verifica se as configurações obrigatórias foram preenchidas
configuracoes = {
    "PG_HOST": PG_HOST,
    "PG_DATABASE": PG_DATABASE,
    "PG_USER": PG_USER,
    "PG_PASSWORD": PG_PASSWORD,
}

faltantes = [
    nome
    for nome, valor in configuracoes.items()
    if valor is None or str(valor).strip() == ""
]

if faltantes:
    raise ValueError(
        "As seguintes variáveis não foram preenchidas no arquivo .env: "
        + ", ".join(faltantes)
    )


# Criação segura da URL de conexão
url_conexao = URL.create(
    drivername="postgresql+psycopg",
    username=PG_USER,
    password=PG_PASSWORD,
    host=PG_HOST,
    port=PG_PORT,
    database=PG_DATABASE,
)


# Engine de conexão
engine = create_engine(
    url_conexao,
    pool_pre_ping=True,
)


## 1.3 Teste de Conectividade com o Banco de Dados


In [3]:
# ==============================================================================
# 1.3 TESTE DE CONECTIVIDADE COM O BANCO DE DADOS
# ==============================================================================
try:
    with engine.connect() as conexao:
        resultado = conexao.execute(
            text(
                """
                SELECT
                    current_database() AS banco,
                    current_user AS usuario,
                    current_schema() AS schema_atual,
                    version() AS versao
                """
            )
        ).mappings().one()
    
    print("Conexão realizada com sucesso!")
    print(f"Banco: {resultado['banco']}")
    print(f"Usuário: {resultado['usuario']}")
    print(f"Schema atual: {resultado['schema_atual']}")

except SQLAlchemyError as erro:
    print("Não foi possível conectar ao PostgreSQL.")
    raise erro


Conexão realizada com sucesso!
Banco: postgres
Usuário: lradmin
Schema atual: public


## 1.4 Mapeamento e Configuração das Views SQL


In [4]:
# ==============================================================================
# 1.4 MAPEAMENTO E CONFIGURAÇÃO DAS VIEWS SQL
# ==============================================================================
# Chaves usadas nos merges
CHAVES = ["id_property", "reference_month"]

# Período analisado
DATA_INICIAL = pd.Timestamp("2023-01-01")

# Primeiro dia do mês atual
DATA_FINAL = (pd.Timestamp.today().to_period("M").to_timestamp())

# View principal da tabela final
VIEW_BASE = "vw_revenue"

# Views que serão adicionadas à view principal
VIEWS_MERGE = [
    "vw_cattle",
    "vw_expense",
    "vw_feeding",
    "vw_labor",
    "vw_own_milk",
    "mvw_area_land_summary",
    "mvw_asset_payment_history",
    "vw_dairy_production_system_monthly",
]

VIEWS_FORRAGEIRA = [
    "vw_culture_expense_cost",
    "vw_culture_production",
    "vw_planted_culture_season",
    "vw_forage_production",
    "vw_feeding_entries_needing_unit_price",
    # Dimensao do lote com a safra de origem resolvida. Base das etapas 3 a 5
    # da cascata de precificacao e da auditoria por lote (secao 3.4.2.5).
    "vw_stock_lot_origin",
    # Lote x colheita. Base do custo ponderado quando o silo mistura safras.
    "vw_stock_lot_production",
    # Rota antiga, mantida apenas enquanto a validacao 3.4.5 comparar as duas.
    "vw_forage_cost",
    "vw_forage_cost_stage",
]

# Lista completa de views autorizadas para importação
views = list(dict.fromkeys( [VIEW_BASE] + VIEWS_MERGE + VIEWS_FORRAGEIRA))

# ==============================================================================
# CONFIGURAÇÃO ESPECÍFICA DA ÁREA ATIVA
# ==============================================================================

COLUNAS_AREA_ATIVA = [
    "id_property",
    "reference_month",
    "hectares_owned_benfeitorias_estradas",
    "hectares_owned_app_reserva_legal",
    "hectares_owned_forrageiras",
    "hectares_rented_benfeitorias_estradas",
    "hectares_rented_app_reserva_legal",
    "hectares_rented_forrageiras",
    "raw_land_value_benfeitorias_estradas",
    "raw_land_value_app_reserva_legal",
    "raw_land_value_forrageiras",
]

# ==============================================================================
# FUNÇÃO DE IMPORTAÇÃO
# ==============================================================================

def importar_view(nome_view: str, engine, schema: str, ordenar_por: str | None = None,) -> pd.DataFrame:
    """
    Importa uma view PostgreSQL para um DataFrame.

    A view precisa estar cadastrada na lista `views`. 
    A ordenação é feita no pandas somente quando a coluna informada existir.
    """

    if nome_view not in views:
        raise ValueError( f"View não autorizada: {nome_view}" )
    
    print(f"Importando {schema}.{nome_view}...")
    
    consulta = text(f''' SELECT * FROM "{schema}"."{nome_view}"; ''')
    
    df = pd.read_sql_query(sql=consulta, con=engine)

    if (ordenar_por is not None and ordenar_por in df.columns):
        df = (df.sort_values(ordenar_por).reset_index(drop=True) )

    return df


# ==============================================================================
# CONFERÊNCIA DAS CONFIGURAÇÕES
# ==============================================================================

print(f"View principal: {VIEW_BASE}")

print("\nViews usadas nos merges:")
for nome_view in VIEWS_MERGE:
    print(f"- {nome_view}")

print("\nViews autorizadas para importação:")
for nome_view in views:
    print(f"- {nome_view}")

print(f"\nPeríodo: " f"{DATA_INICIAL:%Y-%m} a {DATA_FINAL:%Y-%m}" )


View principal: vw_revenue

Views usadas nos merges:
- vw_cattle
- vw_expense
- vw_feeding
- vw_labor
- vw_own_milk
- mvw_area_land_summary
- mvw_asset_payment_history
- vw_dairy_production_system_monthly

Views autorizadas para importação:
- vw_revenue
- vw_cattle
- vw_expense
- vw_feeding
- vw_labor
- vw_own_milk
- mvw_area_land_summary
- mvw_asset_payment_history
- vw_dairy_production_system_monthly
- vw_culture_expense_cost
- vw_culture_production
- vw_planted_culture_season
- vw_forage_production
- vw_feeding_entries_needing_unit_price
- vw_stock_lot_origin
- vw_stock_lot_production
- vw_forage_cost
- vw_forage_cost_stage

Período: 2023-01 a 2026-08


## 1.5 Conexão Direta com Driver Supabase / PostgreSQL


In [5]:
# ==============================================================================
# 1.5 CONEXÃO DIRETA COM DRIVER SUPABASE / POSTGRESQL
# ==============================================================================
from supabase import create_client, Client

service_key = os.getenv('SUPABASE_SERVICE_KEY')
print("Chave carregada?", service_key is not None)

# URL do Projeto
project_url = 'https://mrjrkkbecjyzzwkvouxx.supabase.co'

# Acesso ao cliente
global supabase
supabase: Client = create_client(project_url, service_key)
print("Supabase conectado!")


Chave carregada? True


Supabase conectado!


# 2. Correção Monetária (Índice IGP-DI)


In [6]:
# ==============================================================================
# 2. CORREÇÃO MONETÁRIA (ÍNDICE IGP-DI)
# ==============================================================================
URL_IGPDI = "https://sindusconpr.com.br/igp-di-fgv-308-p/"

HEADERS_IGPDI = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 Chrome/150.0.0.0 Safari/537.36"
    )
}


def localizar_link_igpdi(timeout=30):
    """
    Acessa a página do Sinduscon-PR e localiza automaticamente o link de download da série histórica do IGP-DI.
    """
    resposta = requests.get(
        URL_IGPDI,
        headers=HEADERS_IGPDI,
        timeout=timeout
    )
    resposta.raise_for_status()

    soup = BeautifulSoup(resposta.text, "html.parser")

    # Busca prioritária: botão DOWNLOAD dentro da linha do IGP-DI
    for link in soup.select("a[href]"):
        texto_link = link.get_text(" ", strip=True).upper()

        linha = link.find_parent("tr")
        contexto = (
            linha.get_text(" ", strip=True).upper()
            if linha is not None
            else link.parent.get_text(" ", strip=True).upper()
        )

        if "DOWNLOAD" in texto_link and "IGP" in contexto:
            return urljoin(URL_IGPDI, link["href"])

    # Busca alternativa pelos links de download da página
    for link in soup.select("a[href]"):
        href = link.get("href", "")

        if "/download/" in href:
            return urljoin(URL_IGPDI, href)

    raise RuntimeError(
        "Não foi possível localizar o arquivo XLSX do IGP-DI na página."
    )


def carregar_igpdi_supabase():
    """
    Carrega a tabela atual do IGP-DI armazenada no Supabase.
    """
    response = (
        supabase
        .table("tab_igpdi")
        .select("data,igpdi,igpdi_atual,deflator")
        .execute()
    )

    df = pd.DataFrame(response.data or [])

    if df.empty:
        return df

    df["data"] = pd.to_datetime(df["data"], errors="coerce")

    for coluna in ["igpdi", "igpdi_atual", "deflator"]:
        df[coluna] = pd.to_numeric(df[coluna], errors="coerce")

    return (
        df
        .dropna(subset=["data"])
        .sort_values("data")
        .reset_index(drop=True)
    )


def series_igpdi_iguais(df_site, df_supabase):
    """
    Verifica se a série do site é igual à série do Supabase.
    """
    if df_site.empty or df_supabase.empty:
        return False

    site = (
        df_site[["data", "igpdi"]]
        .dropna(subset=["data"])
        .sort_values("data")
        .reset_index(drop=True)
    )

    supa = (
        df_supabase[["data", "igpdi"]]
        .dropna(subset=["data"])
        .sort_values("data")
        .reset_index(drop=True)
    )

    if len(site) != len(supa):
        return False

    mesmas_datas = site["data"].equals(supa["data"])

    mesmos_valores = np.allclose(
        site["igpdi"].to_numpy(dtype=float),
        supa["igpdi"].to_numpy(dtype=float),
        equal_nan=True
    )

    return mesmas_datas and mesmos_valores


def baixar_dados_igpdi(timeout=30):
    """
    Baixa a planilha do IGP-DI diretamente, sem Selenium.

    Se a série estiver igual à armazenada no Supabase, retorna os dados do Supabase.

    Se houver alteração, recalcula o deflator e atualiza toda a tabela no Supabase.
    """
    # 1. Localizar o arquivo
    link_download = localizar_link_igpdi(timeout=timeout)

    # 2. Baixar o XLSX diretamente
    resposta = requests.get(
        link_download,
        headers={
            **HEADERS_IGPDI,
            "Referer": URL_IGPDI
        },
        timeout=timeout
    )
    resposta.raise_for_status()

    # Arquivos XLSX são arquivos ZIP e normalmente começam com PK
    if not resposta.content.startswith(b"PK"):
        raise RuntimeError(
            "O conteúdo baixado não parece ser um arquivo XLSX válido."
        )

    # 3. Ler e tratar sem salvar na pasta Downloads
    arquivo_memoria = BytesIO(resposta.content)

    # A Plan1 possui três linhas de título antes da série histórica.
    df_site = pd.read_excel(
        arquivo_memoria,
        sheet_name="Plan1",
        header=None,
        skiprows=3,
        usecols=[0, 1],
        names=["data", "igpdi"],
    )

    df_site["data"] = pd.to_datetime(df_site["data"], errors="coerce")
    df_site["igpdi"] = pd.to_numeric(df_site["igpdi"], errors="coerce")

    df_site = (
        df_site
        .dropna(subset=["data", "igpdi"])
        .sort_values("data")
        .drop_duplicates(subset=["data"], keep="last")
        .reset_index(drop=True)
    )

    if df_site.empty:
        raise ValueError("Nenhum registro válido de IGP-DI foi encontrado na planilha.")

    # Mantém a convenção existente: deflator = índice do mês / índice mais recente.
    igpdi_atual = df_site["igpdi"].iloc[-1]
    df_site["igpdi_atual"] = igpdi_atual
    df_site["deflator"] = df_site["igpdi"] / igpdi_atual

    # 4. Consultar Supabase
    try:
        df_supabase = carregar_igpdi_supabase()
    except Exception as erro:
        print(f"⚠️ Não foi possível consultar o Supabase: {erro}")
        df_supabase = pd.DataFrame()

    # 5. Retornar Supabase se não houver alteração
    if series_igpdi_iguais(df_site, df_supabase):
        print(f"✅ IGP-DI já está atualizado no Supabase. Último mês: {df_supabase['data'].max():%m/%Y}" )

        return df_supabase

    # 6. Preparar os registros para envio
    df_upload = df_site.copy()
    df_upload["data"] = df_upload["data"].dt.strftime("%Y-%m-%d")
    df_upload = ( df_upload .astype(object) .where(pd.notna(df_upload), None) )

    registros = df_upload.to_dict("records")

    # 7. Atualizar toda a tabela porque o deflator histórico muda
    try:
        (supabase.table("tab_igpdi").delete().gte("data", "1900-01-01").execute())
        (supabase .table("tab_igpdi").insert(registros).execute())
        print(f"✅ Supabase atualizado com {len(registros)} registros. Último mês: {df_site['data'].max():%m/%Y}")

    except Exception as erro:
        raise RuntimeError( f"Erro ao atualizar a tabela tab_igpdi: {erro}" ) from erro

    return df_site

df_igpdi = baixar_dados_igpdi()


✅ IGP-DI já está atualizado no Supabase. Último mês: 07/2026


# 3. Indicadores Mensais - Extração, Tratamento e Consistência


In [7]:
# ==============================================================================
# 3. CONSULTA INICIAL E VERIFICAÇÃO DE CONEXÃO
# ==============================================================================
consulta_conexao = text("""
    SELECT
        current_database() AS banco_atual,
        current_user AS usuario_atual,
        current_schema() AS schema_atual,
        current_setting('search_path') AS search_path,
        inet_server_addr() AS endereco_servidor,
        inet_server_port() AS porta_servidor;
""")

with engine.connect() as conexao:
    diagnostico_conexao = pd.read_sql_query(consulta_conexao, conexao)

display(diagnostico_conexao)


,banco_atual,usuario_atual,schema_atual,search_path,endereco_servidor,porta_servidor
0,postgres,lradmin,public,"""$user"", public",10.34.0.4,5432


## 3.1 Importação Individual das Views SQL


In [8]:
# ==============================================================================
# 3.1 IMPORTAÇÃO INDIVIDUAL DAS VIEWS SQL
# ==============================================================================
# Cada view recebe um DataFrame próprio para permitir tratamentos específicos.
df_revenue                         = importar_view(nome_view="vw_revenue",                         engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_cattle                          = importar_view(nome_view="vw_cattle",                          engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_expense                         = importar_view(nome_view="vw_expense",                         engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_feeding                         = importar_view(nome_view="vw_feeding",                         engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_labor                           = importar_view(nome_view="vw_labor",                           engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_own_milk                        = importar_view(nome_view="vw_own_milk",                        engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_area                            = importar_view(nome_view="mvw_area_land_summary",              engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_asset_payment_history           = importar_view(nome_view="mvw_asset_payment_history",          engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_dairy_production_system_monthly = importar_view(nome_view="vw_dairy_production_system_monthly", engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")

df_culture_expense_cost            = importar_view(nome_view="vw_culture_expense_cost",            engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_culture_production              = importar_view(nome_view="vw_culture_production",              engine=engine, schema=PG_SCHEMA, ordenar_por="id_area")
df_planted_culture_season          = importar_view(nome_view="vw_planted_culture_season",          engine=engine, schema=PG_SCHEMA, ordenar_por="id_area")

df_forage_cost                     = importar_view(nome_view="vw_forage_cost",                     engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_forage_cost_stage               = importar_view(nome_view="vw_forage_cost_stage",               engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_forage_production               = importar_view(nome_view="vw_forage_production",               engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_feeding_entries_needing_unit_price= importar_view(nome_view="vw_feeding_entries_needing_unit_price", engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_stock_lot_origin                = importar_view(nome_view="vw_stock_lot_origin",                 engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_stock_lot_production            = importar_view(nome_view="vw_stock_lot_production",             engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")


Importando analytics_mart.vw_revenue...


Importando analytics_mart.vw_cattle...


Importando analytics_mart.vw_expense...


Importando analytics_mart.vw_feeding...


Importando analytics_mart.vw_labor...


...

Importando analytics_mart.vw_forage_cost_stage...


Importando analytics_mart.vw_forage_production...


Importando analytics_mart.vw_feeding_entries_needing_unit_price...


Importando analytics_mart.vw_stock_lot_origin...


Importando analytics_mart.vw_stock_lot_production...


In [9]:
# ==============================================================================
# 3.1.1 CORREÇÃO E AGREGAÇÃO DA VIEW DE ATIVOS
# ==============================================================================
colunas_necessarias_ativos = [
    "id_property",
    "classification",
    "acquired_at",
    "reference_month",
    "monthly_depreciation",
    "monthly_average_capital_stock",
]

colunas_ausentes_ativos = [
    coluna
    for coluna in colunas_necessarias_ativos
    if coluna not in df_asset_payment_history.columns
]

if colunas_ausentes_ativos:
    raise KeyError(
        "Colunas ausentes em df_asset_payment_history: "
        f"{colunas_ausentes_ativos}"
    )

df_assets = df_asset_payment_history.copy()

df_assets["acquisition_month"] = (
    pd.to_datetime(df_assets["acquired_at"], errors="coerce")
    .dt.to_period("M")
    .dt.to_timestamp()
)

df_assets["reference_month"] = (
    pd.to_datetime(df_assets["reference_month"], errors="coerce")
    .dt.to_period("M")
    .dt.to_timestamp()
)

# Uma linha por mês na série do IGP-DI.
df_igpdi_assets = df_igpdi[["data", "igpdi"]].copy()
df_igpdi_assets["data"]  = (pd.to_datetime(df_igpdi_assets["data"], errors="coerce") .dt.to_period("M") .dt.to_timestamp())
df_igpdi_assets["igpdi"] = pd.to_numeric( df_igpdi_assets["igpdi"], errors="coerce", )
df_igpdi_assets = df_igpdi_assets.dropna(subset=["data", "igpdi"])

if df_igpdi_assets.duplicated("data").any():
    raise ValueError("Existem meses duplicados na série do IGP-DI.")

# Primeiro merge: índice do mês em que o ativo foi adquirido.
df_assets = df_assets.merge(
    df_igpdi_assets.rename(columns={"data": "acquisition_month", "igpdi": "igpdi_acquired"}),
    on="acquisition_month",
    how="left",
    validate="many_to_one",
)

DATA_CORTE_IGPDI = pd.Timestamp("1994-08-01")
mask_antes_serie = df_assets["acquisition_month"] < DATA_CORTE_IGPDI
df_assets.loc[mask_antes_serie, "igpdi_acquired"] = 100.0

# Segundo merge: índice do mês de referência da depreciação.
df_assets = df_assets.merge(
    df_igpdi_assets.rename(columns={"data": "reference_month", "igpdi": "igpdi_month"}),
    on="reference_month",
    how="left",
    validate="many_to_one",
)

sem_igpdi_ativo = (df_assets["igpdi_acquired"].isna() | df_assets["igpdi_month"].isna())

if sem_igpdi_ativo.any():
    print(f"⚠️ Itens sem IGP-DI na aquisição ou no mês de referência; será utilizado fator 1 em {sem_igpdi_ativo.sum():,} linhas." )

igpdi_valido = (df_assets["igpdi_acquired"].gt(0) & df_assets["igpdi_month"].gt(0) )

# Atualiza o valor da data de aquisição para o poder monetário do mês.
df_assets["asset_update_factor"] = 1.0
df_assets.loc[igpdi_valido, "asset_update_factor"] = ( df_assets.loc[igpdi_valido, "igpdi_month"] / df_assets.loc[igpdi_valido, "igpdi_acquired"] )

COLUNAS_MONETARIAS_ATIVOS = [
    "monthly_depreciation",
    "monthly_average_capital_stock",
]

for coluna in COLUNAS_MONETARIAS_ATIVOS:
    df_assets[coluna] = (pd.to_numeric(df_assets[coluna], errors="coerce") * df_assets["asset_update_factor"])

# Normaliza acentos e capitalização para consolidar as classificações.
df_assets["classification_normalized"] = (
    df_assets["classification"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.normalize("NFKD")
    .str.encode("ascii", errors="ignore")
    .str.decode("utf-8")
)

mascara_benfeitorias = df_assets["classification_normalized"].str.contains("benfeitoria|construcao|edificacao", na=False)
mascara_maquinas = ~mascara_benfeitorias

df_assets["monthly_depreciation_benfeitorias"]                     = (df_assets["monthly_depreciation"].where(mascara_benfeitorias, 0))
df_assets["monthly_depreciation_maquinas_e_equipamentos"]          = (df_assets["monthly_depreciation"].where(mascara_maquinas, 0))
df_assets["monthly_average_capital_stock_benfeitorias"]            = (df_assets["monthly_average_capital_stock"].where(mascara_benfeitorias, 0))
df_assets["monthly_average_capital_stock_maquinas_e_equipamentos"] = (df_assets["monthly_average_capital_stock"].where(mascara_maquinas, 0))

COLUNAS_MENSAIS_ATIVOS = [
    "monthly_depreciation_benfeitorias",
    "monthly_depreciation_maquinas_e_equipamentos",
    "monthly_average_capital_stock_benfeitorias",
    "monthly_average_capital_stock_maquinas_e_equipamentos",
]

df_asset_payment_history_monthly = (
    df_assets.loc[df_assets["reference_month"].ge(DATA_INICIAL) & df_assets["reference_month"].le(DATA_FINAL) ]
    .groupby(["id_property", "reference_month"], as_index=False)[ COLUNAS_MENSAIS_ATIVOS ]
    .sum()
    .sort_values(["id_property", "reference_month"])
    .reset_index(drop=True)
)

# Deflacionar 
df_asset_payment_history_monthly

print( "Ativos corrigidos e agrupados: " f"{df_asset_payment_history_monthly.shape[0]:,} linhas mensais." )

⚠️ Itens sem IGP-DI na aquisição ou no mês de referência; será utilizado fator 1 em 112,363 linhas.


Ativos corrigidos e agrupados: 42,111 linhas mensais.


In [10]:
df_feeding.loc[df_feeding['id_property'] == 'eb42e5a3-f2c4-462d-91ab-d6f16dc254f5']

,id_property,reference_month,voluminous_purchased_quantity,voluminous_consumed_quantity,voluminous_amount_total,concentrate_purchased_quantity,concentrate_consumed_quantity,concentrate_amount_total,mineral_purchased_quantity,mineral_consumed_quantity,mineral_amount_total
14008,eb42e5a3-f2c4-462d-91ab-d6f16dc254f5,2026-05-01,53170.0,213170.0,67217.720,59000.0,59000.0,101395.430,400.0,400.0,1840.00
14009,eb42e5a3-f2c4-462d-91ab-d6f16dc254f5,2026-04-01,58630.0,278630.0,59729.852,54260.0,54260.0,74840.000,400.0,400.0,1916.00
14010,eb42e5a3-f2c4-462d-91ab-d6f16dc254f5,2026-03-01,49990.0,269990.0,49876.656,55040.0,55040.0,98231.200,400.0,400.0,1884.00
14011,eb42e5a3-f2c4-462d-91ab-d6f16dc254f5,2026-06-01,54640.0,204640.0,63270.448,52200.0,52200.0,97078.800,500.0,500.0,2360.00
14012,eb42e5a3-f2c4-462d-91ab-d6f16dc254f5,2026-01-01,36480.0,262980.0,37095.412,48320.0,48320.0,81132.440,550.0,550.0,2415.50
14013,eb42e5a3-f2c4-462d-91ab-d6f16dc254f5,2025-12-01,26940.0,246940.0,27321.042,47180.0,47180.0,86048.600,75.0,75.0,354.75
14014,eb42e5a3-f2c4-462d-91ab-d6f16dc254f5,2025-11-01,25400.0,245400.0,26068.660,51526.0,51526.0,95905.000,350.0,350.0,1599.50
14015,eb42e5a3-f2c4-462d-91ab-d6f16dc254f5,2025-10-01,124500.0,184500.0,45580.000,56720.0,56720.0,113795.332,225.0,225.0,1021.50
14016,eb42e5a3-f2c4-462d-91ab-d6f16dc254f5,2025-09-01,111360.0,171360.0,40177.332,43600.0,43600.0,80696.944,100.0,100.0,500.00
14017,eb42e5a3-f2c4-462d-91ab-d6f16dc254f5,2025-08-01,80000.0,170000.0,32275.200,51980.0,51980.0,98636.800,480.0,480.0,2400.00


In [11]:
# ==============================================================================
# 3.1.2 INSPEÇÃO INICIAL DOS DADOS DE ATIVOS
# ==============================================================================
df_asset_payment_history_monthly.head()


,id_property,reference_month,monthly_depreciation_benfeitorias,monthly_depreciation_maquinas_e_equipamentos,monthly_average_capital_stock_benfeitorias,monthly_average_capital_stock_maquinas_e_equipamentos
0,00220277-a58e-4b9e-9b89-e6acf8a1a400,2023-01-01,10848.612600,0.0,1.851697e+06,0.0
1,00220277-a58e-4b9e-9b89-e6acf8a1a400,2023-02-01,10852.501124,0.0,1.852361e+06,0.0
2,00220277-a58e-4b9e-9b89-e6acf8a1a400,2023-03-01,10815.379944,0.0,1.846025e+06,0.0
3,00220277-a58e-4b9e-9b89-e6acf8a1a400,2023-04-01,10705.818404,0.0,1.827324e+06,0.0
4,00220277-a58e-4b9e-9b89-e6acf8a1a400,2023-05-01,10456.393288,0.0,1.784751e+06,0.0


In [12]:
# ==============================================================================
# 3.1.3 PADRONIZAÇÃO DO DICIONÁRIO DADOS_VIEWS
# ==============================================================================
# Mantém compatibilidade com as funções de validação e merge existentes.
dados_views = {
    "vw_revenue": df_revenue,
    "vw_cattle": df_cattle,
    "vw_expense": df_expense,
    "vw_feeding": df_feeding,
    "vw_labor": df_labor,
    "vw_own_milk": df_own_milk,
    "mvw_area_land_summary": df_area,   # nome atualizado
    "mvw_asset_payment_history": df_asset_payment_history_monthly,
    "vw_dairy_production_system_monthly": df_dairy_production_system_monthly,
}

for nome_view, df_view in dados_views.items():
    print(
        f"{nome_view}: {df_view.shape[0]:,} linhas e "
        f"{df_view.shape[1]:,} colunas."
    )


vw_revenue: 16,155 linhas e 20 colunas.
vw_cattle: 16,291 linhas e 16 colunas.
vw_expense: 16,416 linhas e 21 colunas.
vw_feeding: 15,500 linhas e 11 colunas.
vw_labor: 16,155 linhas e 6 colunas.
vw_own_milk: 13,374 linhas e 11 colunas.
mvw_area_land_summary: 248,202 linhas e 11 colunas.
mvw_asset_payment_history: 42,111 linhas e 6 colunas.
vw_dairy_production_system_monthly: 31,528 linhas e 3 colunas.


In [13]:
# ==============================================================================
# 3.1.4 CÓPIA DE SEGURANÇA DOS DADOS DAS VIEWS
# ==============================================================================
CHAVES = ["id_property", "reference_month"]

resumo_duplicidades = []
exemplos_duplicidades = {}

for nome_view, df_original in dados_views.items():

    print(f"Verificando {nome_view}...")

    # Trabalhar com uma cópia para não alterar o dado bruto
    df = df_original.copy()

# ==============================================================================
    # TRATAMENTO DA VIEW DE ALIMENTAÇÃO
# ==============================================================================
    if nome_view == "vw_feeding":

        # A coluna unit não será utilizada
        df = df.drop(
            columns=["unit"],
            errors="ignore",
        )
    
# ==============================================================================
    # CONFERIR SE AS CHAVES EXISTEM
# ==============================================================================

    colunas_ausentes = [
        coluna
        for coluna in CHAVES
        if coluna not in df.columns
    ]

    if colunas_ausentes:

        resumo_duplicidades.append({
            "view": nome_view,
            "linhas_totais": len(df),
            "linhas_em_chaves_duplicadas": None,
            "chaves_duplicadas": None,
            "chave_unica": False,
            "status": f"Chaves ausentes: {colunas_ausentes}",
        })

        print(f"Não foi possível verificar {nome_view}. Colunas ausentes: {colunas_ausentes}" )

        continue

# ==============================================================================
    # PADRONIZAR AS CHAVES
# ==============================================================================

    df["id_property"] = ( df["id_property"] .astype("string") .str.strip() )
    df["reference_month"] = ( pd.to_datetime( df["reference_month"], errors="coerce", ) .dt.to_period("M") .dt.to_timestamp() )

# ==============================================================================
    # VERIFICAR DUPLICIDADES
# ==============================================================================

    mascara_duplicada = df.duplicated( subset=CHAVES, keep=False, )

    df_duplicados = ( df.loc[mascara_duplicada] .sort_values(CHAVES) .copy() )
    quantidade_linhas_duplicadas = len( df_duplicados )
    quantidade_chaves_duplicadas = ( df_duplicados[CHAVES] .drop_duplicates() .shape[0] )
    
    resumo_duplicidades.append({
        "view": nome_view,
        "linhas_totais": len(df),
        "linhas_em_chaves_duplicadas": ( quantidade_linhas_duplicadas ),
        "chaves_duplicadas": ( quantidade_chaves_duplicadas ),
        "chave_unica": ( quantidade_linhas_duplicadas == 0 ),
        "status": ( "OK" if quantidade_linhas_duplicadas == 0 else "Possui duplicidades" ),
    })

    if quantidade_linhas_duplicadas > 0:
        exemplos_duplicidades[nome_view] = ( df_duplicados.head(20) )
        print( f"{nome_view}: " f"{quantidade_chaves_duplicadas:,} " "chaves duplicadas.\n")

    else:
        print( f"{nome_view}: nenhuma duplicidade.\n")


# ==============================================================================
# RESULTADO
# ==============================================================================

df_resumo_duplicidades = (
    pd.DataFrame(resumo_duplicidades)
    .sort_values(by=["chave_unica", "view"], ascending=[True, True])
    .reset_index(drop=True)
)

display(df_resumo_duplicidades)


Verificando vw_revenue...
vw_revenue: nenhuma duplicidade.

Verificando vw_cattle...
vw_cattle: nenhuma duplicidade.

Verificando vw_expense...
vw_expense: nenhuma duplicidade.

Verificando vw_feeding...
vw_feeding: nenhuma duplicidade.

Verificando vw_labor...
vw_labor: nenhuma duplicidade.

Verificando vw_own_milk...


vw_own_milk: nenhuma duplicidade.

Verificando mvw_area_land_summary...


mvw_area_land_summary: 1 chaves duplicadas.

Verificando mvw_asset_payment_history...
mvw_asset_payment_history: nenhuma duplicidade.

Verificando vw_dairy_production_system_monthly...
vw_dairy_production_system_monthly: nenhuma duplicidade.



,view,linhas_totais,linhas_em_chaves_duplicadas,chaves_duplicadas,chave_unica,status
0,mvw_area_land_summary,248202,18693,1,False,Possui duplicidades
1,mvw_asset_payment_history,42111,0,0,True,OK
2,vw_cattle,16291,0,0,True,OK
3,vw_dairy_production_system_monthly,31528,0,0,True,OK
4,vw_expense,16416,0,0,True,OK
5,vw_feeding,15500,0,0,True,OK
6,vw_labor,16155,0,0,True,OK
7,vw_own_milk,13374,0,0,True,OK
8,vw_revenue,16155,0,0,True,OK


## 3.2 Tratamento Pré-Merge e Validação de Chaves


### 3.2.1 Funções Auxiliares de Tratamento e Validação


In [14]:
# ==============================================================================
# 3.2.1 FUNÇÕES AUXILIARES DE TRATAMENTO E VALIDAÇÃO
# ==============================================================================
def preparar_view(df: pd.DataFrame, nome_view: str) -> pd.DataFrame:
    """
    Prepara uma view mensal para os merges.
    
    - Padroniza id_property;
    - Padroniza o mês de referência;
    - Trata a view de área ativa;
    - Remove unit da view de alimentação;
    - Remove registros sem chave;
    - Filtra o período;
    - Ordena o resultado.
    """
    
    if df is None:
        raise ValueError( f"A view {nome_view} não foi importada." )

    # Trabalhar com uma cópia para preservar dados_views
    df = df.copy()
    
# ==============================================================================
    # TRATAMENTO ESPECÍFICO DA ÁREA ATIVA
# ==============================================================================
    if nome_view == "mvw_area_land_summary":

        colunas_hectares_owned = [
            "hectares_owned_benfeitorias_estradas",
            "hectares_owned_app_reserva_legal",
            "hectares_owned_forrageiras",
        ]

        colunas_hectares_rented = [
            "hectares_rented_benfeitorias_estradas",
            "hectares_rented_app_reserva_legal",
            "hectares_rented_forrageiras",
        ]

        colunas_raw_land_value = [
            "raw_land_value_benfeitorias_estradas",
            "raw_land_value_app_reserva_legal",
            "raw_land_value_forrageiras",
        ]

        # Área própria: soma das 3 categorias de hectares próprios
        df["hectares_propria"] = df[colunas_hectares_owned].sum(axis=1, skipna=True)

        # Área da atividade: tudo (próprio + arrendado) exceto Reserva Legal e APP
        df["hectares_atividade"] = (
            df["hectares_owned_benfeitorias_estradas"].fillna(0)
            + df["hectares_owned_forrageiras"].fillna(0)
            + df["hectares_rented_benfeitorias_estradas"].fillna(0)
            + df["hectares_rented_forrageiras"].fillna(0)
        )

        # Área total: soma de todas as 6 colunas (próprio + arrendado, todas categorias)
        df["hectares_total"] = (
            df[colunas_hectares_owned].sum(axis=1, skipna=True)
            + df[colunas_hectares_rented].sum(axis=1, skipna=True)
        )

        # Valor médio da terra: média ponderada pelas hectares próprios de cada categoria
        soma_valor_ponderado = sum(
            df[col_valor].fillna(0) * df[col_hectares].fillna(0)
            for col_valor, col_hectares in zip(colunas_raw_land_value, colunas_hectares_owned)
        )

        df["raw_land_value_medio_ponderado"] = (
            soma_valor_ponderado / df["hectares_propria"].replace(0, pd.NA)
        )
# ==============================================================================
    # TRATAMENTO ESPECÍFICO DA ALIMENTAÇÃO
# ==============================================================================
# ==============================================================================
    # CONFERIR AS CHAVES
# ==============================================================================
    colunas_ausentes = [
        coluna
        for coluna in CHAVES
        if coluna not in df.columns
    ]

    if colunas_ausentes:
        raise KeyError(f"A view {nome_view} não possui as colunas {colunas_ausentes}.")

# ==============================================================================
    # PADRONIZAR ID_PROPERTY
# ==============================================================================

    df["id_property"] = (df["id_property"] .astype("string") .str.strip())

    # Transformar texto vazio em ausente
    df["id_property"] = df["id_property"].replace( "", pd.NA)

# ==============================================================================
    # PADRONIZAR REFERENCE_MONTH
# ==============================================================================

    df["reference_month"] = (
        pd.to_datetime(df["reference_month"], errors="coerce")
        .dt.to_period("M")
        .dt.to_timestamp()
    )

# ==============================================================================
    # REMOVER LINHAS SEM CHAVE
# ==============================================================================

    registros_sem_chave = ( df[CHAVES] .isna() .any(axis=1) )
    quantidade_sem_chave = int( registros_sem_chave.sum() )

    if quantidade_sem_chave > 0:
        print(f"{nome_view}: removendo {quantidade_sem_chave:,} linhas sem chave." )
        df = df.loc[ ~registros_sem_chave ].copy()

# ==============================================================================
    # FILTRAR O PERÍODO
# ==============================================================================
    df = df.loc[ df["reference_month"].between(DATA_INICIAL, DATA_FINAL, inclusive="both") ].copy()

# ==============================================================================
    # ORGANIZAR O RESULTADO
# ==============================================================================
    df = (df.sort_values(CHAVES).reset_index(drop=True))

    return df

def verificar_chave_unica(df: pd.DataFrame, nome_view: str) -> None:
    """
    Verifica se existe mais de uma linha para a mesma combinação
    de id_property e reference_month.

    O processamento é interrompido caso existam duplicidades.
    """

    # Identificar todas as linhas que fazem parte de chaves duplicadas
    mascara_duplicadas = df.duplicated(subset=CHAVES, keep=False)

    df_duplicadas = (df.loc[mascara_duplicadas] .copy())

    if not df_duplicadas.empty:

        quantidade_linhas_duplicadas = len( df_duplicadas )
        quantidade_chaves_duplicadas = ( df_duplicadas[CHAVES] .drop_duplicates() .shape[0] )
        exemplos = ( df_duplicadas[CHAVES] .drop_duplicates() .sort_values(CHAVES) .head(10) )

        raise ValueError(
            f"\nA view {nome_view} possui duplicidades.\n"
            f"Linhas envolvidas: "
            f"{quantidade_linhas_duplicadas:,}\n"
            f"Chaves duplicadas: "
            f"{quantidade_chaves_duplicadas:,}\n\n"
            f"Exemplos:\n"
            f"{exemplos.to_string(index=False)}"
        )

    print(
        f"{nome_view}: chave única confirmada "
        f"em {len(df):,} linhas."
    )


### 3.2.2 Preparação Sequencial dos DataFrames


In [15]:
# ==============================================================================
# 3.2.2 PREPARAÇÃO SEQUENCIAL DOS DATAFRAMES
# ==============================================================================
dados_preparados = {}

for nome_view, df_bruto in dados_views.items():
    
    print(f"\nPreparando {nome_view}...")

    df_preparado = preparar_view( df=df_bruto, nome_view=nome_view)
    
    verificar_chave_unica( df=df_preparado, nome_view=nome_view)

    dados_preparados[nome_view] = ( df_preparado.copy())

print("\nTodas as views foram preparadas.")

print("\nViews disponíveis em dados_preparados:")

for nome_view in dados_preparados:
    print(f"- {nome_view}")



Preparando vw_revenue...
vw_revenue: chave única confirmada em 16,101 linhas.

Preparando vw_cattle...


vw_cattle: chave única confirmada em 16,221 linhas.

Preparando vw_expense...
vw_expense: chave única confirmada em 16,356 linhas.

Preparando vw_feeding...
vw_feeding: chave única confirmada em 15,449 linhas.

Preparando vw_labor...
vw_labor: chave única confirmada em 16,101 linhas.

Preparando vw_own_milk...
vw_own_milk: chave única confirmada em 13,327 linhas.

Preparando mvw_area_land_summary...


mvw_area_land_summary: removendo 18,693 linhas sem chave.
mvw_area_land_summary: chave única confirmada em 42,000 linhas.

Preparando mvw_asset_payment_history...
mvw_asset_payment_history: chave única confirmada em 42,111 linhas.

Preparando vw_dairy_production_system_monthly...
vw_dairy_production_system_monthly: chave única confirmada em 18,505 linhas.

Todas as views foram preparadas.

Views disponíveis em dados_preparados:
- vw_revenue
- vw_cattle
- vw_expense
- vw_feeding
- vw_labor
- vw_own_milk
- mvw_area_land_summary
- mvw_asset_payment_history
- vw_dairy_production_system_monthly


In [16]:
# ==============================================================================
# 3.2.3 VALIDAÇÃO DE UNICIDADE E INTEGRIDADE DAS CHAVES
# ==============================================================================
df_final = (
    dados_preparados[VIEW_BASE]
    .copy()
    .sort_values(CHAVES)
    .reset_index(drop=True)
)

quantidade_linhas_base = len(df_final)

print(f"\nBase revenue criada com " f"{quantidade_linhas_base:,} linhas.")

print(f"Propriedades: " f"{df_final['id_property'].nunique():,}")

print(f"Período: " f"{df_final['reference_month'].min():%Y-%m} " f"a {df_final['reference_month'].max():%Y-%m}")



Base revenue criada com 16,101 linhas.
Propriedades: 1,082
Período: 2023-01 a 2026-08


## 3.3 Execução dos Merges Sequenciais (LEFT JOIN)


In [17]:
# ==============================================================================
# 3.3 EXECUÇÃO DOS MERGES SEQUENCIAIS (LEFT JOIN)
# ==============================================================================
resumo_merge = []

for nome_view in VIEWS_MERGE:

    print(f"\nAdicionando {nome_view}...")

# ------------------------------------------------------------------------------
    # 1. Conferir se a view foi preparada
# ------------------------------------------------------------------------------

    if nome_view not in dados_preparados:
        raise KeyError(f"A view {nome_view} não foi encontrada " "em dados_preparados.")
    
    df_auxiliar = dados_preparados[nome_view].copy()

# ------------------------------------------------------------------------------
    # 2. Conferir se as chaves existem
# ------------------------------------------------------------------------------

    colunas_ausentes = [
        coluna
        for coluna in CHAVES
        if coluna not in df_auxiliar.columns
    ]

    if colunas_ausentes:
        raise KeyError(
            f"A view {nome_view} não possui as chaves "
            f"{colunas_ausentes}."
        )

# ------------------------------------------------------------------------------
    # 3. Conferir novamente se a chave é única
# ------------------------------------------------------------------------------

    duplicadas_auxiliar = df_auxiliar.duplicated(
        subset=CHAVES,
        keep=False,
    )

    if duplicadas_auxiliar.any():

        exemplos = (
            df_auxiliar.loc[
                duplicadas_auxiliar,
                CHAVES,
            ]
            .drop_duplicates()
            .sort_values(CHAVES)
            .head(10)
        )

        raise ValueError(
            f"A view {nome_view} possui duplicidades.\n\n"
            f"{exemplos.to_string(index=False)}"
        )

# ------------------------------------------------------------------------------
    # 4. Renomear colunas que já existem no df_final
# ------------------------------------------------------------------------------

    import re

    prefixo = re.sub(r"^(vw_|mvw_)", "", nome_view)

    colunas_conflitantes = [
        coluna
        for coluna in df_auxiliar.columns
        if coluna not in CHAVES
        and coluna in df_final.columns
    ]

    if colunas_conflitantes:

        df_auxiliar = df_auxiliar.rename(
            columns={
                coluna: f"{prefixo}_{coluna}"
                for coluna in colunas_conflitantes
            }
        )

        print("Colunas renomeadas:", colunas_conflitantes)

# ------------------------------------------------------------------------------
    # 5. Verificar a cobertura antes do merge
# ------------------------------------------------------------------------------

    cobertura = (
        df_final[CHAVES]
        .merge(
            df_auxiliar[CHAVES],
            on=CHAVES,
            how="left",
            indicator=True,
            validate="one_to_one",
        )
    )

    chaves_encontradas = int(
        cobertura["_merge"]
        .eq("both")
        .sum()
    )

    chaves_sem_correspondencia = int(
        cobertura["_merge"]
        .eq("left_only")
        .sum()
    )

    linhas_antes = len(df_final)

# ------------------------------------------------------------------------------
    # 6. Realizar o left merge
# ------------------------------------------------------------------------------

    df_final = df_final.merge(
        df_auxiliar,
        on=CHAVES,
        how="left",
        validate="one_to_one",
    )

    linhas_depois = len(df_final)

# ------------------------------------------------------------------------------
    # 7. Validar se a quantidade de linhas foi preservada
# ------------------------------------------------------------------------------

    if linhas_antes != linhas_depois:
        raise ValueError(
            f"O merge com {nome_view} alterou a quantidade "
            f"de linhas de {linhas_antes:,} para "
            f"{linhas_depois:,}."
        )

# ------------------------------------------------------------------------------
    # 8. Registrar o resumo
# ------------------------------------------------------------------------------

    cobertura_percentual = round(
        chaves_encontradas / linhas_antes * 100,
        2,
    )

    resumo_merge.append({
        "view": nome_view,
        "linhas_view": len(df_auxiliar),
        "chaves_encontradas": chaves_encontradas,
        "chaves_sem_correspondencia": (
            chaves_sem_correspondencia
        ),
        "cobertura_percentual": cobertura_percentual,
        "colunas_adicionadas": (
            len(df_auxiliar.columns)
            - len(CHAVES)
        ),
        "linhas_antes": linhas_antes,
        "linhas_depois": linhas_depois,
    })

    print( f"Correspondências: " f"{chaves_encontradas:,}" )
    print( f"Sem correspondência: " f"{chaves_sem_correspondencia:,}" )
    print( f"Cobertura: " f"{cobertura_percentual:.2f}%" )
    print( f"Linhas antes/depois: " f"{linhas_antes:,} / {linhas_depois:,}" )


# ==============================================================================
# ORGANIZAR O RESULTADO FINAL
# ==============================================================================

df_final = (
    df_final
    .sort_values(CHAVES)
    .reset_index(drop=True)
)


# ==============================================================================
# VALIDAÇÕES FINAIS
# ==============================================================================

assert len(df_final) == quantidade_linhas_base, (
    "Os merges alteraram a quantidade de linhas da revenue."
)

assert not df_final.duplicated(CHAVES).any(), (
    "A tabela final possui duplicidades por "
    "id_property e reference_month."
)


# ==============================================================================
# RESUMO DOS MERGES
# ==============================================================================

df_resumo_merge = pd.DataFrame(
    resumo_merge
)

print("\nTodos os merges foram concluídos com sucesso.")
print(f"Linhas da base revenue: {quantidade_linhas_base:,}" )
print(f"Linhas da tabela final: {df_final.shape[0]:,}" )
print(f"Colunas da tabela final: {df_final.shape[1]:,}" )
print(f"Duplicidades por propriedade e mês: {df_final.duplicated(CHAVES).sum()}")

display(df_resumo_merge)
display(df_final.head())



Adicionando vw_cattle...
Correspondências: 15,231
Sem correspondência: 870
Cobertura: 94.60%
Linhas antes/depois: 16,101 / 16,101

Adicionando vw_expense...
Correspondências: 15,809
Sem correspondência: 292
Cobertura: 98.19%
Linhas antes/depois: 16,101 / 16,101

Adicionando vw_feeding...
Correspondências: 15,059
Sem correspondência: 1,042
Cobertura: 93.53%
Linhas antes/depois: 16,101 / 16,101

Adicionando vw_labor...


Correspondências: 15,681
Sem correspondência: 420
Cobertura: 97.39%
Linhas antes/depois: 16,101 / 16,101

Adicionando vw_own_milk...
Colunas renomeadas: ['hired_labor_quantity', 'family_labor_quantity']
Correspondências: 13,185
Sem correspondência: 2,916
Cobertura: 81.89%
Linhas antes/depois: 16,101 / 16,101

Adicionando mvw_area_land_summary...


Correspondências: 15,734
Sem correspondência: 367
Cobertura: 97.72%
Linhas antes/depois: 16,101 / 16,101

Adicionando mvw_asset_payment_history...
Correspondências: 15,568
Sem correspondência: 533
Cobertura: 96.69%
Linhas antes/depois: 16,101 / 16,101

Adicionando vw_dairy_production_system_monthly...


Correspondências: 10,760
Sem correspondência: 5,341
Cobertura: 66.83%
Linhas antes/depois: 16,101 / 16,101

Todos os merges foram concluídos com sucesso.
Linhas da base revenue: 16,101
Linhas da tabela final: 16,101
Colunas da tabela final: 93
Duplicidades por propriedade e mês: 0


,view,linhas_view,chaves_encontradas,chaves_sem_correspondencia,cobertura_percentual,colunas_adicionadas,linhas_antes,linhas_depois
0,vw_cattle,16221,15231,870,94.60,14,16101,16101
1,vw_expense,16356,15809,292,98.19,19,16101,16101
2,vw_feeding,15449,15059,1042,93.53,9,16101,16101
3,vw_labor,16101,15681,420,97.39,4,16101,16101
4,vw_own_milk,13327,13185,2916,81.89,9,16101,16101
5,mvw_area_land_summary,42000,15734,367,97.72,13,16101,16101
6,mvw_asset_payment_history,42111,15568,533,96.69,4,16101,16101
7,vw_dairy_production_system_monthly,18505,10760,5341,66.83,1,16101,16101


,id_property,reference_month,milk_sold_revenue,milk_volume_sold,milk_unit_price,ccs,cpp,fat,protein,unit_price_derivative,...,raw_land_value_forrageiras,hectares_propria,hectares_atividade,hectares_total,raw_land_value_medio_ponderado,monthly_depreciation_benfeitorias,monthly_depreciation_maquinas_e_equipamentos,monthly_average_capital_stock_benfeitorias,monthly_average_capital_stock_maquinas_e_equipamentos,production_system
0,00220277-a58e-4b9e-9b89-e6acf8a1a400,2024-05-01,0.00,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,10663.864045,0.000000,1.888052e+06,0.000000,NaN
1,00220277-a58e-4b9e-9b89-e6acf8a1a400,2024-08-01,501822.00,167274.0,3.00,159.0,33.0,3.85,3.35,NaN,...,NaN,NaN,NaN,NaN,NaN,10849.970268,0.000000,2.007212e+06,0.000000,NaN
2,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-04-01,112025.78,35677.0,3.14,421.0,77.0,3.48,3.40,NaN,...,NaN,0.0,24.0,25.18,<NA>,0.000000,1215.722920,0.000000e+00,48346.370420,UNSTRUCTURED_CONFINMENT
3,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-05-01,161358.96,54884.0,2.94,338.0,54.0,3.55,3.29,NaN,...,NaN,0.0,24.0,25.18,<NA>,0.000000,1205.387920,0.000000e+00,53990.366800,UNSTRUCTURED_CONFINMENT
4,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-06-01,157107.69,56311.0,2.79,192.0,102.0,3.46,3.32,NaN,...,NaN,0.0,24.0,25.18,<NA>,0.000000,1183.632817,0.000000e+00,58961.649491,UNSTRUCTURED_CONFINMENT


In [18]:
# ==============================================================================
# 3.3.1 RESUMO DE ESTRUTURA DO DATAFRAME INTEGRADO
# ==============================================================================
df_final.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16101 entries, 0 to 16100
Data columns (total 93 columns):
 #   Column                                                 Non-Null Count  Dtype         
---  ------                                                 --------------  -----         
 0   id_property                                            16101 non-null  string        
 1   reference_month                                        16101 non-null  datetime64[ns]
 2   milk_sold_revenue                                      16101 non-null  float64       
 3   milk_volume_sold                                       16101 non-null  float64       
 4   milk_unit_price                                        15874 non-null  float64       
 5   ccs                                                    15105 non-null  float64       
 6   cpp                                                    15105 non-null  float64       
 7   fat                                                    15106 non-nu

## 3.4 Aplicação da Correção Monetária (Deflação por IGP-DI)


In [19]:
# ==============================================================================
# 3.4 APLICAÇÃO DA CORREÇÃO MONETÁRIA (DEFLAÇÃO POR IGP-DI)
# ==============================================================================
# A deflação ocorre antes do Feature Engineering.
df_integrada = df_final.copy()

# ==============================================================================
# PREPARAR A BASE DO IGP-DI
# ==============================================================================

# Seleciona somente as colunas necessárias.
df_igpdi_aux = df_igpdi[[ "data", "deflator"] ].copy()

# Padroniza a data do IGP-DI para o primeiro dia de cada mês.
df_igpdi_aux["data"] = (
    pd.to_datetime(df_igpdi_aux["data"], errors="coerce")
    .dt.to_period("M")
    .dt.to_timestamp()
)

# Converte o deflator para formato numérico.
df_igpdi_aux["deflator"] = pd.to_numeric(df_igpdi_aux["deflator"], errors="coerce")

# Remove linhas sem data válida.
df_igpdi_aux = df_igpdi_aux.dropna(subset=["data"])

# Verifica se existem meses duplicados na tabela do IGP-DI.
if df_igpdi_aux.duplicated("data").any():
    meses_duplicados = (
        df_igpdi_aux.loc[df_igpdi_aux.duplicated("data", keep=False), "data"]
        .dt.strftime("%Y-%m")
        .unique()
        .tolist()
    )
    raise ValueError( "Existem meses duplicados na base do IGP-DI: " f"{meses_duplicados}" )


# ==============================================================================
# PADRONIZAR O MÊS DA BASE INTEGRADA
# ==============================================================================

df_integrada["reference_month"] = (
    pd.to_datetime(df_integrada["reference_month"], errors="coerce")
    .dt.to_period("M")
    .dt.to_timestamp()
)

# ==============================================================================
# ADICIONAR O DEFLATOR
# ==============================================================================

df_integrada = df_integrada.drop(
    columns=["data", "deflator"],
    errors="ignore",
)

df_integrada = df_integrada.merge(
    df_igpdi_aux,
    left_on="reference_month",
    right_on="data",
    how="left",
    validate="many_to_one",
)


# ==============================================================================
# VALIDAR O DEFLATOR
# ==============================================================================

meses_sem_deflator = (
    df_integrada.loc[ df_integrada["deflator"].isna(), "reference_month", ]
    .dropna()
    .dt.strftime("%Y-%m")
    .unique()
    .tolist()
)

if meses_sem_deflator:
    print(f"⚠️ Meses sem IGP-DI; o deflator 1 será utilizado: {meses_sem_deflator}" )

# Na ausência de IGP-DI, mantém o valor nominal usando deflator igual a 1.
df_integrada["deflator"] = df_integrada["deflator"].fillna(1.0)

if df_integrada["deflator"].le(0).any():
    raise ValueError( "Foram encontrados valores de deflator iguais ou inferiores a zero." )

# ==============================================================================
# DEFINIR AS COLUNAS MONETÁRIAS DE RECEITA
# ==============================================================================

COLUNAS_DEFLACIONAR_RECEITAS = [
    "milk_sold_revenue",         # Receita total da venda de leite.
    "milk_unit_price",           # Preço unitário do leite.
    "received_loans",            # Empréstimos recebidos.
    "animal_sale",               # Receita com venda de animais.
    "other_revenues",            # Outras receitas.
    "price_bonus",               # Bonificação do preço do leite.
    "price_penalty",             # Penalização ou desconto aplicado ao leite.
    "milk_derivatives_revenue",  # Receita total com derivados.
    "unit_price_derivative",     # Preço unitário dos derivados.
    "voluminous_sold",           # Receita com venda de volumoso.
    "concentrated_sold",          
    "surplus_division",          # Receita com divisão de sobras.
]

COLUNAS_DEFLACIONAR_DESPESAS = [
    "general_expenses",
    "advance_payment",
    "administration",
    "land_lease",
    "technical_assistance",
    "animal_purchase",
    "land_purchase",
    "repairs",
    "loan_interest",
    "hormones",
    "taxes_fees",
    "medicines_vaccines",
    "bedding_replacement",
    "reproduction",
    "milk_replacer",
    "milking_material",
    "milk_calves",
    "energy",
    "fuel",
]

COLUNAS_DEFLACIONAR_MDO = [
    'family_labor_expenses',
    'hired_labor_expenses'
]

COUNAS_DEFLACIONAR_LEITE_CONSUMIDO = [
    'hired_labor_amount_total',
    'discarded_amount_total',
    'calves_amount_total',
    'family_labor_amount_total'
]

COLUNAS_DEFLACIONAR_ANIMAIS = [
    "lactating_cows_value",
    "dry_cows_value",
    "nursing_value",
    "rearing_value",
    "males_value",
    "other_categories_value",
]

COLUNAS_DEFLACIONAR_MAQ_BEN = [
    'monthly_depreciation_benfeitorias',
    'monthly_depreciation_maquinas_e_equipamentos',
    'monthly_average_capital_stock_benfeitorias',
    'monthly_average_capital_stock_maquinas_e_equipamentos'
]

COLUNAS_DEFLACIONAR_FORRAGEIRA = [
    "pre_planting_cost",
    "planting_cost",
    "cultural_treatments_cost",
    "harvest_grain_cost",
    "harvest_whole_plant_cost",
    "total_forage_cost",
]

# Custo do alimento comprado, vindo de vw_feeding. Depois da correcao de
# 02/08/2026 essas colunas trazem o custo do que foi CONSUMIDO no mes, e nao o
# valor da compra, entao o deflator do proprio mes de referencia e o correto.
# Precisam ser deflacionadas aqui, antes da secao 3.4.2, porque la o custo da
# forrageira propria - ja deflacionado - e somado nelas. Sem isto a mesma coluna
# misturava valor nominal com valor deflacionado.
COLUNAS_DEFLACIONAR_ALIMENTACAO = [
    "voluminous_amount_total",
    "concentrate_amount_total",
    "mineral_amount_total",
]

COLUNAS_DEFLACIONAR = (
    COLUNAS_DEFLACIONAR_RECEITAS
    + COLUNAS_DEFLACIONAR_DESPESAS
    + COLUNAS_DEFLACIONAR_ALIMENTACAO
    + COLUNAS_DEFLACIONAR_MDO
    + COUNAS_DEFLACIONAR_LEITE_CONSUMIDO
    # + COLUNAS_DEFLACIONAR_MAQ_BEN
    # + COLUNAS_DEFLACIONAR_FORRAGEIRA
    # + COLUNAS_DEFLACIONAR_ANIMAIS
)

# ==============================================================================
# CONFERIR SE AS COLUNAS EXISTEM
# ==============================================================================

colunas_ausentes = [
    coluna
    for coluna in COLUNAS_DEFLACIONAR
    if coluna not in df_integrada.columns
]

if colunas_ausentes:
    raise KeyError(f"As seguintes colunas monetárias não foram encontradas: {colunas_ausentes}" )


# ==============================================================================
# DEFLACIONAR AS COLUNAS
# ==============================================================================

for coluna in COLUNAS_DEFLACIONAR:
    
    # Converte o valor monetário para número e substitui a própria coluna.
    df_integrada[coluna] = (
        pd.to_numeric(df_integrada[coluna], errors="coerce")
        / df_integrada["deflator"]
    )

# ==============================================================================
# ORGANIZAR A BASE
# ==============================================================================

# Remove a coluna auxiliar de data proveniente do IGP-DI. O deflator é mantido para rastreabilidade.
df_integrada = df_integrada.drop(columns=["data"], errors="ignore")

# Substitui eventuais infinitos por NaN.
df_integrada[COLUNAS_DEFLACIONAR] = (
    df_integrada[COLUNAS_DEFLACIONAR] .replace( [np.inf, -np.inf], np.nan)
)

# ==============================================================================
# CONFERÊNCIA
# ==============================================================================

display(
    df_integrada[
        [
            "id_property",
            "reference_month",
            "deflator",
            "milk_unit_price",
            "milk_sold_revenue",
            "general_expenses",
            "lactating_cows_value",
        ]
    ].head(20)
)

print("Deflação das receitas, despesas e valores dos animais concluída com sucesso.")

⚠️ Meses sem IGP-DI; o deflator 1 será utilizado: ['2026-08']


,id_property,reference_month,deflator,milk_unit_price,milk_sold_revenue,general_expenses,lactating_cows_value
0,00220277-a58e-4b9e-9b89-e6acf8a1a400,2024-05-01,0.933092,NaN,0.000000,NaN,NaN
1,00220277-a58e-4b9e-9b89-e6acf8a1a400,2024-08-01,0.946639,3.169106,530109.101788,21127.375914,10000.0
2,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-04-01,1.000054,3.139831,112019.765592,0.000000,0.0
3,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-05-01,0.991552,2.965048,162733.714940,2521.299637,0.0
4,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-06-01,0.973656,2.865487,161358.457208,0.000000,8000.0
5,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-07-01,0.973017,2.857093,199716.486371,0.000000,8000.0
6,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-08-01,0.975004,2.779477,199666.489408,721.843437,10000.0
7,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-09-01,0.978468,2.708314,188311.806035,109.865583,8000.0
8,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-10-01,0.978162,2.586483,178291.455363,0.000000,8000.0
9,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-11-01,0.978239,2.371610,144148.810170,0.000000,8000.0


Deflação das receitas, despesas e valores dos animais concluída com sucesso.


In [20]:
# ==============================================================================
# CONSULTA DIRETA DE ATIVOS PELO ID_PROPERTY
# ==============================================================================
id_morro_feio = "99ef0160-5fd8-44f9-93ed-6b6822f01c47"

# Filtrar os registros de ativos da Fazenda Morro Feio
ativos_morro_feio = df_assets[df_assets["id_property"].astype(str).str.strip() == id_morro_feio].copy()

# Totalizadores de preenchimento da coluna acquired_at
total_registros = len(ativos_morro_feio)
sem_data = ativos_morro_feio["acquired_at"].isna().sum()
com_data = ativos_morro_feio["acquired_at"].notna().sum()

print("=" * 65)
print(f"ANÁLISE DE ATIVOS DA FAZENDA MORRO FEIO")
print(f"ID: {id_morro_feio}")
print("=" * 65)
print(f"Total de registros de ativos: {total_registros:,}")
print(f"Registros SEM data de aquisição (acquired_at NULO): {sem_data:,} ({sem_data/total_registros:.1%})")
print(f"Registros COM data de aquisição preenchida:        {com_data:,} ({com_data/total_registros:.1%})")
print("=" * 65)

# Exibir os detalhes dos itens
colunas_exibicao = [
    "classification",
    "acquired_at",
    "reference_month",
    "monthly_average_capital_stock",
    "igpdi_acquired",
    "igpdi_month",
    "asset_update_factor"
]

display(ativos_morro_feio[colunas_exibicao].drop_duplicates().head(20))

ANÁLISE DE ATIVOS DA FAZENDA MORRO FEIO
ID: 99ef0160-5fd8-44f9-93ed-6b6822f01c47
Total de registros de ativos: 3,971
Registros SEM data de aquisição (acquired_at NULO): 0 (0.0%)
Registros COM data de aquisição preenchida:        3,971 (100.0%)


,classification,acquired_at,reference_month,monthly_average_capital_stock,igpdi_acquired,igpdi_month,asset_update_factor
1449307,Benfeitorias,1975-01-01 08:00:00,1980-02-01,5000.0,100.0,NaN,1.0
1449308,Benfeitorias,1975-01-01 08:00:00,1979-12-01,5000.0,100.0,NaN,1.0
1449309,Benfeitorias,1975-01-01 08:00:00,1982-12-01,5000.0,100.0,NaN,1.0
1449310,Benfeitorias,1975-01-01 08:00:00,1980-04-01,5000.0,100.0,NaN,1.0
1449311,Benfeitorias,1975-01-01 08:00:00,1979-11-01,5000.0,100.0,NaN,1.0
1449312,Benfeitorias,1975-01-01 08:00:00,1980-01-01,5000.0,100.0,NaN,1.0
1449313,Benfeitorias,1975-01-01 08:00:00,1980-03-01,5000.0,100.0,NaN,1.0
1449314,Benfeitorias,1975-01-01 08:00:00,1982-07-01,5000.0,100.0,NaN,1.0
1449315,Benfeitorias,1975-01-01 08:00:00,1979-09-01,5000.0,100.0,NaN,1.0
1449316,Benfeitorias,1975-01-01 08:00:00,1979-08-01,5000.0,100.0,NaN,1.0


In [21]:
# ==============================================================================
# INSPECIONAR VALORES DE IGP-DI E FATOR DE ATUALIZAÇÃO DA MORRO FEIO
# ==============================================================================
id_morro_feio = "99ef0160-5fd8-44f9-93ed-6b6822f01c47"

ativos_morro_feio = df_assets[df_assets["id_property"].astype(str).str.strip() == id_morro_feio].copy()

colunas_debug = [
    "classification",
    "acquired_at",
    "acquisition_month",
    "reference_month",
    "igpdi_acquired",
    "igpdi_month",
    "asset_update_factor",
    "monthly_average_capital_stock"
]

print("=" * 70)
print("AMOSTRA DE ATIVOS E FATORES CALCULADOS - FAZENDA MORRO FEIO")
print("=" * 70)
display(ativos_morro_feio[colunas_debug].drop_duplicates().head(20))

print("\n" + "=" * 70)
print("RESUMO DO FATOR DE ATUALIZAÇÃO (asset_update_factor):")
print("=" * 70)
print(ativos_morro_feio["asset_update_factor"].describe())

AMOSTRA DE ATIVOS E FATORES CALCULADOS - FAZENDA MORRO FEIO


,classification,acquired_at,acquisition_month,reference_month,igpdi_acquired,igpdi_month,asset_update_factor,monthly_average_capital_stock
1449307,Benfeitorias,1975-01-01 08:00:00,1975-01-01,1980-02-01,100.0,NaN,1.0,5000.0
1449308,Benfeitorias,1975-01-01 08:00:00,1975-01-01,1979-12-01,100.0,NaN,1.0,5000.0
1449309,Benfeitorias,1975-01-01 08:00:00,1975-01-01,1982-12-01,100.0,NaN,1.0,5000.0
1449310,Benfeitorias,1975-01-01 08:00:00,1975-01-01,1980-04-01,100.0,NaN,1.0,5000.0
1449311,Benfeitorias,1975-01-01 08:00:00,1975-01-01,1979-11-01,100.0,NaN,1.0,5000.0
1449312,Benfeitorias,1975-01-01 08:00:00,1975-01-01,1980-01-01,100.0,NaN,1.0,5000.0
1449313,Benfeitorias,1975-01-01 08:00:00,1975-01-01,1980-03-01,100.0,NaN,1.0,5000.0
1449314,Benfeitorias,1975-01-01 08:00:00,1975-01-01,1982-07-01,100.0,NaN,1.0,5000.0
1449315,Benfeitorias,1975-01-01 08:00:00,1975-01-01,1979-09-01,100.0,NaN,1.0,5000.0
1449316,Benfeitorias,1975-01-01 08:00:00,1975-01-01,1979-08-01,100.0,NaN,1.0,5000.0



RESUMO DO FATOR DE ATUALIZAÇÃO (asset_update_factor):
count    3971.000000
mean        2.542641
std         2.611075
min         0.945672
25%         1.007629
50%         1.161166
75%         3.107735
max        12.118310
Name: asset_update_factor, dtype: float64


### 3.4.1 Deflação Específica dos Custos de Forrageira


In [22]:
# ==============================================================================
# 3.4.1 DEFLAÇÃO DOS CUSTOS DE CULTURA (PELA DATA DE COMPRA)
# ==============================================================================
# 1. Preparar a tabela auxiliar do IGP-DI
df_igpdi_aux = df_igpdi[["data", "deflator"]].copy()
df_igpdi_aux["data"] = (
    pd.to_datetime(df_igpdi_aux["data"], errors="coerce")
    .dt.to_period("M")
    .dt.to_timestamp()
)
df_igpdi_aux["deflator"] = pd.to_numeric(df_igpdi_aux["deflator"], errors="coerce")
df_igpdi_aux = df_igpdi_aux.dropna(subset=["data"])

if df_igpdi_aux.duplicated("data").any():
    raise ValueError("Existem meses duplicados na série do IGP-DI.")

# ------------------------------------------------------------------------------
# 2. Custo de cultura deflacionado pela data de compra (entry_date)
# ------------------------------------------------------------------------------
# vw_culture_expense_cost entrega o grão de item justamente para permitir
# deflacionar cada linha pelo mês em que o insumo foi adquirido, antes de somar.
# Na origem entry_date é sempre dia 1 às 12:00: a granularidade já é mensal.
df_custo_cultura = df_culture_expense_cost.copy()

df_custo_cultura["periodo_compra"] = (
    pd.to_datetime(df_custo_cultura["entry_date"], errors="coerce")
    .dt.to_period("M")
    .dt.to_timestamp()
)

df_custo_cultura = df_custo_cultura.merge(
    df_igpdi_aux,
    left_on="periodo_compra",
    right_on="data",
    how="left",
    validate="m:1",
)

meses_sem_deflator_custo = (
    df_custo_cultura.loc[df_custo_cultura["deflator"].isna(), "periodo_compra"]
    .dropna()
    .dt.strftime("%Y-%m")
    .unique()
    .tolist()
)

if meses_sem_deflator_custo:
    print(
        "⚠️ Meses de compra sem IGP-DI; o valor nominal é mantido com deflator 1: "
        f"{sorted(meses_sem_deflator_custo)}"
    )

# Mesma regra da seção 3.4: sem índice, mantém o nominal.
df_custo_cultura["deflator"] = df_custo_cultura["deflator"].fillna(1.0)

df_custo_cultura["custo_deflacionado"] = (
    pd.to_numeric(df_custo_cultura["custo"], errors="coerce")
    / df_custo_cultura["deflator"]
)

df_custo_cultura = df_custo_cultura.drop(columns=["data"], errors="ignore")


# ------------------------------------------------------------------------------
# 3. Agregar na chave de integração com a produção
# ------------------------------------------------------------------------------
# id_area + id_culture + harvest_season é a chave possível: as tabelas de
# despesa de cultura não guardam id_planted_culture. id_property entra junto
# porque todo o restante do fluxo é por fazenda.
CHAVE_CUSTO_CULTURA = ["id_property", "id_area", "id_culture", "harvest_season"]

chave_incompleta = (
    df_custo_cultura[CHAVE_CUSTO_CULTURA]
    .replace(r"^\s*$", np.nan, regex=True)
    .isna()
    .any(axis=1)
)

if chave_incompleta.any():
    print(
        f"Aviso: {int(chave_incompleta.sum()):,} linhas de custo com chave "
        "incompleta ficaram fora da agregação (R$ "
        f"{df_custo_cultura.loc[chave_incompleta, 'custo_deflacionado'].sum():,.2f})."
    )

df_custo_cultura_safra = (
    df_custo_cultura
    .loc[~chave_incompleta]
    .groupby(CHAVE_CUSTO_CULTURA, as_index=False)["custo_deflacionado"]
    .sum(min_count=1)
)

# Mesma base, preservando a etapa do gasto. O rateio entre volumoso e
# concentrado da secao 3.4.2.1-B depende de stage, e stage nulo viraria linha
# descartada no groupby.
df_custo_cultura_safra_etapa = (
    df_custo_cultura
    .loc[~chave_incompleta]
    .assign(
        stage=lambda quadro: quadro["stage"]
        .astype("string")
        .str.strip()
        .str.upper()
        .fillna("ETAPA_NAO_INFORMADA")
    )
    .groupby(CHAVE_CUSTO_CULTURA + ["stage"], as_index=False)["custo_deflacionado"]
    .sum(min_count=1)
)

print(
    f"Custo de cultura deflacionado: {len(df_custo_cultura):,} itens em "
    f"{len(df_custo_cultura_safra):,} chaves propriedade/área/cultura/safra."
)


# ------------------------------------------------------------------------------
# 4. Rota antiga (vw_forage_cost), mantida só para a comparação da seção 3.4.5
# ------------------------------------------------------------------------------
if "df_forage_cost" in locals() and df_forage_cost is not None:
    df_forage_cost_deflacionado = df_forage_cost.copy()
    
    df_forage_cost_deflacionado["reference_month"] = (
        pd.to_datetime(df_forage_cost_deflacionado["reference_month"], errors="coerce")
        .dt.to_period("M")
        .dt.to_timestamp()
    )
    
    df_forage_cost_deflacionado = df_forage_cost_deflacionado.drop(columns=["data", "deflator"], errors="ignore")
    df_forage_cost_deflacionado = df_forage_cost_deflacionado.merge(
        df_igpdi_aux,
        left_on="reference_month",
        right_on="data",
        how="left"
    )
    df_forage_cost_deflacionado["deflator"] = df_forage_cost_deflacionado["deflator"].fillna(1.0)
    
    colunas_cost = [
        "pre_planting_cost",
        "planting_cost",
        "cultural_treatments_cost",
        "harvest_grain_cost",
        "harvest_whole_plant_cost",
        "total_forage_cost"
    ]
    
    for col in colunas_cost:
        if col in df_forage_cost_deflacionado.columns:
            df_forage_cost_deflacionado[col] = (
                pd.to_numeric(df_forage_cost_deflacionado[col], errors="coerce") 
                / df_forage_cost_deflacionado["deflator"]
            )
            
    # Remove as colunas auxiliares do novo DataFrame
    df_forage_cost_deflacionado = df_forage_cost_deflacionado.drop(columns=["data", "deflator"], errors="ignore")


⚠️ Meses de compra sem IGP-DI; o valor nominal é mantido com deflator 1: ['2026-08', '2026-11', '2026-12']


Aviso: 72 linhas de custo com chave incompleta ficaram fora da agregação (R$ 409,585.81).
Custo de cultura deflacionado: 18,108 itens em 1,522 chaves propriedade/área/cultura/safra.


In [23]:
# ==============================================================================
# 3.4.2 INSPEÇÃO DA PRODUÇÃO DE FORRAGEIRA
# ==============================================================================
df_forage_production.head(50)


,id_property,id_production,id_planted_culture,id_culture,id_culture_harvest_product,reference_month,planted_at,harvest_date,planted_area,harvested_area,total_harvested_area,production,culture_name,harvest_product_name,feeding_category,yield_kg_per_ha
0,004cde8f-4688-4b22-acd9-76151b7dcc7a,26799627-0c4b-4814-b128-a8c0019257f7,7c02c790-1c7d-49b3-a5c1-ba76c75d246d,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,689761c8-f128-496c-88be-fa39f446b506,2026-02-01,2025-11-01 12:00:00,2026-02-01 12:00:00,22.00,22.00,22.00,1210.00,Milho,Milho (silagem),VOLUMOSO,55.00
1,004cde8f-4688-4b22-acd9-76151b7dcc7a,b0e485dd-2940-42a9-96ea-5b35a854e3cc,80474f30-2fa9-4d24-bb3b-79fe2ef7deeb,5663ac91-dad8-4edf-b835-1eca6bd4482b,0d9a77e5-93aa-491c-8e55-4b58847dc6bf,2026-07-01,2026-03-01 12:00:00,2026-07-01 12:00:00,7.68,7.68,7.68,570.62,Sorgo,Sorgo (grão seco),CONCENTRADO,74.30
2,0179ab93-910c-4aad-898b-a9006b2fd0da,6bf4c4dc-fa37-43b7-a306-0bc13b2542c5,7080bbe7-97e6-49bf-aa24-ee5cd64b4df2,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,689761c8-f128-496c-88be-fa39f446b506,2025-04-01,2024-12-04 08:00:00,2025-04-08 07:00:00,14.60,14.60,14.60,540.20,Milho,Milho (silagem),VOLUMOSO,37.00
3,01957b94-005f-473e-8dbd-7d569a9aed4a,8cfd7aab-b891-467b-a94a-6463d9e350eb,654f5858-cabf-40d0-88f9-69220e1a5675,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,689761c8-f128-496c-88be-fa39f446b506,2025-02-01,2024-10-28 07:00:00,2025-02-12 08:00:00,150.00,150.00,150.00,8250000.00,Milho,Milho (silagem),VOLUMOSO,55000.00
4,01957b94-005f-473e-8dbd-7d569a9aed4a,a1701e1a-7198-40b2-b5b7-a2f002d2ff82,f1688441-487b-44b8-9725-2957049874cf,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,689761c8-f128-496c-88be-fa39f446b506,2025-06-01,2025-02-11 08:00:00,2025-06-01 07:00:00,100.00,100.00,100.00,4000000.00,Milho,Milho (silagem),VOLUMOSO,40000.00
5,01f268e2-339c-464b-b0a4-c04f97d080ab,3eb51bbf-7a1c-42b6-9e5a-8fa629f0b9bd,c0b43fcd-b7ab-469d-825c-061b27fe3b53,9e5fa3d8-0bf3-46aa-8834-e8e9c22ec8e5,6e15e0f5-8817-4b42-b3b8-d793d5706803,2026-06-01,2024-03-01 12:00:00,2026-06-01 12:00:00,4.00,2.00,2.00,80640.00,Capim elefante,Capim-elefante (silagem),VOLUMOSO,40320.00
6,027d7c06-0ab1-4a29-8383-98a7e2d36a80,cce58b1a-6b97-49f8-b7f5-6ddb75a50abb,28c13363-90b2-420c-9994-092fe87f30c2,143b4444-08e4-4cc6-9601-cd8b83b34d2a,2ce71f82-8048-4208-af3d-bed7083274ca,2025-07-01,2025-03-27 07:00:00,2025-07-10 07:00:00,8.50,8.50,8.50,39000.00,Milheto,Milheto (silagem),VOLUMOSO,4588.24
7,027d7c06-0ab1-4a29-8383-98a7e2d36a80,dd933219-bad9-4ec9-bf63-508d48fdebe3,d90a4c75-4b1c-43dc-80b6-69cd5e81a2be,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,689761c8-f128-496c-88be-fa39f446b506,2026-03-01,2025-11-03 08:00:00,2026-03-10 07:00:00,38.80,34.80,34.80,1701537.00,Milho,Milho (silagem),VOLUMOSO,48894.74
8,027d7c06-0ab1-4a29-8383-98a7e2d36a80,0774fc0c-7b67-4218-8a6c-f393fc8df070,db9786a3-daad-4231-950f-3fb6e10093f3,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,689761c8-f128-496c-88be-fa39f446b506,2025-02-01,2024-10-28 07:00:00,2025-02-27 08:00:00,40.00,40.00,40.00,1535320.00,Milho,Milho (silagem),VOLUMOSO,38383.00
9,027d7c06-0ab1-4a29-8383-98a7e2d36a80,2117cce1-cd22-4418-b2c2-38260563aaf2,6e3f8baf-dd38-4853-b826-1239a2124f48,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,689761c8-f128-496c-88be-fa39f446b506,2025-07-01,2025-03-20 07:00:00,2025-07-07 07:00:00,18.50,18.50,18.50,656500.00,Milho,Milho (silagem),VOLUMOSO,35486.49


In [24]:
# ==============================================================================
# 3.4.3 INSPEÇÃO DOS CUSTOS DEFLACIONADOS DE FORRAGEIRA
# ==============================================================================
df_forage_cost_deflacionado.head()


,id_property,id_planted_culture,id_culture,id_area,reference_month,culture_name,pre_planting_cost,planting_cost,cultural_treatments_cost,harvest_grain_cost,harvest_whole_plant_cost,total_forage_cost
0,004cde8f-4688-4b22-acd9-76151b7dcc7a,7c02c790-1c7d-49b3-a5c1-ba76c75d246d,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,3ac0355d-8f4b-4a8e-8ac6-c503671de9e6,2025-08-01,Milho,1489.225163,0.000000,0.000000,0.0,0.0,1489.225163
1,004cde8f-4688-4b22-acd9-76151b7dcc7a,7c02c790-1c7d-49b3-a5c1-ba76c75d246d,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,3ac0355d-8f4b-4a8e-8ac6-c503671de9e6,2025-10-01,Milho,0.000000,31020.418072,2878.275235,0.0,0.0,33898.693307
2,004cde8f-4688-4b22-acd9-76151b7dcc7a,2f042645-1b00-4766-b456-f52c78b3e55e,c3adae35-3a13-441e-b543-ad9c4886de2f,4d8a982a-9d3e-4208-b723-7dedbf0493d7,2025-11-01,Brachiaria,0.000000,552.012606,0.000000,0.0,0.0,552.012606
3,004cde8f-4688-4b22-acd9-76151b7dcc7a,7c02c790-1c7d-49b3-a5c1-ba76c75d246d,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,3ac0355d-8f4b-4a8e-8ac6-c503671de9e6,2025-11-01,Milho,0.000000,0.000000,4666.346566,0.0,0.0,4666.346566
4,004cde8f-4688-4b22-acd9-76151b7dcc7a,7c02c790-1c7d-49b3-a5c1-ba76c75d246d,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,3ac0355d-8f4b-4a8e-8ac6-c503671de9e6,2026-01-01,Milho,0.000000,0.000000,4569.414954,0.0,0.0,4569.414954


In [25]:
# ==============================================================================
# 3.4.4 ENTRADAS DE ALIMENTAÇÃO QUE NECESSITAM PREÇO UNITÁRIO
# ==============================================================================
df_feeding_entries_needing_unit_price.head()


,id_expense_entry,id_property,category_code,consumed_quantity_kg,current_unit_price,consumption_reference_month,id_production,id_planted_culture,id_culture_harvest_product,id_stock_control_item,...,stock_unit_factor_kg,stock_unit_price_kg,stock_item_name,stock_in_stock_quantity_kg,stock_purchased_quantity_kg,lot_id_production,lot_id_planted_culture,lot_id_area,lot_id_culture,lot_harvest_season
0,c2b40a2f-9c72-4a0a-bdf4-fe86c878b226,01957b94-005f-473e-8dbd-7d569a9aed4a,CONCENTRADO,47966.0,0.0,2026-05-01,None,None,None,f7a21cab-8f53-491c-a8c2-715855019396,...,1.0,1.4180,Algodão (caroço),0.0,0.0,None,None,None,None,None
1,425421bd-2c03-4edc-b02e-266bb9dde113,01957b94-005f-473e-8dbd-7d569a9aed4a,CONCENTRADO,23843.0,0.0,2026-05-01,None,None,None,31f4894f-6a1b-44df-b06b-dd735939ebd4,...,1.0,1.4700,Polpa cítrica,0.0,0.0,None,None,None,None,None
2,425421bd-2c03-4edc-b02e-266bb9dde113,01957b94-005f-473e-8dbd-7d569a9aed4a,CONCENTRADO,23843.0,0.0,2026-05-01,None,None,None,25a12bc4-16e4-4a11-a011-a547104a7e22,...,1.0,1.4300,Polpa cítrica,0.0,0.0,None,None,None,None,None
3,8dd5a597-5d8c-45ce-9f3c-b2344f21cb53,01957b94-005f-473e-8dbd-7d569a9aed4a,CONCENTRADO,5042.0,0.0,2026-05-01,None,None,None,705c1be6-f5b5-41dc-b3f4-6b12f2bd82a5,...,1.0,1.7600,DDG/DDGS/WDG/WDGS,0.0,0.0,None,None,None,None,None
4,766b6a3d-bf3f-41e5-848f-ee92dbe5ab52,027d7c06-0ab1-4a29-8383-98a7e2d36a80,VOLUMOSO,250000.0,0.0,2026-01-01,2117cce1-cd22-4418-b2c2-38260563aaf2,6e3f8baf-dd38-4853-b826-1239a2124f48,689761c8-f128-496c-88be-fa39f446b506,cc811904-6009-4e53-810f-6e2192e09e5b,...,1.0,0.1629,Milho (silagem),0.0,656500.0,2117cce1-cd22-4418-b2c2-38260563aaf2,6e3f8baf-dd38-4853-b826-1239a2124f48,c80ddfec-b204-4036-8d7f-ccfa249ee2b7,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,24/25


### 3.4.5 Validação da Troca de Views de Custo de Cultura


In [26]:
# ==============================================================================
# 3.4.5.1 CASO CONHECIDO E INTEGRIDADE DA CHAVE DE CUSTO
# ==============================================================================
# Caso de teste da metodologia: área 6cbdeeca... na safra 25/26 deve trazer
# Milho (silagem), VOLUMOSO, 1.887.500 kg, 41 ha, 46.036,6 kg/ha, 1 plantio,
# 0 linhas sem conversão e ciclo fechado.
ID_AREA_CASO_CONHECIDO = "6cbdeeca-26da-4d35-94ad-e8499b9dedcc"

display(
    df_culture_production.loc[
        df_culture_production["id_area"].eq(ID_AREA_CASO_CONHECIDO)
        & df_culture_production["harvest_season"].eq("25/26")
    ]
)


# ------------------------------------------------------------------------------
# Chaves nulas ou vazias no custo
# ------------------------------------------------------------------------------
for coluna in ["id_area", "id_culture", "harvest_season"]:
    vazias = (
        df_culture_expense_cost[coluna]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
        .isna()
    )
    print(
        f"{coluna:>16}: {int(vazias.sum()):>6,} linhas sem valor | "
        f"R$ {pd.to_numeric(df_culture_expense_cost.loc[vazias, 'custo'], errors='coerce').sum():,.2f}"
    )

print(f"\nTotal de linhas de custo: {len(df_culture_expense_cost):,}")

# Operações presentes: ESTOCAR puro não deve aparecer, vira custo só no consumo.
print("\nCusto por operação:")
print(
    df_culture_expense_cost
    .groupby(["tipo_custo", "operation"], dropna=False)["custo"]
    .agg(["size", "sum"])
    .round(2)
    .to_string()
)


,id_area,id_culture,harvest_season,product_name,feeding_category,quantidade_kg,area_colhida,produtividade_kg_ha,ms_media,qtd_plantios,qtd_producoes,linhas_sem_conversao,tem_plantio_em_andamento,unidades_origem
505,6cbdeeca-26da-4d35-94ad-e8499b9dedcc,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,25/26,Milho (silagem),VOLUMOSO,1887500.0,41.0,46036.6,NaN,1,1,0,False,(nulo)


         id_area:      5 linhas sem valor | R$ 3,141.00
      id_culture:     62 linhas sem valor | R$ 262,610.11
  harvest_season:     10 linhas sem valor | R$ 122,537.00

Total de linhas de custo: 18,108

Custo por operação:
                                size           sum
tipo_custo   operation                            
fertilizante CONSUMIR            185  5.006882e+06
             ESTOCAR_CONSUMIR   3158  5.841272e+08
             NaN                   1  0.000000e+00
manejo       CONSUMIR            333  1.162607e+06
             ESTOCAR_CONSUMIR  14427  1.325907e+08
             NaN                   4  0.000000e+00


In [27]:
# ==============================================================================
# 3.4.5.2 COBERTURA CUSTO x PRODUÇÃO E MESES SEM ÍNDICE
# ==============================================================================
chaves_custo = (
    df_custo_cultura_safra[CHAVE_CUSTO_CULTURA[1:]]
    .drop_duplicates()
)

chaves_producao = (
    df_culture_production[["id_area", "id_culture", "harvest_season"]]
    .drop_duplicates()
    .assign(tem_producao=True)
)

cobertura = chaves_custo.merge(
    chaves_producao,
    on=["id_area", "id_culture", "harvest_season"],
    how="left",
)

sem_par = cobertura["tem_producao"].isna()

print(
    f"Chaves de custo: {len(cobertura):,} | "
    f"com produção: {int((~sem_par).sum()):,} | "
    f"sem produção: {int(sem_par.sum()):,}"
)

valor_sem_par = (
    df_custo_cultura_safra
    .merge(cobertura.loc[sem_par], on=["id_area", "id_culture", "harvest_season"])
    ["custo_deflacionado"]
    .sum()
)

print(f"Custo deflacionado sem produção correspondente: R$ {valor_sem_par:,.2f}")

print("\nSafras das chaves sem produção:")
print(
    df_custo_cultura_safra
    .merge(cobertura.loc[sem_par], on=["id_area", "id_culture", "harvest_season"])
    .groupby("harvest_season")["custo_deflacionado"]
    .agg(["size", "sum"])
    .round(2)
    .to_string()
)


# ------------------------------------------------------------------------------
# Faixa de datas de compra x série do IGP-DI
# ------------------------------------------------------------------------------
print(
    "\nPeríodo de compra: "
    f"{df_custo_cultura['periodo_compra'].min():%Y-%m} a "
    f"{df_custo_cultura['periodo_compra'].max():%Y-%m}"
)
print(
    "Série IGP-DI: "
    f"{df_igpdi_aux['data'].min():%Y-%m} a {df_igpdi_aux['data'].max():%Y-%m}"
)


Chaves de custo: 1,522 | com produção: 1,005 | sem produção: 517
Custo deflacionado sem produção correspondente: R$ 526,108,457.01

Safras das chaves sem produção:
                  size           sum
harvest_season                      
03/04                2  1.557290e+03
04/05                3  1.074672e+04
06/07                1  1.530281e+04
11/12                3  4.407960e+03
13/14                2  3.063556e+04
14/15                7  2.162834e+05
15/16                1  2.055558e+04
16/17                3  1.688546e+04
17/18                7  1.309248e+05
18/19                5  9.238009e+04
19/20                2  1.079061e+04
20/21               10  4.791345e+04
21/22               11  2.626510e+05
22/23               22  3.468845e+05
23/24               37  5.275384e+05
24/25              116  4.134135e+06
25/26              245  5.195701e+08
26/27               35  6.317909e+05
94/95                1  1.039944e+04
98/99                1  6.269660e+03
99/00                1

In [28]:
# ==============================================================================
# 3.4.5.3 ESCALA DO CUSTO POR ITEM (SEM FILTRAR NADA)
# ==============================================================================
# Registro suspeito é reportado, nunca descartado nem corrigido aqui: a decisão
# sobre unidade digitada errada na origem é do negócio, não do código.
custo_item = pd.to_numeric(df_culture_expense_cost["custo"], errors="coerce")

print("Custo por item (R$):")
print(f"  linhas...: {custo_item.notna().sum():>16,}")
print(f"  mediana..: {custo_item.median():>16,.2f}")
print(f"  p99......: {custo_item.quantile(0.99):>16,.2f}")
print(f"  máximo...: {custo_item.max():>16,.2f}")
print(f"  total....: {custo_item.sum():>16,.2f}")

print("\nMaiores lançamentos:")
display(
    df_culture_expense_cost
    .nlargest(10, "custo")
    [
        [
            "tipo_custo",
            "product_name",
            "unit",
            "quantity",
            "unit_cost",
            "custo",
            "entry_date",
            "harvest_season",
        ]
    ]
)


Custo por item (R$):
  linhas...:           18,108
  mediana..:         2,351.00
  p99......:       129,279.32
  máximo...:   235,200,000.00
  total....:   722,887,380.83

Maiores lançamentos:


,tipo_custo,product_name,unit,quantity,unit_cost,custo,entry_date,harvest_season
56,fertilizante,8.40.0+5%S ROBUSTO,KG,48000.0,4900.00,235200000.0,2026-05-01 12:00:00,25/26
5297,fertilizante,Fertilizante mineral,KG,32000.0,3900.00,124800000.0,2025-10-01 12:00:00,25/26
15032,fertilizante,Fertilizante mineral,KG,7300.0,3600.00,26280000.0,2025-12-01 12:00:00,25/26
15031,fertilizante,Fertilizante mineral,KG,6000.0,4086.29,24517740.0,2025-12-01 12:00:00,25/26
9007,fertilizante,Composto orgânico,KG,95000.0,231.06,21950700.0,2025-08-01 12:00:00,25/26
11461,fertilizante,Fertilizante mineral,KG,5000.0,4020.00,20100000.0,2025-09-01 12:00:00,25/26
4944,fertilizante,Fertilizante mineral,KG,6000.0,3000.00,18000000.0,2024-02-01 12:00:00,23/24
11465,fertilizante,Fertilizante mineral,KG,5000.0,3445.00,17225000.0,2025-09-01 12:00:00,25/26
2218,fertilizante,Fertilizante mineral,KG,4000.0,3000.00,12000000.0,2025-11-01 12:00:00,25/26
13350,fertilizante,Fertilizante mineral,KG,2500.0,3596.28,8990700.0,2025-01-01 12:00:00,24/25


### 3.4.2 Rateio e Custo da Forrageira Própria na Alimentação


In [29]:
# ==============================================================================
# 3.4.2.1 CUSTO UNITÁRIO DA FORRAGEIRA PRODUZIDA (RATEIO POR ETAPA E ÁREA)
# ==============================================================================
# ATENÇÃO: rota antiga, resultado INCOMPLETO. Lê vw_forage_cost, que só cobre
# CultureExpenseManagement — nenhuma linha de fertilizante entra, e a diferença
# chega a R$ 617 milhões no total. Serve apenas à comparação da seção 3.4.2.1-C.
# O fluxo real usa df_custo_unitario_forrageira, montado na seção 3.4.2.1-B.
#
# Regra reproduzida de ELABORE_IND_ANUAIS.ipynb, seção
# "Encontrar o Valor Unitário da Forrageira Produzida" (etapas 1 e 2).
#
# Entradas:
#   df_forage_cost_deflacionado -> id_property + reference_month + id_planted_culture
#   df_forage_production        -> uma linha por id_production
#
# Saída:
#   df_custo_producao_forrageira -> uma linha por id_property + id_production

# Colunas de custo de vw_forage_cost e a etapa correspondente.
MAPA_ETAPAS_CUSTO = {
    "pre_planting_cost":        "PRE_PLANTIO",
    "planting_cost":            "PLANTIO",
    "cultural_treatments_cost": "TRATOS_CULTURAIS",
    "harvest_grain_cost":       "COLHEITA_ENSILAGEM_GRAO",
    "harvest_whole_plant_cost": "COLHEITA_ENSILAGEM_PLANTA_INTEIRA",
}

CATEGORIA_VOLUMOSO    = "VOLUMOSO"
CATEGORIA_CONCENTRADO = "CONCENTRADO"


# ------------------------------------------------------------------------------
# 1. Custo por plantio e etapa
# ------------------------------------------------------------------------------
# Lançamento de custo sem plantio identificado não pode ser rateado. Na origem
# são os registros de CultureExpenseManagement sem id_culture informado.
custo_sem_plantio = df_forage_cost_deflacionado["id_planted_culture"].isna()

if custo_sem_plantio.any():
    print(
        f"Aviso: {int(custo_sem_plantio.sum()):,} linhas de custo sem "
        "id_planted_culture descartadas do rateio (R$ "
        f"{df_forage_cost_deflacionado.loc[custo_sem_plantio, 'total_forage_cost'].sum():,.2f})."
    )

# vw_forage_cost entrega o custo somado por etapa em colunas separadas.
# O melt volta ao formato longo e a soma consolida os meses de aplicação
# do insumo dentro do mesmo plantio.
df_custo_etapa = (
    df_forage_cost_deflacionado
    .loc[~custo_sem_plantio]
    .melt(
        id_vars=["id_property", "id_planted_culture"],
        value_vars=list(MAPA_ETAPAS_CUSTO),
        var_name="coluna_etapa",
        value_name="valor_total",
    )
)

df_custo_etapa["stage"] = df_custo_etapa["coluna_etapa"].map(MAPA_ETAPAS_CUSTO)

df_custo_etapa["valor_total"] = pd.to_numeric(
    df_custo_etapa["valor_total"],
    errors="coerce",
)

df_custo_etapa = (
    df_custo_etapa
    .groupby(
        ["id_property", "id_planted_culture", "stage"],
        as_index=False,
    )["valor_total"]
    .sum(min_count=1)
)


# ------------------------------------------------------------------------------
# 2. Produção: garantir uma linha por id_production
# ------------------------------------------------------------------------------
COLUNAS_PRODUCAO_RATEIO = [
    "id_property",
    "id_production",
    "id_planted_culture",
    "harvested_area",
    "total_harvested_area",
    "production",
    "feeding_category",
    "harvest_date",
]

producoes_duplicadas = df_forage_production.duplicated(
    subset=["id_property", "id_production"],
    keep=False,
)

if producoes_duplicadas.any():
    print(
        "Aviso: "
        f"{df_forage_production.loc[producoes_duplicadas, 'id_production'].nunique():,} "
        "id_production com mais de uma linha em vw_forage_production "
        "(fan-out do LEFT JOIN com CultureHarvestProduct). "
        "Mantida a primeira ocorrência."
    )

df_producao_rateio = (
    df_forage_production[COLUNAS_PRODUCAO_RATEIO]
    .drop_duplicates(subset=["id_property", "id_production"], keep="first")
    .copy()
)

df_producao_rateio["feeding_category"] = (
    df_producao_rateio["feeding_category"]
    .astype("string")
    .str.strip()
    .str.upper()
)

print(
    "Categorias encontradas na produção:",
    df_producao_rateio["feeding_category"].dropna().unique().tolist(),
)


# ------------------------------------------------------------------------------
# 3. Proporção da área colhida dentro do plantio
# ------------------------------------------------------------------------------
df_producao_rateio["prop_area_produzida"] = (
    pd.to_numeric(df_producao_rateio["harvested_area"], errors="coerce")
    / pd.to_numeric(
        df_producao_rateio["total_harvested_area"],
        errors="coerce",
    ).replace(0, np.nan)
)

sem_proporcao = df_producao_rateio["prop_area_produzida"].isna()

if sem_proporcao.any():
    print(
        f"Aviso: {int(sem_proporcao.sum()):,} produções sem área colhida total "
        "válida; o rateio por área fica indefinido (NaN)."
    )


# ------------------------------------------------------------------------------
# 4. Quantidade de produções por plantio e categoria
# ------------------------------------------------------------------------------
df_contagem_producoes = (
    df_producao_rateio
    .groupby(
        ["id_planted_culture", "feeding_category"],
        as_index=False,
        dropna=False,
    )["id_production"]
    .nunique()
    .rename(columns={"id_production": "quantidade_producoes"})
)

df_producao_rateio = df_producao_rateio.merge(
    df_contagem_producoes,
    on=["id_planted_culture", "feeding_category"],
    how="left",
)


# ------------------------------------------------------------------------------
# 5. Cruzar cada produção com as etapas de custo do plantio
# ------------------------------------------------------------------------------
# Expansão intencional: uma linha por produção x etapa.
df_rateio = df_producao_rateio.merge(
    df_custo_etapa,
    on=["id_property", "id_planted_culture"],
    how="left",
)


# ------------------------------------------------------------------------------
# 6. Fator de rateio
# ------------------------------------------------------------------------------
# Quando o plantio tem uma única produção da categoria, o custo de colheita
# vai integralmente para a categoria correspondente à etapa. Nos demais casos
# o custo é rateado pela proporção da área colhida.
def combinar_mascaras(*mascaras):
    """Combina máscaras booleanas anuláveis em um array booleano puro."""

    resultado = mascaras[0]

    for mascara in mascaras[1:]:
        resultado = resultado & mascara

    return resultado.fillna(False).to_numpy(dtype=bool)


condicoes_rateio = [
    combinar_mascaras(
        df_rateio["stage"].eq("COLHEITA_ENSILAGEM_PLANTA_INTEIRA"),
        df_rateio["feeding_category"].eq(CATEGORIA_VOLUMOSO),
        df_rateio["quantidade_producoes"].eq(1),
    ),
    combinar_mascaras(
        df_rateio["stage"].eq("COLHEITA_ENSILAGEM_PLANTA_INTEIRA"),
        df_rateio["feeding_category"].eq(CATEGORIA_CONCENTRADO),
        df_rateio["quantidade_producoes"].eq(1),
    ),
    combinar_mascaras(
        df_rateio["stage"].eq("COLHEITA_ENSILAGEM_GRAO"),
        df_rateio["feeding_category"].eq(CATEGORIA_CONCENTRADO),
        df_rateio["quantidade_producoes"].eq(1),
    ),
    combinar_mascaras(
        df_rateio["stage"].eq("COLHEITA_ENSILAGEM_GRAO"),
        df_rateio["feeding_category"].eq(CATEGORIA_VOLUMOSO),
        df_rateio["quantidade_producoes"].eq(1),
    ),
]

resultado_rateio = [1.0, 0.0, 1.0, 0.0]

df_rateio["fator_rateio"] = np.select(
    condicoes_rateio,
    resultado_rateio,
    default=df_rateio["prop_area_produzida"],
)

df_rateio["valor_total_rateado"] = (
    df_rateio["valor_total"] * df_rateio["fator_rateio"]
)


# ------------------------------------------------------------------------------
# 7. Custo unitário por produção
# ------------------------------------------------------------------------------
df_custo_producao_forrageira = (
    df_rateio
    .groupby(
        ["id_property", "id_production", "feeding_category", "production"],
        as_index=False,
        dropna=False,
    )["valor_total_rateado"]
    .sum(min_count=1)
)

df_custo_producao_forrageira["custo_unitario_forrageira"] = (
    df_custo_producao_forrageira["valor_total_rateado"]
    / pd.to_numeric(
        df_custo_producao_forrageira["production"],
        errors="coerce",
    ).replace(0, np.nan)
)

if df_custo_producao_forrageira.duplicated(
    subset=["id_property", "id_production"]
).any():
    raise ValueError(
        "df_custo_producao_forrageira não está com uma linha por "
        "id_property e id_production."
    )

print(
    f"\nProduções com custo unitário: "
    f"{int(df_custo_producao_forrageira['custo_unitario_forrageira'].notna().sum()):,} "
    f"de {len(df_custo_producao_forrageira):,}"
)

print("\nDistribuição do custo unitário (R$/kg):")
print(
    df_custo_producao_forrageira["custo_unitario_forrageira"]
    .describe()
    .round(4)
    .to_string()
)

# Valores absurdos indicam production em unidade divergente do custo.
# Conferir antes de aceitar o resultado: nada é filtrado automaticamente.
print("\nMaiores custos unitários (conferir antes de aceitar):")
display(
    df_custo_producao_forrageira
    .nlargest(10, "custo_unitario_forrageira")
    [
        [
            "id_property",
            "id_production",
            "feeding_category",
            "production",
            "valor_total_rateado",
            "custo_unitario_forrageira",
        ]
    ]
)


Aviso: 23 linhas de custo sem id_planted_culture descartadas do rateio (R$ 141,897.11).


Categorias encontradas na produção: ['VOLUMOSO', 'CONCENTRADO']
Aviso: 8 produções sem área colhida total válida; o rateio por área fica indefinido (NaN).

Produções com custo unitário: 1,134 de 1,413

Distribuição do custo unitário (R$/kg):
count    1.134000e+03
mean     3.237154e+03
std      1.053646e+05
min      0.000000e+00
25%      7.540000e-02
50%      1.202000e-01
75%      2.467000e-01
max      3.547811e+06

Maiores custos unitários (conferir antes de aceitar):


,id_property,id_production,feeding_category,production,valor_total_rateado,custo_unitario_forrageira
704,773ceb89-f6ae-4bac-8394-11904bee3862,c5a86dca-b731-429f-b11a-c7c6aec66900,VOLUMOSO,0.01,3.547846e+04,3.547811e+06
1222,d8b1aac4-a108-4278-bb29-0b0babc10888,3e71a78c-a3b7-4535-991a-746f634a9515,VOLUMOSO,260.00,1.254473e+07,4.824895e+04
428,46ca764b-5c1b-4d72-b6fe-757fe5d45abc,619af921-5d8a-4bb0-88a1-a2a965544dbf,CONCENTRADO,1.00,1.947639e+04,1.947639e+04
115,10fb6de6-7c13-4dda-9c0b-e8e01bddb02b,52f1cdb3-1b7b-4848-bc9c-ec5b0271cefb,VOLUMOSO,1.00,1.705050e+04,1.705050e+04
258,2c135321-99af-47f3-b03e-629d3ba40c55,102cee2b-f640-4caf-a4d9-96470939581c,VOLUMOSO,1.00,8.410849e+03,8.410849e+03
1234,dba03118-00ca-4806-9db2-67b0cff9347c,21ba52b8-9883-4df3-96ce-89569b2791ac,VOLUMOSO,1.00,6.776068e+03,6.776068e+03
1124,c555d241-bc85-4047-b79b-ea947220cba1,ac8aad5f-b4c7-4411-83b1-4dec41e3e468,VOLUMOSO,1.00,3.695843e+03,3.695843e+03
408,4610197a-071c-47c6-b11d-7e4fc74f70ec,fd5bc64f-af9c-4fe4-b08d-f480e00e258d,CONCENTRADO,3.00,7.695794e+03,2.565265e+03
680,704583a5-13d6-4e8b-a769-9eeda2de9abb,2116c8af-8f08-41ed-9019-316e208cbcfb,VOLUMOSO,784.00,7.355510e+05,9.382028e+02
1209,d53c389f-3eaf-4666-8362-19553fc48ffe,0f9c90f9-2856-4171-9482-f9416947ea91,VOLUMOSO,40.00,3.647593e+04,9.118983e+02


In [30]:
df_custo_producao_forrageira.loc[df_custo_producao_forrageira['id_property']=='663897f4-86f7-4c6d-b600-189086a5543d']

,id_property,id_production,feeding_category,production,valor_total_rateado,custo_unitario_forrageira
634,663897f4-86f7-4c6d-b600-189086a5543d,fa39b04b-8751-4a2e-a659-5af0b155061c,VOLUMOSO,1620000.0,238023.171841,0.146928


In [31]:
# ==============================================================================
# 3.4.2.1-B CUSTO UNITÁRIO DA FORRAGEIRA (ROTA vw_culture_expense_cost)
# ==============================================================================
# Rateio do custo da safra entre VOLUMOSO e CONCENTRADO pela etapa do gasto,
# mesma regra da rota antiga (seção 3.4.2.1), agora sobre a base que tem
# fertilizante:
#   - colheita/ensilagem de planta inteira -> 100% VOLUMOSO
#   - colheita/ensilagem de grão           -> 100% CONCENTRADO
#   - pré-plantio, plantio, tratos culturais e etapa não informada
#                                          -> proporção da área colhida
#
# Os fatores são normalizados dentro de propriedade + área + cultura + safra +
# etapa. Duas consequências desejadas: chave com uma única categoria recebe 100%
# de toda etapa (não há o que ratear), e custo de colheita de grão numa chave sem
# CONCENTRADO cai para a proporção de área em vez de evaporar.
#
# Entradas:
#   df_custo_cultura_safra_etapa -> id_property + id_area + id_culture +
#                                   harvest_season + stage
#   df_culture_production        -> id_area + id_culture + harvest_season +
#                                   product_name
#
# Saída:
#   df_custo_unitario_forrageira -> uma linha por
#   id_area + id_culture + harvest_season + feeding_category

CHAVE_SAFRA  = ["id_area", "id_culture", "harvest_season"]
CHAVE_RATEIO = CHAVE_CUSTO_CULTURA + ["stage"]

CATEGORIA_VOLUMOSO    = "VOLUMOSO"
CATEGORIA_CONCENTRADO = "CONCENTRADO"

ETAPA_COLHEITA_GRAO           = "COLHEITA_ENSILAGEM_GRAO"
ETAPA_COLHEITA_PLANTA_INTEIRA = "COLHEITA_ENSILAGEM_PLANTA_INTEIRA"

df_producao_safra = df_culture_production.copy()

df_producao_safra["feeding_category"] = (
    df_producao_safra["feeding_category"]
    .astype("string")
    .str.strip()
    .str.upper()
)

producao_sem_categoria = df_producao_safra["feeding_category"].isna()

if producao_sem_categoria.any():
    print(
        f"Aviso: {int(producao_sem_categoria.sum()):,} linhas de produção sem "
        "feeding_category ficaram fora do rateio ("
        f"{df_producao_safra.loc[producao_sem_categoria, 'quantidade_kg'].sum():,.0f} kg)."
    )


# ------------------------------------------------------------------------------
# 1. Produção e área colhida por chave e categoria
# ------------------------------------------------------------------------------
df_producao_categoria = (
    df_producao_safra
    .loc[~producao_sem_categoria]
    .groupby(CHAVE_SAFRA + ["feeding_category"], as_index=False)
    [["quantidade_kg", "area_colhida"]]
    .sum(min_count=1)
)

for coluna in ["quantidade_kg", "area_colhida"]:
    df_producao_categoria[coluna] = pd.to_numeric(
        df_producao_categoria[coluna],
        errors="coerce",
    )
    df_producao_categoria[f"{coluna}_chave"] = (
        df_producao_categoria
        .groupby(CHAVE_SAFRA)[coluna]
        .transform("sum")
    )

# Base do rateio das etapas comuns é a área colhida: kg de silagem úmida e kg de
# grão seco não são comparáveis. Sem área na origem, cai para a quantidade, senão
# o custo da chave ficaria sem destino.
df_producao_categoria["prop_area"] = (
    df_producao_categoria["area_colhida"]
    / df_producao_categoria["area_colhida_chave"].replace(0, np.nan)
)

df_producao_categoria["prop_quantidade"] = (
    df_producao_categoria["quantidade_kg"]
    / df_producao_categoria["quantidade_kg_chave"].replace(0, np.nan)
)

df_producao_categoria["prop_base"] = (
    df_producao_categoria["prop_area"]
    .fillna(df_producao_categoria["prop_quantidade"])
)

sem_area = df_producao_categoria["prop_area"].isna()

if sem_area.any():
    print(
        f"Aviso: {int(sem_area.sum()):,} categorias sem área colhida; nelas o "
        "rateio das etapas comuns usa a proporção da quantidade produzida."
    )

# Duas categorias saindo do mesmo talhão inflam a área da chave e diluem o
# rateio das etapas comuns. Nada é filtrado: só o aviso.
chaves_multicategoria = (
    df_producao_categoria
    .loc[df_producao_categoria.duplicated(CHAVE_SAFRA, keep=False), CHAVE_SAFRA]
    .drop_duplicates()
)

print(
    f"Chaves com mais de uma categoria (área somada): {len(chaves_multicategoria):,} "
    f"de {df_producao_categoria[CHAVE_SAFRA].drop_duplicates().shape[0]:,}"
)


# ------------------------------------------------------------------------------
# 2. Cruzar custo por etapa com produção por categoria
# ------------------------------------------------------------------------------
# Expansão intencional: uma linha de custo/etapa vira uma linha por categoria.
df_rateio_safra = df_custo_cultura_safra_etapa.merge(
    df_producao_categoria,
    on=CHAVE_SAFRA,
    how="left",
)

custo_sem_producao = df_rateio_safra["quantidade_kg"].isna()

if custo_sem_producao.any():
    print(
        f"Aviso: {int(custo_sem_producao.sum()):,} linhas de custo por etapa sem "
        "produção correspondente ficaram sem custo unitário (R$ "
        f"{df_rateio_safra.loc[custo_sem_producao, 'custo_deflacionado'].sum():,.2f})."
    )


# ------------------------------------------------------------------------------
# 3. Fator de rateio por etapa
# ------------------------------------------------------------------------------
def mascara_pura(serie):
    """Converte máscara booleana anulável em array booleano puro."""

    return serie.fillna(False).to_numpy(dtype=bool)


eh_volumoso    = mascara_pura(df_rateio_safra["feeding_category"].eq(CATEGORIA_VOLUMOSO))
eh_concentrado = mascara_pura(df_rateio_safra["feeding_category"].eq(CATEGORIA_CONCENTRADO))
etapa_grao     = mascara_pura(df_rateio_safra["stage"].eq(ETAPA_COLHEITA_GRAO))
etapa_planta   = mascara_pura(df_rateio_safra["stage"].eq(ETAPA_COLHEITA_PLANTA_INTEIRA))

prop_base = pd.to_numeric(
    df_rateio_safra["prop_base"],
    errors="coerce",
).to_numpy(dtype=float)

df_rateio_safra["fator_bruto"] = np.select(
    [
        etapa_grao & eh_concentrado,
        etapa_grao,
        etapa_planta & eh_volumoso,
        etapa_planta,
    ],
    [1.0, 0.0, 1.0, 0.0],
    default=prop_base,
)

df_rateio_safra["soma_fator"] = (
    df_rateio_safra
    .groupby(CHAVE_RATEIO, dropna=False)["fator_bruto"]
    .transform("sum")
)

# Etapa cuja regra não achou a categoria alvo (colheita de grão em chave só de
# volumoso, por exemplo) volta para a proporção de área, senão o custo sumiria.
df_rateio_safra["fator_rateio"] = np.where(
    df_rateio_safra["soma_fator"] > 0,
    df_rateio_safra["fator_bruto"] / df_rateio_safra["soma_fator"],
    prop_base,
)

df_rateio_safra["custo_rateado"] = (
    pd.to_numeric(df_rateio_safra["custo_deflacionado"], errors="coerce")
    * df_rateio_safra["fator_rateio"]
)


# ------------------------------------------------------------------------------
# 4. Custo unitário por chave e categoria
# ------------------------------------------------------------------------------
df_custo_unitario_forrageira = (
    df_rateio_safra
    .loc[~custo_sem_producao]
    .groupby(
        CHAVE_CUSTO_CULTURA + ["feeding_category"],
        as_index=False,
        dropna=False,
    )["custo_rateado"]
    .sum(min_count=1)
    .merge(
        df_producao_categoria[
            CHAVE_SAFRA
            + ["feeding_category", "quantidade_kg", "area_colhida"]
        ],
        on=CHAVE_SAFRA + ["feeding_category"],
        how="left",
        validate="m:1",
    )
)

df_custo_unitario_forrageira["custo_unitario_forrageira"] = (
    df_custo_unitario_forrageira["custo_rateado"]
    / df_custo_unitario_forrageira["quantidade_kg"].replace(0, np.nan)
)

if df_custo_unitario_forrageira.duplicated(
    subset=CHAVE_SAFRA + ["feeding_category"]
).any():
    raise ValueError(
        "df_custo_unitario_forrageira não está com uma linha por "
        "área, cultura, safra e categoria; o merge do lançamento de "
        "alimentação exige essa unicidade."
    )


# ------------------------------------------------------------------------------
# 5. Conservação: o rateio redistribui dentro da chave, não cria nem perde valor
# ------------------------------------------------------------------------------
custo_por_chave = (
    df_rateio_safra
    .loc[~custo_sem_producao]
    .drop_duplicates(subset=CHAVE_RATEIO)
    .groupby(CHAVE_CUSTO_CULTURA, as_index=False)["custo_deflacionado"]
    .sum(min_count=1)
)

conferencia = custo_por_chave.merge(
    df_custo_unitario_forrageira
    .groupby(CHAVE_CUSTO_CULTURA, as_index=False)["custo_rateado"]
    .sum(min_count=1),
    on=CHAVE_CUSTO_CULTURA,
    how="outer",
)

conferencia["diferenca"] = (
    conferencia["custo_rateado"] - conferencia["custo_deflacionado"]
)

fora_da_tolerancia = conferencia["diferenca"].abs() > 0.01

if fora_da_tolerancia.any():
    display(conferencia.loc[fora_da_tolerancia].head(20))
    raise ValueError(
        f"{int(fora_da_tolerancia.sum()):,} chaves com custo rateado diferente "
        "do custo deflacionado da chave."
    )

print(
    f"\nChaves com custo unitário: "
    f"{int(df_custo_unitario_forrageira['custo_unitario_forrageira'].notna().sum()):,} "
    f"de {len(df_custo_unitario_forrageira):,}"
)

print("\nDistribuição do custo unitário (R$/kg):")
print(
    df_custo_unitario_forrageira["custo_unitario_forrageira"]
    .describe()
    .round(4)
    .to_string()
)

# Nada é filtrado: valores extremos são reportados para conferência manual.
print("\nMaiores custos unitários (conferir antes de aceitar):")
display(
    df_custo_unitario_forrageira
    .nlargest(10, "custo_unitario_forrageira")
    [
        CHAVE_SAFRA
        + [
            "feeding_category",
            "quantidade_kg",
            "area_colhida",
            "custo_rateado",
            "custo_unitario_forrageira",
        ]
    ]
)


Aviso: 2 linhas de produção sem feeding_category ficaram fora do rateio (3,044 kg).
Aviso: 7 categorias sem área colhida; nelas o rateio das etapas comuns usa a proporção da quantidade produzida.
Chaves com mais de uma categoria (área somada): 64 de 1,143
Aviso: 836 linhas de custo por etapa sem produção correspondente ficaram sem custo unitário (R$ 526,108,457.01).



Chaves com custo unitário: 1,044 de 1,058

Distribuição do custo unitário (R$/kg):
count      1044.0000
mean        197.5182
std        4346.2628
min           0.0002
25%           0.1247
50%           0.1703
75%           0.2321
max      137332.3352

Maiores custos unitários (conferir antes de aceitar):


,id_area,id_culture,harvest_season,feeding_category,quantidade_kg,area_colhida,custo_rateado,custo_unitario_forrageira
315,57be7222-058f-4254-a888-bcb11a6dd556,5663ac91-dad8-4edf-b835-1eca6bd4482b,25/26,CONCENTRADO,1.0,8.00,1.373323e+05,137332.335153
764,7256c78d-7e79-4999-92af-db892c620938,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,24/25,VOLUMOSO,435.3,22.00,9.117860e+06,20946.151418
77,117333cf-7478-46bf-a916-095b339737cd,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,24/25,VOLUMOSO,1.0,1.50,1.705050e+04,17050.504989
199,56dd6d42-0ac7-44b2-afe5-0281396704df,d6e99d12-073f-4a3c-9d3e-df7e3467e2dc,24/25,VOLUMOSO,1.0,0.01,8.410849e+03,8410.849014
901,ff0f3c34-2bbd-4210-b2c0-a171bd02f7b6,143b4444-08e4-4cc6-9601-cd8b83b34d2a,24/25,VOLUMOSO,1.0,9.96,6.776068e+03,6776.067741
821,ebfb0b84-bbef-4fa3-b956-2331ba2d09f9,5663ac91-dad8-4edf-b835-1eca6bd4482b,24/25,VOLUMOSO,1.0,4.50,5.268414e+03,5268.414452
663,9c505c1f-f635-4792-8284-e6ccb9f16d39,13d18705-54ad-4e44-ba6b-fbd8fd03fbf1,23/24,VOLUMOSO,1.0,2.28,2.300053e+03,2300.052527
509,52f5f4c5-35b7-47d9-8452-b2f77a19fc9b,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,23/24,VOLUMOSO,784.0,28.00,8.806860e+05,1123.323965
879,ba08273a-1ce3-4e22-9a9b-f794b9a58733,5663ac91-dad8-4edf-b835-1eca6bd4482b,23/24,VOLUMOSO,40.0,20.00,3.647593e+04,911.898340
767,7c1d3516-c4b5-4aa4-bb7f-ff2368617c5e,143b4444-08e4-4cc6-9601-cd8b83b34d2a,24/25,VOLUMOSO,1.0,1.80,7.282109e+02,728.210902


In [32]:
# ==============================================================================
# 3.4.2.1-C COMPARAÇÃO ENTRE A ROTA ANTIGA E A NOVA
# ==============================================================================
# Não altera nada do fluxo: só mede a diferença entre vw_forage_cost e
# vw_culture_expense_cost antes de a rota antiga ser retirada.
#
# Atenção: desde a correção D7 (02/08/2026) a rota nova lança o custo do insumo
# CONSUMIDO e a rota antiga continua lançando o COMPRADO. Parte da diferença
# abaixo é essa troca de base, não só o fertilizante que falta na rota antiga.

total_antigo = pd.to_numeric(
    df_forage_cost_deflacionado["total_forage_cost"],
    errors="coerce",
).sum()

total_novo = pd.to_numeric(
    df_custo_cultura["custo_deflacionado"],
    errors="coerce",
).sum()

total_novo_manejo = pd.to_numeric(
    df_custo_cultura.loc[df_custo_cultura["tipo_custo"].eq("manejo"), "custo_deflacionado"],
    errors="coerce",
).sum()

total_novo_fertilizante = pd.to_numeric(
    df_custo_cultura.loc[
        df_custo_cultura["tipo_custo"].eq("fertilizante"), "custo_deflacionado"
    ],
    errors="coerce",
).sum()

print("Custo deflacionado total (R$)")
print(f"  rota antiga (vw_forage_cost).......: {total_antigo:>20,.2f}")
print(f"  rota nova, ramo manejo.............: {total_novo_manejo:>20,.2f}")
print(f"  rota nova, ramo fertilizante.......: {total_novo_fertilizante:>20,.2f}")
print(f"  rota nova, total...................: {total_novo:>20,.2f}")
print(f"  diferença total....................: {total_novo - total_antigo:>20,.2f}")

# Custo unitário: a rota antiga é por id_production, a nova por safra.
# A comparação é feita na propriedade, que é o grão comum às duas.
comparativo_unitario = (
    df_custo_producao_forrageira
    .groupby("id_property", as_index=False)["custo_unitario_forrageira"]
    .median()
    .rename(columns={"custo_unitario_forrageira": "unitario_mediano_antigo"})
    .merge(
        df_custo_unitario_forrageira
        .groupby("id_property", as_index=False)["custo_unitario_forrageira"]
        .median()
        .rename(columns={"custo_unitario_forrageira": "unitario_mediano_novo"}),
        on="id_property",
        how="outer",
    )
)

comparativo_unitario["variacao"] = (
    comparativo_unitario["unitario_mediano_novo"]
    - comparativo_unitario["unitario_mediano_antigo"]
)

print(
    f"\nPropriedades comparadas: {len(comparativo_unitario):,} | "
    f"só na rota antiga: {int(comparativo_unitario['unitario_mediano_novo'].isna().sum()):,} | "
    f"só na rota nova: {int(comparativo_unitario['unitario_mediano_antigo'].isna().sum()):,}"
)

display(
    comparativo_unitario
    .reindex(comparativo_unitario["variacao"].abs().sort_values(ascending=False).index)
    .head(15)
)


Custo deflacionado total (R$)
  rota antiga (vw_forage_cost).......:       168,824,552.40
  rota nova, ramo manejo.............:       137,299,163.22
  rota nova, ramo fertilizante.......:       594,870,195.74
  rota nova, total...................:       732,169,358.96
  diferença total....................:       563,344,806.56

Propriedades comparadas: 509 | só na rota antiga: 51 | só na rota nova: 57


,id_property,unitario_mediano_antigo,unitario_mediano_novo,variacao
430,d8b1aac4-a108-4278-bb29-0b0babc10888,48248.948086,0.150685,-48248.797401
365,bc781616-f1e7-4973-9291-fd9fb9c3b500,79.718085,20946.151418,20866.433333
98,2c135321-99af-47f3-b03e-629d3ba40c55,0.384567,4205.530539,4205.145972
317,9f993e17-eb5d-4764-8a01-98a634e2203c,0.179835,1150.043623,1149.863788
75,22ebdfda-2528-44e1-b9d4-668eba79c36a,243.035195,0.295440,-242.739755
311,9b101631-e909-4e83-9765-7fdc003cceb5,237.461191,0.382692,-237.078499
249,7b3b4a6b-36bc-42cf-91df-876d90abbb0f,0.053424,218.445025,218.391601
233,704583a5-13d6-4e8b-a769-9eeda2de9abb,938.202779,1123.323965,185.121186
200,611f1b92-5e1d-49da-9685-f09894bde773,132.029190,316.874247,184.845057
281,8b690fe9-bb02-4c00-9bc9-05d8bd39744b,137.704800,0.191297,-137.513503


In [33]:
# ==============================================================================
# 3.4.2.2 CUSTO DA ALIMENTAÇÃO COM FORRAGEIRA PRÓPRIA (PREÇO UNITÁRIO = 0)
# ==============================================================================
# Entrada: df_feeding_entries_needing_unit_price
#   grão real: id_expense_entry x lote de estoque (id_stock_control_item)
# Saída: df_custo_alimentacao_forrageira
#   grão: id_property + reference_month

df_entradas_forrageira = df_feeding_entries_needing_unit_price.copy()


# ------------------------------------------------------------------------------
# 1. Lançamento que consome mais de um lote de estoque
# ------------------------------------------------------------------------------
# A view repete consumed_quantity em cada lote consumido. Sem informação da
# divisão real entre lotes, a quantidade é dividida em partes iguais para não
# inflar o consumo total. Hipótese assumida, não é regra do sistema.
df_entradas_forrageira["quantidade_lotes"] = (
    df_entradas_forrageira
    .groupby("id_expense_entry")["id_expense_entry"]
    .transform("size")
)

lancamentos_multilote = df_entradas_forrageira["quantidade_lotes"].gt(1)

if lancamentos_multilote.any():
    print(
        "Aviso: "
        f"{df_entradas_forrageira.loc[lancamentos_multilote, 'id_expense_entry'].nunique():,} "
        "lançamentos consomem mais de um lote; a quantidade foi dividida "
        "igualmente entre os lotes."
    )

df_entradas_forrageira["consumed_quantity_rateada"] = (
    pd.to_numeric(df_entradas_forrageira["consumed_quantity_kg"], errors="coerce")
    / df_entradas_forrageira["quantidade_lotes"]
)


# ------------------------------------------------------------------------------
# 2. Trazer o custo unitário da safra
# ------------------------------------------------------------------------------
# O lançamento traz id_planted_culture; o custo é por área + cultura + safra.
# vw_planted_culture_season faz essa tradução. A categoria vem do produto
# colhido (id_culture_harvest_product), que é único por feeding_category.
df_mapa_categoria_produto = (
    df_forage_production[["id_culture_harvest_product", "feeding_category"]]
    .dropna(subset=["id_culture_harvest_product"])
    .drop_duplicates()
)

df_mapa_categoria_produto["feeding_category"] = (
    df_mapa_categoria_produto["feeding_category"]
    .astype("string")
    .str.strip()
    .str.upper()
)

if df_mapa_categoria_produto.duplicated("id_culture_harvest_product").any():
    raise ValueError(
        "id_culture_harvest_product com mais de uma feeding_category; "
        "a categoria do lançamento ficaria ambígua."
    )

df_entradas_forrageira = df_entradas_forrageira.merge(
    df_planted_culture_season[
        ["id_planted_culture", "id_area", "id_culture", "harvest_season"]
    ],
    on="id_planted_culture",
    how="left",
    validate="m:1",
)

df_entradas_forrageira = df_entradas_forrageira.merge(
    df_mapa_categoria_produto,
    on="id_culture_harvest_product",
    how="left",
    validate="m:1",
)

df_entradas_forrageira = df_entradas_forrageira.merge(
    df_custo_unitario_forrageira[
        ["id_area", "id_culture", "harvest_season", "feeding_category",
         "custo_unitario_forrageira"]
    ],
    on=["id_area", "id_culture", "harvest_season", "feeding_category"],
    how="left",
    validate="m:1",
)

# Renomeada para deixar explicito que esta e a rota 2 (custo da safra apontada
# pelo proprio lancamento). A coluna custo_unitario_forrageira passa a ser o
# resultado final da cascata, decidido no fim desta secao.
df_entradas_forrageira = df_entradas_forrageira.rename(
    columns={"custo_unitario_forrageira": "preco_kg_safra"}
)

sem_custo = df_entradas_forrageira["preco_kg_safra"].isna()

if sem_custo.any():
    print(
        f"Aviso: {int(sem_custo.sum()):,} de {len(df_entradas_forrageira):,} "
        "lançamentos ficaram sem custo unitário de forrageira."
    )


# ------------------------------------------------------------------------------
# 2-B. Demais rotas de preço, aplicadas em cascata
# ------------------------------------------------------------------------------
# Nem todo consumo com preço zero veio de colheita própria com produção
# registrada. Em 02/08/2026 eram 973 lançamentos e 62,2 milhões de kg sem
# id_production: lotes de COLHEITA sem movimento de entrada, lotes de COMPRA
# estocados e consumidos depois, e casos sem lote. Antes desta correção esse
# alimento entrava no indicador custando zero.
#
# stock_unit_price_kg vem de StockControlItem.unit_value / unit_factor_kg.
# Conferido no banco em 02/08/2026: unit_value é o mesmo unit_price do
# lançamento que originou o lote, na unidade do lançamento, então dividir pelo
# fator entrega R$/kg.
#
# O custo da safra continua tendo prioridade; o fallback só age onde ele falta.

# A categoria do produto colhido não existe sem produção associada. Nesse caso
# vale a categoria digitada no próprio lançamento de alimentação.
df_entradas_forrageira["feeding_category"] = (
    df_entradas_forrageira["feeding_category"]
    .astype("string")
    .fillna(
        df_entradas_forrageira["category_code"]
        .astype("string")
        .str.strip()
        .str.upper()
    )
)

# O preço do lote é nominal na data da compra do lote, que é quando o dinheiro
# saiu. Sem data de compra, usa o mês do consumo.
df_entradas_forrageira["mes_preco_lote"] = (
    pd.to_datetime(df_entradas_forrageira["stock_purchase_date"], errors="coerce")
    .fillna(
        pd.to_datetime(
            df_entradas_forrageira["consumption_reference_month"], errors="coerce"
        )
    )
    .dt.to_period("M")
    .dt.to_timestamp()
)

df_entradas_forrageira = df_entradas_forrageira.drop(
    columns=["deflator_lote"], errors="ignore"
).merge(
    df_igpdi_aux.rename(
        columns={"data": "mes_preco_lote", "deflator": "deflator_lote"}
    ),
    on="mes_preco_lote",
    how="left",
    validate="m:1",
)

# Sem IGP-DI do mês, mantém o valor nominal.
df_entradas_forrageira["deflator_lote"] = (
    df_entradas_forrageira["deflator_lote"].fillna(1.0)
)

df_entradas_forrageira["preco_kg_lote"] = (
    pd.to_numeric(df_entradas_forrageira["stock_unit_price_kg"], errors="coerce")
    / df_entradas_forrageira["deflator_lote"]
)

# Faixa de sanidade por categoria. Medido em 02/08/2026: o volumoso do fallback
# se concentra abaixo de R$ 2/kg (683 de 699 lançamentos) e os 7 acima de
# R$ 5/kg somam R$ 2,7 milhões. Preço de volumoso nessa ordem é erro de
# digitação, não custo. Os suspeitos ficam de fora e são reportados.
LIMITE_PRECO_KG_LOTE = {
    "VOLUMOSO":    5.0,
    "CONCENTRADO": 20.0,
    "MINERAIS":    50.0,
}

limite_da_linha = (
    df_entradas_forrageira["feeding_category"]
    .map(LIMITE_PRECO_KG_LOTE)
    .astype("float64")
)

preco_lote_suspeito = (
    df_entradas_forrageira["preco_kg_lote"].notna()
    & limite_da_linha.notna()
    & df_entradas_forrageira["preco_kg_lote"].gt(limite_da_linha)
)

if preco_lote_suspeito.any():
    print(
        f"Aviso: {int(preco_lote_suspeito.sum()):,} lotes com preço acima da "
        "faixa de sanidade da categoria; a rota do lote foi descartada e o "
        "lançamento segue para a próxima etapa da cascata. Somavam "
        f"{df_entradas_forrageira.loc[preco_lote_suspeito, 'consumed_quantity_rateada'].sum():,.0f} kg."
    )

# Descarta só esta rota. Antes o preço suspeito zerava o lançamento inteiro.
df_entradas_forrageira.loc[preco_lote_suspeito, "preco_kg_lote"] = np.nan


# ------------------------------------------------------------------------------
# 2-C. Custo da safra pelo lote, quando o lote mistura mais de uma colheita
# ------------------------------------------------------------------------------
# O lançamento aponta uma produção só, mas o silo pode ter recebido duas colheitas.
# Nesse caso o custo do que saiu do lote é a média dos custos das safras que
# entraram nele, ponderada pela quantidade colhida — não o custo de uma delas.
# Em 05/08/2026 eram 12 lotes com mais de uma colheita, 2 deles com consumo.
#
# Só as colheitas que têm custo entram na ponderação. Quando parte do lote não
# tem custo lançado na lavoura, o custo unitário sai subestimado, e é por isso
# que safras_sem_custo_no_lote acompanha o lançamento até a auditoria.
df_lote_colheita = df_stock_lot_production.copy()

df_lote_colheita["quantity_produced"] = pd.to_numeric(
    df_lote_colheita["quantity_produced"], errors="coerce"
)

# A categoria vem do produto colhido, mesma chave usada no custo unitário da safra.
df_lote_colheita = df_lote_colheita.merge(
    df_forage_production[["id_production", "feeding_category"]].drop_duplicates("id_production"),
    on="id_production",
    how="left",
    validate="m:1",
)

df_lote_colheita["feeding_category"] = (
    df_lote_colheita["feeding_category"].astype("string").str.strip().str.upper()
)

df_lote_colheita = df_lote_colheita.merge(
    df_custo_unitario_forrageira[
        ["id_area", "id_culture", "harvest_season", "feeding_category",
         "custo_unitario_forrageira"]
    ].rename(columns={"custo_unitario_forrageira": "custo_unitario_safra_do_lote"}),
    on=["id_area", "id_culture", "harvest_season", "feeding_category"],
    how="left",
    validate="m:1",
)

df_lote_colheita["peso_ponderacao"] = np.where(
    df_lote_colheita["custo_unitario_safra_do_lote"].notna(),
    df_lote_colheita["quantity_produced"].fillna(0.0),
    0.0,
)

df_lote_colheita["custo_ponderado_parcial"] = (
    df_lote_colheita["custo_unitario_safra_do_lote"].fillna(0.0)
    * df_lote_colheita["peso_ponderacao"]
)

df_custo_unitario_lote = (
    df_lote_colheita
    .groupby("id_stock_control_item", as_index=False)
    .agg(
        custo_ponderado_parcial=("custo_ponderado_parcial", "sum"),
        peso_ponderacao=("peso_ponderacao", "sum"),
        colheitas_no_lote=("id_production", "nunique"),
        colheitas_com_custo=("custo_unitario_safra_do_lote", "count"),
    )
)

df_custo_unitario_lote["preco_kg_safra_lote"] = (
    df_custo_unitario_lote["custo_ponderado_parcial"]
    / df_custo_unitario_lote["peso_ponderacao"].replace(0, np.nan)
)

df_custo_unitario_lote["safras_sem_custo_no_lote"] = (
    df_custo_unitario_lote["colheitas_no_lote"]
    - df_custo_unitario_lote["colheitas_com_custo"]
)

df_entradas_forrageira = df_entradas_forrageira.drop(
    columns=["preco_kg_safra_lote", "colheitas_no_lote", "colheitas_com_custo",
             "safras_sem_custo_no_lote"],
    errors="ignore",
).merge(
    df_custo_unitario_lote[
        ["id_stock_control_item", "preco_kg_safra_lote", "colheitas_no_lote",
         "colheitas_com_custo", "safras_sem_custo_no_lote"]
    ],
    on="id_stock_control_item",
    how="left",
    validate="m:1",
)

# Lote com uma colheita só continua usando o custo da safra apontada pelo próprio
# lançamento; lote misto usa a média ponderada, que é o que está fisicamente no silo.
lote_misto = df_entradas_forrageira["colheitas_no_lote"].fillna(0).gt(1)

df_entradas_forrageira["custo_safra_ponderado_no_lote"] = (
    lote_misto & df_entradas_forrageira["preco_kg_safra_lote"].notna()
)

df_entradas_forrageira["preco_kg_safra"] = np.where(
    df_entradas_forrageira["custo_safra_ponderado_no_lote"],
    df_entradas_forrageira["preco_kg_safra_lote"],
    df_entradas_forrageira["preco_kg_safra"],
)

# Lançamento sem produção própria ainda pode ser custeado pela safra do lote.
df_entradas_forrageira["preco_kg_safra"] = (
    pd.to_numeric(df_entradas_forrageira["preco_kg_safra"], errors="coerce")
    .fillna(pd.to_numeric(df_entradas_forrageira["preco_kg_safra_lote"], errors="coerce"))
)

if lote_misto.any():
    print(
        f"Aviso: {int(lote_misto.sum()):,} consumos saem de lote que recebeu mais de "
        "uma colheita; nesses o custo é a média ponderada das safras do lote."
    )


# ------------------------------------------------------------------------------
# 2-D. Cascata: primeira rota que entrega preço positivo vence
# ------------------------------------------------------------------------------
# Só duas rotas, ambas ancoradas em valor real: a despesa da lavoura e o valor do
# próprio lote. Estimativa por mediana de preço foi descartada de propósito —
# preencher custo com preço de outro lote ou de outra fazenda entrega ao cliente
# um número que ninguém lançou. O que não tem âncora fica sem custo e aparece
# nominalmente na aba de pendências da seção 3.4.2.5, para cobrança do lançamento.
ROTAS_CUSTO_FORRAGEIRA = [
    ("Custo da safra", "preco_kg_safra"),
    ("Preço do lote",  "preco_kg_lote"),
]

df_entradas_forrageira["custo_unitario_forrageira"] = np.nan
df_entradas_forrageira["metodo_custo_forrageira"] = "Sem custo"

for nome_rota, coluna_rota in ROTAS_CUSTO_FORRAGEIRA:

    if coluna_rota not in df_entradas_forrageira.columns:
        continue

    preco_rota = pd.to_numeric(df_entradas_forrageira[coluna_rota], errors="coerce")

    # A faixa de sanidade da categoria vale para toda rota de preço, não só o lote.
    dentro_da_faixa = ~(limite_da_linha.notna() & preco_rota.gt(limite_da_linha))

    aplicavel = (
        df_entradas_forrageira["custo_unitario_forrageira"].isna()
        & preco_rota.gt(0)
        & dentro_da_faixa
    )

    df_entradas_forrageira.loc[aplicavel, "custo_unitario_forrageira"] = preco_rota[aplicavel]
    df_entradas_forrageira.loc[aplicavel, "metodo_custo_forrageira"] = nome_rota

ainda_sem_custo = df_entradas_forrageira["custo_unitario_forrageira"].isna()

print("\nCascata de precificação da forrageira própria:")
print(
    df_entradas_forrageira
    .assign(kg=df_entradas_forrageira["consumed_quantity_rateada"])
    .groupby("metodo_custo_forrageira")
    .agg(lancamentos=("id_expense_entry", "size"), kg=("kg", "sum"))
    .sort_values("lancamentos", ascending=False)
    .to_string()
)

if ainda_sem_custo.any():
    print(
        f"\nAviso: {int(ainda_sem_custo.sum()):,} lançamentos sem âncora de custo; "
        f"{df_entradas_forrageira.loc[ainda_sem_custo, 'consumed_quantity_rateada'].sum():,.0f} kg "
        "entram no indicador valendo zero e estão na aba de pendências (seção 3.4.2.5)."
    )

df_entradas_forrageira["custo_forrageira"] = (
    df_entradas_forrageira["consumed_quantity_rateada"]
    * df_entradas_forrageira["custo_unitario_forrageira"]
)


# ------------------------------------------------------------------------------
# 3. Mês de referência e coluna de destino
# ------------------------------------------------------------------------------
# consumption_reference_month vem de ExpenseEntry.reference_month, a mesma
# competência mensal usada por vw_feeding. Conferido no DDL do banco em
# 02/08/2026: as duas rotas pousam no mesmo mês.
df_entradas_forrageira["reference_month"] = (
    pd.to_datetime(
        df_entradas_forrageira["consumption_reference_month"],
        errors="coerce",
    )
    .dt.to_period("M")
    .dt.to_timestamp()
)

MAPA_CATEGORIA_COLUNA_FEEDING = {
    "VOLUMOSO":    "voluminous_amount_total",
    "CONCENTRADO": "concentrate_amount_total",
    "MINERAIS":    "mineral_amount_total",
}

COLUNAS_ALIMENTACAO_FORRAGEIRA = list(MAPA_CATEGORIA_COLUNA_FEEDING.values())

df_entradas_forrageira["coluna_destino"] = (
    df_entradas_forrageira["feeding_category"]
    .map(MAPA_CATEGORIA_COLUNA_FEEDING)
)

sem_destino = df_entradas_forrageira["coluna_destino"].isna()

if sem_destino.any():
    print(
        f"Aviso: {int(sem_destino.sum()):,} lançamentos sem categoria mapeada; "
        f"R$ {df_entradas_forrageira.loc[sem_destino, 'custo_forrageira'].sum():,.2f} "
        "ficaram de fora."
    )


# ------------------------------------------------------------------------------
# 5. Agregar por propriedade e mês
# ------------------------------------------------------------------------------
# Base por lançamento preservada para a auditoria da seção 3.4.2.5.
df_auditoria_forrageira_lancamento = df_entradas_forrageira.copy()

df_custo_alimentacao_forrageira = (
    df_entradas_forrageira.loc[~sem_destino]
    .pivot_table(
        index=CHAVES,
        columns="coluna_destino",
        values="custo_forrageira",
        aggfunc="sum",
    )
    .reset_index()
)

df_custo_alimentacao_forrageira.columns.name = None

# Ausência de consumo de forrageira própria na categoria significa custo
# adicional igual a zero, e não valor desconhecido.
for coluna in COLUNAS_ALIMENTACAO_FORRAGEIRA:

    if coluna not in df_custo_alimentacao_forrageira.columns:
        df_custo_alimentacao_forrageira[coluna] = 0.0

    df_custo_alimentacao_forrageira[coluna] = (
        df_custo_alimentacao_forrageira[coluna].fillna(0.0)
    )

df_custo_alimentacao_forrageira = df_custo_alimentacao_forrageira[
    CHAVES + COLUNAS_ALIMENTACAO_FORRAGEIRA
]

if df_custo_alimentacao_forrageira.duplicated(CHAVES).any():
    raise ValueError(
        "df_custo_alimentacao_forrageira possui duplicidades por "
        "id_property e reference_month."
    )

print(
    f"Linhas: {len(df_custo_alimentacao_forrageira):,} | "
    f"Propriedades: {df_custo_alimentacao_forrageira['id_property'].nunique():,}"
)

print(
    df_custo_alimentacao_forrageira[COLUNAS_ALIMENTACAO_FORRAGEIRA]
    .sum()
    .round(2)
    .to_string()
)

display(df_custo_alimentacao_forrageira.head())


Aviso: 80 lançamentos consomem mais de um lote; a quantidade foi dividida igualmente entre os lotes.


Aviso: 895 de 4,484 lançamentos ficaram sem custo unitário de forrageira.


Aviso: 13 lotes com preço acima da faixa de sanidade da categoria; a rota do lote foi descartada e o lançamento segue para a próxima etapa da cascata. Somavam 273,241 kg.
Aviso: 5 consumos saem de lote que recebeu mais de uma colheita; nesses o custo é a média ponderada das safras do lote.



Cascata de precificação da forrageira própria:
                         lancamentos            kg
metodo_custo_forrageira                           
Custo da safra                  3586  3.452188e+08
Preço do lote                    727  3.785528e+07
Sem custo                        171  1.395017e+07

Aviso: 171 lançamentos sem âncora de custo; 13,950,167 kg entram no indicador valendo zero e estão na aba de pendências (seção 3.4.2.5).
Linhas: 3,625 | Propriedades: 371
voluminous_amount_total     63787674.92
concentrate_amount_total     4393969.88
mineral_amount_total          241957.83


,id_property,reference_month,voluminous_amount_total,concentrate_amount_total,mineral_amount_total
0,01957b94-005f-473e-8dbd-7d569a9aed4a,2026-05-01,0.000000,110617.220155,0.0
1,027d7c06-0ab1-4a29-8383-98a7e2d36a80,2025-04-01,43812.500332,0.000000,0.0
2,027d7c06-0ab1-4a29-8383-98a7e2d36a80,2025-05-01,43812.500332,0.000000,0.0
3,027d7c06-0ab1-4a29-8383-98a7e2d36a80,2025-07-01,40833.250310,0.000000,0.0
4,027d7c06-0ab1-4a29-8383-98a7e2d36a80,2025-08-01,41987.976341,0.000000,0.0


In [34]:
# ==============================================================================
# 3.4.2.3 CONCATENAÇÃO DO CUSTO DE FORRAGEIRA PRÓPRIA NA VW_FEEDING
# ==============================================================================
# df_feeding tem uma linha por id_property + reference_month. O append
# acrescenta as linhas de custo e reagrega pelas chaves, somando o custo da
# forrageira própria às colunas de alimentação já existentes.
#
# Serve para conferência. O que chega aos indicadores é a célula seguinte,
# porque df_feeding já foi consumida no merge das views.

df_feeding_corrigido = df_feeding.copy()

df_feeding_corrigido["reference_month"] = (
    pd.to_datetime(df_feeding_corrigido["reference_month"], errors="coerce")
    .dt.to_period("M")
    .dt.to_timestamp()
)

chaves_novas = (
    df_custo_alimentacao_forrageira
    .merge(
        df_feeding_corrigido[CHAVES],
        on=CHAVES,
        how="left",
        indicator=True,
    )
    .query("_merge == 'left_only'")
)

if len(chaves_novas) > 0:
    print(
        f"Aviso: {len(chaves_novas):,} pares propriedade/mês do custo de "
        "forrageira não existem em vw_feeding e serão criados pelo append. "
        "Indica mês com consumo de forrageira própria e nenhum lançamento "
        "de alimentação comprada."
    )

COLUNAS_VALOR_FEEDING = [
    coluna
    for coluna in df_feeding_corrigido.columns
    if coluna not in CHAVES
]

df_feeding_corrigido = (
    pd.concat(
        [df_feeding_corrigido, df_custo_alimentacao_forrageira],
        ignore_index=True,
    )
    .groupby(CHAVES, as_index=False)[COLUNAS_VALOR_FEEDING]
    .sum(min_count=1)
)

if df_feeding_corrigido.duplicated(CHAVES).any():
    raise ValueError(
        "df_feeding_corrigido possui duplicidades por "
        "id_property e reference_month."
    )

print(
    f"df_feeding: {len(df_feeding):,} linhas | "
    f"df_feeding_corrigido: {len(df_feeding_corrigido):,} linhas"
)

display(df_feeding_corrigido.head())


df_feeding: 15,500 linhas | df_feeding_corrigido: 15,500 linhas


,id_property,reference_month,voluminous_purchased_quantity,voluminous_consumed_quantity,voluminous_amount_total,concentrate_purchased_quantity,concentrate_consumed_quantity,concentrate_amount_total,mineral_purchased_quantity,mineral_consumed_quantity,mineral_amount_total
0,00220277-a58e-4b9e-9b89-e6acf8a1a400,2024-08-01,77550.0,77550.0,9306.0,60246.0,60246.0,80346.360,2646.0,2646.0,10584.0
1,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-04-01,84000.0,84000.0,21184.8,16090.0,16090.0,42191.960,800.0,800.0,1692.8
2,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-05-01,90000.0,90000.0,22680.0,15596.0,13596.0,28848.752,0.0,0.0,0.0
3,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-06-01,90000.0,90000.0,22680.0,21115.0,23115.0,51860.440,450.0,450.0,2191.5
4,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-07-01,96000.0,96000.0,24192.0,35502.0,35502.0,74514.728,600.0,600.0,1314.0


In [35]:
df_feeding_corrigido.loc[df_feeding_corrigido['id_property'] =='663897f4-86f7-4c6d-b600-189086a5543d']

,id_property,reference_month,voluminous_purchased_quantity,voluminous_consumed_quantity,voluminous_amount_total,concentrate_purchased_quantity,concentrate_consumed_quantity,concentrate_amount_total,mineral_purchased_quantity,mineral_consumed_quantity,mineral_amount_total
6373,663897f4-86f7-4c6d-b600-189086a5543d,2025-07-01,195000.0,195000.0,39000.00000,46490.0,46490.0,107830.400000,1450.0,1450.0,8594.00
6374,663897f4-86f7-4c6d-b600-189086a5543d,2025-08-01,195000.0,196200.0,40740.00000,37850.0,41230.0,95738.800000,0.0,0.0,0.00
6375,663897f4-86f7-4c6d-b600-189086a5543d,2025-09-01,195000.0,198200.0,43640.00000,43750.0,51910.0,115726.300000,50.0,50.0,46.00
6376,663897f4-86f7-4c6d-b600-189086a5543d,2025-10-01,195000.0,198360.0,43872.00000,41410.0,49810.0,109460.800000,906.0,906.0,4353.96
6377,663897f4-86f7-4c6d-b600-189086a5543d,2025-11-01,195000.0,198280.0,43756.00000,45540.0,54140.0,117545.140000,240.0,240.0,2352.00
6378,663897f4-86f7-4c6d-b600-189086a5543d,2025-12-01,195000.0,198280.0,43756.00000,52230.0,60330.0,125224.200000,1500.0,1500.0,5257.50
6379,663897f4-86f7-4c6d-b600-189086a5543d,2026-01-01,195000.0,199500.0,45525.00000,43010.0,47210.0,95684.700000,390.0,390.0,3350.40
6380,663897f4-86f7-4c6d-b600-189086a5543d,2026-02-01,195000.0,199200.0,45090.00000,42330.0,48880.0,97879.500000,220.0,220.0,1881.80
6381,663897f4-86f7-4c6d-b600-189086a5543d,2026-03-01,195000.0,199000.0,44800.00000,40060.0,46610.0,93560.500000,970.0,970.0,5244.80
6382,663897f4-86f7-4c6d-b600-189086a5543d,2026-04-01,0.0,198030.0,48692.17858,42150.0,49830.0,97960.200000,240.0,240.0,2260.80


In [36]:
# ==============================================================================
# 3.4.2.4 REPASSE DA CORREÇÃO PARA A BASE INTEGRADA
# ==============================================================================
# df_integrada já recebeu vw_feeding no merge das views. Sem esta etapa a
# correção fica apenas em df_feeding_corrigido e não chega aos indicadores
# calculados adiante (feeding_cost, feeding_cost_liter, COE etc.).

SUFIXO_FORRAGEIRA = "_forrageira_propria"

colunas_ja_aplicadas = [
    coluna
    for coluna in COLUNAS_ALIMENTACAO_FORRAGEIRA
    if f"{coluna}{SUFIXO_FORRAGEIRA}" in df_integrada.columns
]

if colunas_ja_aplicadas:
    raise RuntimeError(
        "A correção de forrageira já foi aplicada em df_integrada nesta "
        "execução. Reexecute a partir da célula de merge das views antes "
        "de repetir esta etapa."
    )

cobertura_forrageira = df_custo_alimentacao_forrageira.merge(
    df_integrada[CHAVES],
    on=CHAVES,
    how="left",
    indicator=True,
)

fora_da_base = int(cobertura_forrageira["_merge"].eq("left_only").sum())

if fora_da_base > 0:
    print(
        f"Aviso: {fora_da_base:,} pares propriedade/mês do custo de forrageira "
        "não existem em df_integrada e serão descartados no merge."
    )

df_integrada = df_integrada.merge(
    df_custo_alimentacao_forrageira.rename(
        columns={
            coluna: f"{coluna}{SUFIXO_FORRAGEIRA}"
            for coluna in COLUNAS_ALIMENTACAO_FORRAGEIRA
        }
    ),
    on=CHAVES,
    how="left",
    validate="one_to_one",
)

for coluna in COLUNAS_ALIMENTACAO_FORRAGEIRA:
    df_integrada[coluna] = (
        df_integrada[[coluna, f"{coluna}{SUFIXO_FORRAGEIRA}"]]
        .sum(axis=1, min_count=1)
    )

print("Custo de forrageira própria incorporado (R$):")

print(
    df_integrada[ [ f"{coluna}{SUFIXO_FORRAGEIRA}" for coluna in COLUNAS_ALIMENTACAO_FORRAGEIRA ] ]
    .sum()
    .round(2)
    .to_string()
)


Aviso: 57 pares propriedade/mês do custo de forrageira não existem em df_integrada e serão descartados no merge.
Custo de forrageira própria incorporado (R$):
voluminous_amount_total_forrageira_propria     62988553.26
concentrate_amount_total_forrageira_propria     4268922.96
mineral_amount_total_forrageira_propria          235395.17


In [37]:
# ==============================================================================
# 3.4.2.5 AUDITORIA DA FORRAGEIRA PRÓPRIA: LANÇAMENTO, LOTE E SAFRA
# ==============================================================================
# Três visões da mesma cascata, para responder "por que este alimento consumido
# não virou custo?" sem abrir o banco:
#   - por lançamento: qual rota precificou cada consumo e quanto atribuiu;
#   - por lote: quanto entrou pela colheita, quanto já foi consumido, quanto
#     sobrou em estoque e quanto do valor do lote já virou custo de alimentação;
#   - por safra: custo rateado da lavoura contra custo já apropriado no consumo.
#
# As duas primeiras vão para o Excel mensal. A terceira fecha a conservação:
# custo apropriado maior que o custo da lavoura é erro, não estoque.

COLUNAS_AUDITORIA_LANCAMENTO = [
    "id_property", "property_name", "reference_month", "category_code",
    "stock_item_name", "id_stock_control_item", "stock_origin",
    "harvest_season", "lot_harvest_season",
    "consumed_quantity_kg", "quantidade_lotes", "consumed_quantity_rateada",
    "preco_kg_safra", "preco_kg_lote",
    "colheitas_no_lote", "safras_sem_custo_no_lote", "custo_safra_ponderado_no_lote",
    "custo_unitario_forrageira", "custo_forrageira",
    "metodo_custo_forrageira",
]

df_auditoria_forrageira = df_auditoria_forrageira_lancamento.copy()

# Nome da fazenda, quando a base integrada tiver a coluna.
if "property_name" in df_integrada.columns and "property_name" not in df_auditoria_forrageira.columns:
    df_auditoria_forrageira = df_auditoria_forrageira.merge(
        df_integrada[["id_property", "property_name"]].drop_duplicates("id_property"),
        on="id_property",
        how="left",
        validate="m:1",
    )

df_auditoria_forrageira = df_auditoria_forrageira[
    [c for c in COLUNAS_AUDITORIA_LANCAMENTO if c in df_auditoria_forrageira.columns]
].sort_values(["id_property", "reference_month"]).reset_index(drop=True)


# ------------------------------------------------------------------------------
# 1. Resumo por método
# ------------------------------------------------------------------------------
df_resumo_metodo_forrageira = (
    df_auditoria_forrageira
    .groupby("metodo_custo_forrageira", as_index=False)
    .agg(
        lancamentos=("consumed_quantity_rateada", "size"),
        kg=("consumed_quantity_rateada", "sum"),
        custo=("custo_forrageira", "sum"),
    )
    .sort_values("kg", ascending=False)
)

df_resumo_metodo_forrageira["custo_kg"] = (
    df_resumo_metodo_forrageira["custo"]
    / df_resumo_metodo_forrageira["kg"].replace(0, np.nan)
)

print("Custo da forrageira própria por rota de precificação:")
display(df_resumo_metodo_forrageira.round(2))


# ------------------------------------------------------------------------------
# 2. Por lote: colheita, consumo, saldo e valor apropriado
# ------------------------------------------------------------------------------
# purchased_quantity_kg do lote é o que entrou; in_stock_quantity_kg é o saldo que
# o app mantém. A diferença entre o consumo somado aqui e (entrada - saldo) indica
# consumo lançado fora do módulo de alimentação ou baixa manual de estoque.
df_consumo_por_lote = (
    df_auditoria_forrageira_lancamento
    .groupby("id_stock_control_item", as_index=False)
    .agg(
        consumos=("consumed_quantity_rateada", "size"),
        kg_consumido_alimentacao=("consumed_quantity_rateada", "sum"),
        custo_apropriado=("custo_forrageira", "sum"),
        primeiro_consumo=("reference_month", "min"),
        ultimo_consumo=("reference_month", "max"),
    )
)

COLUNAS_LOTE = [
    "id_stock_control_item", "id_property", "item_name", "category", "origin",
    "situation", "purchase_date", "harvest_season", "id_planted_culture",
    "unit_price_kg", "purchased_quantity_kg", "in_stock_quantity_kg",
]

df_auditoria_forrageira_lote = (
    df_stock_lot_origin[[c for c in COLUNAS_LOTE if c in df_stock_lot_origin.columns]]
    .merge(df_consumo_por_lote, on="id_stock_control_item", how="inner", validate="1:m")
)

df_auditoria_forrageira_lote["valor_lote"] = (
    pd.to_numeric(df_auditoria_forrageira_lote["purchased_quantity_kg"], errors="coerce")
    * pd.to_numeric(df_auditoria_forrageira_lote["unit_price_kg"], errors="coerce")
)

# Quanto do lote ainda não foi consumido pela alimentação, em kg.
df_auditoria_forrageira_lote["kg_nao_consumido"] = (
    pd.to_numeric(df_auditoria_forrageira_lote["purchased_quantity_kg"], errors="coerce")
    - df_auditoria_forrageira_lote["kg_consumido_alimentacao"]
)

df_auditoria_forrageira_lote["custo_kg_apropriado"] = (
    df_auditoria_forrageira_lote["custo_apropriado"]
    / df_auditoria_forrageira_lote["kg_consumido_alimentacao"].replace(0, np.nan)
)

df_auditoria_forrageira_lote = (
    df_auditoria_forrageira_lote
    .sort_values(["id_property", "purchase_date"])
    .reset_index(drop=True)
)

lotes_sem_valor = df_auditoria_forrageira_lote["unit_price_kg"].fillna(0).le(0)

if lotes_sem_valor.any():

    # Sem valor no lote, resta a despesa da lavoura. Quem não tem nem uma nem
    # outra fica sem custo — por isso a contagem é separada, e não uma promessa
    # de que "as demais rotas resolveram".
    custo_por_lote_sem_valor = (
        df_auditoria_forrageira_lancamento
        .loc[
            df_auditoria_forrageira_lancamento["id_stock_control_item"].isin(
                df_auditoria_forrageira_lote.loc[lotes_sem_valor, "id_stock_control_item"]
            )
        ]
        .groupby("metodo_custo_forrageira")["consumed_quantity_rateada"]
        .agg(["size", "sum"])
    )

    print(
        f"\n{int(lotes_sem_valor.sum()):,} lotes consumidos estão sem valor unitário "
        f"no estoque ({df_auditoria_forrageira_lote.loc[lotes_sem_valor, 'kg_consumido_alimentacao'].sum():,.0f} kg "
        "consumidos). Como esses consumos foram resolvidos:"
    )
    print(
        custo_por_lote_sem_valor
        .rename(columns={"size": "lançamentos", "sum": "kg"})
        .to_string()
    )


# ------------------------------------------------------------------------------
# 3. Conservação por safra: custo da lavoura x custo apropriado no consumo
# ------------------------------------------------------------------------------
CHAVE_AUDITORIA_SAFRA = ["id_area", "id_culture", "harvest_season", "feeding_category"]

df_custo_apropriado_safra = (
    df_auditoria_forrageira_lancamento
    .loc[df_auditoria_forrageira_lancamento["metodo_custo_forrageira"].eq("Custo da safra")]
    .groupby(CHAVE_AUDITORIA_SAFRA, as_index=False, dropna=False)["custo_forrageira"]
    .sum(min_count=1)
    .rename(columns={"custo_forrageira": "custo_apropriado_consumo"})
)

df_auditoria_forrageira_safra = (
    df_custo_unitario_forrageira[CHAVE_AUDITORIA_SAFRA + ["custo_rateado", "quantidade_kg", "custo_unitario_forrageira"]]
    .merge(df_custo_apropriado_safra, on=CHAVE_AUDITORIA_SAFRA, how="left")
)

df_auditoria_forrageira_safra["custo_apropriado_consumo"] = (
    df_auditoria_forrageira_safra["custo_apropriado_consumo"].fillna(0.0)
)

# Positivo = custo da lavoura ainda em estoque. Negativo = apropriou mais do que a
# lavoura custou, o que só acontece por erro de quantidade ou de safra.
df_auditoria_forrageira_safra["custo_nao_apropriado"] = (
    df_auditoria_forrageira_safra["custo_rateado"]
    - df_auditoria_forrageira_safra["custo_apropriado_consumo"]
)

safras_estouradas = df_auditoria_forrageira_safra["custo_nao_apropriado"].lt(-0.01)

print(
    f"\nSafras com custo unitário: {len(df_auditoria_forrageira_safra):,} | "
    f"custo da lavoura ainda não apropriado: R$ "
    f"{df_auditoria_forrageira_safra['custo_nao_apropriado'].clip(lower=0).sum():,.2f}"
)

if safras_estouradas.any():
    print(
        f"Aviso: {int(safras_estouradas.sum()):,} safras apropriaram mais custo do que "
        "a lavoura custou (consumo maior que a produção registrada):"
    )
    display(
        df_auditoria_forrageira_safra
        .loc[safras_estouradas]
        .nsmallest(10, "custo_nao_apropriado")
        .round(2)
    )

# ------------------------------------------------------------------------------
# 4. Pendências: consumo que ficou sem custo, com o motivo de cada caso
# ------------------------------------------------------------------------------
# Esta é a lista de cobrança. Cada linha é alimento consumido que entrou no
# indicador valendo zero, com o motivo classificado para o consultor saber o que
# pedir ao cliente: lançar a despesa da lavoura, cadastrar a origem do lote ou
# digitar o valor da compra.
pendencia = df_auditoria_forrageira_lancamento["metodo_custo_forrageira"].eq("Sem custo")

df_pendencia_custo_forrageira = df_auditoria_forrageira_lancamento.loc[pendencia].copy()

sem_lote = df_pendencia_custo_forrageira["id_stock_control_item"].isna()
origem_ausente = df_pendencia_custo_forrageira["stock_origin"].isna()
lote_de_compra = df_pendencia_custo_forrageira["stock_origin"].astype(str).str.upper().eq("COMPRA")
tem_safra = df_pendencia_custo_forrageira["lot_harvest_season"].notna()

df_pendencia_custo_forrageira["motivo"] = np.select(
    [sem_lote, tem_safra, lote_de_compra, origem_ausente],
    [
        "Consumo sem lote de estoque",
        "Safra sem custo lançado na lavoura",
        "Lote de compra sem valor digitado",
        "Lote sem origem cadastrada",
    ],
    default="Lote sem valor e sem safra",
)

COLUNAS_PENDENCIA = [
    "id_property", "property_name", "reference_month", "category_code",
    "stock_item_name", "lot_harvest_season", "stock_origin",
    "consumed_quantity_rateada", "motivo",
]

if "property_name" in df_integrada.columns and "property_name" not in df_pendencia_custo_forrageira.columns:
    df_pendencia_custo_forrageira = df_pendencia_custo_forrageira.merge(
        df_integrada[["id_property", "property_name"]].drop_duplicates("id_property"),
        on="id_property",
        how="left",
        validate="m:1",
    )

df_pendencia_custo_forrageira = (
    df_pendencia_custo_forrageira[
        [c for c in COLUNAS_PENDENCIA if c in df_pendencia_custo_forrageira.columns]
    ]
    .sort_values("consumed_quantity_rateada", ascending=False)
    .reset_index(drop=True)
)

if len(df_pendencia_custo_forrageira):
    print(
        f"\nPendências de custo: {len(df_pendencia_custo_forrageira):,} consumos, "
        f"{df_pendencia_custo_forrageira['consumed_quantity_rateada'].sum() / 1000:,.0f} t, "
        f"{df_pendencia_custo_forrageira['id_property'].nunique():,} propriedades."
    )
    display(
        df_pendencia_custo_forrageira
        .groupby("motivo", as_index=False)
        .agg(consumos=("consumed_quantity_rateada", "size"), kg=("consumed_quantity_rateada", "sum"))
        .sort_values("kg", ascending=False)
    )
else:
    print("\nNenhuma pendência de custo: todo consumo de alimento próprio foi custeado.")


# ------------------------------------------------------------------------------
# 5. Safras produzidas, estocadas e consumidas sem nenhuma despesa lançada
# ------------------------------------------------------------------------------
# O nível que se cobra do cliente não é o lote, é a safra: a lavoura foi plantada,
# colhida e entrou no silo sem uma linha de despesa em vw_culture_expense_cost.
CHAVES_SAFRA_PENDENTE = ["id_property", "id_area", "id_culture", "harvest_season"]

df_lote_sem_custo_safra = (
    df_stock_lot_origin
    .loc[
        df_stock_lot_origin["id_stock_control_item"].isin(
            df_auditoria_forrageira_lancamento.loc[pendencia, "id_stock_control_item"]
        )
        & df_stock_lot_origin["harvest_season"].notna()
    ]
    .groupby(CHAVES_SAFRA_PENDENTE, as_index=False)
    .agg(
        lotes=("id_stock_control_item", "nunique"),
        kg_estocado=("purchased_quantity_kg", "sum"),
        produto=("item_name", "first"),
    )
    .sort_values("kg_estocado", ascending=False)
    .reset_index(drop=True)
)

if len(df_lote_sem_custo_safra):
    print(
        f"\nSafras com colheita estocada e consumida sem despesa lançada: "
        f"{len(df_lote_sem_custo_safra):,} "
        f"({df_lote_sem_custo_safra['kg_estocado'].sum() / 1000:,.0f} t)."
    )
    display(df_lote_sem_custo_safra.head(15))

display(df_auditoria_forrageira.head())


Custo da forrageira própria por rota de precificação:


,metodo_custo_forrageira,lancamentos,kg,custo,custo_kg
0,Custo da safra,3586,3.452188e+08,58186000.31,0.17
1,Preço do lote,727,3.785528e+07,10237602.32,0.27
2,Sem custo,171,1.395017e+07,0.00,0.00



85 lotes consumidos estão sem valor unitário no estoque (13,724,420 kg consumidos). Como esses consumos foram resolvidos:
                         lançamentos            kg
metodo_custo_forrageira                           
Sem custo                        160  1.372442e+07



Safras com custo unitário: 1,058 | custo da lavoura ainda não apropriado: R$ 147,935,695.87
Aviso: 21 safras apropriaram mais custo do que a lavoura custou (consumo maior que a produção registrada):


,id_area,id_culture,harvest_season,feeding_category,custo_rateado,quantidade_kg,custo_unitario_forrageira,custo_apropriado_consumo,custo_nao_apropriado
601,1845d8b8-4b70-4e9f-a5d0-47112b3cfc86,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,24/25,VOLUMOSO,967389.80,3520000.0,0.27,1133550.00,-166160.19
43,4ac35b8f-2072-43e5-a857-5cb5e0d452ca,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,23/24,VOLUMOSO,55735.48,330000.0,0.17,108701.07,-52965.59
63,52d2c8fe-85bf-4e8c-b6fc-004b1b5baabe,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,24/25,VOLUMOSO,158949.03,1180000.0,0.13,206633.74,-47684.71
5,c80ddfec-b204-4036-8d7f-ccfa249ee2b7,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,24/25,VOLUMOSO,384116.46,2191820.0,0.18,425738.33,-41621.88
739,5ef76d8b-067d-4525-8841-af1553ce60c9,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,23/24,VOLUMOSO,29803.57,100000.0,0.30,56626.78,-26823.21
746,bc12744f-ddc9-4e75-af87-3320c5ecfb62,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,23/24,VOLUMOSO,26795.13,150000.0,0.18,53590.26,-26795.13
442,49e4319e-ac06-45c9-8017-ba8e655d858b,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,23/24,VOLUMOSO,24884.40,135000.0,0.18,44238.93,-19354.53
725,b77ddd44-15e0-4f23-9c4b-5551d1dbfd7b,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,24/25,CONCENTRADO,101541.34,184320.0,0.55,118068.25,-16526.91
741,6f66ceaf-9d02-42d8-b4fa-8c73ac5a13bd,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,23/24,VOLUMOSO,14601.24,80000.0,0.18,31027.63,-16426.39
591,6ffab552-02a2-47c2-a00e-269bc74e3a1e,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,23/24,VOLUMOSO,14624.60,80000.0,0.18,30949.30,-16324.71



Pendências de custo: 171 consumos, 13,950 t, 61 propriedades.


,motivo,consumos,kg
3,Safra sem custo lançado na lavoura,81,8.242277e+06
1,Lote sem origem cadastrada,63,4.528193e+06
2,Lote sem valor e sem safra,19,9.820780e+05
0,Lote de compra sem valor digitado,8,1.976200e+05



Safras com colheita estocada e consumida sem despesa lançada: 19 (11,940 t).


,id_property,id_area,id_culture,harvest_season,lotes,kg_estocado,produto
0,7fdc4768-8f72-4fd4-ad6d-3aeeb8649874,baff5c7a-cac4-4d0f-9640-caa3c6a60cea,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,23/24,1,3974250.0,Milho (silagem)
1,c814b703-5ab5-4217-ab4d-86aec65cdd86,b748b735-4702-44d4-9158-42678f8765bf,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,24/25,1,2135000.0,Milho (silagem)
2,7351f173-63c6-4941-be3d-09aa802f92a4,ca8728f9-2b47-4540-a422-465e9062d957,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,24/25,1,1045000.0,Milho (silagem)
3,7400bd5a-b490-47d4-8676-d722815e01bb,64c382cc-2c4b-4bc9-996f-54ebe41a1159,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,23/24,1,941160.0,Milho (silagem)
4,0e9e4f8e-df78-425d-82be-9f5a387a2332,816dbdee-98d9-4cd9-80fd-92d6b9bffd04,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,24/25,1,770000.0,Milho (silagem)
5,87133f7e-1326-4386-aef1-e2f55679bc1e,9b77c589-9c72-48cd-950b-ce054d319e9e,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,25/26,1,715000.0,Milho (silagem)
6,f8a9584b-5906-422a-a356-407a327ce2ee,cec2e3f2-2630-4669-8ee1-b534307b60b3,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,24/25,1,600000.0,Milho (silagem)
7,27dee6c3-a4a7-4711-b049-908ec5da026f,8973a3ff-2daf-449b-8096-7c176834ae1a,13d18705-54ad-4e44-ba6b-fbd8fd03fbf1,24/25,1,600000.0,Mombaça (silagem)
8,02b4352e-c80f-47d8-8618-fa6545a74f10,e9b4d84d-24e5-4784-a4f5-21dcffba50c2,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,23/24,1,382030.0,Milho (silagem)
9,17cea1fd-b3d3-4de8-b7cf-7c6927635e46,17471f45-55c3-4338-bddb-ebb3e52a6979,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,23/24,1,350000.0,Milho (silagem)


,id_property,reference_month,category_code,stock_item_name,id_stock_control_item,stock_origin,harvest_season,lot_harvest_season,consumed_quantity_kg,quantidade_lotes,consumed_quantity_rateada,preco_kg_safra,preco_kg_lote,colheitas_no_lote,safras_sem_custo_no_lote,custo_safra_ponderado_no_lote,custo_unitario_forrageira,custo_forrageira,metodo_custo_forrageira
0,01957b94-005f-473e-8dbd-7d569a9aed4a,2026-05-01,CONCENTRADO,Algodão (caroço),f7a21cab-8f53-491c-a8c2-715855019396,COMPRA,NaN,None,47966.0,1,47966.0,NaN,1.394813,NaN,NaN,False,1.394813,66903.586006,Preço do lote
1,01957b94-005f-473e-8dbd-7d569a9aed4a,2026-05-01,CONCENTRADO,Polpa cítrica,31f4894f-6a1b-44df-b06b-dd735939ebd4,COMPRA,NaN,None,23843.0,2,11921.5,NaN,1.458563,NaN,NaN,False,1.458563,17388.257685,Preço do lote
2,01957b94-005f-473e-8dbd-7d569a9aed4a,2026-05-01,CONCENTRADO,Polpa cítrica,25a12bc4-16e4-4a11-a011-a547104a7e22,COMPRA,NaN,None,23843.0,2,11921.5,NaN,1.469656,NaN,NaN,False,1.469656,17520.498543,Preço do lote
3,01957b94-005f-473e-8dbd-7d569a9aed4a,2026-05-01,CONCENTRADO,DDG/DDGS/WDG/WDGS,705c1be6-f5b5-41dc-b3f4-6b12f2bd82a5,COMPRA,NaN,None,5042.0,1,5042.0,NaN,1.746307,NaN,NaN,False,1.746307,8804.877921,Preço do lote
4,027d7c06-0ab1-4a29-8383-98a7e2d36a80,2025-04-01,VOLUMOSO,Milho (silagem),84d9d410-00b8-4746-8782-399681570935,COLHEITA,24/25,24/25,250000.0,1,250000.0,0.17525,0.176829,1.0,0.0,False,0.175250,43812.500332,Custo da safra


## 3.5 Engenharia de Recursos e Cálculo de Indicadores Mensais


In [38]:
# ==============================================================================
# 3.5 ENGENHARIA DE RECURSOS E CÁLCULO DE INDICADORES MENSAIS
# ==============================================================================
CHAVES = ["id_property", "reference_month"]

linhas_iniciais = len(df_integrada)

print(f"Linhas: {df_integrada.shape[0]:,}")
print(f"Colunas antes das flags: {df_integrada.shape[1]:,}")

# ==============================================================================
# 2. MAPA DAS VIEWS E COLUNAS DE PRESENÇA
# ==============================================================================
MAPA_PRESENCA = {
    "vw_cattle": "has_cattle_data",
    "vw_expense": "has_expense_data",
    "vw_feeding": "has_feeding_data",
    "vw_labor": "has_labor_data",
    "vw_own_milk": "has_own_milk_data",
    "mvw_asset_payment_history": "has_asset_data",
    "mvw_area_land_summary": "has_active_area_month",
    "vw_dairy_production_system_monthly": "has_dairy_production_system_data",
}

# A revenue é a base da tabela final. Portanto, todas as linhas possuem revenue.
df_integrada["has_revenue_data"] = 1

# ==============================================================================
# 3. CRIAR AS FLAGS DE PRESENÇA
# ==============================================================================
for nome_view, nome_flag in MAPA_PRESENCA.items():

    print(f"Criando indicador de presença: {nome_flag}")

    if nome_view not in dados_preparados:
        raise KeyError( f"A view {nome_view} não foi encontrada em dados_preparados." )

    # Evita duplicação caso a célula seja executada novamente
    df_integrada = df_integrada.drop(
        columns=[nome_flag],
        errors="ignore",
    )

    # Uma linha por propriedade e mês presente na view
    chaves_disponiveis = (
        dados_preparados[nome_view][CHAVES]
        .dropna(subset=CHAVES)
        .drop_duplicates(subset=CHAVES)
        .assign(**{nome_flag: 1})
    )

    linhas_antes = len(df_integrada)

    df_integrada = df_integrada.merge(
        chaves_disponiveis,
        on=CHAVES,
        how="left",
        validate="one_to_one",
    )
    
    linhas_depois = len(df_integrada)

    if linhas_antes != linhas_depois:
        raise ValueError(
            f"O merge da flag {nome_flag} alterou a quantidade de linhas: "
            f"{linhas_antes:,} para {linhas_depois:,}."
        )

    df_integrada[nome_flag] = (
        df_integrada[nome_flag]
        .fillna(0)
        .astype("int8")
    )

def somar_colunas_preservando_ausencia(df: pd.DataFrame, colunas: list[str]) -> pd.Series:
    """
    Soma as colunas informadas.
    Quando todas as colunas estiverem ausentes na linha, mantém o resultado como NaN em vez de transformar em zero.
    """

    colunas_ausentes = [
        coluna
        for coluna in colunas
        if coluna not in df.columns
    ]
    
    if colunas_ausentes:
        raise KeyError("Colunas necessárias não encontradas: " + ", ".join(colunas_ausentes))

    return df[colunas].sum(axis=1, min_count=1)

# ==============================================================================
# 4. ORGANIZAR AS COLUNAS DE PRESENÇA
# ==============================================================================

colunas_presenca = [
    "has_revenue_data",
    "has_cattle_data",
    "has_expense_data",
    "has_feeding_data",
    "has_labor_data",
    "has_own_milk_data",
    "has_asset_data",
    "has_active_area_month",
    "has_dairy_production_system_data"
]


# ==============================================================================
# 5. VALIDAR O RESULTADO
# ==============================================================================

assert len(df_integrada) == linhas_iniciais, ("A criação das flags alterou a quantidade de linhas.")

assert not df_integrada.duplicated(CHAVES).any(), ("Foram geradas duplicidades por id_property e reference_month.")

print("\nFlags de presença criadas com sucesso.")
print(f"Linhas finais: {df_integrada.shape[0]:,}")
print(f"Colunas finais: {df_integrada.shape[1]:,}")


# ==============================================================================
# 6. CALCULAR A COBERTURA DE CADA VIEW
# ==============================================================================
df_cobertura_views = (
    df_integrada[colunas_presenca]
    .mean()
    .mul(100)
    .round(2)
    .rename("coverage_percentage")
    .rename_axis("source")
    .reset_index()
)

display(df_cobertura_views)

# ==============================================================================
# 7. CALCULAR INDICADORES PADRÃO
# ==============================================================================

# Renda do leite consumido
df_integrada['discarded_quantity_milk_revenue']   = df_integrada['discarded_quantity'] * df_integrada['milk_unit_price']
df_integrada['hired_labor_quantity_milk_revenue'] = df_integrada['own_milk_hired_labor_quantity'] * df_integrada['milk_unit_price']
df_integrada['discarded_quantity_milk_revenue']   = df_integrada['discarded_quantity'] * df_integrada['milk_unit_price']
df_integrada['family_labor_milk_revenue']         = df_integrada['own_milk_family_labor_quantity'] * df_integrada['milk_unit_price']
df_integrada['calves_quantity_milk_revenue']      = df_integrada['calves_quantity'] * df_integrada['milk_unit_price']

# Renda do leite
df_integrada['total_milk_revenue'] = df_integrada[[
    'milk_sold_revenue',
    'milk_derivatives_revenue',
    'price_bonus',
    'discarded_quantity_milk_revenue',
    'hired_labor_quantity_milk_revenue',
    'family_labor_milk_revenue',
    'calves_quantity_milk_revenue'
]].sum(axis=1).round(2) - df_integrada['price_penalty']

# Leite produzido
df_integrada['milk_produced'] = df_integrada[[
    'milk_volume_sold',
    'milk_volume_derivatives',
    'discarded_quantity',
    'own_milk_hired_labor_quantity',
    'own_milk_family_labor_quantity',
    'calves_quantity']].sum(axis=1).round(2)

# Renda da Atividade
df_integrada['total_activity_revenue'] = (
    df_integrada[[
        'total_milk_revenue',
        'received_loans',
        'animal_sale',
        'other_revenues',
        'voluminous_sold',
        'concentrated_sold',
        'surplus_division',
    ]].sum(axis=1)
)
# Preço do Leite
df_integrada['milk_revenue_liter'] = df_integrada['total_milk_revenue'] / df_integrada['milk_produced']

# Leite diário
# Quantidade de dias do mês
df_integrada["days_in_month"] = ( df_integrada["reference_month"].dt.days_in_month)

# Produção média diária de leite
df_integrada["milk_daily"] = (df_integrada["milk_produced"] / df_integrada["days_in_month"].replace(0, np.nan))

# Produção diária por vaca em lactação
df_integrada["milk_lactating_cow_day"] = (df_integrada["milk_daily"] / df_integrada["lactating_cows"].replace(0, np.nan))

# Vacas em lactação sobre o total de vacas
df_integrada["lactating_cows_total_cows"] = (df_integrada["lactating_cows"] / df_integrada["total_cows"].replace(0, np.nan)) * 100

# Vacas em lactação sobre o total do rebanho
df_integrada["lactating_cows_total_cattle"] = (df_integrada["lactating_cows"] / df_integrada["total_cattle"].replace(0, np.nan) ) * 100

# Ajustar mão de obra para dias/homem
df_integrada['hired_labor_quantity']  = df_integrada['hired_labor_quantity'] / df_integrada["days_in_month"]
df_integrada['family_labor_quantity'] = df_integrada['family_labor_quantity'] / df_integrada["days_in_month"]

# Quantidade total de mão de obra
df_integrada["total_labor_quantity"] = ( df_integrada[[ "hired_labor_quantity", "family_labor_quantity"]] .sum(axis=1, min_count=1) )

# Produção diária por unidade de mão de obra total
df_integrada["milk_total_labor_day"] = (df_integrada["milk_daily"] / df_integrada["total_labor_quantity"].replace(0, np.nan))

# Vacas em lactação por unidade de mão de obra
df_integrada["lactating_cows_total_labor"] = (df_integrada["lactating_cows"] / df_integrada["total_labor_quantity"].replace(0, np.nan))

# Custo de concentrado e minerais
df_integrada["concentrate_mineral_cost"] = (df_integrada["concentrate_amount_total"].fillna(0) + df_integrada["mineral_amount_total"].fillna(0))

# Custo total da alimentação
df_integrada["feeding_cost"] = (df_integrada["voluminous_amount_total"].fillna(0) + df_integrada["concentrate_amount_total"].fillna(0) + df_integrada["mineral_amount_total"].fillna(0) )

# Quantidade de concentrado 
df_integrada["concentrate_mineral_quantity"] = (df_integrada["concentrate_purchased_quantity"].fillna(0) + df_integrada["mineral_purchased_quantity"].fillna(0))

# Custo da alimentação por litro
df_integrada["feeding_cost_liter"] = (df_integrada["feeding_cost"] / df_integrada["milk_produced"].replace(0, np.nan))

# Custo de volumoso por litro
df_integrada["voluminous_cost_liter"] = (df_integrada["voluminous_amount_total"] / df_integrada["milk_produced"].replace(0, np.nan))

# Custo de concentrado e minerais por litro
df_integrada["concentrate_mineral_cost_liter"] = (df_integrada["concentrate_mineral_cost"] / df_integrada["milk_produced"].replace(0, np.nan))

# Participação do custo da alimentação no preço do leite
df_integrada["feeding_cost_milk_price"] = (df_integrada["feeding_cost_liter"] / df_integrada["milk_revenue_liter"].replace(0, np.nan) ) * 100

# Custo total da mão de obra
df_integrada["total_labor_expenses"] = (df_integrada[["hired_labor_expenses", "family_labor_expenses"]] .sum(axis=1, min_count=1))

# Custo da mão de obra contratada por litro
df_integrada["hired_labor_cost_liter"] = (df_integrada["hired_labor_expenses"] / df_integrada["milk_produced"].replace(0, np.nan))

# Custo da mão de obra familiar por litro
df_integrada["family_labor_cost_liter"] = (df_integrada["family_labor_expenses"] / df_integrada["milk_produced"].replace(0, np.nan))

# Custo total da mão de obra por litro
df_integrada["total_labor_cost_liter"] = (df_integrada["total_labor_expenses"] / df_integrada["milk_produced"].replace(0, np.nan))

# Participação da mão de obra na receita do leite
df_integrada["labor_cost_milk_revenue"] = ( df_integrada["total_labor_expenses"] / df_integrada["total_milk_revenue"].replace(0, np.nan) ) * 100

# Agregação das demais despesas operacionais
COLUNAS_OUTRAS_DESPESAS = [
    "milk_replacer",
    "milking_material",
    "reproduction",
    "hormones",
    "medicines_vaccines",
    "technical_assistance",
    "taxes_fees",
    "land_lease",
    "repairs",
    "administration",
    "general_expenses",
    "bedding_replacement",
]

df_integrada["other_operating_expenses"] = (somar_colunas_preservando_ausencia(df=df_integrada, colunas=COLUNAS_OUTRAS_DESPESAS) )

# Estoque de capital de animais
df_integrada["animal_capital_stock"] = (df_integrada["lactating_cows"] * df_integrada["lactating_cows_value"]) + (df_integrada["dry_cows"] * df_integrada["dry_cows_value"]) + (df_integrada["nursing"] * df_integrada["nursing_value"]) + (df_integrada["rearing"] * df_integrada["rearing_value"]) + (df_integrada["males"] * df_integrada["males_value"]) + ((df_integrada["other_categories"] * df_integrada["other_categories_value"])/2)

# Estoque de capital da terra
df_integrada["land_capital_stock"] = df_integrada["hectares_propria"] * df_integrada["raw_land_value_medio_ponderado"]

# Define os componentes do estoque de capital fixo.
COLUNAS_CAPITAL_FIXO = [
    "monthly_average_capital_stock_benfeitorias",
    "monthly_average_capital_stock_maquinas_e_equipamentos",
]

# Soma o estoque médio de benfeitorias com o estoque médio de máquinas e equipamentos.
#
# min_count=2 exige que os dois valores estejam disponíveis.
df_integrada["fixed_capital_stock"] = (
    df_integrada[COLUNAS_CAPITAL_FIXO]
    .sum(
        axis=1,
        min_count=len(COLUNAS_CAPITAL_FIXO),
    )
)

# Estoque de capital total
COLUNAS_CAPITAL_TOTAL = [
    "animal_capital_stock",
    "land_capital_stock",
    "fixed_capital_stock",
]

df_integrada["total_capital_stock"] = (
    df_integrada[COLUNAS_CAPITAL_TOTAL]
    .sum(axis=1)
)

# Calcula o estoque de capital total por litro de produção diária.
#
# Esse indicador é o equivalente novo de:
# estoqueCapitalcomTerra_leiteDiario.
#
# A produção diária igual a zero é substituída por NaN
# para evitar divisão por zero.
df_integrada["total_capital_stock_milk_daily"] = (
    df_integrada["total_capital_stock"]
    / df_integrada["milk_daily"].replace(0, np.nan)
)


# Substitui eventuais resultados infinitos por NaN.
df_integrada[
    [
        "total_capital_stock",
        "total_capital_stock_milk_daily",
    ]
] = (
    df_integrada[
        [
            "total_capital_stock",
            "total_capital_stock_milk_daily",
        ]
    ]
    .replace([np.inf, -np.inf], np.nan)
)


# ==============================================================================
# INDICADORES DE ÁREA
# ==============================================================================

# Vacas em lactação por hectare de atividade
df_integrada["lactating_cows_hectare_activity"] = (df_integrada["lactating_cows"] / df_integrada["hectares_atividade"].replace(0, np.nan))

# Percentual da área arrendada
df_integrada["hectares_arrendada"] = df_integrada[[
    "hectares_rented_benfeitorias_estradas",
    "hectares_rented_app_reserva_legal",
    "hectares_rented_forrageiras",
]].sum(axis=1, skipna=True)

df_integrada["rented_area_percentage"] = (
    df_integrada["hectares_arrendada"] / df_integrada["hectares_total"].replace(0, np.nan) * 100
)

df_integrada.head(20)


Linhas: 16,101
Colunas antes das flags: 97
Criando indicador de presença: has_cattle_data
Criando indicador de presença: has_expense_data
Criando indicador de presença: has_feeding_data


Criando indicador de presença: has_labor_data


Criando indicador de presença: has_own_milk_data
Criando indicador de presença: has_asset_data


Criando indicador de presença: has_active_area_month
Criando indicador de presença: has_dairy_production_system_data

Flags de presença criadas com sucesso.
Linhas finais: 16,101
Colunas finais: 106


,source,coverage_percentage
0,has_revenue_data,100.00
1,has_cattle_data,94.60
2,has_expense_data,98.19
3,has_feeding_data,93.53
4,has_labor_data,97.39
5,has_own_milk_data,81.89
6,has_asset_data,96.69
7,has_active_area_month,97.72
8,has_dairy_production_system_data,66.83


C:\Users\analy\AppData\Local\Temp\ipykernel_58032\2233461704.py:327: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace([np.inf, -np.inf], np.nan)


,id_property,reference_month,milk_sold_revenue,milk_volume_sold,milk_unit_price,ccs,cpp,fat,protein,unit_price_derivative,...,labor_cost_milk_revenue,other_operating_expenses,animal_capital_stock,land_capital_stock,fixed_capital_stock,total_capital_stock,total_capital_stock_milk_daily,lactating_cows_hectare_activity,hectares_arrendada,rented_area_percentage
0,00220277-a58e-4b9e-9b89-e6acf8a1a400,2024-05-01,0.000000,0.00,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1.888052e+06,1.888052e+06,NaN,NaN,0.00,NaN
1,00220277-a58e-4b9e-9b89-e6acf8a1a400,2024-08-01,530109.101788,167274.00,3.169106,159.0,33.0,3.85,3.35,NaN,...,5.950626,45423.858214,2790000.0,NaN,2.007212e+06,4.797212e+06,811.870480,NaN,0.00,NaN
2,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-04-01,112019.765592,35677.00,3.139831,421.0,77.0,3.48,3.40,NaN,...,13.958394,11920.489982,0.0,<NA>,4.834637e+04,4.834637e+04,40.653393,3.500000,25.18,100.0
3,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-05-01,162733.714940,54884.00,2.965048,338.0,54.0,3.55,3.29,NaN,...,13.820995,22728.043793,0.0,<NA>,5.399037e+04,5.399037e+04,29.810868,3.250000,25.18,100.0
4,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-06-01,161358.457208,56311.00,2.865487,192.0,102.0,3.46,3.32,NaN,...,14.313373,11459.350879,862500.0,<NA>,5.896165e+04,9.214616e+05,465.611990,3.375000,25.18,100.0
5,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-07-01,199716.486371,69902.00,2.857093,116.0,38.0,3.39,3.25,NaN,...,9.190334,37162.552774,868000.0,<NA>,6.486475e+04,9.328647e+05,406.037561,3.500000,25.18,100.0
6,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-08-01,199666.489408,71836.00,2.779477,132.0,41.0,3.47,3.24,NaN,...,10.525011,103554.112749,1160000.0,<NA>,7.095112e+04,1.230951e+06,504.221520,3.375000,25.18,100.0
7,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-09-01,188311.806035,69531.00,2.708314,112.0,3.0,3.51,3.20,NaN,...,9.461966,47892.155122,840000.0,<NA>,7.120325e+04,9.112032e+05,367.168975,3.541667,25.18,100.0
8,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-10-01,178291.455363,68932.00,2.586483,145.0,8.0,3.58,3.25,NaN,...,10.050908,29532.934689,805000.0,<NA>,7.118096e+04,8.761810e+05,370.069348,2.962963,28.18,100.0
9,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-11-01,144148.810170,60781.00,2.371610,137.0,3.0,3.50,3.27,NaN,...,12.908788,67996.902633,760000.0,<NA>,7.516898e+04,8.351690e+05,397.882667,2.592593,28.18,100.0


In [39]:
# ==============================================================================
# 3.5.1 LISTAGEM DE COLUNAS DA BASE INTEGRADA FINAL
# ==============================================================================
df_integrada.loc[df_integrada['id_property']=='99ef0160-5fd8-44f9-93ed-6b6822f01c47']


,id_property,reference_month,milk_sold_revenue,milk_volume_sold,milk_unit_price,ccs,cpp,fat,protein,unit_price_derivative,...,labor_cost_milk_revenue,other_operating_expenses,animal_capital_stock,land_capital_stock,fixed_capital_stock,total_capital_stock,total_capital_stock_milk_daily,lactating_cows_hectare_activity,hectares_arrendada,rented_area_percentage
9734,99ef0160-5fd8-44f9-93ed-6b6822f01c47,2024-01-01,33618.399139,14199.0,2.367660,NaN,NaN,NaN,NaN,NaN,...,8.852975,5442.546768,NaN,1507280.0,449613.247636,1.956893e+06,4252.624653,NaN,0.0,0.0
9735,99ef0160-5fd8-44f9-93ed-6b6822f01c47,2024-02-01,33207.804634,13243.0,2.507574,NaN,NaN,NaN,NaN,NaN,...,9.007674,5273.450174,NaN,1507280.0,447787.996567,1.955068e+06,4260.047479,NaN,0.0,0.0
9736,99ef0160-5fd8-44f9-93ed-6b6822f01c47,2024-03-01,31343.192664,12146.0,2.580536,NaN,NaN,NaN,NaN,NaN,...,9.574520,5293.942790,NaN,1507280.0,446429.252920,1.953709e+06,4959.465021,NaN,0.0,0.0
9737,99ef0160-5fd8-44f9-93ed-6b6822f01c47,2024-04-01,35886.098798,12817.0,2.799883,NaN,NaN,NaN,NaN,NaN,...,8.324700,12991.056553,NaN,1507280.0,449649.540608,1.956930e+06,4557.004286,NaN,0.0,0.0
9738,99ef0160-5fd8-44f9-93ed-6b6822f01c47,2024-05-01,35576.363171,12817.0,2.775717,NaN,NaN,NaN,NaN,NaN,...,10.482522,8821.496151,NaN,1507280.0,454564.288209,1.961844e+06,4720.730648,NaN,0.0,0.0
9739,99ef0160-5fd8-44f9-93ed-6b6822f01c47,2024-06-01,45995.880803,15571.0,2.953945,NaN,NaN,NaN,NaN,NaN,...,6.077849,14235.669799,NaN,1507280.0,456973.091745,1.964253e+06,3564.671995,NaN,0.0,0.0
9740,99ef0160-5fd8-44f9-93ed-6b6822f01c47,2024-07-01,49403.377954,16925.0,2.918959,NaN,NaN,NaN,NaN,NaN,...,11.361979,14913.087500,NaN,1507280.0,460817.286313,1.968097e+06,3412.249210,NaN,0.0,0.0
9741,99ef0160-5fd8-44f9-93ed-6b6822f01c47,2024-08-01,52520.332510,16740.0,3.137415,197.0,17.0,3.62,3.39,NaN,...,10.858513,11412.269010,0.0,1507280.0,461388.105631,1.968668e+06,3500.356253,1.474531,0.0,0.0
9742,99ef0160-5fd8-44f9-93ed-6b6822f01c47,2024-09-01,48516.226898,15313.0,3.168303,257.0,40.0,3.65,3.41,NaN,...,11.793105,10277.923298,0.0,1507280.0,466120.121486,1.973400e+06,3759.334750,1.541555,0.0,0.0
9743,99ef0160-5fd8-44f9-93ed-6b6822f01c47,2024-10-01,46907.711247,14789.0,3.171797,189.0,91.0,3.57,3.34,NaN,...,11.690304,8662.847980,0.0,1507280.0,473289.904430,1.980570e+06,3925.933054,1.541555,0.0,0.0


## 3.6 Integração da Dimensão Produtor (SharePoint)


In [40]:
# ==============================================================================
# 3.6 INTEGRAÇÃO DA DIMENSÃO PRODUTOR (SHAREPOINT)
# ==============================================================================
from sqlalchemy import text

# ==============================================================================
# IMPORTAR APENAS AS COLUNAS DIMENSIONAIS NECESSÁRIAS
# ==============================================================================
consulta_dim_property = text(
    """
    SELECT
        id_property,
        property_name,
        labor_rural_code,
        entrepreneur_name,
        agroindustry_name,
        dairy_region,
        property_status
    FROM analytics_mart.vw_dim_property; 
    """
)

df_dim_property = pd.read_sql_query(consulta_dim_property, con=engine)

# Padronizar a chave
df_dim_property["id_property"] = (df_dim_property["id_property"].astype("string").str.strip())

# Remover linhas sem chave
df_dim_property = (df_dim_property.dropna(subset=["id_property"]).reset_index(drop=True) )

# Validar uma linha por propriedade
if df_dim_property.duplicated("id_property").any():
    raise ValueError( "A dimensão possui mais de uma linha por id_property." )

# Torna o merge idempotente: remove versões anteriores das colunas dimensionais.
colunas_dimensionais = [
    coluna
    for coluna in df_dim_property.columns
    if coluna != "id_property"
]

colunas_dimensionais_antigas = [
    nome
    for coluna in colunas_dimensionais
    for nome in (coluna, f"{coluna}_x", f"{coluna}_y")
    if nome in df_integrada.columns
]

df_integrada = df_integrada.drop(
    columns=colunas_dimensionais_antigas,
    errors="ignore",
)

# Merge dimensional
linhas_antes = len(df_integrada)

df_integrada = df_integrada.merge(
    df_dim_property,
    on="id_property",
    how="left"
    # validate="many_to_one",
)

if len(df_integrada) != linhas_antes:
    raise ValueError( "O merge dimensional alterou a quantidade de linhas." )

# Opção A: Remover lançamentos históricos de fazendas inativas (não presentes em vw_dim_property)
inativas_descartadas = df_integrada["property_name"].isna().sum()
if inativas_descartadas > 0:
    print(f"Filtrando apenas fazendas ativas: {inativas_descartadas:,} linhas de fazendas inativas foram removidas.")
    df_integrada = df_integrada.dropna(subset=["property_name"]).reset_index(drop=True)

# Rótulo combinado fazenda - produtor
if {"property_name", "entrepreneur_name"}.issubset(df_integrada.columns):
    df_integrada["property_entrepreneur_label"] = (
        df_integrada["property_name"].fillna("") + " - " + df_integrada["entrepreneur_name"].fillna("")
    )

print("Merge dimensional concluído.")
print(f"Linhas: {df_integrada.shape[0]:,}")
print(f"Colunas: {df_integrada.shape[1]:,}")

df_integrada.head()

# ==============================================================================
# CONSULTOR: IMPORTAR AQUI, MAS NÃO MESCLAR AGORA
# ==============================================================================
# Uma propriedade pode ter mais de um consultor em vw_dim_property_consultant.
# Se mesclássemos isso em df_integrada agora (grão mensal), cada mês de uma
# fazenda com 2+ consultores viraria 2+ linhas ANTES da Feature Engineering e
# da janela anual - o que faria toda soma anual (COE, receita, produção etc.)
# ser contada 2x, 3x etc. Por isso só importamos aqui; o merge (que duplica de
# propósito) acontece só no final, em df_anuais, depois que as somas já estão
# fechadas - ver célula anuais-indicadores.
# A view vw_dim_property_consultant está em analytics_mart.
consulta_dim_consultor = text(
    """
    SELECT
        id_consultant,
        consultant_name,
        id_property
    FROM analytics_mart.vw_dim_property_consultant;
    """
)

df_dim_consultor = pd.read_sql_query(consulta_dim_consultor, con=engine)
df_dim_consultor["id_property"] = (df_dim_consultor["id_property"].astype("string").str.strip())
df_dim_consultor = (df_dim_consultor.dropna(subset=["id_property"]).reset_index(drop=True) )

print(f"Consultores importados: {df_dim_consultor.shape[0]:,} linhas.")
print(f"Propriedades com mais de um consultor: {df_dim_consultor['id_property'].duplicated().sum():,}")

# Agregação de consultores por propriedade para o relatório mensal (evita duplicação de linhas)
if not df_dim_consultor.empty:
    df_consultor_agg = (
        df_dim_consultor.groupby("id_property")["consultant_name"]
        .apply(lambda s: ", ".join(s.dropna().unique()))
        .reset_index()
    )
    df_integrada = df_integrada.drop(columns=["consultant_name"], errors="ignore")
    df_integrada["id_property"] = df_integrada["id_property"].astype("string").str.strip()
    df_consultor_agg["id_property"] = df_consultor_agg["id_property"].astype("string").str.strip()
    df_integrada = df_integrada.merge(df_consultor_agg, on="id_property", how="left")
    df_integrada["consultant_name"] = df_integrada["consultant_name"].fillna("Sem Consultor Vinculado")


Filtrando apenas fazendas ativas: 2,779 linhas de fazendas inativas foram removidas.
Merge dimensional concluído.
Linhas: 13,322
Colunas: 150


Consultores importados: 982 linhas.
Propriedades com mais de um consultor: 148


## 3.7 Regras Mensais de Consistência e Qualidade dos Dados


In [41]:
# ==============================================================================
# 3.7 REGRAS MENSAIS DE CONSISTÊNCIA E QUALIDADE DOS DADOS
# ==============================================================================
df_consistencia = df_integrada.copy(deep=True)

# ==============================================================================
# 1. DEFINIR AS COLUNAS NECESSÁRIAS
# ==============================================================================

# Lista das colunas que precisam existir antes de calcular os indicadores e as regras de consistência.
COLUNAS_NECESSARIAS_CONSISTENCIA = [
    "ccs",
    "cpp",
    "fat",
    "protein",
    "lactating_cows",
    "lactating_cows_total_cows",
    "lactating_cows_total_cattle",
    "milk_daily",
    "milk_total_labor_day",
    "lactating_cows_total_labor",
    "feeding_cost_milk_price",
    "voluminous_cost_liter",
    "concentrate_mineral_cost_liter",
    "hired_labor_cost_liter",
    # "total_capital_stock_milk_daily"
]


# Verifica quais colunas da lista acima não existem na df_consistencia.
colunas_ausentes = [
    coluna
    for coluna in COLUNAS_NECESSARIAS_CONSISTENCIA
    if coluna not in df_consistencia.columns
]

# Interrompe a execução caso alguma coluna obrigatória esteja ausente.
if colunas_ausentes:
    raise KeyError( "As seguintes colunas necessárias para a consistência " f"não foram encontradas: {colunas_ausentes}" )


# ==============================================================================
# 2. GARANTIR QUE AS COLUNAS SEJAM NUMÉRICAS
# ==============================================================================

# Percorre cada coluna usada nas regras de consistência.
for coluna in COLUNAS_NECESSARIAS_CONSISTENCIA:
    
    # Converte a coluna para número.
    # Valores que não puderem ser convertidos serão transformados em NaN.
    df_consistencia[coluna] = pd.to_numeric( df_consistencia[coluna], errors="coerce", )


# Substitui valores infinitos positivos e negativos por NaN.
# Isso impede que divisões inválidas sejam classificadas como consistentes.
df_consistencia[COLUNAS_NECESSARIAS_CONSISTENCIA] = (
    df_consistencia[COLUNAS_NECESSARIAS_CONSISTENCIA]
    .replace([np.inf, -np.inf], np.nan)
)

# ==============================================================================
# 4. FUNÇÃO PARA PADRONIZAR AS REGRAS
# ==============================================================================

def preparar_regra_consistencia(condicao: pd.Series) -> pd.Series:
    """
    Converte o resultado de uma regra em verdadeiro ou falso.

    Valores ausentes são classificados como False, ou seja,
    o critério não é considerado atendido quando não há informação.
    """

    # Substitui resultados ausentes por False.
    condicao = condicao.fillna(False)

    # Garante que o resultado final tenha tipo booleano.
    return condicao.astype(bool)


# ==============================================================================
# 5. QUALIDADE DO LEITE
# ==============================================================================

# CCS é considerada consistente quando for maior que 50.
df_consistencia["cons_ccs"] = preparar_regra_consistencia( df_consistencia["ccs"].gt(50) )

# CPP é considerada consistente quando for maior que 1.
df_consistencia["cons_cpp"] = preparar_regra_consistencia( df_consistencia["cpp"].gt(1) )

# Gordura é consistente quando estiver acima de 2,5 e abaixo de 5,5.
df_consistencia["cons_fat"] = preparar_regra_consistencia( df_consistencia["fat"].gt(2.5) & df_consistencia["fat"].lt(5.5) )

# Proteína é consistente quando estiver acima de 2,4 e abaixo de 4,5.
df_consistencia["cons_protein"] = preparar_regra_consistencia( df_consistencia["protein"].gt(2.4) & df_consistencia["protein"].lt(4.5) )


# ==============================================================================
# 6. ESTRUTURA DO REBANHO
# ==============================================================================

# Na base nova, lactating_cows_total_cows está em percentual. 
# O limite antigo de 0,20 a 0,99 corresponde agora a 20% a 99%.
df_consistencia["cons_lactating_cows_total_cows"] = (
    preparar_regra_consistencia(
        df_consistencia["lactating_cows_total_cows"].gt(20)
        & df_consistencia["lactating_cows_total_cows"].lt(99)
    )
)

# Na base nova, lactating_cows_total_cattle também está em percentual. 
# O limite antigo de 0,15 a 0,99 corresponde agora a 15% a 99%.
df_consistencia["cons_lactating_cows_total_cattle"] = (
    preparar_regra_consistencia(
        df_consistencia["lactating_cows_total_cattle"].gt(15)
        & df_consistencia["lactating_cows_total_cattle"].lt(99)
    )
)


# ==============================================================================
# 7. PRODUTIVIDADE E MÃO DE OBRA
# ==============================================================================

# A produção diária por vaca em lactação deve ser maior que 3 e menor que 45 litros por vaca por dia.
df_consistencia["cons_milk_lactating_cow_day"] = (
    preparar_regra_consistencia(
        df_consistencia["milk_lactating_cow_day"].gt(3)
        & df_consistencia["milk_lactating_cow_day"].lt(45)
    )
)


# A produção diária por unidade de mão de obra deve ser maior que 20 e menor que 1.500 litros por trabalhador por dia.
df_consistencia["cons_milk_total_labor_day"] = (
    preparar_regra_consistencia(
        df_consistencia["milk_total_labor_day"].gt(20)
        & df_consistencia["milk_total_labor_day"].lt(1500)
    )
)


# O número de vacas em lactação por unidade de mão de obra deve ser positivo e menor que 70.
# O limite inferior positivo evita que propriedades com zero vacas em lactação sejam classificadas como consistentes.
df_consistencia["cons_lactating_cows_total_labor"] = (
    preparar_regra_consistencia(
        df_consistencia["lactating_cows_total_labor"].gt(0)
        & df_consistencia["lactating_cows_total_labor"].lt(70)
    )
)


# ==============================================================================
# 8. ALIMENTAÇÃO E CUSTOS
# ==============================================================================

# feeding_cost_milk_price está em percentual na base nova.
# O limite antigo de 0,15 a 1,50 corresponde a 15% a 150%.
df_consistencia["cons_feeding_cost_milk_price"] = (
    preparar_regra_consistencia(
        df_consistencia["feeding_cost_milk_price"].gt(15)
        & df_consistencia["feeding_cost_milk_price"].lt(150)
    )
)

# O custo do volumoso deve ser menor que R$ 3,00 por litro em todos os sistemas.
# - Para sistemas à pasto ("PASTURE") ou semi-confinado ("SEMI_CONFINED"): pode ser >= 0.
# - Para os demais sistemas (Compost Barn, Free Stall, Confinado, etc.): deve ser > 0,10.

is_pasto_semi = df_consistencia["production_system"].astype(str).str.upper().isin(["PASTURE", "SEMI_CONFINED"])

# Condição 1: Pasto ou Semi-confinado (0 <= custo < 3.00)
cond_pasto_semi = is_pasto_semi & df_consistencia["voluminous_cost_liter"].ge(0) & df_consistencia["voluminous_cost_liter"].lt(3.0)

# Condição 2: Demais sistemas (0.10 < custo < 3.00)
cond_demais_sistemas = (~is_pasto_semi) & df_consistencia["voluminous_cost_liter"].gt(0.10) & df_consistencia["voluminous_cost_liter"].lt(3.0)

# Aplicação da regra combinada
df_consistencia["cons_voluminous_cost_liter"] = preparar_regra_consistencia(
    cond_pasto_semi | cond_demais_sistemas
)

# O custo de concentrado e minerais deve ser maior que R$ 0,30 e menor que R$ 3,50 por litro.
df_consistencia["cons_concentrate_mineral_cost_liter"] = (
    preparar_regra_consistencia(
        df_consistencia["concentrate_mineral_cost_liter"].gt(0.3)
        & df_consistencia["concentrate_mineral_cost_liter"].lt(3.5)
    )
)

# O custo da mão de obra contratada deve ser menor que R$ 1 por litro.
df_consistencia["cons_hired_labor_cost_liter"] = (
    preparar_regra_consistencia(
        df_consistencia["hired_labor_cost_liter"].lt(1)
    )
)

# ==============================================================================
# 9. ESTOQUE DE CAPITAL
# ==============================================================================

# Verifica se o estoque total de capital por produção diária
# está abaixo do limite de consistência.
# df_consistencia["cons_total_capital_stock"] = (
#     preparar_regra_consistencia(
#         df_consistencia["total_capital_stock_milk_daily"].lt(30000)
#     )
# )

# ==============================================================================
# 10. MAPA DOS CRITÉRIOS
# ==============================================================================

# Relaciona cada coluna booleana ao nome que será exibido
# no relatório de critérios violados.
MAPA_CRITERIOS_CONSISTENCIA = {
    "cons_ccs": "CCS",
    "cons_cpp": "CPP",
    "cons_fat": "Gordura",
    "cons_protein": "Proteína",
    "cons_lactating_cows_total_cows": "VL/Total de vacas",
    "cons_lactating_cows_total_cattle": "VL/Rebanho total",
    "cons_milk_lactating_cow_day": "Produção/VL",
    "cons_milk_total_labor_day": "Produção/MDO",
    "cons_lactating_cows_total_labor": "VL/MDO",
    "cons_feeding_cost_milk_price": "Alimentação/Preço do leite",
    "cons_voluminous_cost_liter": "Custo de volumoso",
    "cons_concentrate_mineral_cost_liter": "Custo de concentrado",
    "cons_hired_labor_cost_liter": "Custo da MDO contratada",
    # "cons_total_capital_stock": "Estoque de capital fixo",
}


# Transforma as chaves do dicionário em uma lista.
# Essa lista contém todas as colunas de consistência.
colunas_criterios = list(
    MAPA_CRITERIOS_CONSISTENCIA.keys()
)


# ==============================================================================
# 11. CONTAR CRITÉRIOS ATENDIDOS E VIOLADOS
# ==============================================================================

# Soma os valores True de cada linha.
# No pandas, True equivale a 1 e False equivale a 0.
df_consistencia["total_consistency_criteria_ok"] = (
    df_consistencia[colunas_criterios]
    .sum(axis=1)
    .astype("int8")
)


# Salva a quantidade total de critérios avaliados.
df_consistencia["total_consistency_criteria"] = len(
    colunas_criterios
)


# Calcula quantos critérios não foram atendidos.
df_consistencia["total_consistency_criteria_violated"] = (
    df_consistencia["total_consistency_criteria"]
    - df_consistencia["total_consistency_criteria_ok"]
)


# ==============================================================================
# 12. CLASSIFICAÇÃO GERAL
# ==============================================================================

# Classifica como Consistente quando nenhum critério foi violado.
# Caso exista pelo menos uma violação, classifica como Inconsistente.
df_consistencia["consistency_status"] = np.where(
    df_consistencia["total_consistency_criteria_violated"].eq(0),
    "Consistente",
    "Inconsistente",
)


# Cria uma classificação numérica.
# Zero representa consistente e um representa inconsistente.
df_consistencia["consistency_id"] = np.where(
    df_consistencia["total_consistency_criteria_violated"].eq(0),
    0,
    1,
).astype("int8")


# ==============================================================================
# 13. LISTAR OS CRITÉRIOS VIOLADOS E DETALHAMENTO DOS MOTIVOS/VALORES
# ==============================================================================

# Cria inicialmente uma coluna vazia.
df_consistencia["violated_consistency_criteria"] = ""


# Percorre cada critério e seu respectivo nome de exibição.
for coluna_criterio, nome_criterio in MAPA_CRITERIOS_CONSISTENCIA.items():

    # Identifica as linhas em que o critério não foi atendido.
    mascara_violacao = ~df_consistencia[coluna_criterio]

    # Adiciona o nome do critério à lista de violações da linha.
    df_consistencia.loc[
        mascara_violacao,
        "violated_consistency_criteria",
    ] += nome_criterio + "; "


# Remove o último ponto e vírgula e os espaços excedentes.
df_consistencia["violated_consistency_criteria"] = (
    df_consistencia["violated_consistency_criteria"]
    .str.rstrip("; ")
    .replace("", "Nenhum")
)

# ==============================================================================
# 13.1 COLUNA DETALHADA COM VALORES E MOTIVOS EXATOS DA VIOLAÇÃO
# ==============================================================================
REGRAS_DETALHADAS = [
    ("cons_ccs", "ccs", 50, None, "CCS", "", ".0f"),
    ("cons_cpp", "cpp", 1, None, "CPP", "", ".0f"),
    ("cons_fat", "fat", 2.5, 5.5, "Gordura", "%", ".2f"),
    ("cons_protein", "protein", 2.4, 4.5, "Proteína", "%", ".2f"),
    ("cons_lactating_cows_total_cows", "lactating_cows_total_cows", 20, 99, "VL/Total de vacas", "%", ".1f"),
    ("cons_lactating_cows_total_cattle", "lactating_cows_total_cattle", 15, 99, "VL/Rebanho total", "%", ".1f"),
    ("cons_milk_lactating_cow_day", "milk_lactating_cow_day", 3, 45, "Produção/VL", " L/vaca/dia", ".1f"),
    ("cons_milk_total_labor_day", "milk_total_labor_day", 20, 1500, "Produção/MDO", " L/trabalhador/dia", ".1f"),
    ("cons_lactating_cows_total_labor", "lactating_cows_total_labor", 0, 70, "VL/MDO", " vacas/trabalhador", ".1f"),
    ("cons_feeding_cost_milk_price", "feeding_cost_milk_price", 15, 150, "Alimentação/Preço do leite", "%", ".1f"),
    ("cons_voluminous_cost_liter", "voluminous_cost_liter", None, 3.0, "Custo de volumoso", " R$/L", ".2f"),
    ("cons_concentrate_mineral_cost_liter", "concentrate_mineral_cost_liter", 0.30, 3.50, "Custo de concentrado", " R$/L", ".2f"),
    ("cons_hired_labor_cost_liter", "hired_labor_cost_liter", None, 1.0, "Custo da MDO contratada", " R$/L", ".2f"),
]

detalhes_lista = []
for _, row in df_consistencia.iterrows():
    motivos = []
    for col_bool, col_val, min_v, max_v, nome, unit, fmt in REGRAS_DETALHADAS:
        if not row.get(col_bool, True):
            val = row.get(col_val, np.nan)
            if pd.isna(val):
                motivos.append(f"{nome} (Ausente/NaN)")
            elif min_v is not None and val <= min_v:
                motivos.append(f"{nome} (Abaixo do mín <{min_v}: {val:{fmt}}{unit})")
            elif max_v is not None and val >= max_v:
                motivos.append(f"{nome} (Acima do máx >{max_v}: {val:{fmt}}{unit})")
            else:
                motivos.append(f"{nome} (Inconsistente: {val:{fmt}}{unit})")
    
    detalhes_lista.append("; ".join(motivos) if motivos else "Nenhum")

df_consistencia["violated_consistency_details"] = detalhes_lista

# Atualiza df_integrada para que a exportação mensal contenha todas as colunas de consistência
df_integrada = df_consistencia.copy()

# ==============================================================================
# 14. CONFERÊNCIA DO RESULTADO
# ==============================================================================

# Cria uma tabela resumida com a quantidade de registros
# consistentes e inconsistentes.
df_resumo_consistencia = (
    df_consistencia["consistency_status"]
    .value_counts(dropna=False)
    .rename_axis("consistency_status")
    .reset_index(name="records")
)


# Calcula o percentual de cada classificação.
df_resumo_consistencia["percentage"] = (
    df_resumo_consistencia["records"]
    .div(len(df_consistencia))
    .mul(100)
    .round(2)
)


# Exibe o resumo da classificação.
display(df_resumo_consistencia)


# Exibe uma amostra das principais colunas geradas.
display(
    df_consistencia[
        [
            "id_property",
            "reference_month",
            "consistency_status",
            "consistency_id",
            "total_consistency_criteria_ok",
            "total_consistency_criteria_violated",
            "violated_consistency_criteria",
            "violated_consistency_details",
        ]
    ]
    .head(20)
)


,consistency_status,records,percentage
0,Consistente,9382,70.42
1,Inconsistente,3940,29.58


,id_property,reference_month,consistency_status,consistency_id,total_consistency_criteria_ok,total_consistency_criteria_violated,violated_consistency_criteria,violated_consistency_details
0,00220277-a58e-4b9e-9b89-e6acf8a1a400,2024-05-01,Inconsistente,1,0,13,CCS; CPP; Gordura; Proteína; VL/Total de vacas...,CCS (Ausente/NaN); CPP (Ausente/NaN); Gordura ...
1,00220277-a58e-4b9e-9b89-e6acf8a1a400,2024-08-01,Inconsistente,1,12,1,Custo de volumoso,Custo de volumoso (Inconsistente: 0.05 R$/L)
2,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-04-01,Consistente,0,13,0,Nenhum,Nenhum
3,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-05-01,Consistente,0,13,0,Nenhum,Nenhum
4,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-06-01,Consistente,0,13,0,Nenhum,Nenhum
5,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-07-01,Consistente,0,13,0,Nenhum,Nenhum
6,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-08-01,Consistente,0,13,0,Nenhum,Nenhum
7,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-09-01,Consistente,0,13,0,Nenhum,Nenhum
8,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-10-01,Consistente,0,13,0,Nenhum,Nenhum
9,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-11-01,Consistente,0,13,0,Nenhum,Nenhum


In [42]:
df_consistencia

,id_property,reference_month,milk_sold_revenue,milk_volume_sold,milk_unit_price,ccs,cpp,fat,protein,unit_price_derivative,...,cons_voluminous_cost_liter,cons_concentrate_mineral_cost_liter,cons_hired_labor_cost_liter,total_consistency_criteria_ok,total_consistency_criteria,total_consistency_criteria_violated,consistency_status,consistency_id,violated_consistency_criteria,violated_consistency_details
0,00220277-a58e-4b9e-9b89-e6acf8a1a400,2024-05-01,0.000000e+00,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,0,13,13,Inconsistente,1,CCS; CPP; Gordura; Proteína; VL/Total de vacas...,CCS (Ausente/NaN); CPP (Ausente/NaN); Gordura ...
1,00220277-a58e-4b9e-9b89-e6acf8a1a400,2024-08-01,5.301091e+05,167274.0,3.169106,159.0,33.0,3.85,3.35,NaN,...,False,True,True,12,13,1,Inconsistente,1,Custo de volumoso,Custo de volumoso (Inconsistente: 0.05 R$/L)
2,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-04-01,1.120198e+05,35677.0,3.139831,421.0,77.0,3.48,3.40,NaN,...,True,True,True,13,13,0,Consistente,0,Nenhum,Nenhum
3,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-05-01,1.627337e+05,54884.0,2.965048,338.0,54.0,3.55,3.29,NaN,...,True,True,True,13,13,0,Consistente,0,Nenhum,Nenhum
4,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-06-01,1.613585e+05,56311.0,2.865487,192.0,102.0,3.46,3.32,NaN,...,True,True,True,13,13,0,Consistente,0,Nenhum,Nenhum
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13317,fff52e9f-83b6-44db-b7a4-01da7e77aeaa,2026-03-01,2.161324e+06,765080.0,2.824964,158.0,3.0,3.82,3.30,NaN,...,True,True,True,13,13,0,Consistente,0,Nenhum,Nenhum
13318,fff52e9f-83b6-44db-b7a4-01da7e77aeaa,2026-04-01,2.293030e+06,733654.0,3.125492,167.0,9.0,4.04,3.33,NaN,...,True,True,True,13,13,0,Consistente,0,Nenhum,Nenhum
13319,fff52e9f-83b6-44db-b7a4-01da7e77aeaa,2026-05-01,2.315243e+06,751991.0,3.078818,167.0,9.0,4.04,3.33,NaN,...,True,True,True,13,13,0,Consistente,0,Nenhum,Nenhum
13320,fff52e9f-83b6-44db-b7a4-01da7e77aeaa,2026-06-01,2.333599e+06,751991.0,3.103227,160.0,9.0,3.90,3.33,NaN,...,True,True,True,13,13,0,Consistente,0,Nenhum,Nenhum


## 3.8 Verificação de Variação >±50% entre Períodos Consecutivos (Mensal)

Diagnóstico **relativo** e **estritamente aditivo**, no mesmo espírito da análise de
outliers da 4.4: marca todo indicador que variou mais de +50% ou menos de −50% em relação
ao mês calendário imediatamente anterior (M−1) da própria fazenda.

Cobre a lista de indicadores **estruturais e de custo agregado** aprovada na re-medição do
ticket 02 sobre a base de 18/08 (`.scratch/variacao-50-periodo/`): Total de Vacas, Preço
Unitário do Leite, Área Total da Propriedade, Produção Total de Leite, Estoque de Capital
Total, Custo Total Mão de Obra e Custo Total com Alimentação — todos abaixo do corte de 10%
das linhas marcadas. As rubricas de custo detalhadas ficaram de fora (acima do corte) e
seguem cobertas só pela 4.4 (contra o grupo).

Nada de consistência muda aqui (`consistency_id` e `consistency_status` são apenas lidos);
`df_consistencia` não é alterado — a função lê uma cópia e devolve três DataFrames novos.
As colunas novas não sobem para o Supabase. A coloração na aba **Indicadores Mensais**
(colunas auxiliares `_var_`) chega no ticket 04; a aba **Variações por Consultor**, no
ticket 05 — ambos de `.scratch/variacao-50-periodo/`. O ticket 09 (mesma pasta) estende a
coloração ao bloco de identificação (IdFazenda .. Mês de Referência) via a coluna auxiliar
`_var_qualquer`, quando pelo menos um indicador da linha foi marcado.

In [43]:
# ==============================================================================
# 3.8 VERIFICAÇÃO DE VARIAÇÃO >±50% ENTRE PERÍODOS CONSECUTIVOS (MENSAL)
# ==============================================================================
# Diagnóstico RELATIVO e ADITIVO, mesmo espírito da 4.4: marca todo indicador que
# variou mais de +50% ou menos de -50% em relação ao mês calendário imediatamente
# anterior (M-1) da própria fazenda.
#
# Nada de consistência muda aqui: consistency_id e consistency_status são apenas
# LIDOS. A célula lê df_consistencia (cópia defensiva) e devolve três DataFrames
# novos - NÃO altera df_consistencia, que segue intocado para a 3.9. As colunas
# "_var_" que entram na aba "Indicadores Mensais" chegam no ticket 04
# (.scratch/variacao-50-periodo/); a aba "Variações por Consultor", no ticket 05.
# ==============================================================================
LIMITE_VARIACAO_PERCENTUAL = 50

# Lista final aprovada na re-medição sobre a base de 18/08 (ticket 02 de
# .scratch/variacao-50-periodo/, tabela documentada na spec): estruturais + custos
# agregados, todos abaixo do corte de 10% de linhas marcadas. Mesmo shape de
# INDICADORES_OUTLIER (4.4): coluna técnica -> (rótulo em português, unidade, formato).
INDICADORES_VARIACAO_MENSAL = {
    "total_cows":           ("Total de Vacas", "cabeças", ".0f"),
    "milk_unit_price":      ("Preço Unitário do Leite", "R$/litro", ".2f"),
    "hectares_total":       ("Área Total da Propriedade", "ha", ".1f"),
    "milk_produced":        ("Produção Total de Leite", "litros", ".0f"),
    "total_capital_stock":  ("Estoque de Capital Total", "R$", ".2f"),
    "total_labor_expenses": ("Custo Total Mão de Obra", "R$", ".2f"),
    "feeding_cost":         ("Custo Total com Alimentação", "R$", ".2f"),
}

# Helpers no mesmo padrão da 4.4 (formatação numérica brasileira, renomeação do
# consolidado, aviso do resumo por consultor). A 3.8 roda ANTES da 4.4 na ordem
# do notebook, então não há como importar os objetos de lá - ficam redefinidos
# aqui com o MESMO conteúdo. Se um dia isso for extraído para um lugar comum
# (fora de functions/, que fica fora de escopo desta spec), os dois pontos de
# definição colapsam em um só.
def _formatar_br(valor, formato):
    """Formata um número no padrão brasileiro (vírgula decimal, ponto de milhar)."""
    if valor is None or pd.isna(valor):
        return "-"
    texto = f"{valor:{formato}}"
    return texto.replace(",", "\x00").replace(".", ",").replace("\x00", ".")


RENOMEAR_CONSOLIDADO = {
    "id_property": "IDFazenda",
    "property_entrepreneur_label": "Fazenda - Produtor",
    "property_name": "Fazenda",
    "labor_rural_code": "Código LR",
    "consultant_name": "Consultor",
}

AVISO_SOMA_CONSULTOR = (
    "ATENÇÃO: cada vínculo conta - fazenda com mais de um consultor aparece "
    "para todos eles, logo a soma desta coluna EXCEDE o total de fazendas distintas"
)


def calcular_variacao_periodo(
    df,
    indicadores=None,
    coluna_chave="id_property",
    coluna_ordem="reference_month",
    limite_percentual=LIMITE_VARIACAO_PERCENTUAL,
    prefixo="variacao",
    colunas_informativas=None,
):
    """
    Marca variação >+-limite_percentual% entre meses calendário CONSECUTIVOS da
    mesma fazenda (comparação contra M-1, nunca atravessando uma lacuna).

    Função pura: faz cópia defensiva do DataFrame de entrada, não faz I/O, não
    acessa banco/SharePoint e não usa estado global além da configuração recebida
    por parâmetro. Nasce parametrizada (coluna-chave, coluna de ordem, limite,
    prefixo) para poder ser reaproveitada por uma extensão anual futura sem
    reescrita, embora hoje só o mensal seja chamado.

    No grão mensal, `consultant_name` é uma string com os nomes separados por
    ", " quando a fazenda tem mais de um consultor vinculado (agregação feita na
    3.6, para não duplicar linha por consultor no mensal - só o anual duplica de
    propósito). O resumo por consultor abaixo separa (`str.split`) essa string
    para que cada consultor veja a fazenda na própria carteira, sem duplicar a
    linha correspondente na aba "Variações".

    Parâmetros
    ----------
    df : DataFrame com uma linha por fazenda-mês. Precisa de `coluna_chave`,
        `coluna_ordem` (data/timestamp) e as colunas de indicador.
    indicadores : dict {coluna: (rótulo, unidade, formato)}. Default
        INDICADORES_VARIACAO_MENSAL.
    coluna_chave : coluna que identifica a fazenda.
    coluna_ordem : coluna de data usada para ordenar e checar consecutividade
        (mês calendário com passo de exatamente 1).
    limite_percentual : limiar de variação (50 -> razão > 1,5 ou < 0,5 marca).
    prefixo : prefixo das colunas de categoria anexadas ao DataFrame anotado.
    colunas_informativas : colunas de identificação copiadas para a aba
        "Variações". Default: id_property, property_entrepreneur_label,
        property_name, labor_rural_code, consultant_name (as que existirem).

    Retorna
    -------
    (df_anotado, df_consolidado, df_por_consultor)
    """
    indicadores = INDICADORES_VARIACAO_MENSAL if indicadores is None else indicadores
    if colunas_informativas is None:
        colunas_informativas = [
            "id_property", "property_entrepreneur_label", "property_name",
            "labor_rural_code", "consultant_name",
        ]

    df_anotado = df.copy(deep=True)

    colunas_indicador = [c for c in indicadores if c in df_anotado.columns]
    if not colunas_indicador:
        raise KeyError("Nenhuma coluna de indicador de variação encontrada no DataFrame.")

    limite_superior = 1 + (limite_percentual / 100.0)   # 1,5 para 50%
    limite_inferior = 1 - (limite_percentual / 100.0)   # 0,5 para 50%

    # --------------------------------------------------------------------------
    # 1. CONTINUIDADE: mês calendário consecutivo (passo de exatamente 1) dentro
    #    da mesma fazenda. Lacuna de mês -> a comparação simplesmente não roda.
    # --------------------------------------------------------------------------
    df_anotado = df_anotado.sort_values([coluna_chave, coluna_ordem]).copy()

    periodo_dt = pd.to_datetime(df_anotado[coluna_ordem], errors="coerce")
    ordem_num = periodo_dt.dt.year * 12 + periodo_dt.dt.month

    mesma_fazenda = df_anotado[coluna_chave].eq(df_anotado[coluna_chave].shift(1))
    passo = ordem_num - ordem_num.shift(1)
    consecutivo = (mesma_fazenda & passo.eq(1) & ordem_num.notna()).to_numpy()

    # --------------------------------------------------------------------------
    # 2. CLASSIFICAÇÃO POR INDICADOR (todas as linhas)
    # --------------------------------------------------------------------------
    marcado_por_coluna = {}
    for coluna in colunas_indicador:
        valores = pd.to_numeric(df_anotado[coluna], errors="coerce").to_numpy(dtype=float)
        valor_anterior = np.roll(valores, 1)
        valor_anterior[0] = np.nan
        valor_anterior = np.where(consecutivo, valor_anterior, np.nan)
        valor_atual = np.where(consecutivo, valores, np.nan)

        ambos_validos = np.isfinite(valor_anterior) & np.isfinite(valor_atual)
        ambos_zero = ambos_validos & (valor_anterior == 0) & (valor_atual == 0)
        entrada = ambos_validos & (valor_anterior == 0) & (valor_atual > 0)
        saida = ambos_validos & (valor_anterior > 0) & (valor_atual == 0)

        with np.errstate(divide="ignore", invalid="ignore"):
            razao = valor_atual / valor_anterior
        avaliavel = ambos_validos & ~ambos_zero & ~entrada & ~saida & np.isfinite(razao)
        aumento = avaliavel & (razao > limite_superior)
        reducao = avaliavel & (razao < limite_inferior)

        # Exatamente no limite (1,5 ou 0,5) NÃO marca; imediatamente fora dele marca.
        categoria = np.full(len(df_anotado), "Não avaliado", dtype=object)
        categoria[avaliavel] = "Normal"
        categoria[entrada] = "Entrada"
        categoria[saida] = "Saída"
        categoria[aumento] = "Aumento"
        categoria[reducao] = "Redução"

        df_anotado[f"{prefixo}_{coluna}"] = categoria

        marcado_por_coluna[coluna] = {
            "marcado": aumento | reducao,
            "lado": categoria,
            "valor_anterior": valor_anterior,
            "valor_atual": valor_atual,
            "variacao_pct": (razao - 1) * 100,
        }

    df_anotado = df_anotado.copy()  # desfragmenta após as inserções (mesmo cuidado da 4.2/4.4)

    # --------------------------------------------------------------------------
    # 3. ABA "VARIAÇÕES": série inteira, uma linha por fazenda x indicador x mês
    #    marcado (só Aumento/Redução), do mês mais recente para o mais antigo.
    # --------------------------------------------------------------------------
    colunas_id_presentes = [c for c in colunas_informativas if c in df_anotado.columns]
    linhas_consolidado = []
    for coluna in colunas_indicador:
        rotulo, _unidade, _formato = indicadores[coluna]
        dados = marcado_por_coluna[coluna]
        indices_marcados = np.flatnonzero(dados["marcado"])
        if indices_marcados.size == 0:
            continue
        recorte = df_anotado.iloc[indices_marcados]
        registro = pd.DataFrame(index=recorte.index)
        for c in colunas_id_presentes:
            registro[c] = recorte[c].values
        registro["_periodo"] = recorte[coluna_ordem].values
        registro["Indicador"] = rotulo
        registro["Valor Anterior"] = dados["valor_anterior"][indices_marcados]
        registro["Valor Atual"] = dados["valor_atual"][indices_marcados]
        registro["Variação (%)"] = dados["variacao_pct"][indices_marcados]
        registro["Lado"] = dados["lado"][indices_marcados]
        linhas_consolidado.append(registro)

    colunas_saida_consolidado = (
        [RENOMEAR_CONSOLIDADO.get(c, c) for c in colunas_id_presentes]
        + ["Indicador", "Período", "Valor Anterior", "Valor Atual", "Variação (%)", "Lado"]
    )

    if linhas_consolidado:
        df_consolidado = pd.concat(linhas_consolidado, ignore_index=True)
        df_consolidado = df_consolidado.rename(columns=RENOMEAR_CONSOLIDADO)
        df_consolidado = df_consolidado.rename(columns={"_periodo": "Período"})
        df_consolidado = df_consolidado.sort_values(
            ["Período", "Indicador"], ascending=[False, True]
        ).reset_index(drop=True)
        df_consolidado = df_consolidado[
            [c for c in colunas_saida_consolidado if c in df_consolidado.columns]
        ]
    else:
        df_consolidado = pd.DataFrame(columns=colunas_saida_consolidado)

    # --------------------------------------------------------------------------
    # 4. RESUMO POR CONSULTOR (ocorrência mais recente de cada fazenda). Cada
    #    vínculo conta: consultant_name é dividida em nomes individuais, e cada
    #    um vê a fazenda na própria carteira - sem duplicar linha na 3.
    # --------------------------------------------------------------------------
    if coluna_chave in df_anotado.columns and coluna_ordem in df_anotado.columns:
        ultima_ocorrencia = df_anotado.groupby(coluna_chave)[coluna_ordem].transform("max")
        df_ultima = df_anotado.loc[df_anotado[coluna_ordem].eq(ultima_ocorrencia)].copy()
    else:
        df_ultima = df_anotado.iloc[0:0].copy()

    if "consultant_name" in df_ultima.columns:
        df_ultima["consultant_name"] = df_ultima["consultant_name"].fillna("(Sem consultor)")
    else:
        df_ultima["consultant_name"] = "(Sem consultor)"

    df_ultima["_consultor_individual"] = (
        df_ultima["consultant_name"].astype(str).str.split(",")
    )
    df_ultima = df_ultima.explode("_consultor_individual")
    df_ultima["_consultor_individual"] = df_ultima["_consultor_individual"].str.strip()

    linhas_consultor = []
    for consultor, grupo in df_ultima.groupby("_consultor_individual", sort=True):
        atendidas = grupo[coluna_chave].nunique()
        marcado_geral = pd.Series(False, index=grupo.index)
        registro = {
            "Consultor": consultor,
            f"Fazendas atendidas [{AVISO_SOMA_CONSULTOR}]": atendidas,
        }
        for coluna in colunas_indicador:
            rotulo, _unidade, _formato = indicadores[coluna]
            marcado_coluna = grupo[f"{prefixo}_{coluna}"].isin(["Aumento", "Redução"])
            marcado_geral = marcado_geral | marcado_coluna
            registro[rotulo] = grupo.loc[marcado_coluna, coluna_chave].nunique()
        registro["Fazendas marcadas"] = grupo.loc[marcado_geral, coluna_chave].nunique()
        registro["% de fazendas marcadas"] = (
            100.0 * registro["Fazendas marcadas"] / atendidas if atendidas else np.nan
        )
        linhas_consultor.append(registro)

    df_por_consultor = pd.DataFrame(linhas_consultor)
    if not df_por_consultor.empty:
        colunas_ordem_consultor = (
            ["Consultor", f"Fazendas atendidas [{AVISO_SOMA_CONSULTOR}]", "Fazendas marcadas",
             "% de fazendas marcadas"]
            + [rotulo for _coluna, (rotulo, _u, _f) in indicadores.items()]
        )
        df_por_consultor = df_por_consultor[
            [c for c in colunas_ordem_consultor if c in df_por_consultor.columns]
        ].sort_values("Fazendas marcadas", ascending=False).reset_index(drop=True)

    return df_anotado, df_consolidado, df_por_consultor


# ==============================================================================
# EXECUÇÃO SOBRE OS INDICADORES MENSAIS JÁ CONSOLIDADOS
# ==============================================================================
# Só LÊ df_consistencia (cópia defensiva dentro da função) - a 3.9 continua
# recebendo df_consistencia exatamente como saiu da 3.7.
(
    df_variacao_anotado_mensal,
    df_variacoes,
    df_variacoes_por_consultor,
) = calcular_variacao_periodo(df_consistencia, INDICADORES_VARIACAO_MENSAL)

_total_linhas_mensal = len(df_consistencia)

print("=" * 78)
print(f"VERIFICAÇÃO DE VARIAÇÃO >±{LIMITE_VARIACAO_PERCENTUAL}% ENTRE MESES CONSECUTIVOS (MENSAL)")
print("=" * 78)
print(f"Base: {_total_linhas_mensal:,} linhas | "
      f"{df_consistencia['id_property'].nunique():,} fazendas distintas")
print()
print("Linhas marcadas (Aumento/Redução) por indicador:")
for _coluna, (_rotulo, _unidade, _formato) in INDICADORES_VARIACAO_MENSAL.items():
    _col_categoria = f"variacao_{_coluna}"
    if _col_categoria not in df_variacao_anotado_mensal.columns:
        continue
    _marcadas = int(df_variacao_anotado_mensal[_col_categoria].isin(["Aumento", "Redução"]).sum())
    _taxa = 100.0 * _marcadas / _total_linhas_mensal if _total_linhas_mensal else 0.0
    print(f"  {_rotulo:<32} {_marcadas:>5,}  ({_formatar_br(_taxa, '.2f')}%)")
print()
print(f"Aba 'Variações': {len(df_variacoes):,} linhas (série inteira, mês mais recente primeiro)")
print(f"Aba 'Variações por Consultor': {len(df_variacoes_por_consultor):,} consultores")

display(df_variacoes.head())
display(df_variacoes_por_consultor.head())

VERIFICAÇÃO DE VARIAÇÃO >±50% ENTRE MESES CONSECUTIVOS (MENSAL)
Base: 13,322 linhas | 812 fazendas distintas

Linhas marcadas (Aumento/Redução) por indicador:
  Total de Vacas                      25  (0,19%)
  Preço Unitário do Leite             32  (0,24%)
  Área Total da Propriedade           83  (0,62%)
  Produção Total de Leite            188  (1,41%)
  Estoque de Capital Total           203  (1,52%)
  Custo Total Mão de Obra            515  (3,87%)
  Custo Total com Alimentação      1,125  (8,44%)

Aba 'Variações': 2,171 linhas (série inteira, mês mais recente primeiro)
Aba 'Variações por Consultor': 66 consultores


,IDFazenda,Fazenda - Produtor,Fazenda,Código LR,Consultor,Indicador,Período,Valor Anterior,Valor Atual,Variação (%),Lado
0,aea396c2-be47-42ee-a1c7-00503e09f296,Sitio Corrego dos Coimbras - Antônio Domingos ...,Sitio Corrego dos Coimbras,LR03243,João Pedro Sillos Damitto Tinoco,Custo Total com Alimentação,2026-08-01,1.592400e+04,24899.496,56.364582,Aumento
1,e2a7cc1a-c3d0-48cc-8fa5-ba491c69b70d,São Domingos dos Teixeira - Osvander da Silva ...,São Domingos dos Teixeira,LR09678,Paulo Sergio da Silva Lopes,Custo Total com Alimentação,2026-08-01,2.298831e+04,9679.412,-57.894196,Redução
2,e2a7cc1a-c3d0-48cc-8fa5-ba491c69b70d,São Domingos dos Teixeira - Osvander da Silva ...,São Domingos dos Teixeira,LR09678,Paulo Sergio da Silva Lopes,Estoque de Capital Total,2026-08-01,2.658549e+06,1191850.000,-55.169161,Redução
3,1da27c18-3d06-4bc1-995d-71658ec0c8e6,"Fazenda Morro Feio, Serra Negra e Chapadão de ...","Fazenda Morro Feio, Serra Negra e Chapadão de ...",LR05329,Amanda Roriz dos Reis Ferreira,Custo Total Mão de Obra,2026-07-01,5.032582e+03,10960.150,117.783839,Aumento
4,925f0bd2-4476-4516-b68e-ec1333c50ff3,Fazenda Cachoeira - Juscelio Lopes de Miranda,Fazenda Cachoeira,LR02562,Paulo Sergio da Silva Lopes,Custo Total Mão de Obra,2026-07-01,5.948678e+03,9500.000,59.699340,Aumento


,Consultor,"Fazendas atendidas [ATENÇÃO: cada vínculo conta - fazenda com mais de um consultor aparece para todos eles, logo a soma desta coluna EXCEDE o total de fazendas distintas]",Fazendas marcadas,% de fazendas marcadas,Total de Vacas,Preço Unitário do Leite,Área Total da Propriedade,Produção Total de Leite,Estoque de Capital Total,Custo Total Mão de Obra,Custo Total com Alimentação
0,Sem Consultor Vinculado,68,13,19.117647,0,2,0,2,1,1,11
1,Antonio Aélito Madeiro Filho,16,7,43.750000,0,0,1,3,0,3,1
2,MATHEUS VARGAS DE CARVALHO,24,6,25.000000,0,0,0,0,0,1,5
3,Amanda Roriz dos Reis Ferreira,23,6,26.086957,0,0,0,1,0,1,5
4,Luís Felipe Pereira Cruciol,10,6,60.000000,0,0,0,1,1,1,5


### 3.8.1 Verificação da Regra de Variação Mensal com Amostra Sintética

Exercita os casos de fronteira do método contra a função pura `calcular_variacao_periodo`,
com uma amostra sintética pequena — sem banco, sem SharePoint e sem dado real de cliente
(`AGENTS.md`: "prefira amostras sintéticas e pequenas"). Verifica comportamento externo
observável — os valores e categorias que saem da função —, não como as colunas foram
montadas internamente.

In [44]:
# ==============================================================================
# 3.8.1 VERIFICAÇÃO DA REGRA DE VARIAÇÃO MENSAL COM AMOSTRA SINTÉTICA
# ==============================================================================
# Exercita os casos de fronteira do método contra a função pura, sem banco, sem
# SharePoint e sem dado real de cliente (AGENTS.md: "prefira amostras sintéticas
# e pequenas"). Verifica comportamento externo observável - os valores e
# categorias que saem da função -, não como as colunas foram montadas internamente.
# ==============================================================================

def _amostra_sintetica_variacao_mensal():
    """Monta um DataFrame mensal sintético cobrindo os casos de fronteira do spec."""
    MES = lambda ano, mes: pd.Timestamp(year=ano, month=mes, day=1)

    linhas = []

    def adicionar(id_property, ano, mes, valor, consultor="Consultor A", nome=None):
        linhas.append({
            "id_property": id_property,
            "property_name": nome or id_property,
            "property_entrepreneur_label": f"{nome or id_property} - Produtor {id_property}",
            "labor_rural_code": f"LR-{id_property}",
            "reference_month": MES(ano, mes),
            "consultant_name": consultor,
            "total_cows": valor,
        })

    # --- Razão exatamente 1,5 e exatamente 0,5: NÃO marca -----------------------
    adicionar("SONDA-LIM-SUP", 2026, 1, 1.0)
    adicionar("SONDA-LIM-SUP", 2026, 2, 1.5)                              # razão == 1,5 exato
    adicionar("SONDA-LIM-INF", 2026, 1, 1.0)
    adicionar("SONDA-LIM-INF", 2026, 2, 0.5)                              # razão == 0,5 exato

    # --- Menor incremento além de cada limite: marca -----------------------------
    adicionar("SONDA-FORA-SUP", 2026, 1, 1.0)
    adicionar("SONDA-FORA-SUP", 2026, 2, np.nextafter(1.5, np.inf))       # razão > 1,5
    adicionar("SONDA-FORA-INF", 2026, 1, 1.0)
    adicionar("SONDA-FORA-INF", 2026, 2, np.nextafter(0.5, -np.inf))      # razão < 0,5

    # --- Entrada (0 -> positivo) e Saída (positivo -> 0) -------------------------
    adicionar("SONDA-ENTRADA", 2026, 1, 0)
    adicionar("SONDA-ENTRADA", 2026, 2, 50)
    adicionar("SONDA-SAIDA", 2026, 1, 50)
    adicionar("SONDA-SAIDA", 2026, 2, 0)

    # --- 0 -> 0 e nulo: Não avaliado ---------------------------------------------
    adicionar("SONDA-ZERO-ZERO", 2026, 1, 0)
    adicionar("SONDA-ZERO-ZERO", 2026, 2, 0)
    adicionar("SONDA-NULO", 2026, 1, 100)
    adicionar("SONDA-NULO", 2026, 2, np.nan)
    adicionar("SONDA-NULO", 2026, 3, 100)                                  # mês seguinte ao nulo

    # --- Lacuna de mês: Não avaliado, NUNCA comparação atravessando o buraco -----
    # jan -> mar pulando fev; se a lacuna vazasse, 100 -> 400 marcaria Aumento.
    adicionar("SONDA-LACUNA", 2026, 1, 100)
    adicionar("SONDA-LACUNA", 2026, 3, 400)

    # --- Primeira linha da fazenda: Não avaliado (fazenda com um único mês) ------
    adicionar("SONDA-PRIMEIRA", 2026, 1, 100)

    # --- Fazenda com dois consultores: aparece nos dois no resumo, sem duplicar
    #     a linha de variação em si (mensal agrega consultor em uma string única).
    adicionar("SONDA-DUPLA", 2026, 1, 100, consultor="Consultor B, Consultor C")
    adicionar("SONDA-DUPLA", 2026, 2, 250, consultor="Consultor B, Consultor C")  # razão 2,5 -> Aumento

    # --- Caso normal, de controle: variação pequena não marca ---------------------
    adicionar("SONDA-NORMAL", 2026, 1, 100)
    adicionar("SONDA-NORMAL", 2026, 2, 110)

    return pd.DataFrame(linhas)


INDICADORES_TESTE_VARIACAO = {
    "total_cows": ("Total de Vacas", "cabeças", ".0f"),
}


def verificar_variacao_periodo_com_amostra_sintetica():
    """Roda a função pura contra a amostra sintética e confere os casos de fronteira."""
    df_sintetico = _amostra_sintetica_variacao_mensal()
    colunas_antes = list(df_sintetico.columns)
    snapshot_antes = df_sintetico.copy(deep=True)

    df_anotado, df_cons, df_por_cons = calcular_variacao_periodo(
        df_sintetico, INDICADORES_TESTE_VARIACAO,
    )

    falhas = []

    def conferir(descricao, condicao, obtido=""):
        if condicao:
            print(f"  OK   {descricao}")
        else:
            falhas.append(f"{descricao} -> {obtido}")
            print(f"  FALHA {descricao} -> {obtido}")

    def categoria(id_property, ano, mes):
        linha = df_anotado.loc[
            df_anotado["id_property"].eq(id_property)
            & df_anotado["reference_month"].eq(pd.Timestamp(year=ano, month=mes, day=1))
        ]
        return linha["variacao_total_cows"].iloc[0]

    # 1. Razão exatamente 1,5 e 0,5: NÃO marca (fica Normal).
    conferir("razão exatamente 1,5 NÃO marca", categoria("SONDA-LIM-SUP", 2026, 2) == "Normal",
             categoria("SONDA-LIM-SUP", 2026, 2))
    conferir("razão exatamente 0,5 NÃO marca", categoria("SONDA-LIM-INF", 2026, 2) == "Normal",
             categoria("SONDA-LIM-INF", 2026, 2))

    # 2. Menor incremento além de cada limite: marca.
    conferir("razão logo acima de 1,5 marca Aumento", categoria("SONDA-FORA-SUP", 2026, 2) == "Aumento",
             categoria("SONDA-FORA-SUP", 2026, 2))
    conferir("razão logo abaixo de 0,5 marca Redução", categoria("SONDA-FORA-INF", 2026, 2) == "Redução",
             categoria("SONDA-FORA-INF", 2026, 2))

    # 3. Entrada e Saída.
    conferir("0 -> positivo = Entrada", categoria("SONDA-ENTRADA", 2026, 2) == "Entrada",
             categoria("SONDA-ENTRADA", 2026, 2))
    conferir("positivo -> 0 = Saída", categoria("SONDA-SAIDA", 2026, 2) == "Saída",
             categoria("SONDA-SAIDA", 2026, 2))

    # 4. 0 -> 0 e nulo: Não avaliado.
    conferir("0 -> 0 = Não avaliado", categoria("SONDA-ZERO-ZERO", 2026, 2) == "Não avaliado",
             categoria("SONDA-ZERO-ZERO", 2026, 2))
    conferir("valor nulo = Não avaliado", categoria("SONDA-NULO", 2026, 2) == "Não avaliado",
             categoria("SONDA-NULO", 2026, 2))
    conferir("mês seguinte a um nulo (sem M-1 válido) = Não avaliado",
             categoria("SONDA-NULO", 2026, 3) == "Não avaliado", categoria("SONDA-NULO", 2026, 3))

    # 5. Lacuna de mês: Não avaliado, nunca comparação atravessando o buraco.
    conferir("lacuna de mês (jan -> mar) = Não avaliado, não compara 100 -> 400",
             categoria("SONDA-LACUNA", 2026, 3) == "Não avaliado", categoria("SONDA-LACUNA", 2026, 3))

    # 6. Primeira linha da fazenda: Não avaliado.
    conferir("primeira (e única) linha da fazenda = Não avaliado",
             categoria("SONDA-PRIMEIRA", 2026, 1) == "Não avaliado", categoria("SONDA-PRIMEIRA", 2026, 1))

    # 7. Fazenda com dois consultores: aparece nos dois no resumo por consultor,
    #    sem duplicar a linha de variação em si na aba "Variações".
    linhas_dupla_consolidado = df_cons.loc[df_cons["IDFazenda"].eq("SONDA-DUPLA")]
    conferir("SONDA-DUPLA gera UMA única linha na aba Variações (não duplica por consultor)",
             len(linhas_dupla_consolidado) == 1, len(linhas_dupla_consolidado))
    resumo = df_por_cons.set_index("Consultor")
    conferir("SONDA-DUPLA marcada aparece para os DOIS consultores no resumo",
             "Consultor B" in resumo.index and "Consultor C" in resumo.index
             and int(resumo.loc["Consultor B", "Fazendas marcadas"]) >= 1
             and int(resumo.loc["Consultor C", "Fazendas marcadas"]) >= 1,
             resumo.index.tolist() if "Consultor B" not in resumo.index or "Consultor C" not in resumo.index
             else {c: int(resumo.loc[c, "Fazendas marcadas"]) for c in ("Consultor B", "Consultor C")})

    # 8. Caso normal: variação pequena não entra na aba Variações.
    conferir("SONDA-NORMAL (variação de 10%) não marca e não aparece na aba Variações",
             categoria("SONDA-NORMAL", 2026, 2) == "Normal"
             and df_cons.loc[df_cons["IDFazenda"].eq("SONDA-NORMAL")].empty)

    # 9. A aba "Variações" só contém Aumento/Redução (nunca Entrada/Saída/Normal/Não avaliado).
    conferir("aba Variações só contém linhas Aumento/Redução",
             df_cons["Lado"].isin(["Aumento", "Redução"]).all(), df_cons["Lado"].unique().tolist())

    # 10. Pureza: a função não altera o DataFrame de entrada.
    conferir("a função não altera o DataFrame de entrada (mesmas colunas)",
             list(df_sintetico.columns) == colunas_antes)
    conferir("a função não altera os valores do DataFrame de entrada",
             df_sintetico.equals(snapshot_antes))

    if falhas:
        raise AssertionError(
            f"{len(falhas)} verificação(ões) da amostra sintética falharam:\n  - "
            + "\n  - ".join(falhas)
        )
    print(f"\nAmostra sintética: {len(df_sintetico):,} linhas, "
          f"{df_sintetico['id_property'].nunique()} fazendas - todas as verificações passaram.")


verificar_variacao_periodo_com_amostra_sintetica()

  OK   razão exatamente 1,5 NÃO marca
  OK   razão exatamente 0,5 NÃO marca
  OK   razão logo acima de 1,5 marca Aumento
  OK   razão logo abaixo de 0,5 marca Redução
  OK   0 -> positivo = Entrada
  OK   positivo -> 0 = Saída
  OK   0 -> 0 = Não avaliado
  OK   valor nulo = Não avaliado
  OK   mês seguinte a um nulo (sem M-1 válido) = Não avaliado
  OK   lacuna de mês (jan -> mar) = Não avaliado, não compara 100 -> 400
  OK   primeira (e única) linha da fazenda = Não avaliado
  OK   SONDA-DUPLA gera UMA única linha na aba Variações (não duplica por consultor)
  OK   SONDA-DUPLA marcada aparece para os DOIS consultores no resumo
  OK   SONDA-NORMAL (variação de 10%) não marca e não aparece na aba Variações
  OK   aba Variações só contém linhas Aumento/Redução
  OK   a função não altera o DataFrame de entrada (mesmas colunas)
  OK   a função não altera os valores do DataFrame de entrada

Amostra sintética: 24 linhas, 12 fazendas - todas as verificações passaram.


## 3.9 Preparação e Formatação dos Dados Mensais para Excel


In [45]:
# ==============================================================================
# 3.9 PREPARAÇÃO E FORMATAÇÃO DOS DADOS MENSAIS PARA EXCEL
# ==============================================================================
def preparar_dataframe_para_excel(df: pd.DataFrame) -> pd.DataFrame:
    """
    Prepara um DataFrame para exportação pelo openpyxl.
    """

    df_excel = df.copy()

    # Excel não trabalha com infinito
    df_excel = df_excel.replace([np.inf, -np.inf], np.nan)
    
    # Excel não aceita datas com timezone
    for coluna in df_excel.columns:
        if isinstance(df_excel[coluna].dtype, pd.DatetimeTZDtype, ):
            df_excel[coluna] = ( df_excel[coluna] .dt.tz_localize(None) )
    
    return df_excel


# ==============================================================================
# 3.9.0.1 PREENCHIMENTO DE NUMERICOS VAZIOS COM ZERO
# ==============================================================================
def preencher_numericos_vazios_com_zero(df: pd.DataFrame) -> pd.DataFrame:
    """
    Substitui vazio por 0 apenas em colunas numericas que ja possuem algum dado real.

    Regras:
    - so dtype numerico (select_dtypes "number"): texto, data, booleano e os placeholders
      object=None ficam intactos;
    - coluna 100% vazia e preservada vazia (placeholder sem fonte de dados no modelo novo,
      ex.: investimentos, exames laboratoriais, transporte, vacinas);
    - infinito NAO e tratado aqui: continua virando vazio em preparar_dataframe_para_excel,
      por isso este passo roda ANTES daquele.
    """
    df_preenchido = df.copy()

    for coluna in df_preenchido.select_dtypes(include="number").columns:
        if df_preenchido[coluna].notna().sum() == 0:
            continue  # placeholder sem fonte de dados: mantem vazio
        df_preenchido[coluna] = df_preenchido[coluna].fillna(0)

    return df_preenchido


# ==============================================================================
# 3.9.1 TRADUÇÃO E MAPEAMENTO 1-A-1 DOS INDICADORES MENSAIS
# ==============================================================================
COLUNAS_TECNICAS_MENSAIS_EM_ORDEM = [
    "id_property",
    "property_entrepreneur_label",
    "labor_rural_code",
    "agroindustry_name",
    "dairy_region",
    "consultant_name",
    "reference_month",
    "milk_sold_revenue", "milk_volume_sold", "milk_unit_price", "ccs", "cpp", "fat", "protein",
    "unit_price_derivative", "milk_volume_derivatives", "milk_derivatives_revenue", "received_loans", "animal_sale",
    "other_revenues", "price_bonus", "price_penalty", "voluminous_sold", "concentrated_sold", "surplus_division",
    "lactating_cows", "dry_cows", "nursing", "rearing", "males", "other_categories", "total_cows", "total_cattle",
    "lactating_cows_value", "dry_cows_value", "nursing_value", "rearing_value", "males_value", "other_categories_value",
    "general_expenses", "advance_payment", "administration", "land_lease", "technical_assistance", "animal_purchase",
    "land_purchase", "repairs", "loan_interest", "hormones", "taxes_fees", "medicines_vaccines", "bedding_replacement",
    "reproduction", "milk_replacer", "milking_material", "milk_calves", "energy", "fuel",
    "voluminous_purchased_quantity", "voluminous_consumed_quantity", "voluminous_amount_total",
    "concentrate_purchased_quantity", "concentrate_consumed_quantity", "concentrate_amount_total",
    "mineral_purchased_quantity", "mineral_consumed_quantity", "mineral_amount_total",
    "family_labor_expenses", "family_labor_quantity", "hired_labor_expenses", "hired_labor_quantity", "unit_price",
    "own_milk_hired_labor_quantity", "hired_labor_amount_total", "discarded_quantity", "discarded_amount_total",
    "calves_quantity", "calves_amount_total", "own_milk_family_labor_quantity", "family_labor_amount_total",
    "hectares_owned_benfeitorias_estradas", "hectares_owned_app_reserva_legal", "hectares_owned_forrageiras",
    "hectares_rented_benfeitorias_estradas", "hectares_rented_app_reserva_legal", "hectares_rented_forrageiras",
    "raw_land_value_benfeitorias_estradas", "raw_land_value_app_reserva_legal", "raw_land_value_forrageiras",
    "hectares_propria", "hectares_atividade", "hectares_total", "raw_land_value_medio_ponderado",
    "monthly_depreciation_benfeitorias", "monthly_depreciation_maquinas_e_equipamentos",
    "monthly_average_capital_stock_benfeitorias", "monthly_average_capital_stock_maquinas_e_equipamentos",
    "production_system", "deflator", "voluminous_amount_total_forrageira_propria",
    "concentrate_amount_total_forrageira_propria",
    "has_revenue_data", "has_cattle_data", "has_expense_data", "has_feeding_data", "has_labor_data", "has_own_milk_data",
    "has_asset_data", "has_active_area_month", "has_dairy_production_system_data",
    "discarded_quantity_milk_revenue", "hired_labor_quantity_milk_revenue", "family_labor_milk_revenue",
    "calves_quantity_milk_revenue", "total_milk_revenue", "milk_produced", "total_activity_revenue",
    "milk_revenue_liter", "days_in_month", "milk_daily", "milk_lactating_cow_day", "lactating_cows_total_cows",
    "lactating_cows_total_cattle", "total_labor_quantity", "milk_total_labor_day", "lactating_cows_total_labor",
    "concentrate_mineral_cost", "feeding_cost", "concentrate_mineral_quantity", "feeding_cost_liter",
    "voluminous_cost_liter", "concentrate_mineral_cost_liter", "feeding_cost_milk_price", "total_labor_expenses",
    "hired_labor_cost_liter", "family_labor_cost_liter", "total_labor_cost_liter", "labor_cost_milk_revenue",
    "other_operating_expenses", "animal_capital_stock", "land_capital_stock", "fixed_capital_stock",
    "total_capital_stock", "total_capital_stock_milk_daily", "lactating_cows_hectare_activity", "hectares_arrendada",
    "rented_area_percentage", "property_status", "cons_ccs", "cons_cpp", "cons_fat", "cons_protein",
    "cons_lactating_cows_total_cows", "cons_lactating_cows_total_cattle", "cons_milk_lactating_cow_day",
    "cons_milk_total_labor_day", "cons_lactating_cows_total_labor", "cons_feeding_cost_milk_price",
    "cons_voluminous_cost_liter", "cons_concentrate_mineral_cost_liter", "cons_hired_labor_cost_liter",
    "total_consistency_criteria_ok", "total_consistency_criteria", "total_consistency_criteria_violated",
    "consistency_status", "consistency_id", "violated_consistency_criteria", "violated_consistency_details"
]

NOMES_PORTUGUES_EM_ORDEM_MENSAIS = [
    "IdFazenda",
    "Fazenda - Produtor",
    "Código LR",
    "Agroindústria",
    "Região",
    "Consultor",
    "Mês de Referência",
    "Receita com Venda de Leite (R$)", "Volume de Leite Vendido (litros)", "Preço Unitário do Leite (R$/litro)", "CCS (x1000 células/ml)", "CPP (x1000 UFC/ml)", "Gordura (%)", "Proteína (%)",
    "Preço Unitário Derivados (R$)", "Volume de Leite em Derivados (litros)", "Receita de Derivados (R$)", "Empréstimos Recebidos (R$)", "Venda de Animais (R$)",
    "Outras Receitas (R$)", "Bonificação no Preço (R$)", "Penalidade no Preço (R$)", "Volumoso Vendido (R$)", "Concentrado Vendido (R$)", "Divisão de Sobra (R$)",
    "Vacas em Lactação (cabeças)", "Vacas Secas (cabeças)", "Bezerras (cabeças)", "Novilhas (cabeças)", "Machos (cabeças)", "Outras Categorias (cabeças)", "Total de Vacas (cabeças)", "Total de Animais (cabeças)",
    "Valor Vacas em Lactação (R$)", "Valor Vacas Secas (R$)", "Valor Bezerras (R$)", "Valor Novilhas (R$)", "Valor Machos (R$)", "Valor Outras Categorias (R$)",
    "Acessórios e Despesas Gerais (R$)", "Adiantamento (R$)", "Despesas Administrativas (R$)", "Arrendamento (R$)", "Assistência Técnica (R$)", "Compra de Animais (R$)",
    "Compra de Terra (R$)", "Reparos e Consertos (R$)", "Juros de Empréstimos (R$)", "Hormônios (R$)", "Impostos e Taxas (R$)", "Medicamentos e Vacinas (R$)", "Reposição da cama (R$)",
    "Reprodução (R$)", "Sucedâneo (R$)", "Material de Ordenha (R$)", "Leite para Bezerras (R$)", "Energia Elétrica (R$)", "Combustível (R$)",
    "Qtd Comprada Volumoso (kg)", "Qtd Consumida Volumoso (kg)", "Gasto Total Volumoso (R$)",
    "Qtd Comprada Concentrado (kg)", "Qtd Consumida Concentrado (kg)", "Gasto Total Concentrado (R$)",
    "Qtd Comprada Mineral (kg)", "Qtd Consumida Mineral (kg)", "Gasto Total Mineral (R$)",
    "Despesas Mão de Obra Familiar (R$)", "Qtd Mão de Obra Familiar (trabalhadores)", "Despesas Mão de Obra Contratada (R$)", "Qtd Mão de Obra Contratada (trabalhadores)", "Preço Unitário do Leite Próprio (R$/litro)",
    "Leite Próprio MDO Contratada (litros)", "Custo Total MDO Contratada (R$)", "Leite Descartado (litros)", "Valor Lançado Leite Descartado (R$)",
    "Leite Bezerras Consumido (litros)", "Valor Leite Bezerras (R$)", "Leite Próprio MDO Familiar (litros)", "Custo com Leite Consumido (R$)",
    "Área Própria Benfeitorias/Estradas (ha)", "Área Própria APP/Reserva (ha)", "Área Própria Forrageiras (ha)",
    "Área Arrendada Benfeitorias/Estradas (ha)", "Área Arrendada APP/Reserva (ha)", "Área Arrendada Forrageiras (ha)",
    "Valor Terra Nua Benfeitorias/Estradas (R$)", "Valor Terra Nua APP/Reserva (R$)", "Valor Terra Nua Forrageiras (R$)",
    "Área Própria Total (ha)", "Área Destinada à Atividade (ha)", "Área Total da Propriedade (ha)", "Valor Média Ponderada Terra Nua (R$/ha)",
    "Depreciação Mensal Benfeitorias (R$)", "Depreciação Mensal Máquinas (R$)",
    "Estoque de Capital Benfeitorias (R$)", "Estoque de Capital Máquinas (R$)",
    "Sistema de Produção", "Fator Deflator (IGP-DI)", "Custo Forrageira Própria Volumoso (R$)",
    "Custo Forrageira Própria Concentrado (R$)",
    "Possui Dados de Receita", "Possui Dados de Rebanho", "Possui Dados de Despesas", "Possui Dados de Alimentação", "Possui Dados de Mão de Obra", "Possui Dados de Leite Próprio",
    "Possui Dados de Ativos", "Possui Dados de Área Ativa", "Possui Dados Sistema Produção",
    "Valor Leite Descartado a Preço de Venda do Leite (R$)", "Valor Leite MDO Contratada a Preço de Leite (R$)", "Custo com Leite consumido pela MDO familiar (R$)",
    "Custo leite consumido pelas Bezerras (R$)", "Receita Total do Leite (R$)", "Produção Total de Leite (litros)", "Receita Bruta da Atividade (R$)",
    "Receita da Atividade por litro de leite (R$/litro)", "Dias no Mês", "Produção Diária de Leite (litros/dia)", "Produção por Vaca em Lactação (litros/vaca/dia)", "Vacas em Lactação / Total de Vacas (%)",
    "Vacas em Lactação / Rebanho Total (%)", "Mão de Obra Total (trabalhadores)", "Produção por Mão de Obra (litros/trabalhador/dia)", "Vacas em Lactação por Mão de Obra (vacas/trabalhador)",
    "Custo de Concentrado e Mineral (R$)", "Custo Total com Alimentação (R$)", "Quantidade de Concentrado e Mineral (kg)", "Custo de Alimentação por Litro (R$/litro)",
    "Custo de Volumoso por Litro (R$/litro)", "Custo de Concentrado por Litro (R$/litro)", "Alimentação / Preço do Leite (%)", "Custo Total Mão de Obra (R$)",
    "Custo MDO Contratada por Litro (R$/litro)", "Custo MDO Familiar por Litro (R$/litro)", "Custo MDO Total por Litro (R$/litro)", "Mão de Obra / Receita do Leite (%)",
    "Outras Despesas Operacionais (R$)", "Estoque de Capital em Animais (R$)", "Estoque de Capital em Terra (R$)", "Estoque de Capital Fixo (R$)",
    "Estoque de Capital Total (R$)", "Estoque de Capital / Produção Diária (R$/litro/dia)", "Vacas em Lactação / Área (vacas/ha)", "Área Arrendada (ha)",
    "Área Arrendada (%)", "Status Cadastral", "Consistência: CCS", "Consistência: CPP", "Consistência: Gordura", "Consistência: Proteína",
    "Consistência: VL/Total Vacas", "Consistência: VL/Rebanho Total", "Consistência: Produção/VL",
    "Consistência: Produção/MDO", "Consistência: VL/MDO", "Consistência: Alimentação/Preço Leite",
    "Consistência: Custo Volumoso/L", "Consistência: Custo Concentrado/L", "Consistência: Custo MDO Contratada/L",
    "Total Critérios Atendidos", "Total Critérios Avaliados", "Total Critérios Violados",
    "Status de Consistência", "ID de Consistência", "Critérios Violados", "Detalhamento das Inconsistências"
]

assert len(COLUNAS_TECNICAS_MENSAIS_EM_ORDEM) == len(NOMES_PORTUGUES_EM_ORDEM_MENSAIS), (
    f"Incompatibilidade de tamanho: {len(COLUNAS_TECNICAS_MENSAIS_EM_ORDEM)} vs {len(NOMES_PORTUGUES_EM_ORDEM_MENSAIS)}"
)

# Garantir a criação da coluna de rótulo combinado se não existir em df_consistencia
if "property_entrepreneur_label" not in df_consistencia.columns and {"property_name", "entrepreneur_name"}.issubset(df_consistencia.columns):
    df_consistencia["property_entrepreneur_label"] = (
        df_consistencia["property_name"].fillna("") + " - " + df_consistencia["entrepreneur_name"].fillna("")
    )

dados_renomeados_mensais = {}
colunas_mensais_ausentes = []

for col_tecnica, col_pt in zip(COLUNAS_TECNICAS_MENSAIS_EM_ORDEM, NOMES_PORTUGUES_EM_ORDEM_MENSAIS):
    if col_tecnica in df_consistencia.columns:
        dados_renomeados_mensais[col_pt] = df_consistencia[col_tecnica].values
    else:
        dados_renomeados_mensais[col_pt] = pd.Series([None] * len(df_consistencia), index=df_consistencia.index).values
        colunas_mensais_ausentes.append(col_tecnica)

df_exportacao_mensal = pd.DataFrame(dados_renomeados_mensais, index=df_consistencia.index)
print(f"Indicadores mensais mapeados para português: {df_exportacao_mensal.shape[1]} colunas.")


Indicadores mensais mapeados para português: 168 colunas.


In [46]:
df_exportacao_mensal

,IdFazenda,Fazenda - Produtor,Código LR,Agroindústria,Região,Consultor,Mês de Referência,Receita com Venda de Leite (R$),Volume de Leite Vendido (litros),Preço Unitário do Leite (R$/litro),...,Consistência: Custo Volumoso/L,Consistência: Custo Concentrado/L,Consistência: Custo MDO Contratada/L,Total Critérios Atendidos,Total Critérios Avaliados,Total Critérios Violados,Status de Consistência,ID de Consistência,Critérios Violados,Detalhamento das Inconsistências
0,00220277-a58e-4b9e-9b89-e6acf8a1a400,FAZENDA RIO DO PEIXE - Jaci Guimaraes De Siqueira,LR05385,NESTLÉ,Goiânia - 9655,"MARLUS MARRA CARVALHO PINHEIRO, Pablo Freitas ...",2024-05-01,0.000000e+00,0.0,NaN,...,False,False,False,0,13,13,Inconsistente,1,CCS; CPP; Gordura; Proteína; VL/Total de vacas...,CCS (Ausente/NaN); CPP (Ausente/NaN); Gordura ...
1,00220277-a58e-4b9e-9b89-e6acf8a1a400,FAZENDA RIO DO PEIXE - Jaci Guimaraes De Siqueira,LR05385,NESTLÉ,Goiânia - 9655,"MARLUS MARRA CARVALHO PINHEIRO, Pablo Freitas ...",2024-08-01,5.301091e+05,167274.0,3.169106,...,False,True,True,12,13,1,Inconsistente,1,Custo de volumoso,Custo de volumoso (Inconsistente: 0.05 R$/L)
2,004cde8f-4688-4b22-acd9-76151b7dcc7a,Fazenda São Bartolomeu - EDGAR JOSE DE AZEVEDO...,LR01964,NESTLÉ,Ibiá - 1215,LORENA VIRGINIA ARAUJO,2025-04-01,1.120198e+05,35677.0,3.139831,...,True,True,True,13,13,0,Consistente,0,Nenhum,Nenhum
3,004cde8f-4688-4b22-acd9-76151b7dcc7a,Fazenda São Bartolomeu - EDGAR JOSE DE AZEVEDO...,LR01964,NESTLÉ,Ibiá - 1215,LORENA VIRGINIA ARAUJO,2025-05-01,1.627337e+05,54884.0,2.965048,...,True,True,True,13,13,0,Consistente,0,Nenhum,Nenhum
4,004cde8f-4688-4b22-acd9-76151b7dcc7a,Fazenda São Bartolomeu - EDGAR JOSE DE AZEVEDO...,LR01964,NESTLÉ,Ibiá - 1215,LORENA VIRGINIA ARAUJO,2025-06-01,1.613585e+05,56311.0,2.865487,...,True,True,True,13,13,0,Consistente,0,Nenhum,Nenhum
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13317,fff52e9f-83b6-44db-b7a4-01da7e77aeaa,Fazenda Campo Alegre - Luiz Alexandre de Avelar,LR05853,NESTLÉ,Patos de Minas - 9188,GISELE MARIA CARVALHO MAGALHAES,2026-03-01,2.161324e+06,765080.0,2.824964,...,True,True,True,13,13,0,Consistente,0,Nenhum,Nenhum
13318,fff52e9f-83b6-44db-b7a4-01da7e77aeaa,Fazenda Campo Alegre - Luiz Alexandre de Avelar,LR05853,NESTLÉ,Patos de Minas - 9188,GISELE MARIA CARVALHO MAGALHAES,2026-04-01,2.293030e+06,733654.0,3.125492,...,True,True,True,13,13,0,Consistente,0,Nenhum,Nenhum
13319,fff52e9f-83b6-44db-b7a4-01da7e77aeaa,Fazenda Campo Alegre - Luiz Alexandre de Avelar,LR05853,NESTLÉ,Patos de Minas - 9188,GISELE MARIA CARVALHO MAGALHAES,2026-05-01,2.315243e+06,751991.0,3.078818,...,True,True,True,13,13,0,Consistente,0,Nenhum,Nenhum
13320,fff52e9f-83b6-44db-b7a4-01da7e77aeaa,Fazenda Campo Alegre - Luiz Alexandre de Avelar,LR05853,NESTLÉ,Patos de Minas - 9188,GISELE MARIA CARVALHO MAGALHAES,2026-06-01,2.333599e+06,751991.0,3.103227,...,True,True,True,13,13,0,Consistente,0,Nenhum,Nenhum


## 3.10 Exportação dos Arquivos Excel Mensais


In [47]:
# ==============================================================================
# 3.10 EXPORTAÇÃO DOS ARQUIVOS EXCEL MENSAIS
# ==============================================================================
# Garante que a raiz do projeto seja a pasta pai caso o notebook rode dentro da pasta /app
RAIZ_PROJETO = Path.cwd().parent if Path.cwd().name == 'app' else Path.cwd()
PASTA_SAIDA = RAIZ_PROJETO / 'data' / 'outputs' / 'monthly'
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

DATA_EXPORTACAO = datetime.now().strftime("%Y_%m_%d_%H%M%S")
CAMINHO_ARQUIVO = PASTA_SAIDA / f'{DATA_EXPORTACAO}_indicadores_mensais.xlsx'

# Preparar os dados em português para o Excel
df_exportacao = preparar_dataframe_para_excel(df_exportacao_mensal)

# ------------------------------------------------------------------------------
# Colunas auxiliares "_var_" (ticket 04 de .scratch/variacao-50-periodo/): uma
# por indicador coberto pela regra de variação da 3.8, agrupadas no fim da aba
# "Indicadores Mensais". Ficam ocultas (mas desocultáveis) e alimentam a
# formatação condicional laranja aplicada depois da exportação, mais abaixo
# nesta mesma célula. Acréscimo puro - nenhuma coluna existente muda de nome,
# cabeçalho, ordem ou tipo.
# O cabeçalho exportado do indicador inclui a unidade entre parênteses (ex.:
# "Preço Unitário do Leite (R$/litro)") - vem do mapa técnico -> português da
# 3.9, não do rótulo "nu" da configuração da 3.8. Usar o rótulo nu aqui faria a
# formatação condicional nunca encontrar a coluna e virar um no-op silencioso.
MAPA_TECNICO_PARA_PT_MENSAL = dict(zip(COLUNAS_TECNICAS_MENSAIS_EM_ORDEM, NOMES_PORTUGUES_EM_ORDEM_MENSAIS))

COLUNAS_VAR_MENSAL = {}  # nome da coluna auxiliar -> cabeçalho exportado do indicador correspondente
for _coluna, (_rotulo, _unidade, _formato) in INDICADORES_VARIACAO_MENSAL.items():
    _nome_var = f"_var_{_coluna}"
    df_exportacao[_nome_var] = (
        df_variacao_anotado_mensal[f"variacao_{_coluna}"]
        .reindex(df_exportacao_mensal.index)
        .to_numpy()
    )
    if _coluna not in MAPA_TECNICO_PARA_PT_MENSAL:
        raise KeyError(
            f"Indicador '{_coluna}' não está em COLUNAS_TECNICAS_MENSAIS_EM_ORDEM (3.9) - "
            "a coloração da 3.10 não vai achar a coluna correspondente na aba exportada."
        )
    COLUNAS_VAR_MENSAL[_nome_var] = MAPA_TECNICO_PARA_PT_MENSAL[_coluna]

# ------------------------------------------------------------------------------
# Coluna auxiliar "_var_qualquer" (ticket 09 de .scratch/variacao-50-periodo/): OR
# das 7 colunas "_var_" acima - alimenta a coloração do bloco de identificação
# (7 primeiras colunas da aba, antes do primeiro indicador), pintado quando pelo
# menos um indicador da linha foi marcado Aumento/Redução. Fica agrupada e oculta
# junto das demais colunas "_var_", desocultável pelo auditor como as outras.
df_exportacao["_var_qualquer"] = np.zeros(len(df_exportacao), dtype=bool)
for _nome_var in COLUNAS_VAR_MENSAL:
    df_exportacao["_var_qualquer"] |= df_exportacao[_nome_var].isin(["Aumento", "Redução"])

# Bloco de identificação da aba "Indicadores Mensais": as 7 colunas antes do
# primeiro indicador (IdFazenda .. Mês de Referência), coloridas pela regra acima.
COLUNAS_IDENTIFICACAO_MENSAL = NOMES_PORTUGUES_EM_ORDEM_MENSAIS[:7]

# Abas novas da regra de variação mensal (tickets 03 e 05): sem preenchimento de
# numéricos vazios com zero, para "Não avaliado" chegar como texto, nunca "0".
df_exportacao_variacoes = preparar_dataframe_para_excel(df_variacoes)
df_exportacao_variacoes_consultor = preparar_dataframe_para_excel(df_variacoes_por_consultor)

# Auditoria da forrageira própria: uma aba por lançamento, uma por lote e uma de
# pendências. As três são montadas na seção 3.4.2.5, que roda antes de
# df_dim_property existir, então o nome da fazenda é colado aqui — sem ele o
# consultor recebe só o UUID e não consegue cobrar o lançamento de ninguém.
COLUNAS_IDENTIFICACAO_FORRAGEIRA = ["id_property", "property_name", "labor_rural_code"]


def identificar_fazenda(df: pd.DataFrame) -> pd.DataFrame:
    """Acrescenta nome e código da fazenda e os traz para as primeiras colunas."""

    if "id_property" not in df.columns:
        return df

    colunas_dim = [
        coluna
        for coluna in COLUNAS_IDENTIFICACAO_FORRAGEIRA
        if coluna in df_dim_property.columns
    ]

    df = df.drop(
        columns=[c for c in colunas_dim if c != "id_property" and c in df.columns],
        errors="ignore",
    ).merge(
        df_dim_property[colunas_dim].drop_duplicates("id_property"),
        on="id_property",
        how="left",
        validate="m:1",
    )

    identificacao = [c for c in colunas_dim if c in df.columns]

    return df[identificacao + [c for c in df.columns if c not in identificacao]]


df_exportacao_forrageira_lancamento = preparar_dataframe_para_excel(identificar_fazenda(df_auditoria_forrageira))
df_exportacao_forrageira_lote = preparar_dataframe_para_excel(identificar_fazenda(df_auditoria_forrageira_lote))
df_exportacao_forrageira_pendencia = preparar_dataframe_para_excel(identificar_fazenda(df_pendencia_custo_forrageira))

# Exportar com a formatação padrão
exportar_varias_abas_xlsx(
    abas={
        'Indicadores Mensais': df_exportacao,
        'Variações': df_exportacao_variacoes,
        'Variações por Consultor': df_exportacao_variacoes_consultor,
        'Forrageira por Lançamento': df_exportacao_forrageira_lancamento,
        'Forrageira por Lote': df_exportacao_forrageira_lote,
        'Pendências de Custo': df_exportacao_forrageira_pendencia,
    },
    caminho_saida=CAMINHO_ARQUIVO,
    fonte='Aptos'
)

# ------------------------------------------------------------------------------
# Coloração laranja pós-exportação (ticket 04 de .scratch/variacao-50-periodo/):
# formatação condicional por fórmula via openpyxl, lendo as colunas "_var_" -
# não estilo célula a célula, porque a aba tem ~168 colunas x ~13 mil linhas.
# Roda DEPOIS de exportar_varias_abas_xlsx porque essa função já recarrega e
# resalva o arquivo (aplicar_estilo_listrado_xlsx, em functions/) - fazer antes
# seria sobrescrito. functions/ não é tocado; esta etapa fica local ao notebook.
# ------------------------------------------------------------------------------
from openpyxl import load_workbook
from openpyxl.formatting.rule import FormulaRule
from openpyxl.styles import PatternFill
from openpyxl.utils import get_column_letter

_wb_cor = load_workbook(CAMINHO_ARQUIVO)
_ws_indicadores = _wb_cor['Indicadores Mensais']
_preenchimento_laranja = PatternFill(start_color='FFA500', end_color='FFA500', fill_type='solid')

_colunas_por_nome = {celula.value: idx + 1 for idx, celula in enumerate(_ws_indicadores[1])}
_max_linha = _ws_indicadores.max_row
_regras_aplicadas = 0

for _nome_var, _rotulo in COLUNAS_VAR_MENSAL.items():
    if _nome_var not in _colunas_por_nome or _rotulo not in _colunas_por_nome:
        raise KeyError(
            f"Coloração da 3.10: coluna '{_nome_var}' ou '{_rotulo}' não encontrada no "
            f"cabeçalho exportado. Colunas disponíveis (amostra): "
            f"{list(_colunas_por_nome)[:5]} ... {list(_colunas_por_nome)[-10:]}"
        )
    _letra_var = get_column_letter(_colunas_por_nome[_nome_var])
    _letra_indicador = get_column_letter(_colunas_por_nome[_rotulo])

    # Coluna auxiliar oculta por padrão - o auditor consegue desocultá-la.
    _ws_indicadores.column_dimensions[_letra_var].hidden = True

    if _max_linha >= 2:
        _intervalo = f"{_letra_indicador}2:{_letra_indicador}{_max_linha}"
        _formula = f'OR(${_letra_var}2="Aumento",${_letra_var}2="Redução")'
        _ws_indicadores.conditional_formatting.add(
            _intervalo,
            FormulaRule(formula=[_formula], fill=_preenchimento_laranja),
        )
        _regras_aplicadas += 1

if _regras_aplicadas != len(COLUNAS_VAR_MENSAL):
    raise AssertionError(
        f"Coloração da 3.10: só {_regras_aplicadas} de {len(COLUNAS_VAR_MENSAL)} regras "
        "condicionais foram aplicadas."
    )

# ------------------------------------------------------------------------------
# Coloração laranja do bloco de identificação (ticket 09 de
# .scratch/variacao-50-periodo/): mesma técnica acima, lendo a coluna auxiliar
# única "_var_qualquer" - pinta o bloco de identificação quando pelo menos um
# indicador da linha foi marcado Aumento/Redução.
# ------------------------------------------------------------------------------
if "_var_qualquer" not in _colunas_por_nome:
    raise KeyError(
        "Coloração da 3.10 (ticket 09): coluna '_var_qualquer' não encontrada no "
        "cabeçalho exportado."
    )
_letra_var_qualquer = get_column_letter(_colunas_por_nome["_var_qualquer"])
_ws_indicadores.column_dimensions[_letra_var_qualquer].hidden = True

_regras_identificacao_aplicadas = 0
for _rotulo_id in COLUNAS_IDENTIFICACAO_MENSAL:
    if _rotulo_id not in _colunas_por_nome:
        raise KeyError(
            f"Coloração da 3.10 (ticket 09): coluna de identificação '{_rotulo_id}' "
            "não encontrada no cabeçalho exportado."
        )
    _letra_id = get_column_letter(_colunas_por_nome[_rotulo_id])

    if _max_linha >= 2:
        _intervalo_id = f"{_letra_id}2:{_letra_id}{_max_linha}"
        _formula_id = f"${_letra_var_qualquer}2"
        _ws_indicadores.conditional_formatting.add(
            _intervalo_id,
            FormulaRule(formula=[_formula_id], fill=_preenchimento_laranja),
        )
        _regras_identificacao_aplicadas += 1

if _regras_identificacao_aplicadas != len(COLUNAS_IDENTIFICACAO_MENSAL):
    raise AssertionError(
        f"Coloração da 3.10 (ticket 09): só {_regras_identificacao_aplicadas} de "
        f"{len(COLUNAS_IDENTIFICACAO_MENSAL)} regras de identificação foram aplicadas."
    )

_wb_cor.save(CAMINHO_ARQUIVO)
print(f"Coloração aplicada: {_regras_aplicadas} colunas de indicador + "
      f"{_regras_identificacao_aplicadas} colunas de identificação com formatação condicional laranja.")

print('Exportação mensal concluída com sucesso.')
print(f'Linhas exportadas: {len(df_exportacao):,}')
print(f'Aba Variações: {len(df_exportacao_variacoes):,} linhas | '
      f'Variações por Consultor: {len(df_exportacao_variacoes_consultor):,} linhas')
print(f'Arquivo: {CAMINHO_ARQUIVO}')

Coloração aplicada: 7 colunas de indicador + 7 colunas de identificação com formatação condicional laranja.
Exportação mensal concluída com sucesso.
Linhas exportadas: 13,322
Aba Variações: 2,171 linhas | Variações por Consultor: 66 linhas
Arquivo: C:\Users\analy\LABOR RURAL\Analytics - Departamento Analytics\TEMP\GUILHERME\SCRIPTS\projetcs\elabore-views\data\outputs\monthly\2026_08_19_120415_indicadores_mensais.xlsx


In [48]:
# # ==============================================================================
# # 3.10 EXPORTAÇÃO DOS ARQUIVOS EXCEL MENSAIS
# # ==============================================================================
# RAIZ_PROJETO = Path.cwd().parent if Path.cwd().name == "app" else Path.cwd()

# PASTA_SAIDA = RAIZ_PROJETO / "data" / "outputs" / "monthly"
# PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
# DATA_EXPORTACAO = datetime.now().strftime("%Y_%m_%d_%H%M%S")
# CAMINHO_ARQUIVO = ( PASTA_SAIDA / f"{DATA_EXPORTACAO}_indicadores_mensais.xlsx" )

# # Preparar os dados em português para o Excel
# df_exportacao = preparar_dataframe_para_excel( df_exportacao_mensal)

# # Criar o arquivo com a formatação padrão
# exportar_varias_abas_xlsx(abas={"Indicadores Mensais": df_exportacao}, caminho_saida=CAMINHO_ARQUIVO, fonte="Aptos")

# # Aplicar cabeçalho colorido e linhas alternadas
# aplicar_estilo_listrado_xlsx(
#     caminho_arquivo=CAMINHO_ARQUIVO,
#     cor_cabecalho="#247B72",
#     cor_texto_cabecalho="#FFFFFF",
#     cor_linha_alternada="#F2F2F2",
#     cor_linha_base="#FFFFFF",
#     primeira_linha_cinza=True,
# )

# print("Exportação mensal concluída com sucesso.")
# print(f"Linhas exportadas: {len(df_exportacao):,}")
# print(f"Arquivo: {CAMINHO_ARQUIVO}")

# 4. Indicadores Anuais - Janela Móvel, Regras e Exportação


## 4.1 Agregação por Janela Móvel de 12 Meses Completos


In [49]:
# ==============================================================================
# 4.1 AGREGAÇÃO POR JANELA MÓVEL DE 12 MESES COMPLETOS
# ==============================================================================
df_mensal_anuais = df_consistencia.copy()
df_mensal_anuais["reference_month"] = (pd.to_datetime(df_mensal_anuais["reference_month"], errors="coerce").dt.to_period("M").dt.to_timestamp())
df_mensal_anuais = (df_mensal_anuais.dropna(subset=["id_property", "reference_month"]).sort_values(["id_property", "reference_month"]).drop_duplicates(["id_property", "reference_month"], keep="last").reset_index(drop=True))

# Criar coluna de CCS, CPP, Gordura, Proteína ainda não calculados
df_mensal_anuais['abs_ccs'] = df_mensal_anuais['ccs'] * df_mensal_anuais['milk_produced']
df_mensal_anuais['abs_cpp'] = df_mensal_anuais['cpp'] * df_mensal_anuais['milk_produced']
df_mensal_anuais['abs_fat'] = df_mensal_anuais['fat'] * df_mensal_anuais['milk_produced']
df_mensal_anuais['abs_protein'] = df_mensal_anuais['protein'] * df_mensal_anuais['milk_produced']

# ==============================================================================
# ÁREA: RECONSTRUIR "COM RESERVA" E "FORRAGEIRA" (ainda não calculadas na Feature Engineering)
# ==============================================================================
# hectares_atividade já é a área "sem reserva" (benfeitorias/estradas + forrageiras, próprio + arrendado).
# Somando de volta a Reserva Legal e APP chegamos na área "com reserva" (equivalente à antiga areaAtividadeReserva).
COLUNAS_RESERVA_LEGAL = [c for c in ["hectares_owned_app_reserva_legal", "hectares_rented_app_reserva_legal"] if c in df_mensal_anuais.columns]
if COLUNAS_RESERVA_LEGAL:
    df_mensal_anuais["hectares_atividade_com_reserva"] = (
        df_mensal_anuais["hectares_atividade"].fillna(0)
        + df_mensal_anuais[COLUNAS_RESERVA_LEGAL].sum(axis=1, skipna=True)
    )

# Área de forrageira (própria + arrendada), equivalente à antiga areaForrageira.
COLUNAS_FORRAGEIRA = [c for c in ["hectares_owned_forrageiras", "hectares_rented_forrageiras"] if c in df_mensal_anuais.columns]
if COLUNAS_FORRAGEIRA:
    df_mensal_anuais["hectares_forrageira"] = df_mensal_anuais[COLUNAS_FORRAGEIRA].sum(axis=1, skipna=True)

COLUNAS_SOMA_ANUAL = [
    # Produção de Leite #
    "milk_produced", "milk_volume_sold", "milk_volume_derivatives", "discarded_quantity",
    # Renda #
    "total_milk_revenue", "total_activity_revenue", "concentrated_sold", "voluminous_sold",
    # COE #
    "general_expenses", "administration", "land_lease", "technical_assistance",
    "repairs", "hormones", "taxes_fees", "medicines_vaccines", "bedding_replacement",
    "reproduction", "milk_replacer", "milking_material", "milk_calves", "energy", "fuel",
    "voluminous_amount_total", "concentrate_amount_total", "mineral_amount_total",
    "hired_labor_expenses", "other_operating_expenses",
    "feeding_cost", "concentrate_mineral_cost",
    # COT #
    "family_labor_expenses", "monthly_depreciation_benfeitorias", "monthly_depreciation_maquinas_e_equipamentos",
    # Alimentação (quantidades consumidas) #
    "concentrate_consumed_quantity",
    "voluminous_consumed_quantity",
    "mineral_consumed_quantity",
    # MDO #
    "family_labor_quantity",
    "hired_labor_quantity",
    # QUALIDADE (auxiliares para média ponderada) #
    "abs_ccs", "abs_cpp", "abs_fat", "abs_protein",
    # Estoque de Capital #
    'monthly_depreciation_benfeitorias',
    'monthly_depreciation_maquinas_e_equipamentos'
]
COLUNAS_MEDIA_ANUAL = [
    # Animais #
    "lactating_cows", "total_cows", "total_cattle",
    # Quantidade de MDO #
    "hired_labor_quantity", "family_labor_quantity", "total_labor_quantity",
    # Área #
    "hectares_atividade", "hectares_atividade_com_reserva", "hectares_forrageira",
    "hectares_arrendada", "hectares_propria", "hectares_total",
    "raw_land_value_medio_ponderado",
    # Estoque de Capital (é um estoque médio, não um fluxo: usar média, não soma) === #
    "animal_capital_stock", "land_capital_stock",
    'monthly_average_capital_stock_benfeitorias',
    'monthly_average_capital_stock_maquinas_e_equipamentos'
]
COLUNAS_QUALIDADE = ["ccs", "cpp", "fat", "protein"]
COLUNAS_SOMA_ANUAL = [c for c in COLUNAS_SOMA_ANUAL if c in df_mensal_anuais.columns]
COLUNAS_MEDIA_ANUAL = [c for c in COLUNAS_MEDIA_ANUAL if c in df_mensal_anuais.columns]
COLUNAS_QUALIDADE = [c for c in COLUNAS_QUALIDADE if c in df_mensal_anuais.columns]

# ==============================================================================
# COBERTURA DE LANÇAMENTO DE DADOS POR JANELA ANUAL
# ==============================================================================
# A continuidade da janela (12 meses seguidos) só é garantida para a vw_revenue,
# que é a base de df_mensal_anuais. Isso não impede que uma propriedade tenha
# lançado receita todo mês mas deixado de lançar despesa/mão de obra/alimentação
# em alguns meses daquela mesma janela - o que subestima COE/COT/CT sem gerar
# nenhum erro (a soma anual simplesmente usa os meses que existem).
# Este relatório mede, para cada janela, quantos dos 12 meses têm cada fonte.
MAPA_COBERTURA = {
    "has_revenue_data": "months_with_revenue_data",
    "has_cattle_data": "months_with_cattle_data",
    "has_expense_data": "months_with_expense_data",
    "has_feeding_data": "months_with_feeding_data",
    "has_labor_data": "months_with_labor_data",
    "has_own_milk_data": "months_with_own_milk_data",
    "has_asset_data": "months_with_asset_data",
    "has_active_area_month": "months_with_active_area_data",
    "has_dairy_production_system_data": "months_with_dairy_production_system_data",
}
MAPA_COBERTURA = {k: v for k, v in MAPA_COBERTURA.items() if k in df_mensal_anuais.columns}

# Fontes usadas diretamente no cálculo de COE/COT/CT: são as que mais importam
# para explicar uma possível subestimação de custo.
FONTES_CRITICAS_PARA_CUSTO = {
    "has_expense_data": "Despesas (vw_expense)",
    "has_feeding_data": "Alimentação (vw_feeding)",
    "has_labor_data": "Mão de obra (vw_labor)",
    "has_asset_data": "Patrimônio/Depreciação (mvw_asset_payment_history)",
}

registros_anuais = []
registros_cobertura = []
for id_property, grupo in df_mensal_anuais.groupby("id_property", sort=False):
    grupo = grupo.sort_values("reference_month").reset_index(drop=True)
    for fim in range(11, len(grupo)):
        janela = grupo.iloc[fim - 11:fim + 1]
        meses = janela["reference_month"].dt.to_period("M")
        esperado = pd.period_range(meses.iloc[0], meses.iloc[-1], freq="M")
        if len(esperado) != 12 or list(meses) != list(esperado):
            continue
        registro = janela.iloc[-1].to_dict()
        registro["annual_period_start"] = janela["reference_month"].iloc[0]
        registro["annual_period_end"] = janela["reference_month"].iloc[-1]
        registro["annual_period"] = f"{registro['annual_period_start']:%b/%y}-{registro['annual_period_end']:%b/%y}"
        registro["months_in_annual_window"] = 12
        for coluna in COLUNAS_SOMA_ANUAL:
            registro[f"{coluna}_annual"] = janela[coluna].sum(min_count=1)
        for coluna in COLUNAS_MEDIA_ANUAL:
            registro[f"{coluna}_annual_average"] = janela[coluna].mean()
        volume = pd.to_numeric(janela["milk_produced"], errors="coerce")
        for coluna in COLUNAS_QUALIDADE:
            valores = pd.to_numeric(janela[coluna], errors="coerce")
            validos = valores.notna() & volume.notna() & volume.gt(0)
            registro[f"{coluna}_annual_weighted_average"] = (np.average(valores[validos], weights=volume[validos]) if validos.any() else np.nan)

        # --- Tolerância configurável de meses inconsistentes na janela anual ---
        # 0 = estrito (1 mês inconsistente já invalida a janela anual)
        # N = permite até N meses inconsistentes antes de marcar a janela anual como Inconsistente
        MAX_INCONSISTENT_MONTHS_ALLOWED = 1
        
        inconsistencias = pd.to_numeric(janela["consistency_id"], errors="coerce")
        registro["inconsistent_months_annual"] = int(inconsistencias.sum())
        registro["annual_consistency_id"] = int(registro["inconsistent_months_annual"] > MAX_INCONSISTENT_MONTHS_ALLOWED)
        registro["annual_consistency_status"] = "Consistente" if registro["annual_consistency_id"] == 0 else "Inconsistente"
        
        # --- Concatenar detalhamento das inconsistências mensais da janela ---
        detalhes_mensais_inconsistentes = []
        for _, linha_mes in janela.iterrows():
            if str(linha_mes.get("consistency_status", "")) == "Inconsistente":
                mes_str = f"{linha_mes['reference_month']:%b/%y}"
                detalhe_mes = str(linha_mes.get("violated_consistency_details", ""))
                if detalhe_mes and detalhe_mes != "Nenhum" and pd.notna(linha_mes.get("violated_consistency_details")):
                    detalhes_mensais_inconsistentes.append(f"{mes_str}: {detalhe_mes}")
                else:
                    criterios_mes = str(linha_mes.get("violated_consistency_criteria", ""))
                    detalhes_mensais_inconsistentes.append(f"{mes_str}: {criterios_mes}")

        registro["annual_monthly_inconsistency_details"] = (" | ".join(detalhes_mensais_inconsistentes) if detalhes_mensais_inconsistentes else "Nenhuma")

        registros_anuais.append(registro)

        # --- Cobertura de lançamento de dados desta mesma janela (relatório separado) ---
        registro_cobertura = {
            "id_property": id_property,
            "annual_period_start": registro["annual_period_start"],
            "annual_period_end": registro["annual_period_end"],
            "annual_period": registro["annual_period"],
        }
        fontes_incompletas = []
        for coluna_flag, nome_coluna_cobertura in MAPA_COBERTURA.items():
            meses_com_dado = int(janela[coluna_flag].sum())
            registro_cobertura[nome_coluna_cobertura] = meses_com_dado
            if coluna_flag in FONTES_CRITICAS_PARA_CUSTO and meses_com_dado < 12:
                fontes_incompletas.append(f"{FONTES_CRITICAS_PARA_CUSTO[coluna_flag]}: {meses_com_dado}/12")
        registro_cobertura["cost_data_status"] = "Completa" if not fontes_incompletas else "Incompleta"
        registro_cobertura["cost_data_gaps"] = "; ".join(fontes_incompletas) if fontes_incompletas else "Nenhum"
        registros_cobertura.append(registro_cobertura)

df_anuais = pd.DataFrame(registros_anuais)
if df_anuais.empty:
    raise ValueError("Nenhuma propriedade possui uma janela completa de 12 meses consecutivos.")
if df_anuais.duplicated(["id_property", "annual_period_end"]).any():
    raise ValueError("Foram geradas janelas anuais duplicadas por propriedade e mês final.")
print(f"Janelas anuais calculadas: {len(df_anuais):,}")

# ==============================================================================
# RELATÓRIO DE COBERTURA DE DADOS (SEPARADO DOS INDICADORES ANUAIS)
# ==============================================================================
# Este relatório não entra em df_anuais / df_indicadores_anuais / df_calculo_medias.
# Ele existe para diagnosticar, por propriedade e janela anual, se algum mês deixou
# de ter dado lançado em despesas/alimentação/mão de obra/patrimônio - o que
# subestimaria COE, COT e CT silenciosamente sem isso ficar visível em nenhum outro lugar.
df_cobertura_anual = pd.DataFrame(registros_cobertura)
df_cobertura_anual = df_cobertura_anual.merge(
    df_anuais[["id_property", "annual_period_end", "property_name", "labor_rural_code"]],
    on=["id_property", "annual_period_end"],
    how="left",
) if {"property_name", "labor_rural_code"}.issubset(df_anuais.columns) else df_cobertura_anual
df_cobertura_anual = df_cobertura_anual.sort_values(["id_property", "annual_period_end"]).reset_index(drop=True)

janelas_incompletas = int((df_cobertura_anual["cost_data_status"] == "Incompleta").sum())
print(f"Janelas anuais com lançamento de custo incompleto: {janelas_incompletas:,} de {len(df_cobertura_anual):,}")
display(df_cobertura_anual.head())

Janelas anuais calculadas: 5,511
Janelas anuais com lançamento de custo incompleto: 492 de 5,511


,id_property,annual_period_start,annual_period_end,annual_period,months_with_revenue_data,months_with_cattle_data,months_with_expense_data,months_with_feeding_data,months_with_labor_data,months_with_own_milk_data,months_with_asset_data,months_with_active_area_data,months_with_dairy_production_system_data,cost_data_status,cost_data_gaps,property_name,labor_rural_code
0,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-04-01,2026-03-01,Apr/25-Mar/26,12,12,12,12,12,10,12,12,12,Completa,Nenhum,Fazenda São Bartolomeu,LR01964
1,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-05-01,2026-04-01,May/25-Apr/26,12,12,12,12,12,11,12,12,12,Completa,Nenhum,Fazenda São Bartolomeu,LR01964
2,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-06-01,2026-05-01,Jun/25-May/26,12,12,12,12,12,11,12,12,12,Completa,Nenhum,Fazenda São Bartolomeu,LR01964
3,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-07-01,2026-06-01,Jul/25-Jun/26,12,12,12,12,12,11,12,12,12,Completa,Nenhum,Fazenda São Bartolomeu,LR01964
4,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-08-01,2026-07-01,Aug/25-Jul/26,12,12,12,12,12,11,12,12,12,Completa,Nenhum,Fazenda São Bartolomeu,LR01964


## 4.2 Indicadores Derivados Anuais e Regras de Consistência Anual


In [50]:
# ==============================================================================
# 4.2 INDICADORES DERIVADOS ANUAIS E REGRAS DE CONSISTÊNCIA ANUAL
# ==============================================================================
pd.set_option('future.no_silent_downcasting', True)
def dividir_seguro(numerador: pd.Series, denominador: pd.Series) -> pd.Series:
    denom_seguro = denominador.where(denominador != 0, np.nan)
    resultado = numerador / denom_seguro
    return resultado.where(resultado.abs() != np.inf, np.nan)

def coluna_ou_none(df: pd.DataFrame, nome: str) -> pd.Series:
    """Retorna a coluna se ela existir; caso contrário, uma série de None (mesmo índice)."""
    if nome in df.columns:
        return df[nome]
    return pd.Series([None] * len(df), index=df.index)

dias_ano = (df_anuais["annual_period_end"] + pd.offsets.MonthEnd(0) - df_anuais["annual_period_start"] + pd.Timedelta(days=1)).dt.days


# ==============================================================================
# 1. RENDA
# ==============================================================================
df_anuais['rbl_rba'] = dividir_seguro(df_anuais['total_milk_revenue_annual'], df_anuais['total_activity_revenue_annual'])
# Versão em percentual (0-100), equivalente ao antigo rbl_rba já multiplicado por 100.
df_anuais['rbl_rba_percentage_annual'] = df_anuais['rbl_rba'] * 100

# ==============================================================================
# 1. PRODUÇÃO, QUALIDADE E MÃO DE OBRA (já existentes, mantidos)
# ==============================================================================
df_anuais["milk_daily_annual"] = dividir_seguro(df_anuais["milk_produced_annual"], dias_ano)
df_anuais["milk_revenue_liter_annual"] = dividir_seguro(df_anuais["total_milk_revenue_annual"], df_anuais["milk_produced_annual"])
df_anuais["milk_lactating_cow_day_annual"] = dividir_seguro(df_anuais["milk_daily_annual"], df_anuais["lactating_cows_annual_average"])
df_anuais["milk_daily_total_cows_annual"] = dividir_seguro(df_anuais["milk_daily_annual"], df_anuais["total_cows_annual_average"])
df_anuais["lactating_cows_total_cows_annual"] = dividir_seguro(df_anuais["lactating_cows_annual_average"], df_anuais["total_cows_annual_average"]) * 100
df_anuais["lactating_cows_total_cattle_annual"] = dividir_seguro(df_anuais["lactating_cows_annual_average"], df_anuais["total_cattle_annual_average"]) * 100
df_anuais["milk_total_labor_day_annual"] = dividir_seguro(df_anuais["milk_daily_annual"], df_anuais["total_labor_quantity_annual_average"])
df_anuais["lactating_cows_total_labor_annual"] = dividir_seguro(df_anuais["lactating_cows_annual_average"], df_anuais["total_labor_quantity_annual_average"])

# ==============================================================================
# 2. ÁREA (com reserva, forrageira e produção por área)
# ==============================================================================
df_anuais["milk_hectare_activity_annual"] = dividir_seguro(df_anuais["milk_produced_annual"], df_anuais["hectares_atividade_annual_average"])
df_anuais["milk_hectare_activity_com_reserva_annual"] = dividir_seguro(df_anuais["milk_produced_annual"], coluna_ou_none(df_anuais, "hectares_atividade_com_reserva_annual_average"))
df_anuais["milk_hectare_forrageira_annual"] = dividir_seguro(df_anuais["milk_produced_annual"], coluna_ou_none(df_anuais, "hectares_forrageira_annual_average"))
df_anuais["lactating_cows_hectare_activity_annual"] = dividir_seguro(df_anuais["lactating_cows_annual_average"], df_anuais["hectares_atividade_annual_average"])
df_anuais["lactating_cows_hectare_activity_com_reserva_annual"] = dividir_seguro(df_anuais["lactating_cows_annual_average"], coluna_ou_none(df_anuais, "hectares_atividade_com_reserva_annual_average"))
df_anuais["rented_area_percentage_annual"] = dividir_seguro(df_anuais["hectares_arrendada_annual_average"], df_anuais["hectares_total_annual_average"]) * 100
df_anuais["land_value_per_hectare_annual"] = coluna_ou_none(df_anuais, "raw_land_value_medio_ponderado_annual_average")

# ==============================================================================
# 3. QUALIDADE DO LEITE: GORDURA/PROTEÍNA POR VACA EM LACTAÇÃO/DIA
# ==============================================================================
# Réplica do indicador antigo Gordura_VL / Proteina_VL:
# (produção anual/365) * 1,032 * (percentual médio ponderado/100) / vacas em lactação médias
df_anuais["fat_lactating_cow_day_annual"] = dividir_seguro(
    (df_anuais["milk_produced_annual"] / 365) * 1.032 * (coluna_ou_none(df_anuais, "fat_annual_weighted_average") / 100),
    df_anuais["lactating_cows_annual_average"],
)
df_anuais["protein_lactating_cow_day_annual"] = dividir_seguro(
    (df_anuais["milk_produced_annual"] / 365) * 1.032 * (coluna_ou_none(df_anuais, "protein_annual_weighted_average") / 100),
    df_anuais["lactating_cows_annual_average"],
)

# ==============================================================================
# 4. ALIMENTAÇÃO: CUSTO, PREÇO DO CONCENTRADO E RELAÇÃO DE TROCA
# ==============================================================================
df_anuais["feeding_cost_liter_annual"] = dividir_seguro(df_anuais["feeding_cost_annual"], df_anuais["milk_produced_annual"])
df_anuais["concentrate_mineral_cost_liter_annual"] = dividir_seguro(df_anuais["concentrate_mineral_cost_annual"], df_anuais["milk_produced_annual"])
df_anuais["feeding_cost_milk_price_annual"] = dividir_seguro(df_anuais["feeding_cost_liter_annual"], df_anuais["milk_revenue_liter_annual"]) * 100

# Gasto com volumoso como percentual da receita total da atividade.
# Denominador e a receita da atividade (mesma base de gross_margin_annual e profitability_annual),
# nao o preco do leite usado em feeding_cost_milk_price_annual.
df_anuais["voluminous_cost_revenue_annual"] = dividir_seguro(
    coluna_ou_none(df_anuais, "voluminous_amount_total_annual"),
    df_anuais["total_activity_revenue_annual"],
) * 100

df_anuais["concentrate_mineral_consumed_quantity_annual"] = (
    df_anuais[["concentrate_consumed_quantity_annual", "mineral_consumed_quantity_annual"]].sum(axis=1, min_count=1)
    if {"concentrate_consumed_quantity_annual", "mineral_consumed_quantity_annual"}.issubset(df_anuais.columns)
    else None
)
df_anuais["concentrate_mineral_price_annual"] = dividir_seguro(df_anuais["concentrate_mineral_cost_annual"], coluna_ou_none(df_anuais, "concentrate_mineral_consumed_quantity_annual"))
df_anuais["milk_concentrate_exchange_ratio_annual"] = dividir_seguro(df_anuais["milk_revenue_liter_annual"], coluna_ou_none(df_anuais, "concentrate_mineral_price_annual"))

# ==============================================================================
# 4b. AGREGAÇÕES DE CUSTO USADAS NO CÁLCULO DE MÉDIAS ANTIGO
# ==============================================================================
# Aleitamento (sucedâneo comprado + leite de bezerro), equivalente ao antigo def_aleitaento_somaMovel.
COMPONENTES_ALEITAMENTO = [c for c in ["milk_replacer_annual", "milk_calves_annual"] if c in df_anuais.columns]
df_anuais["aleitamento_annual"] = df_anuais[COMPONENTES_ALEITAMENTO].sum(axis=1, min_count=1) if COMPONENTES_ALEITAMENTO else None

# Energia + combustível, equivalente ao antigo custoEnergiaCombustivel.
COMPONENTES_ENERGIA_COMBUSTIVEL = [c for c in ["energy_annual", "fuel_annual"] if c in df_anuais.columns]
df_anuais["energy_fuel_cost_annual"] = df_anuais[COMPONENTES_ENERGIA_COMBUSTIVEL].sum(axis=1, min_count=1) if COMPONENTES_ENERGIA_COMBUSTIVEL else None

# Depreciação total do estoque de capital (benfeitorias + máquinas), equivalente ao antigo
# depreciacaoEstoqueCapital_somaMovel. Falta o componente de forrageiras não-anuais/plantio.
COMPONENTES_DEPRECIACAO_TOTAL = [c for c in ["monthly_depreciation_benfeitorias_annual", "monthly_depreciation_maquinas_e_equipamentos_annual"] if c in df_anuais.columns]
df_anuais["total_depreciation_annual"] = df_anuais[COMPONENTES_DEPRECIACAO_TOTAL].sum(axis=1, min_count=1) if COMPONENTES_DEPRECIACAO_TOTAL else None

# ==============================================================================
# 5. COE, COT E CT (CUSTO OPERACIONAL EFETIVO, TOTAL E CUSTO TOTAL)
# ==============================================================================
COMPONENTES_COE_ATIVIDADE_ANUAL = [
    # "other_operating_expenses_annual" foi removida daqui de propósito: ela já é a soma de
    # general_expenses, administration, land_lease, technical_assistance, repairs, hormones,
    # taxes_fees, medicines_vaccines, bedding_replacement, reproduction e milk_replacer
    # (ver COLUNAS_OUTRAS_DESPESAS na Feature Engineering). Mantê-la aqui junto com essas
    # mesmas colunas individuais contava esse bloco de despesas duas vezes no COE.
    "general_expenses_annual", "administration_annual", "land_lease_annual", "technical_assistance_annual",
    "repairs_annual", "hormones_annual", "taxes_fees_annual", "medicines_vaccines_annual", "bedding_replacement_annual",
    "reproduction_annual", "milk_replacer_annual", "milk_calves_annual", "energy_annual", "fuel_annual",
    "voluminous_amount_total_annual", "concentrate_amount_total_annual", "mineral_amount_total_annual",
    "hired_labor_expenses_annual"
]
COMPONENTES_COE_LEITE = [
    # COE do Leite é o COE da Atividade vezes a renda bruta do leite/renda bruta da atividade + Material de Ordenha e sem custos com aleitamento.
    # (mesmo ajuste acima: "other_operating_expenses_annual" removida para não contar essas despesas duas vezes.)
    "general_expenses_annual", "administration_annual", "land_lease_annual", "technical_assistance_annual",
    "repairs_annual", "hormones_annual", "taxes_fees_annual", "medicines_vaccines_annual", "bedding_replacement_annual",
    "reproduction_annual", "energy_annual", "fuel_annual", # Removido aleitamento daqui
    "voluminous_amount_total_annual", "concentrate_amount_total_annual", "mineral_amount_total_annual",
    "hired_labor_expenses_annual"
] # Isso aqui será multiplicado por Renda do Leite / Renda da Atividade.

# COE da Atividade
COMPONENTES_COE_ATIVIDADE_ANUAL = [c for c in COMPONENTES_COE_ATIVIDADE_ANUAL if c in df_anuais.columns]
df_anuais["coe_activity_annual"] = df_anuais[COMPONENTES_COE_ATIVIDADE_ANUAL].sum(axis=1, min_count=1) if COMPONENTES_COE_ATIVIDADE_ANUAL else None

# COE do Leite
COMPONENTES_COE_LEITE = [c for c in COMPONENTES_COE_LEITE if c in df_anuais.columns]
df_anuais["coe_milk_annual"] = (
    (df_anuais[COMPONENTES_COE_LEITE].sum(axis=1, min_count=1)) * df_anuais["rbl_rba"]
    + coluna_ou_none(df_anuais, "milking_material_annual")
) if COMPONENTES_COE_LEITE else None

COMPONENTES_COT_ADICIONAIS = [
    "family_labor_expenses_annual",
    "monthly_depreciation_benfeitorias_annual",
    "monthly_depreciation_maquinas_e_equipamentos_annual",
    # Falta o componente de depreciação de forrageiras não-anuais/plantio: sem fonte de dados ainda -> None.
]

COMPONENTES_COT_ADICIONAIS = [c for c in COMPONENTES_COT_ADICIONAIS if c in df_anuais.columns]
if "coe_activity_annual" in df_anuais.columns and COMPONENTES_COT_ADICIONAIS:
    df_anuais["cot_annual"] = df_anuais[["coe_activity_annual"] + COMPONENTES_COT_ADICIONAIS].sum(axis=1, min_count=1)
else:
    df_anuais["cot_annual"] = None

df_anuais['total_capital_stock_annual_average'] = (
    df_anuais[
        [
            'monthly_average_capital_stock_benfeitorias_annual_average',
            'monthly_average_capital_stock_maquinas_e_equipamentos_annual_average',
            'animal_capital_stock_annual_average',
            'land_capital_stock_annual_average'
        ]
    ]
    .sum(axis=1)
)
# Estoque de capital sem terra (para o custo de oportunidade do capital).
if {"total_capital_stock_annual_average", "land_capital_stock_annual_average"}.issubset(df_anuais.columns):
    df_anuais["capital_stock_no_land_annual_average"] = (
        df_anuais["total_capital_stock_annual_average"] - df_anuais["land_capital_stock_annual_average"]
    )
else:
    df_anuais["capital_stock_no_land_annual_average"] = None

# Custo de oportunidade do capital: 6% ao ano sobre o estoque de capital sem terra.
df_anuais["capital_opportunity_cost_annual"] = coluna_ou_none(df_anuais, "capital_stock_no_land_annual_average") * 0.06

if df_anuais["cot_annual"].notna().any() and df_anuais["capital_opportunity_cost_annual"].notna().any():
    df_anuais["ct_annual"] = df_anuais[["cot_annual", "capital_opportunity_cost_annual"]].sum(axis=1, min_count=1)
else:
    df_anuais["ct_annual"] = None

# ==============================================================================
# 6. RESULTADO ECONÔMICO: MARGENS, LUCRO E RCMA
# ==============================================================================
df_anuais["gross_margin_annual"] = df_anuais["total_activity_revenue_annual"] - coluna_ou_none(df_anuais, "coe_activity_annual")
df_anuais["net_margin_annual"] = df_anuais["total_activity_revenue_annual"] - coluna_ou_none(df_anuais, "cot_annual")
df_anuais["profit_annual"] = df_anuais["total_activity_revenue_annual"] - coluna_ou_none(df_anuais, "ct_annual")

COMPONENTES_RCMA = [c for c in ["concentrate_amount_total_annual", "mineral_amount_total_annual", "voluminous_amount_total_annual"] if c in df_anuais.columns]
if COMPONENTES_RCMA:
    df_anuais["rcma_annual"] = df_anuais["total_activity_revenue_annual"] - df_anuais[COMPONENTES_RCMA].sum(axis=1, min_count=1)
else:
    df_anuais["rcma_annual"] = None
df_anuais["rcma_lactating_cow_day_annual"] = dividir_seguro(coluna_ou_none(df_anuais, "rcma_annual"), df_anuais["lactating_cows_annual_average"] * 365)

# ==============================================================================
# 7. INDICADORES DE EFICIÊNCIA E RETORNO SOBRE O CAPITAL
# ==============================================================================
df_anuais["turnover_rate_annual"] = dividir_seguro(df_anuais["total_activity_revenue_annual"], df_anuais["total_capital_stock_annual_average"]) * 100
df_anuais["profitability_annual"] = dividir_seguro(coluna_ou_none(df_anuais, "net_margin_annual"), df_anuais["total_activity_revenue_annual"]) * 100

taxa_retorno_sem_terra = dividir_seguro(coluna_ou_none(df_anuais, "net_margin_annual"), coluna_ou_none(df_anuais, "capital_stock_no_land_annual_average")) * 100
df_anuais["capital_return_rate_no_land_annual"] = taxa_retorno_sem_terra.clip(lower=0)

taxa_retorno_com_terra = dividir_seguro(coluna_ou_none(df_anuais, "net_margin_annual"), df_anuais["total_capital_stock_annual_average"]) * 100
df_anuais["capital_return_rate_annual"] = taxa_retorno_com_terra.clip(lower=0)
# Versão sem o piso em zero, usada como auxiliar de estratificação (equivalente à antiga taxaRetornoCapitalComTerra_Geral).
df_anuais["capital_return_rate_annual_unfiltered"] = taxa_retorno_com_terra

# Pontos de cobertura (operacional e total), em litros/dia equivalentes.
df_anuais["operating_coverage_point_annual"] = dividir_seguro(coluna_ou_none(df_anuais, "cot_annual") / (12 * 30.42), df_anuais["milk_revenue_liter_annual"])
df_anuais["total_coverage_point_annual"] = dividir_seguro(coluna_ou_none(df_anuais, "ct_annual") / (12 * 30.42), df_anuais["milk_revenue_liter_annual"])

# Cot sobre preço sobre do leite
df_anuais["cot_annual_activity_revenue"] = dividir_seguro(coluna_ou_none(df_anuais, "cot_annual"), df_anuais["total_activity_revenue_annual"]) * 100
# MDO Contratada sobre preço sobre do leite
df_anuais["hired_labor_activity_revenue"] = dividir_seguro(coluna_ou_none(df_anuais, "hired_labor_expenses_annual"), df_anuais["total_activity_revenue_annual"]) * 100
# Gasto com Concentrado sobre preço sobre do leite
df_anuais["concentrate_amount_total_annual_activity_revenue"] = dividir_seguro(coluna_ou_none(df_anuais, "concentrate_amount_total_annual"), df_anuais["total_activity_revenue_annual"]) * 100
# ==============================================================================
df_anuais_final = df_anuais.copy()  # Defragmenta antes do bloco extensivo de colunas
# 8. RB, COE, COT, CT, MARGENS E LUCRO SOB DIFERENTES DENOMINADORES
# ==============================================================================
# Réplica sistemática do antigo bloco "colunas_economicas": cada medida econômica anual
# expressa por litro, % da receita, litros equivalentes, por vaca em lactação, por total
# de vacas e por hectare de área da atividade (sem e com reserva).
MEDIDAS_ECONOMICAS_ANUAIS = {
    "total_activity_revenue_annual": "activity_revenue",
    "coe_activity_annual": "coe",
    "cot_annual": "cot",
    "ct_annual": "ct",
    "gross_margin_annual": "gross_margin",
    "net_margin_annual": "net_margin",
    "profit_annual": "profit",
}
for coluna_base, prefixo in MEDIDAS_ECONOMICAS_ANUAIS.items():
    valores_base = coluna_ou_none(df_anuais_final, coluna_base)
    df_anuais_final[f"{prefixo}_liter_annual"] = dividir_seguro(valores_base, df_anuais_final["milk_produced_annual"])
    df_anuais_final[f"{prefixo}_milk_revenue_percentage_annual"] = dividir_seguro(valores_base, df_anuais_final["total_activity_revenue_annual"]) * 100
    df_anuais_final[f"{prefixo}_milk_equivalent_liters_annual"] = dividir_seguro(valores_base, df_anuais_final["milk_revenue_liter_annual"])
    df_anuais_final[f"{prefixo}_lactating_cow_annual"] = dividir_seguro(valores_base, df_anuais_final["lactating_cows_annual_average"])
    df_anuais_final[f"{prefixo}_total_cows_annual"] = dividir_seguro(valores_base, df_anuais_final["total_cows_annual_average"])
    df_anuais_final[f"{prefixo}_hectare_activity_annual"] = dividir_seguro(valores_base, df_anuais_final["hectares_atividade_annual_average"])
    df_anuais_final[f"{prefixo}_hectare_activity_com_reserva_annual"] = dividir_seguro(valores_base, coluna_ou_none(df_anuais_final, "hectares_atividade_com_reserva_annual_average"))

# ==============================================================================
# 9. ESTOQUE DE CAPITAL: TOTAL E QUEBRA POR COMPONENTE
# ==============================================================================
df_anuais_final["total_capital_stock_milk_daily_annual"] = dividir_seguro(df_anuais_final["total_capital_stock_annual_average"], df_anuais_final["milk_daily_annual"])
df_anuais_final["total_capital_stock_lactating_cow_annual"] = dividir_seguro(df_anuais_final["total_capital_stock_annual_average"], df_anuais_final["lactating_cows_annual_average"])

COMPONENTES_ESTOQUE_CAPITAL = {
    "monthly_average_capital_stock_benfeitorias_annual_average": "benfeitorias",
    "monthly_average_capital_stock_maquinas_e_equipamentos_annual_average": "maquinas_e_equipamentos",
    "animal_capital_stock_annual_average": "animais",
    "land_capital_stock_annual_average": "terra",
}
for coluna_base, sufixo in COMPONENTES_ESTOQUE_CAPITAL.items():
    df_anuais_final[f"capital_stock_{sufixo}_share_annual"] = dividir_seguro(coluna_ou_none(df_anuais_final, coluna_base), df_anuais_final["total_capital_stock_annual_average"]) * 100
# Estoque de capital em forrageiras não-anuais/plantio: sem fonte de dados ainda -> None.
df_anuais_final["capital_stock_forrageira_annual_average"] = None
df_anuais_final["capital_stock_forrageira_share_annual"] = None
df_anuais_final["forrageira_depreciation_annual"] = None

# ==============================================================================
df_anuais_final = df_anuais_final.copy()  # Defragmenta antes de adicionar placeholders
# 10. INVESTIMENTO: SEM FONTE DE DADOS AINDA (colunas mantidas como None)
# ==============================================================================
for coluna in [
    "investment_animals_annual",
    "investment_machinery_annual",
    "investment_improvements_annual",
    "investment_land_annual",
    "investment_total_annual",
    "investment_animals_capital_stock_annual",
    "investment_machinery_capital_stock_annual",
    "investment_improvements_capital_stock_annual",
    "investment_total_capital_stock_annual",
    "investment_revenue_annual",
    "investment_gross_margin_annual",
]:
    df_anuais_final[coluna] = None

# ==============================================================================
# 11. DIMENSÃO E CAMPOS AUXILIARES SEM FONTE DE DADOS NO MODELO NOVO
# ==============================================================================
df_anuais_final["agroindustry_code"] = None     # Código da agroindústria: só o nome está disponível.
df_anuais_final["production_system_annual"] = coluna_ou_none(df_anuais_final, "production_system")

# Piso do gasto com volumoso/receita muda com o sistema de producao: pasto vive de
# pastagem propria e gasta pouco com volumoso, enquanto confinamento compra volumoso.
# Sistema ausente ou desconhecido cai no piso mais rigido (4%), mesma logica do mensal.
PISO_VOLUMOSO_RECEITA_ANUAL = {"PASTURE": 1.0, "SEMI_CONFINED": 2.0}
PISO_VOLUMOSO_RECEITA_PADRAO = 4.0
df_anuais_final["voluminous_cost_revenue_min_annual"] = (
    df_anuais_final["production_system_annual"]
    .astype(str)
    .str.upper()
    .map(PISO_VOLUMOSO_RECEITA_ANUAL)
    .fillna(PISO_VOLUMOSO_RECEITA_PADRAO)
)

# Placeholders herdados do notebook antigo (também eram sempre vazios ou não-informativos lá).
# "filter_1" NAO entra aqui: e calculado abaixo, a partir do merge com o consultor.
for coluna in ["filter_2", "filter_3", "filter_4", "monthly_status_annual", "lab_tests", "milk_quality_notes", "outsourcing", "transport", "vaccines"]:
    df_anuais_final[coluna] = None

# ==============================================================================
# CONSULTOR: MERGE TARDIO (DE PROPOSITO) E FILTRO 1
# Regra de negócio do Cálculo de Médias:
# - a propriedade continua sendo calculada uma única vez;
# - na análise de médias, a mesma propriedade pode ser duplicada para cada consultor vinculado;
# - essa duplicação é intencional e serve para análise por consultor;
# - a base duplicada não deve ser usada para consolidar totais gerais sem reagrupamento por propriedade.
# ==============================================================================
# Feito aqui, sobre df_anuais_final ja agregado -- e nao em df_integrada -- porque um
# merge cedo duplicaria linhas mensais para fazendas com mais de um consultor
# e inflaria todas as somas anuais pelo numero de consultores. Aqui a janela ja
# esta fechada e somada; duplicar a linha so repete o resultado, sem somar de novo.
if "df_dim_consultor" in dir() and not df_dim_consultor.empty:
    df_anuais_final = df_anuais_final.drop(columns=["id_consultant", "consultant_name"], errors="ignore")
    df_anuais_final = df_anuais_final.merge(
        df_dim_consultor[["id_property", "id_consultant", "consultant_name"]].drop_duplicates(),
        on="id_property",
        how="left",
    )
    # consultant_name ja vem padronizado em df_dim_consultor
    
    # Filtro 1: ordem do consultor dentro da mesma propriedade e periodo.
    # 1 = primeiro consultor, 2 = segundo, e assim por diante. Filtrar por
    # filter_1 == 1 devolve uma linha por fazenda-periodo, sem a duplicacao
    # criada pelo merge com os consultores.
    # A ordenacao por consultant_name/id_consultant torna a numeracao deterministica.
    df_anuais_final = df_anuais_final.reset_index(drop=True)
    df_anuais_final["filter_1"] = (
        df_anuais_final
        .sort_values(
            ["id_property", "annual_period", "consultant_name", "id_consultant"],
            kind="mergesort",
            na_position="last",
        )
        .groupby(["id_property", "annual_period"])
        .cumcount()
        + 1
    )
else:
    df_anuais_final["consultant_name"] = None
    df_anuais_final["filter_1"] = None

# Desfragmenta o DataFrame após todas as inserções de colunas (resolve PerformanceWarning).
df_anuais_final = df_anuais_final.copy()
df_anuais_final = df_anuais_final.replace([np.inf, -np.inf], np.nan)

# ==============================================================================
# 11b. REGRAS DE CONSISTÊNCIA NO NÍVEL ANUAL (AVALIAÇÃO DA JANELA DE 12 MESES)
# ==============================================================================
# Cada regra e (coluna, min, max, nome, unidade, formato, coluna_min).
# O 7o campo aponta para uma coluna com o limite minimo linha a linha; quando preenchido,
# ele substitui o minimo fixo. E o que permite um piso diferente por sistema de producao.
REGRAS_ANUAIS_DETALHADAS = [
    ("ccs_annual_weighted_average", 50, None, "CCS Anual", "", ".0f", None),
    ("cpp_annual_weighted_average", 1, None, "CPP Anual", "", ".0f", None),
    ("fat_annual_weighted_average", 2.5, 5.5, "Gordura Anual", "%", ".2f", None),
    ("protein_annual_weighted_average", 2.4, 4.5, "Proteína Anual", "%", ".2f", None),
    ("lactating_cows_total_cows_annual", 20, 99, "VL/Total de vacas Anual", "%", ".1f", None),
    ("lactating_cows_total_cattle_annual", 15, 99, "VL/Rebanho total Anual", "%", ".1f", None),
    ("milk_lactating_cow_day_annual", 3, 45, "Produção/VL Anual", " L/vaca/dia", ".1f", None),
    ("milk_total_labor_day_annual", 20, 1500, "Produção/MDO Anual", " L/trabalhador/dia", ".1f", None),
    ("lactating_cows_total_labor_annual", 0, 70, "VL/MDO Anual", " vacas/trabalhador", ".1f", None),
    ("feeding_cost_milk_price_annual", 15, 150, "Alimentação/Preço do leite Anual", "%", ".1f", None),
    ("feeding_cost_liter_annual", None, 3.0, "Custo de alimentação/L Anual", " R$/L", ".2f", None),
    ("concentrate_mineral_cost_liter_annual", 0.30, 3.50, "Custo de concentrado/L Anual", " R$/L", ".2f", None),
    # Volumoso/Receita: teto de 40% para todos os sistemas; piso vem de
    # voluminous_cost_revenue_min_annual (pasto 1%, semiconfinado 2%, demais 4%).
    ("voluminous_cost_revenue_annual", None, 40, "Custo de volumoso/Receita Anual", "%", ".1f", "voluminous_cost_revenue_min_annual"),
    ("total_labor_quantity_annual_average", 0.5, 40, "Mão de obra total (trabalhador/dia)", "", ".1f", None),
    ("lactating_cows_hectare_activity_annual", None, 20, "Vacas em lactação/área destinada à atividade (animais/hectare)", "", ".1f", None),
    ("lactating_cows_hectare_activity_com_reserva_annual", None, 20, "Vacas em lactação/área destinada à atividade considerando reserva (animais/hectare)", "", ".1f", None),
    ("milk_hectare_activity_annual", None, 70000, "Produção/área destinada à atividade (litros/hectare/ano)", "", ".2f", None),
    ("cot_annual_activity_revenue", 50, 150, "COT/RBA", "%", ".2f", None),
    ("hired_labor_activity_revenue", None, 150, "MDO contratada/RBA", "%", ".2f", None),
    ("concentrate_amount_total_annual_activity_revenue", 10, 70, "Concentrado/RBA", "%", ".2f", None),
    ("rbl_rba_percentage_annual", 60, 100.01, "RBL/RBA", "%", ".2f", None),
    ("total_capital_stock_milk_daily_annual", 800, 30000, "Estoque Capital com Terra/L", "", ",.1f", None),
    ("turnover_rate_annual", None, 180, "Taxa de Giro", "%", ".2f", None),
    ("profitability_annual", -50, 50, "Lucratividade", "%", ".2f", None),
    ("capital_return_rate_annual", None, 40, "Taxa de Retorno Capital com Terra", "%", ".2f", None)
]

detalhes_anuais_lista = []
criterios_anuais_lista = []

for _, row in df_anuais_final.iterrows():
    motivos = []
    criterios = []
    for col_val, min_v, max_v, nome, unit, fmt, col_min in REGRAS_ANUAIS_DETALHADAS:
        val = row.get(col_val, np.nan)
        # Limite minimo por linha quando a regra depende de outra coluna (ex.: sistema de producao).
        limite_min = row.get(col_min, np.nan) if col_min else min_v
        if col_min and pd.isna(limite_min):
            limite_min = PISO_VOLUMOSO_RECEITA_PADRAO
        if pd.isna(val):
            motivos.append(f"{nome} (Ausente/NaN)")
            criterios.append(nome)
        elif limite_min is not None and val <= limite_min:
            # Sem citar o sistema, "Abaixo do mín <2" nao explica por que a fazenda vizinha passou.
            sufixo_sistema = f" para {row.get('production_system_annual')}" if col_min else ""
            motivos.append(f"{nome} (Abaixo do mín <{limite_min:g}{sufixo_sistema}: {val:{fmt}}{unit})")
            criterios.append(nome)
        elif max_v is not None and val >= max_v:
            motivos.append(f"{nome} (Acima do máx >{max_v}: {val:{fmt}}{unit})")
            criterios.append(nome)
            
    detalhes_anuais_lista.append("; ".join(motivos) if motivos else "Nenhum")
    criterios_anuais_lista.append("; ".join(criterios) if criterios else "Nenhum")

df_anuais_final["annual_violated_consistency_criteria"] = criterios_anuais_lista
df_anuais_final["annual_violated_consistency_details"] = detalhes_anuais_lista

# Regra de negocio: qualquer criterio anual violado torna a janela Inconsistente, mesmo que
# os 12 meses tenham passado na consistencia mensal. O status vindo da contagem de meses
# inconsistentes (celula anuais-base-12m) continua valendo e e combinado por OU.
violou_regra_anual = df_anuais_final["annual_violated_consistency_criteria"].ne("Nenhum")
df_anuais_final["annual_consistency_id"] = (
    pd.to_numeric(df_anuais_final["annual_consistency_id"], errors="coerce").fillna(0).astype(int)
    | violou_regra_anual.astype(int)
)
df_anuais_final["annual_consistency_status"] = np.where(
    df_anuais_final["annual_consistency_id"].eq(0),
    "Consistente",
    "Inconsistente",
)

# ==============================================================================
# 12. MONTAR A TABELA FINAL
# ==============================================================================
COLUNAS_RESULTADO_ANUAL = [
    # Identificação e período #
    "id_property", "annual_period_start", "annual_period_end", "annual_period",
    "annual_consistency_status", "annual_consistency_id", "inconsistent_months_annual",
    "annual_monthly_inconsistency_details", "annual_violated_consistency_criteria", "annual_violated_consistency_details",
    "production_system_annual",
    # Produção, qualidade e mão de obra #
    "milk_produced_annual", "milk_daily_annual", "milk_revenue_liter_annual",
    "milk_lactating_cow_day_annual", "lactating_cows_total_cows_annual",
    "lactating_cows_total_cattle_annual", "milk_total_labor_day_annual",
    "lactating_cows_total_labor_annual", "fat_lactating_cow_day_annual", "protein_lactating_cow_day_annual",
    "milk_daily_total_cows_annual", "rbl_rba_percentage_annual",
    "total_labor_quantity_annual_average",
    # Área #
    "milk_hectare_activity_annual", "milk_hectare_activity_com_reserva_annual", "milk_hectare_forrageira_annual",
    "lactating_cows_hectare_activity_annual", "lactating_cows_hectare_activity_com_reserva_annual",
    "rented_area_percentage_annual", "land_value_per_hectare_annual",
    "hectares_atividade_annual_average", "hectares_atividade_com_reserva_annual_average",
    # Alimentação #
    "feeding_cost_liter_annual", "concentrate_mineral_cost_liter_annual", "feeding_cost_milk_price_annual",
    "voluminous_cost_revenue_annual", "voluminous_cost_revenue_min_annual",
    "concentrate_mineral_price_annual", "milk_concentrate_exchange_ratio_annual",
    # Custos agregados #
    "coe_activity_annual", "coe_milk_annual", "cot_annual", "ct_annual", "capital_opportunity_cost_annual",
    "aleitamento_annual", "energy_fuel_cost_annual", "total_depreciation_annual",
    # Resultado econômico #
    "gross_margin_annual", "net_margin_annual", "profit_annual", "rcma_annual", "rcma_lactating_cow_day_annual",
    "turnover_rate_annual", "profitability_annual",
    "cot_annual_activity_revenue", "hired_labor_activity_revenue",
    "concentrate_amount_total_annual_activity_revenue",
    "capital_return_rate_no_land_annual", "capital_return_rate_annual", "capital_return_rate_annual_unfiltered",
    "operating_coverage_point_annual", "total_coverage_point_annual",
    # Estoque de capital #
    "total_capital_stock_milk_daily_annual", "total_capital_stock_lactating_cow_annual",
    "capital_stock_benfeitorias_share_annual", "capital_stock_maquinas_e_equipamentos_share_annual",
    "capital_stock_animais_share_annual", "capital_stock_terra_share_annual",
    "capital_stock_forrageira_annual_average", "capital_stock_forrageira_share_annual", "forrageira_depreciation_annual",
    # Investimento (sem fonte de dados ainda) #
    "investment_animals_annual", "investment_machinery_annual", "investment_improvements_annual",
    "investment_land_annual", "investment_total_annual",
    "investment_animals_capital_stock_annual", "investment_machinery_capital_stock_annual",
    "investment_improvements_capital_stock_annual", "investment_total_capital_stock_annual",
    "investment_revenue_annual", "investment_gross_margin_annual",
    # Auxiliares sem dado disponível #
    "consultant_name", "agroindustry_code",
    "filter_1", "filter_2", "filter_3", "filter_4", "monthly_status_annual",
    "lab_tests", "milk_quality_notes", "outsourcing", "transport", "vaccines",
]
# Mantém apenas as colunas que de fato existem em df_anuais_final (evita KeyError se alguma dependência faltar).
COLUNAS_RESULTADO_ANUAL = [c for c in COLUNAS_RESULTADO_ANUAL if c in df_anuais_final.columns]

# Adiciona a grade sistemática de RB/COE/COT/CT/Margens/Lucro por litro, %, VL, área etc.
for prefixo in MEDIDAS_ECONOMICAS_ANUAIS.values():
    for sufixo in [
        "liter_annual", "milk_revenue_percentage_annual", "milk_equivalent_liters_annual",
        "lactating_cow_annual", "total_cows_annual", "hectare_activity_annual", "hectare_activity_com_reserva_annual",
    ]:
        nome_coluna = f"{prefixo}_{sufixo}"
        if nome_coluna in df_anuais_final.columns and nome_coluna not in COLUNAS_RESULTADO_ANUAL:
            COLUNAS_RESULTADO_ANUAL.append(nome_coluna)

COLUNAS_RESULTADO_ANUAL += [f"{c}_annual_weighted_average" for c in COLUNAS_QUALIDADE if f"{c}_annual_weighted_average" in df_anuais_final.columns]
COLUNAS_DIMENSAO_ANUAL = [c for c in ["property_name", "labor_rural_code", "entrepreneur_name", "agroindustry_name", "dairy_region", "property_status"] if c in df_anuais_final.columns]
df_indicadores_anuais = df_anuais_final[COLUNAS_DIMENSAO_ANUAL + COLUNAS_RESULTADO_ANUAL].copy()

# Rótulo combinado fazenda - produtor, equivalente ao antigo "fazenda-produtor".
if {"property_name", "entrepreneur_name"}.issubset(df_indicadores_anuais.columns):
    df_indicadores_anuais.insert(
        0,
        "property_entrepreneur_label",
        df_indicadores_anuais["property_name"].fillna("") + " - " + df_indicadores_anuais["entrepreneur_name"].fillna(""),
    )

print(f"Colunas no indicador anual: {df_indicadores_anuais.shape[1]:,}")
display(df_indicadores_anuais.head())


Colunas no indicador anual: 155


,property_entrepreneur_label,property_name,labor_rural_code,entrepreneur_name,agroindustry_name,dairy_region,property_status,id_property,annual_period_start,annual_period_end,...,profit_milk_revenue_percentage_annual,profit_milk_equivalent_liters_annual,profit_lactating_cow_annual,profit_total_cows_annual,profit_hectare_activity_annual,profit_hectare_activity_com_reserva_annual,ccs_annual_weighted_average,cpp_annual_weighted_average,fat_annual_weighted_average,protein_annual_weighted_average
0,Fazenda São Bartolomeu - EDGAR JOSE DE AZEVEDO...,Fazenda São Bartolomeu,LR01964,EDGAR JOSE DE AZEVEDO NETO,NESTLÉ,Ibiá - 1215,active_approved,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-04-01,2026-03-01,...,14.085401,109258.809988,3322.002911,2859.835558,11084.199254,10593.968552,213.352095,30.293208,3.462351,3.260178
1,Fazenda São Bartolomeu - EDGAR JOSE DE AZEVEDO...,Fazenda São Bartolomeu,LR01964,EDGAR JOSE DE AZEVEDO NETO,NESTLÉ,Ibiá - 1215,active_approved,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-05-01,2026-04-01,...,16.073836,130765.358708,3963.542809,3408.312340,13070.712370,12497.988991,220.837780,29.223258,3.462035,3.262939
2,Fazenda São Bartolomeu - EDGAR JOSE DE AZEVEDO...,Fazenda São Bartolomeu,LR01964,EDGAR JOSE DE AZEVEDO NETO,NESTLÉ,Ibiá - 1215,active_approved,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-06-01,2026-05-01,...,15.760783,128665.358039,3860.444356,3321.021996,12645.429910,12096.437736,234.413265,27.827960,3.450289,3.260002
3,Fazenda São Bartolomeu - EDGAR JOSE DE AZEVEDO...,Fazenda São Bartolomeu,LR01964,EDGAR JOSE DE AZEVEDO NETO,NESTLÉ,Ibiá - 1215,active_approved,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-07-01,2026-06-01,...,17.302215,140455.204916,4214.838578,3612.718781,13648.048728,13060.928877,238.350990,20.369936,3.450242,3.245073
4,Fazenda São Bartolomeu - EDGAR JOSE DE AZEVEDO...,Fazenda São Bartolomeu,LR01964,EDGAR JOSE DE AZEVEDO NETO,NESTLÉ,Ibiá - 1215,active_approved,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-08-01,2026-07-01,...,16.541285,131732.744066,3981.947768,3394.993993,12747.241596,12203.825950,248.241880,17.792448,3.461203,3.246817


## 4.3 Padronização e Mapeamento de Colunas Anuais


In [51]:
# ==============================================================================
# 4.3 PADRONIZAÇÃO E MAPEAMENTO DE COLUNAS ANUAIS
# ==============================================================================
# Rótulo combinado fazenda - produtor diretamente em df_anuais_final (mesma lógica
# já usada para df_indicadores_anuais), para poder ser puxado pelo mapa abaixo.
if {"property_name", "entrepreneur_name"}.issubset(df_anuais_final.columns) and "property_entrepreneur_label" not in df_anuais_final.columns:
    df_anuais_final["property_entrepreneur_label"] = (
        df_anuais_final["property_name"].fillna("") + " - " + df_anuais_final["entrepreneur_name"].fillna("")
    )

# Mesma ordenação usada na exportação técnica, para que a planilha com nomes
# antigos saia com propriedade/período em ordem crescente.
df_anuais_final = df_anuais_final.sort_values(["id_property", "annual_period_end"]).reset_index(drop=True)

# Mapa (nome técnico antigo -> nome técnico atual em df_anuais_final).
# None indica que o indicador ainda não tem fonte de dados no modelo novo
# (Investimento, Estoque de capital em forrageiras, Consultor etc.) - a
# coluna final é criada mesmo assim, vazia, para manter a mesma estrutura
# do arquivo antigo até que essas fontes existam.

MAPA_NOME_ANTIGO_PARA_ATUAL = {
    "idFazenda": "id_property",
    "fazenda-produtor": "property_entrepreneur_label",
    "codAgroindustria": "labor_rural_code",
    "regiaoLeiteira": "dairy_region",
    "nomeagroindustria": "agroindustry_name",
    "nomeconsultor": "consultant_name",
    "intervalo_movel": "annual_period",
    "consistenciaAnual": "annual_consistency_status",
    "Status_Mensais": "monthly_status_annual",
    "sistema": "production_system_annual",
    "Filtro1": "filter_1",
    "Filtro2": "filter_2",
    "Filtro3": "filter_3",
    "Filtro4": "filter_4",
    "taxaRetornoCapitalComTerra_Geral": "capital_return_rate_annual_unfiltered",
    "areaAtividadeSemReserva_mediaMovel": "hectares_atividade_annual_average",
    "areaAtividadeReserva_mediaMovel": "hectares_atividade_com_reserva_annual_average",
    "areaPropria_mediaMovel": "hectares_propria_annual_average",
    "areaArrendada_mediaMovel": "hectares_arrendada_annual_average",
    "percentualAreaArrendada": "rented_area_percentage_annual",
    "precoTerraNua_somaMovel": "land_value_per_hectare_annual",
    "qtdeVacasEmLactacao_mediaMovel": "lactating_cows_annual_average",
    "totalVacas_mediaMovel": "total_cows_annual_average",
    "totalAnimais_mediaMovel": "total_cattle_annual_average",
    "quantidade_Concentrado_Minerais_Anual": "concentrate_mineral_consumed_quantity_annual",
    "MDOTotalDiaria_mediaMovel": "total_labor_quantity_annual_average",
    "MDOFamiliarDiaria_mediaMovel": "family_labor_quantity_annual_average",
    "MDOContratadaDiaria_mediaMovel": "hired_labor_quantity_annual_average",
    "CCS_mediaMovel": "ccs_annual_weighted_average",
    "CPP_mediaMovel": "cpp_annual_weighted_average",
    "Gordura_mediaMovel": "fat_annual_weighted_average",
    "Proteina_mediaMovel": "protein_annual_weighted_average",
    "absCCS_somaMovel": "abs_ccs_annual",
    "absCPP_somaMovel": "abs_cpp_annual",
    "absGordura_somaMovel": "abs_fat_annual",
    "absProteina_somaMovel": "abs_protein_annual",
    "Gordura_VL": "fat_lactating_cow_day_annual",
    "Proteina_VL": "protein_lactating_cow_day_annual",
    "VL_TV_100": "lactating_cows_total_cows_annual",
    "VL_TA_100": "lactating_cows_total_cattle_annual",
    "VL_areaSemReserva": "lactating_cows_hectare_activity_annual",
    "VL_areaComReserva": "lactating_cows_hectare_activity_com_reserva_annual",
    "VL_MDO": "lactating_cows_total_labor_annual",
    "leiteProduzido_somaMovel": "milk_produced_annual",
    "consumoLeiteDescartado_somaMovel": "discarded_quantity_annual",
    "producaoDiaria": "milk_daily_annual",
    "producaoDiaria_VL": "milk_lactating_cow_day_annual",
    "producaoDiaria_TV": "milk_daily_total_cows_annual",
    "producaoDiaria_MDO": "milk_total_labor_day_annual",
    "producaoAnual_AreaSemReserva": "milk_hectare_activity_annual",
    "producaoAnual_AreaComReserva": "milk_hectare_activity_com_reserva_annual",
    "producaoAnual_AreaForrageira": "milk_hectare_forrageira_annual",
    "areaForrageira_mediaMovel": "hectares_forrageira_annual_average",
    "def_rendaAtividade_somaMovel": "total_activity_revenue_annual",
    "def_rendaLeite_somaMovel": "total_milk_revenue_annual",
    "precoLeiteAnual": "milk_revenue_liter_annual",
    "precoConcentradoAnual": "concentrate_mineral_price_annual",
    "relacaoTroca": "milk_concentrate_exchange_ratio_annual",
    "estoqueCapital_semTerra_mediaMovel": "capital_stock_no_land_annual_average",
    "estoqueCapital_comTerra_mediaMovel": "total_capital_stock_annual_average",
    "def_estoqueCapitalBenfeitorias_mediaMovel": "monthly_average_capital_stock_benfeitorias_annual_average",
    "def_estoqueCapitalMaquinas_mediaMovel": "monthly_average_capital_stock_maquinas_e_equipamentos_annual_average",
    "estoqueCapitalAnimais_Mensal_mediaMovel": "animal_capital_stock_annual_average",
    "def_estoqueTerra_Mensal_mediaMovel": "land_capital_stock_annual_average",
    "estoqueCapitalPlantio_acumulado_mediaMovel": "capital_stock_forrageira_annual_average",
    "def_estoqueCapitalBenfeitorias_mediaMovel_ECTotalcomTerra": "capital_stock_benfeitorias_share_annual",
    "def_estoqueCapitalMaquinas_mediaMovel_ECTotalcomTerra": "capital_stock_maquinas_e_equipamentos_share_annual",
    "estoqueCapitalAnimais_Mensal_mediaMovel_ECTotalcomTerra": "capital_stock_animais_share_annual",
    "def_estoqueTerra_Mensal_mediaMovel_ECTotalcomTerra": "capital_stock_terra_share_annual",
    "estoqueCapitalPlantio_acumulado_mediaMovel_ECTotalcomTerra": "capital_stock_forrageira_share_annual",
    "coe_somaMovel_litrosEquivalente": "coe_milk_equivalent_liters_annual",
    "cotAnual_litrosEquivalente": "cot_milk_equivalent_liters_annual",
    "ctAnual_litrosEquivalente": "ct_milk_equivalent_liters_annual",
    "coe_somaMovel_precoLeite": "coe_milk_revenue_percentage_annual",
    "cotAnual_precoLeite": "cot_milk_revenue_percentage_annual",
    "ctAnual_precoLeite": "ct_milk_revenue_percentage_annual",
    "margemBrutaAnual": "gross_margin_annual",
    "margemBrutaAnual_litro": "gross_margin_liter_annual",
    "margemBrutaAnual_AreaSemReserva": "gross_margin_hectare_activity_annual",
    "margemBrutaAnual_AreaComReserva": "gross_margin_hectare_activity_com_reserva_annual",
    "margemBrutaAnual_VL": "gross_margin_lactating_cow_annual",
    "margemBrutaAnual_TotalVacas": "gross_margin_total_cows_annual",
    "margemLiquidaAnual": "net_margin_annual",
    "margemLiquidaAnual_litro": "net_margin_liter_annual",
    "margemLiquidaAnual_AreaSemReserva": "net_margin_hectare_activity_annual",
    "margemLiquidaAnual_AreaComReserva": "net_margin_hectare_activity_com_reserva_annual",
    "margemLiquidaAnual_VL": "net_margin_lactating_cow_annual",
    "margemLiquidaAnual_TotalVacas": "net_margin_total_cows_annual",
    "lucroAnual": "profit_annual",
    "lucroAnual_litro": "profit_liter_annual",
    "RCMA": "rcma_annual",
    "RCMA_VL": "rcma_lactating_cow_day_annual",
    "rbl_rba": "rbl_rba_percentage_annual",
    "estoqueCapital_comTerra_mediaMovel_VL": "total_capital_stock_lactating_cow_annual",
    "estoqueCapital_comTerra_mediaMovel_litro": "total_capital_stock_milk_daily_annual",
    "investimentoAnimais_ECAnimais": "investment_animals_capital_stock_annual",
    "investimentoMaquinas_ECMaquinas": "investment_machinery_capital_stock_annual",
    "investimentoBenfeitorias_ECBenfeitorias": "investment_improvements_capital_stock_annual",
    "investimento_ECcomTerra": "investment_total_capital_stock_annual",
    "investimento_RBA": "investment_revenue_annual",
    "investimento_MB": "investment_gross_margin_annual",
    "investimentoAnimais_somaMovel": "investment_animals_annual",
    "investimentoMaquinas_somaMovel": "investment_machinery_annual",
    "investimentoBenfeitorias_somaMovel": "investment_improvements_annual",
    "investimento_somaMovel": "investment_total_annual",
    "taxadegiro": "turnover_rate_annual",
    "lucratividade": "profitability_annual",
    "taxaRetornoCapitalSemTerra": "capital_return_rate_no_land_annual",
    "taxaRetornoCapitalComTerra": "capital_return_rate_annual",
    "pcot": "operating_coverage_point_annual",
    "pct": "total_coverage_point_annual",
    "coe_somaMovel": "coe_activity_annual",
    "def_gastoAcessoriosDespesasGerais_somaMovel": "general_expenses_annual",
    "def_aleitaento_somaMovel": "aleitamento_annual",
    "def_gastoArrendamento_somaMovel": "land_lease_annual",
    "custoAlimentacao_Concentrado_Minerais_somaMovel": "concentrate_mineral_cost_annual",
    "custoReceitasConcentrado_somaMovel": "concentrated_sold_annual",
    "def_gastoAdministrativo_somaMovel": "administration_annual",
    "def_gastoAssistenciaTecnica_somaMovel": "technical_assistance_annual",
    "custoEnergiaCombustivel": "energy_fuel_cost_annual",
    "Exames_Laboratoriais": "lab_tests",
    "def_gastoHormonios_somaMovel": "hormones_annual",
    "def_gastoImpostoTaxas_somaMovel": "taxes_fees_annual",
    "custoMDOContratada_somaMovel": "hired_labor_expenses_annual",
    "def_gastoMaterialOrdenha_somaMovel": "milking_material_annual",
    "def_gastoMedicamentosVacinas_somaMovel": "medicines_vaccines_annual",
    "Qualidade_Leite": "milk_quality_notes",
    "def_gastoReparosConsertos_somaMovel": "repairs_annual",
    "def_gastoReproducao_somaMovel": "reproduction_annual",
    "Terceirizacao": "outsourcing",
    "Transporte": "transport",
    "Vacinas": "vaccines",
    "custoAlimentacao_Volumoso_somaMovel": "voluminous_amount_total_annual",
    "custoReceitasVolumoso_somaMovel": "voluminous_sold_annual",
    "cotAnual": "cot_annual",
    "depreciacaoEstoqueCapital_somaMovel": "total_depreciation_annual",
    "def_familiarValortotal_somaMovel": "family_labor_expenses_annual",
    "ctAnual": "ct_annual",
    "custoOportunidadeCapital": "capital_opportunity_cost_annual",
}

# ==============================================================================
# ORDEM E NOMES ANTIGOS (idênticos ao colunas_finais/novos_nomes do notebook antigo)
# ==============================================================================
COLUNAS_ANTIGAS_EM_ORDEM = [
    "idFazenda", "fazenda-produtor", "codAgroindustria", "regiaoLeiteira", "nomeagroindustria",
    "nomeconsultor", "intervalo_movel", "consistenciaAnual", "Status_Mensais", "sistema",
    "Filtro1", "Filtro2", "Filtro3", "Filtro4", "taxaRetornoCapitalComTerra_Geral",
    "areaAtividadeSemReserva_mediaMovel", "areaAtividadeReserva_mediaMovel", "areaPropria_mediaMovel",
    "areaArrendada_mediaMovel", "percentualAreaArrendada", "precoTerraNua_somaMovel",
    "qtdeVacasEmLactacao_mediaMovel", "totalVacas_mediaMovel", "totalAnimais_mediaMovel",
    "quantidade_Concentrado_Minerais_Anual",
    "MDOTotalDiaria_mediaMovel", "MDOFamiliarDiaria_mediaMovel", "MDOContratadaDiaria_mediaMovel",
    "CCS_mediaMovel", "CPP_mediaMovel", "Gordura_mediaMovel", "Proteina_mediaMovel", "absCCS_somaMovel",
    "absCPP_somaMovel", "absGordura_somaMovel", "absProteina_somaMovel", "Gordura_VL", "Proteina_VL",
    "VL_TV_100", "VL_TA_100", "VL_areaSemReserva", "VL_areaComReserva", "VL_MDO",
    "leiteProduzido_somaMovel", "consumoLeiteDescartado_somaMovel", "producaoDiaria", "producaoDiaria_VL", "producaoDiaria_TV",
    "producaoDiaria_MDO", "producaoAnual_AreaSemReserva", "producaoAnual_AreaComReserva", "producaoAnual_AreaForrageira", "areaForrageira_mediaMovel",
    "def_rendaAtividade_somaMovel", "def_rendaLeite_somaMovel", "precoLeiteAnual", "def_rendaLeite_somaMovel",
    "precoConcentradoAnual", "relacaoTroca",
    "estoqueCapital_semTerra_mediaMovel", "estoqueCapital_comTerra_mediaMovel", "def_estoqueCapitalBenfeitorias_mediaMovel",
    "def_estoqueCapitalMaquinas_mediaMovel", "estoqueCapitalAnimais_Mensal_mediaMovel", "def_estoqueTerra_Mensal_mediaMovel", "estoqueCapitalPlantio_acumulado_mediaMovel",
    "def_estoqueCapitalBenfeitorias_mediaMovel_ECTotalcomTerra", "def_estoqueCapitalMaquinas_mediaMovel_ECTotalcomTerra", "estoqueCapitalAnimais_Mensal_mediaMovel_ECTotalcomTerra",
    "def_estoqueTerra_Mensal_mediaMovel_ECTotalcomTerra", "estoqueCapitalPlantio_acumulado_mediaMovel_ECTotalcomTerra",
    "coe_somaMovel_litrosEquivalente", "cotAnual_litrosEquivalente", "ctAnual_litrosEquivalente",
    "coe_somaMovel_precoLeite", "cotAnual_precoLeite", "ctAnual_precoLeite",
    "margemBrutaAnual", "margemBrutaAnual_litro", "margemBrutaAnual_AreaSemReserva", "margemBrutaAnual_AreaComReserva", "margemBrutaAnual_VL", "margemBrutaAnual_TotalVacas",
    "margemLiquidaAnual", "margemLiquidaAnual_litro", "margemLiquidaAnual_AreaSemReserva", "margemLiquidaAnual_AreaComReserva", "margemLiquidaAnual_VL", "margemLiquidaAnual_TotalVacas",
    "lucroAnual", "lucroAnual_litro",
    "RCMA", "RCMA_VL",
    "rbl_rba",
    "estoqueCapital_comTerra_mediaMovel_VL", "estoqueCapital_comTerra_mediaMovel_litro",
    "investimentoAnimais_ECAnimais", "investimentoMaquinas_ECMaquinas", "investimentoBenfeitorias_ECBenfeitorias", "investimento_ECcomTerra",
    "investimento_RBA", "investimento_MB", "investimentoAnimais_somaMovel", "investimentoMaquinas_somaMovel", "investimentoBenfeitorias_somaMovel", "investimento_somaMovel",
    "taxadegiro", "lucratividade", "taxaRetornoCapitalSemTerra", "taxaRetornoCapitalComTerra", "pcot", "pct",
    "coe_somaMovel", "def_gastoAcessoriosDespesasGerais_somaMovel", "def_aleitaento_somaMovel", "def_gastoArrendamento_somaMovel",
    "custoAlimentacao_Concentrado_Minerais_somaMovel", "custoReceitasConcentrado_somaMovel", "def_gastoAdministrativo_somaMovel",
    "def_gastoAssistenciaTecnica_somaMovel", "custoEnergiaCombustivel", "Exames_Laboratoriais", "def_gastoHormonios_somaMovel", "def_gastoImpostoTaxas_somaMovel",
    "custoMDOContratada_somaMovel", "def_gastoMaterialOrdenha_somaMovel", "def_gastoMedicamentosVacinas_somaMovel", "Qualidade_Leite", "def_gastoReparosConsertos_somaMovel",
    "def_gastoReproducao_somaMovel", "Terceirizacao", "Transporte", "Vacinas", "custoAlimentacao_Volumoso_somaMovel", "custoReceitasVolumoso_somaMovel",
    "cotAnual", "depreciacaoEstoqueCapital_somaMovel", "def_familiarValortotal_somaMovel", "ctAnual", "custoOportunidadeCapital",
]

NOMES_PORTUGUES_EM_ORDEM = [
    "IDFazenda", "Fazenda - Produtor", "Código LR", "Região", "Agroindústria",
    "Consultor", "Período", "Status - Ind. Anuais", "Status - Ind. Mensais", "Sistema de produção atual",
    "Filtro 1", "Filtro 2", "Filtro 3", "Filtro 4", "AUXILIAR PARA ESTRATIFICAÇÃO - TRCCT CALCULADA",
    "Área destinada à atividade (hectare)", "Área destinada à atividade considerando reserva (hectare)", "Área própria considerando reserva (hectare)",
    "Área arrendada considerando reserva (hectare)", "Percentual de área arrendada (%)", "Preço médio da terra própria (R$/hectare)",
    "Vacas em lactação (animais/mês)", "Total de vacas (animais/mês)", "Total de animais (animais/mês)",
    "Consumo de concentrado anual (Kg/Ano)",
    "Mão de obra total (trabalhador)", "Mão de obra familiar (trabalhador)", "Mão de obra contratada (trabalhador)",
    "CCS (Contagem de células somáticas) (x1000 células/ml)", "CPP (Contagem padrão em placas) (x1000 UFC/ml)", "Gordura (%)", "Proteína (%)",
    "AUXILIAR PARA MÉDIAS - CCS (Contagem de células somáticas) (x1000 células/ml)", "AUXILIAR PARA MÉDIAS - CPP (Contagem padrão em placas) (x1000 UFC/ml)",
    "AUXILIAR PARA MÉDIAS - Gordura (%)", "AUXILIAR PARA MÉDIAS - Proteína (%)", "Gordura (kg/vaca em lactação/dia)", "Proteína (kg/vaca em lactação/dia)",
    "Vacas em lactação/total de vacas (%)", "Vacas em lactação/total de animais (%)",
    "Vacas em lactação/área destinada à atividade (animais/hectare)", "Vacas em lactação/área destinada à atividade considerando reserva (animais/hectare)",
    "Vacas em lactação/mão de obra total (animais/trabalhador/dia)",
    "Produção anual de leite (litros/ano)", "Leite descartado (litros/ano)", "Produção diária de leite (litros/dia)",
    "Produção/vacas em lactação (litros/animal/dia)", "Produção/total de vacas (litros/animal/dia)",
    "Produção/mão de obra total (litros/trabalhador/dia)", "Produção/área destinada à atividade (litros/hectare/ano)",
    "Produção/área destinada à atividade considerando reserva (litros/hectare/ano)", "Produção / área de produção de forrageira (litros/hectare/ano)",
    "AUXILIAR PARA MÉDIAS - Produção / área de produção de forrageira (litros/hectare/ano)",
    "Renda bruta da atividade leiteira (R$/ano)", "Renda bruta do leite (R$/ano)", "Preço médio do leite (R$/litro)",
    "AUXILIAR PARA MÉDIAS - Preço médio do leite (R$/litro)", "Preço médio do concentrado (R$/Kg)", "Relação de troca leite/concentrado (Kg/L)",
    "Estoque de capital total sem terra (R$)", "Estoque de capital total com terra (R$)", "Estoque de capital em benfeitorias (R$)",
    "Estoque de capital em máquinas (R$)", "Estoque de capital em animais (R$)", "Estoque de capital em terra (R$)", "Estoque de capital em forrageiras não-anuais (R$)",
    "Estoque de capital em benfeitorias/estoque de capital total com terra (%)", "Estoque de capital em máquinas/estoque de capital total com terra (%)",
    "Estoque de capital em animais/estoque de capital total com terra (%)", "Estoque de capital em terra/estoque de capital total com terra (%)",
    "Estoque de capital em forrageiras não-anuais/estoque de capital total com terra (%)",
    "COE da atividade leiteira em equivalentes litros de leite (litros/ano)", "COT da atividade leiteira em equivalentes litros de leite (litros/ano)",
    "CT da atividade leiteira em equivalentes litros de leite (litros/ano)",
    "COE do leite/preço do leite (%)", "COT do leite/preço do leite (%)", "CT do leite/preço do leite (%)",
    "Margem bruta da atividade (R$/ano)", "Margem bruta unitária (R$/litro)", "Margem bruta/área destinada à atividade (R$/hectare/ano)",
    "Margem bruta/área destinada à atividade considerando reserva (R$/hectare/ano)", "Margem bruta/vacas em lactação (R$/animal/ano)", "Margem bruta/total de vacas (R$/animal/ano)",
    "Margem líquida da atividade (R$/ano)", "Margem líquida unitária (R$/litro)", "Margem líquida/área destinada à atividade (R$/hectare/ano)",
    "Margem líquida/área destinada à atividade considerando reserva (R$/hectare/ano)", "Margem líquida/vacas em lactação (R$/animal/ano)", "Margem líquida/total de vacas (R$/animal/ano)",
    "Lucro total (R$/ano)", "Lucro unitário (R$/litro)",
    "RMCA (Receita Menos Custo com Alimentação) (R$/ano)", "RMCA (Receita Menos Custo com Alimentação) (R$/vaca em lactação/dia)",
    "Renda do leite/renda atividade (%)",
    "Estoque de capital total com terra/vaca em lactação (R$/animal)", "Estoque de capital total com terra/produção diária de leite (R$/litro/dia)",
    "Investimento em animais/estoque de capital em animais (%)", "Investimento em máquinas e equipamento/estoque de capital em máquinas e equipamentos (%)",
    "Investimento em benfeitorias/estoque de capital em benfeitorias (%)", "Investimento anual/estoque de capital total com terra (%)",
    "Investimento anual/renda bruta da atividade (%)", "Investimento anual/margem bruta (%)",
    "AUXILIAR PARA MÉDIAS - Investimento em animais", "AUXILIAR PARA MÉDIAS - Investimento em máquinas e equipamento",
    "AUXILIAR PARA MÉDIAS - Investimento em benfeitorias", "AUXILIAR PARA MÉDIAS - Investimento anual",
    "Taxa de giro do estoque de capital total (%)", "Lucratividade operacional (%)",
    "Taxa de remuneração do capital sem terra (% ao ano)", "Taxa de remuneração do capital com terra (% ao ano)",
    "Ponto de cobertura operacional total da atividade (litros/dia)", "Ponto de cobertura total da atividade (litros/dia)",
    "Custo operacional efetivo - R$/ano (Atividade)", "Acessórios e despesas em geral - R$/ano (Atividade)", "Aleitamento - R$/ano (Atividade)",
    "Arrendamento/Aluguel - R$/ano (Atividade)", "Concentrados e Minerais de ingestão livre - R$/ano (Atividade)", "Concentrados (Vendido) - R$/ano (Atividade)",
    "Despesas administrativas - R$/ano (Atividade)", "Despesas com assistência técnica - R$/ano (Atividade)", "Energia e combustível - R$/ano (Atividade)",
    "Exames laboratoriais - R$/ano (Atividade)", "Hormônios - R$/ano (Atividade)", "Impostos e taxas - R$/ano (Atividade)",
    "Mão de obra contratada - R$/ano (Atividade)", "Material de ordenha - R$/ano (Atividade)", "Medicamentos - R$/ano (Atividade)",
    "Qualidade do leite - R$/ano (Atividade)", "Reparos e consertos de máquinas e benfeitorias - R$/ano (Atividade)", "Reprodução - R$/ano (Atividade)",
    "Terceirização de recria - R$/ano (Atividade)", "Transporte e descontos no leite - R$/ano (Atividade)", "Vacinas - R$/ano (Atividade)",
    "Volumosos - R$/ano (Atividade)", "Volumosos (Vendido) - R$/ano (Atividade)",
    "Custo operacional total - R$/ano (Atividade)", "Depreciação - R$/ano (Atividade)", "Mão de obra familiar - R$/ano (Atividade)",
    "Custo total - R$/ano (Atividade)", "Remuneração do capital - R$/ano (Atividade)",
]

assert len(COLUNAS_ANTIGAS_EM_ORDEM) == len(NOMES_PORTUGUES_EM_ORDEM) == 140, (
    "As listas de nomes antigos e novos precisam ter 140 posições cada, igual ao notebook original."
)

# ==============================================================================
# MONTAR A TABELA COM OS NOMES/ORDEM DO CALCULO_MEDIAS.XLSX ANTIGO
# ==============================================================================
dados_renomeados = {}
colunas_sem_correspondencia = []

for nome_antigo, nome_pt in zip(COLUNAS_ANTIGAS_EM_ORDEM, NOMES_PORTUGUES_EM_ORDEM):

    nome_atual = MAPA_NOME_ANTIGO_PARA_ATUAL.get(nome_antigo)

    if nome_atual is not None and nome_atual in df_anuais_final.columns:
        valores = df_anuais_final[nome_atual]
    else:
        valores = pd.Series([None] * len(df_anuais_final), index=df_anuais_final.index)
        colunas_sem_correspondencia.append((nome_antigo, nome_pt))

    # "AUXILIAR PARA MÉDIAS - Preço médio do leite" repete a mesma coluna de origem
    # que "Renda bruta do leite (R$/ano)" - isso é intencional (auxiliar de recálculo
    # de médias ponderadas no BI), igual ao notebook antigo.
    if nome_pt in dados_renomeados:
        nome_pt = f"{nome_pt} (2)"

    dados_renomeados[nome_pt] = valores.values

df_calculo_medias = pd.DataFrame(dados_renomeados, index=df_anuais_final.index)

print(f"Colunas mapeadas com dado real: {140 - len(colunas_sem_correspondencia)} / 140")
print(f"Colunas sem correspondência no modelo novo (ficam vazias): {len(colunas_sem_correspondencia)}")
for nome_antigo, nome_pt in colunas_sem_correspondencia:
    print(f"  - {nome_antigo} -> {nome_pt}")

display(df_calculo_medias.head())

Colunas mapeadas com dado real: 140 / 140
Colunas sem correspondência no modelo novo (ficam vazias): 0


,IDFazenda,Fazenda - Produtor,Código LR,Região,Agroindústria,Consultor,Período,Status - Ind. Anuais,Status - Ind. Mensais,Sistema de produção atual,...,Terceirização de recria - R$/ano (Atividade),Transporte e descontos no leite - R$/ano (Atividade),Vacinas - R$/ano (Atividade),Volumosos - R$/ano (Atividade),Volumosos (Vendido) - R$/ano (Atividade),Custo operacional total - R$/ano (Atividade),Depreciação - R$/ano (Atividade),Mão de obra familiar - R$/ano (Atividade),Custo total - R$/ano (Atividade),Remuneração do capital - R$/ano (Atividade)
0,004cde8f-4688-4b22-acd9-76151b7dcc7a,Fazenda São Bartolomeu - EDGAR JOSE DE AZEVEDO...,LR01964,Ibiá - 1215,NESTLÉ,LORENA VIRGINIA ARAUJO,Apr/25-Mar/26,Inconsistente,None,UNSTRUCTURED_CONFINMENT,...,None,None,None,294913.867972,0.0,1.724020e+06,14302.919536,89511.835823,1.724020e+06,NaN
1,004cde8f-4688-4b22-acd9-76151b7dcc7a,Fazenda São Bartolomeu - EDGAR JOSE DE AZEVEDO...,LR01964,Ibiá - 1215,NESTLÉ,LORENA VIRGINIA ARAUJO,May/25-Apr/26,Inconsistente,None,UNSTRUCTURED_CONFINMENT,...,None,None,None,288551.240497,0.0,1.757334e+06,14312.386657,86556.710096,1.757334e+06,NaN
2,004cde8f-4688-4b22-acd9-76151b7dcc7a,Fazenda São Bartolomeu - EDGAR JOSE DE AZEVEDO...,LR01964,Ibiá - 1215,NESTLÉ,LORENA VIRGINIA ARAUJO,Jun/25-May/26,Inconsistente,None,UNSTRUCTURED_CONFINMENT,...,None,None,None,283710.323871,0.0,1.757290e+06,14342.865422,83317.839230,1.757290e+06,NaN
3,004cde8f-4688-4b22-acd9-76151b7dcc7a,Fazenda São Bartolomeu - EDGAR JOSE DE AZEVEDO...,LR01964,Ibiá - 1215,NESTLÉ,LORENA VIRGINIA ARAUJO,Jul/25-Jun/26,Inconsistente,None,UNSTRUCTURED_CONFINMENT,...,None,None,None,279399.751441,0.0,1.712348e+06,14385.378230,79972.655412,1.712348e+06,NaN
4,004cde8f-4688-4b22-acd9-76151b7dcc7a,Fazenda São Bartolomeu - EDGAR JOSE DE AZEVEDO...,LR01964,Ibiá - 1215,NESTLÉ,LORENA VIRGINIA ARAUJO,Aug/25-Jul/26,Inconsistente,None,UNSTRUCTURED_CONFINMENT,...,None,None,None,275187.752604,0.0,1.704373e+06,14418.180176,76665.093449,1.704373e+06,NaN


## 4.4 Análise de Outliers dos Indicadores Anuais (Média ± 2 Desvios Padrão)

Diagnóstico **relativo** e **estritamente aditivo**: marca a fazenda que está dentro dos
limites fixos da 4.2 e, ainda assim, fora do padrão do grupo naquela janela anual.

- Método: `média ± 2 · desvio padrão amostral (ddof=1)`, agrupado por `annual_period`.
- Base dos parâmetros: `filter_1 == 1` (uma linha por fazenda-período) e
  `annual_consistency_id == 0` (dado ruim não define o próprio limite).
- Classificação aplicada a **todas** as linhas anuais, inclusive inconsistentes e
  duplicadas por consultor: nenhuma fazenda fica sem diagnóstico.
- Períodos com menos de 30 fazendas não geram parâmetro; suas linhas saem como
  `Amostra insuficiente`.

Nada de consistência muda aqui (`consistency_id`, `annual_consistency_id` e os status
correspondentes são apenas lidos), o `Cálculo de Médias` não é tocado e as colunas novas
não sobem para o Supabase. A seção produz três DataFrames que viram abas novas na 4.5:
**Outliers**, **Parâmetros de Outlier** e **Outliers por Consultor**.

In [52]:
# ==============================================================================
# 4.4 ANÁLISE DE OUTLIERS DOS INDICADORES ANUAIS (MÉDIA ± 2 DESVIOS PADRÃO)
# ==============================================================================
# Diagnóstico RELATIVO e ADITIVO: marca a fazenda que está dentro dos limites
# fixos da 4.2 e, ainda assim, fora do padrão do grupo naquela janela anual.
#
# Nada de consistência muda aqui: consistency_id, annual_consistency_id e os
# status correspondentes são apenas LIDOS. A célula recebe df_indicadores_anuais
# e devolve o mesmo DataFrame com colunas novas + três DataFrames de saída.
#
# Método (literal de propósito, para ser refeito à mão no Excel):
#   limite = média ± MULTIPLICADOR_DP_OUTLIER * desvio padrão amostral (ddof=1)
#   agrupado por annual_period, sobre filter_1 == 1 e annual_consistency_id == 0.
# ==============================================================================

# Configuração declarativa única dos indicadores cobertos, no mesmo padrão de
# REGRAS_ANUAIS_DETALHADAS: coluna -> (rótulo em português, unidade, formato).
# Acrescentar ou remover um indicador é editar SÓ esta estrutura.
INDICADORES_OUTLIER = {
    "concentrate_mineral_price_annual":         ("Preço médio do concentrado", "R$/kg", ".2f"),
    "net_margin_liter_annual":                  ("Margem líquida unitária", "R$/litro", ".2f"),
    "milk_total_labor_day_annual":              ("Produção/mão de obra total", "litros/trabalhador/dia", ".1f"),
    "milk_hectare_activity_annual":             ("Produção/área destinada à atividade", "litros/hectare/ano", ".0f"),
    "milk_hectare_activity_com_reserva_annual": ("Produção/área destinada à atividade considerando reserva", "litros/hectare/ano", ".0f"),
    "rbl_rba_percentage_annual":                ("Renda do leite/renda atividade", "%", ".2f"),
}

# Parâmetros nomeados de calibragem (não são literais espalhados pela lógica).
N_MINIMO_OUTLIER = 30          # mínimo de fazendas por annual_period para gerar parâmetro
MULTIPLICADOR_DP_OUTLIER = 2   # quantos desvios padrão definem os limites

# Aviso de cabeçalho do resumo por consultor: cada vínculo conta, então a soma
# das linhas excede o total de fazendas distintas.
AVISO_SOMA_CONSULTOR = (
    "ATENÇÃO: cada vínculo conta - fazenda com mais de um consultor aparece "
    "para todos eles, logo a soma desta coluna EXCEDE o total de fazendas distintas"
)


def _formatar_br(valor, formato):
    """Formata um número no padrão brasileiro (vírgula decimal, ponto de milhar)."""
    if valor is None or pd.isna(valor):
        return "-"
    texto = f"{valor:{formato}}"
    return texto.replace(",", "\x00").replace(".", ",").replace("\x00", ".")


def calcular_outliers_anuais(
    df_anual,
    indicadores=None,
    n_minimo=N_MINIMO_OUTLIER,
    multiplicador=MULTIPLICADOR_DP_OUTLIER,
):
    """
    Marca outliers (média ± N desvios padrão, por annual_period) nos indicadores anuais.

    Função pura: não faz I/O, não acessa banco/SharePoint e não usa estado global
    além da configuração recebida por parâmetro. Não altera o DataFrame de entrada.

    Parâmetros
    ----------
    df_anual : DataFrame com uma linha por fazenda-janela-consultor. Precisa de
        `annual_period`, `annual_period_end`, `id_property`, `filter_1`,
        `annual_consistency_id` e as colunas de indicador.
    indicadores : dict {coluna: (rótulo, unidade, formato)}. Default INDICADORES_OUTLIER.
    n_minimo : mínimo de fazendas por período para o período gerar parâmetro.
    multiplicador : quantos desvios padrão definem os limites.

    Retorna
    -------
    (df_anotado, df_parametros, df_outliers, df_outliers_consultor)
    """
    indicadores = INDICADORES_OUTLIER if indicadores is None else indicadores
    df = df_anual.copy()

    colunas_indicador = [c for c in indicadores if c in df.columns]
    if not colunas_indicador:
        raise KeyError("Nenhuma coluna de indicador de outlier encontrada no DataFrame anual.")

    # --------------------------------------------------------------------------
    # 1. BASE DE CÁLCULO DOS PARÂMETROS
    # --------------------------------------------------------------------------
    # filter_1 == 1  -> uma linha por fazenda-período (sem o peso duplo das
    #                   fazendas com dois ou mais consultores);
    # annual_consistency_id == 0 -> dado reconhecidamente ruim não define o
    #                   próprio limite.
    filtro_1 = pd.to_numeric(df.get("filter_1"), errors="coerce")
    consistencia = pd.to_numeric(df.get("annual_consistency_id"), errors="coerce")
    mascara_base = filtro_1.eq(1) & consistencia.eq(0)
    df_base = df.loc[mascara_base]

    # Ordem cronológica dos períodos, para a aba de parâmetros e para "janela mais recente".
    ordem_periodo = (
        df.groupby("annual_period")["annual_period_end"].max().sort_values()
    )
    posicao_periodo = {p: i for i, p in enumerate(ordem_periodo.index)}

    # Períodos com fazendas suficientes. Como a base tem uma linha por fazenda-período,
    # contar linhas é contar fazendas. Períodos com uma única fazenda (DP indefinido)
    # caem por este mesmo caminho, sem quebrar a execução.
    fazendas_por_periodo = df_base.groupby("annual_period")["id_property"].nunique()
    periodos_validos = set(fazendas_por_periodo[fazendas_por_periodo >= n_minimo].index)

    # --------------------------------------------------------------------------
    # 2. PARÂMETROS POR annual_period × INDICADOR
    # --------------------------------------------------------------------------
    linhas_parametros = []
    parametros = {}  # (periodo, coluna) -> (n, media, dp, lim_inf, lim_sup)

    for periodo in ordem_periodo.index:
        if periodo not in periodos_validos:
            continue
        recorte = df_base.loc[df_base["annual_period"] == periodo]
        for coluna in colunas_indicador:
            rotulo, unidade, _ = indicadores[coluna]
            valores = pd.to_numeric(recorte[coluna], errors="coerce").dropna()
            n = int(valores.shape[0])
            if n < 2:
                continue  # DP amostral indefinido
            media = float(valores.mean())
            desvio = float(valores.std(ddof=1))  # desvio padrão AMOSTRAL
            if not np.isfinite(desvio):
                continue
            limite_inferior = media - multiplicador * desvio
            limite_superior = media + multiplicador * desvio
            parametros[(periodo, coluna)] = (n, media, desvio, limite_inferior, limite_superior)
            linhas_parametros.append({
                "Período": periodo,
                "Indicador": rotulo,
                "Unidade": unidade,
                "Coluna técnica": coluna,
                "n (fazendas)": n,
                "Média": media,
                "Desvio padrão": desvio,
                "Limite inferior": limite_inferior,
                "Limite superior": limite_superior,
            })

    df_parametros = pd.DataFrame(
        linhas_parametros,
        columns=["Período", "Indicador", "Unidade", "Coluna técnica",
                 "n (fazendas)", "Média", "Desvio padrão",
                 "Limite inferior", "Limite superior"],
    )

    # --------------------------------------------------------------------------
    # 3. CLASSIFICAÇÃO POR LINHA (TODAS as linhas anuais, inclusive as
    #    inconsistentes e as duplicadas por consultor)
    # --------------------------------------------------------------------------
    periodos_linha = df["annual_period"].to_numpy()
    tem_parametro = np.array([p in periodos_validos for p in periodos_linha])

    marcado_por_coluna = {}
    for coluna in colunas_indicador:
        rotulo, unidade, formato = indicadores[coluna]
        valores = pd.to_numeric(df[coluna], errors="coerce").to_numpy(dtype=float)

        n_arr = np.full(len(df), np.nan)
        media_arr = np.full(len(df), np.nan)
        desvio_arr = np.full(len(df), np.nan)
        inferior_arr = np.full(len(df), np.nan)
        superior_arr = np.full(len(df), np.nan)
        for periodo in periodos_validos:
            chave = (periodo, coluna)
            if chave not in parametros:
                continue
            n, media, desvio, inferior, superior = parametros[chave]
            posicoes = periodos_linha == periodo
            n_arr[posicoes] = n
            media_arr[posicoes] = media
            desvio_arr[posicoes] = desvio
            inferior_arr[posicoes] = inferior
            superior_arr[posicoes] = superior

        avaliavel = np.isfinite(valores) & np.isfinite(desvio_arr) & tem_parametro
        # Exatamente no limite NÃO marca; imediatamente fora dele marca.
        acima = avaliavel & (valores > superior_arr)
        abaixo = avaliavel & (valores < inferior_arr)

        situacao = np.where(~avaliavel, "Não avaliado",
                    np.where(acima, "Acima",
                     np.where(abaixo, "Abaixo", "Normal")))
        df[f"is_outlier_{coluna}"] = situacao

        marcado_por_coluna[coluna] = {
            "marcado": acima | abaixo,
            "lado": situacao,
            "valor": valores,
            "n": n_arr,
            "media": media_arr,
            "desvio": desvio_arr,
            "inferior": inferior_arr,
            "superior": superior_arr,
            "z": np.divide(valores - media_arr, desvio_arr,
                           out=np.full(len(df), np.nan),
                           where=np.isfinite(desvio_arr) & (desvio_arr != 0)),
        }

    matriz_marcado = np.vstack([marcado_por_coluna[c]["marcado"] for c in colunas_indicador])
    contagem = matriz_marcado.sum(axis=0).astype(int)

    df["outlier_count_annual"] = contagem
    df["outlier_status_annual"] = np.where(
        ~tem_parametro, "Amostra insuficiente",
        np.where(contagem > 0, "Outlier", "Normal"),
    )

    # Lista de rótulos e detalhamento em texto, no mesmo formato de
    # annual_violated_consistency_details ("Nome (motivo); Nome (motivo)").
    lista_indicadores = []
    lista_detalhes = []
    for i in range(len(df)):
        if not tem_parametro[i]:
            lista_indicadores.append("Nenhum")
            lista_detalhes.append(f"Amostra insuficiente (período com menos de {n_minimo} fazendas)")
            continue
        rotulos = []
        motivos = []
        for coluna in colunas_indicador:
            dados = marcado_por_coluna[coluna]
            if not dados["marcado"][i]:
                continue
            rotulo, unidade, formato = indicadores[coluna]
            sufixo_unidade = f" {unidade}" if unidade else ""
            rotulos.append(rotulo)
            motivos.append(
                f"{rotulo} ({dados['lado'][i]}: "
                f"{_formatar_br(dados['valor'][i], formato)}{sufixo_unidade}; "
                f"faixa {_formatar_br(dados['inferior'][i], formato)} a "
                f"{_formatar_br(dados['superior'][i], formato)}; "
                f"z={_formatar_br(dados['z'][i], '+.1f')})"
            )
        lista_indicadores.append("; ".join(rotulos) if rotulos else "Nenhum")
        lista_detalhes.append("; ".join(motivos) if motivos else "Nenhum")

    df["outlier_indicators_annual"] = lista_indicadores
    df["outlier_details_annual"] = lista_detalhes
    # Desfragmenta após as inserções (mesmo cuidado da 4.2).
    df = df.copy()

    # --------------------------------------------------------------------------
    # 4. CONSOLIDADO FAZENDA × INDICADOR MARCADO
    # --------------------------------------------------------------------------
    # Colapsa as até 40 janelas móveis da mesma fazenda em uma linha por indicador.
    colunas_id = [c for c in ["id_property", "property_entrepreneur_label",
                              "property_name", "labor_rural_code"] if c in df.columns]

    # Consultores por fazenda (a fazenda com dois consultores mostra os dois).
    if "consultant_name" in df.columns:
        consultores_por_fazenda = (
            df.dropna(subset=["consultant_name"])
              .groupby("id_property")["consultant_name"]
              .apply(lambda s: "; ".join(sorted(set(s.astype(str)))))
        )
    else:
        consultores_por_fazenda = pd.Series(dtype=object)

    linhas_consolidado = []
    for coluna in colunas_indicador:
        rotulo, unidade, formato = indicadores[coluna]
        dados = marcado_por_coluna[coluna]
        recorte = df.loc[dados["marcado"]].copy()
        if recorte.empty:
            continue
        recorte["_valor"] = dados["valor"][dados["marcado"]]
        recorte["_n"] = dados["n"][dados["marcado"]]
        recorte["_media"] = dados["media"][dados["marcado"]]
        recorte["_desvio"] = dados["desvio"][dados["marcado"]]
        recorte["_inferior"] = dados["inferior"][dados["marcado"]]
        recorte["_superior"] = dados["superior"][dados["marcado"]]
        recorte["_z"] = dados["z"][dados["marcado"]]
        recorte["_lado"] = dados["lado"][dados["marcado"]]
        # Uma fazenda com dois consultores tem a mesma janela repetida: contar
        # janelas distintas, não linhas.
        recorte = recorte.drop_duplicates(subset=["id_property", "annual_period"])
        recorte = recorte.sort_values(["id_property", "annual_period_end"])

        for id_property, grupo in recorte.groupby("id_property", sort=False):
            ultima = grupo.iloc[-1]
            registro = {}
            for coluna_id in colunas_id:
                registro[coluna_id] = ultima[coluna_id]
            registro["consultant_name"] = consultores_por_fazenda.get(id_property, None)
            registro["Indicador"] = rotulo
            registro["Unidade"] = unidade
            registro["Janelas marcadas"] = int(len(grupo))
            registro["Primeira janela marcada"] = grupo["annual_period"].iloc[0]
            registro["Última janela marcada"] = grupo["annual_period"].iloc[-1]
            registro["Lado (janela mais recente)"] = ultima["_lado"]
            registro["Valor (janela mais recente)"] = ultima["_valor"]
            registro["Média do grupo"] = ultima["_media"]
            registro["Desvio padrão"] = ultima["_desvio"]
            registro["Limite inferior"] = ultima["_inferior"]
            registro["Limite superior"] = ultima["_superior"]
            registro["z"] = ultima["_z"]
            registro["n (fazendas no período)"] = ultima["_n"]
            linhas_consolidado.append(registro)

    df_outliers = pd.DataFrame(linhas_consolidado)
    if not df_outliers.empty:
        df_outliers = df_outliers.sort_values(
            ["Janelas marcadas", "Indicador"], ascending=[False, True]
        ).reset_index(drop=True)

    RENOMEAR_CONSOLIDADO = {
        "id_property": "IDFazenda",
        "property_entrepreneur_label": "Fazenda - Produtor",
        "property_name": "Fazenda",
        "labor_rural_code": "Código LR",
        "consultant_name": "Consultor",
    }
    df_outliers = df_outliers.rename(columns=RENOMEAR_CONSOLIDADO)

    # --------------------------------------------------------------------------
    # 5. RESUMO POR CONSULTOR (ÚLTIMA JANELA DE CADA FAZENDA)
    # --------------------------------------------------------------------------
    # Cada vínculo conta: a fazenda com dois consultores aparece para os dois,
    # porque a responsabilidade é compartilhada. Assimetria deliberada em relação
    # à base de parâmetros, que usa filter_1 == 1 para não dobrar o peso na média.
    ultima_janela = df.groupby("id_property")["annual_period_end"].transform("max")
    df_ultima = df.loc[df["annual_period_end"].eq(ultima_janela)].copy()
    if "consultant_name" in df_ultima.columns:
        df_ultima["consultant_name"] = df_ultima["consultant_name"].fillna("(Sem consultor)")
    else:
        df_ultima["consultant_name"] = "(Sem consultor)"

    linhas_consultor = []
    for consultor, grupo in df_ultima.groupby("consultant_name", sort=True):
        atendidas = grupo["id_property"].nunique()
        marcadas = grupo.loc[grupo["outlier_status_annual"].eq("Outlier"), "id_property"].nunique()
        registro = {
            "Consultor": consultor,
            f"Fazendas atendidas [{AVISO_SOMA_CONSULTOR}]": atendidas,
            "Fazendas marcadas": marcadas,
            "% de fazendas marcadas": (100.0 * marcadas / atendidas) if atendidas else np.nan,
        }
        for coluna in colunas_indicador:
            rotulo, unidade, _ = indicadores[coluna]
            marcado = grupo[f"is_outlier_{coluna}"].isin(["Acima", "Abaixo"])
            registro[rotulo] = grupo.loc[marcado, "id_property"].nunique()
        linhas_consultor.append(registro)

    df_outliers_consultor = pd.DataFrame(linhas_consultor)
    if not df_outliers_consultor.empty:
        df_outliers_consultor = df_outliers_consultor.sort_values(
            "Fazendas marcadas", ascending=False
        ).reset_index(drop=True)

    return df, df_parametros, df_outliers, df_outliers_consultor

# ==============================================================================
# EXECUÇÃO SOBRE OS INDICADORES ANUAIS JÁ MONTADOS
# ==============================================================================
# A seção apenas LÊ df_indicadores_anuais e devolve o mesmo DataFrame com as
# colunas de diagnóstico + os três DataFrames que virarão abas novas na 4.5.
(
    df_indicadores_anuais,
    df_parametros_outlier,
    df_outliers,
    df_outliers_por_consultor,
) = calcular_outliers_anuais(df_indicadores_anuais)

_periodos_totais = df_indicadores_anuais["annual_period"].nunique()
_periodos_com_parametro = df_parametros_outlier["Período"].nunique()
_base = df_indicadores_anuais.loc[
    pd.to_numeric(df_indicadores_anuais["filter_1"], errors="coerce").eq(1)
    & pd.to_numeric(df_indicadores_anuais["annual_consistency_id"], errors="coerce").eq(0)
]

print("=" * 78)
print("ANÁLISE DE OUTLIERS ANUAIS (média ± "
      f"{MULTIPLICADOR_DP_OUTLIER} DP amostral, por annual_period)")
print("=" * 78)
print(f"Base de parâmetros (filter_1 == 1 e annual_consistency_id == 0): {len(_base):,} de {len(df_indicadores_anuais):,} linhas")
print(f"Períodos com parâmetro (n >= {N_MINIMO_OUTLIER} fazendas): {_periodos_com_parametro} de {_periodos_totais}")
print()
for _status, _qtd in df_indicadores_anuais["outlier_status_annual"].value_counts().items():
    print(f"  {_status:<22} {_qtd:>6,} linhas ({100 * _qtd / len(df_indicadores_anuais):.1f}%)")
print()
print("Linhas marcadas por indicador:")
for _coluna, (_rotulo, _unidade, _) in INDICADORES_OUTLIER.items():
    if f"is_outlier_{_coluna}" not in df_indicadores_anuais.columns:
        continue
    _marcadas = int(df_indicadores_anuais[f"is_outlier_{_coluna}"].isin(["Acima", "Abaixo"]).sum())
    print(f"  {_rotulo:<58} {_marcadas:>5,}")
print()
print("Distribuição da contagem de indicadores marcados por linha:")
_distribuicao = (
    df_indicadores_anuais.loc[df_indicadores_anuais["outlier_status_annual"].eq("Outlier"),
                              "outlier_count_annual"].value_counts().sort_index()
)
for _n, _qtd in _distribuicao.items():
    print(f"  {_n} indicador(es): {_qtd:,} linhas")
print()
print(f"Consolidado por fazenda x indicador: {len(df_outliers):,} linhas, "
      f"{df_outliers['IDFazenda'].nunique() if not df_outliers.empty else 0} de "
      f"{df_indicadores_anuais['id_property'].nunique()} fazendas marcadas em alguma janela")
print(f"Resumo por consultor: {len(df_outliers_por_consultor):,} consultores")
print(f"Parâmetros: {len(df_parametros_outlier):,} linhas (annual_period x indicador)")

display(df_parametros_outlier.head())
display(df_outliers.head())
display(df_outliers_por_consultor.head())

ANÁLISE DE OUTLIERS ANUAIS (média ± 2 DP amostral, por annual_period)
Base de parâmetros (filter_1 == 1 e annual_consistency_id == 0): 3,228 de 7,527 linhas
Períodos com parâmetro (n >= 30 fazendas): 20 de 33

  Normal                  5,532 linhas (73.5%)
  Outlier                 1,925 linhas (25.6%)
  Amostra insuficiente       70 linhas (0.9%)

Linhas marcadas por indicador:
  Preço médio do concentrado                                   469
  Margem líquida unitária                                      537
  Produção/mão de obra total                                   370
  Produção/área destinada à atividade                          490
  Produção/área destinada à atividade considerando reserva     530
  Renda do leite/renda atividade                               451

Distribuição da contagem de indicadores marcados por linha:
  1 indicador(es): 1,286 linhas
  2 indicador(es): 422 linhas
  3 indicador(es): 172 linhas
  4 indicador(es): 24 linhas
  5 indicador(es): 21 linhas

Cons

,Período,Indicador,Unidade,Coluna técnica,n (fazendas),Média,Desvio padrão,Limite inferior,Limite superior
0,Jan/24-Dec/24,Preço médio do concentrado,R$/kg,concentrate_mineral_price_annual,53,1.879447,0.353405,1.172637,2.586257
1,Jan/24-Dec/24,Margem líquida unitária,R$/litro,net_margin_liter_annual,53,0.579707,0.509392,-0.439078,1.598491
2,Jan/24-Dec/24,Produção/mão de obra total,litros/trabalhador/dia,milk_total_labor_day_annual,53,448.675910,227.310701,-5.945493,903.297313
3,Jan/24-Dec/24,Produção/área destinada à atividade,litros/hectare/ano,milk_hectare_activity_annual,53,14633.203245,10209.711442,-5786.219638,35052.626129
4,Jan/24-Dec/24,Produção/área destinada à atividade consideran...,litros/hectare/ano,milk_hectare_activity_com_reserva_annual,53,12341.039337,9182.529967,-6024.020597,30706.099271


,IDFazenda,Fazenda - Produtor,Fazenda,Código LR,Consultor,Indicador,Unidade,Janelas marcadas,Primeira janela marcada,Última janela marcada,Lado (janela mais recente),Valor (janela mais recente),Média do grupo,Desvio padrão,Limite inferior,Limite superior,z,n (fazendas no período)
0,89d1885f-e48a-4d0d-95a5-99bd8dce5fe3,Fazenda Santa Cruz - Adilson Antonio Coelho,Fazenda Santa Cruz,LR05258,THUANY LANCA PEREIRA SILVA,Produção/mão de obra total,litros/trabalhador/dia,20,Jan/24-Dec/24,Aug/25-Jul/26,Acima,1165.636999,430.772683,253.406321,-76.039959,937.585326,2.899945,76.0
1,fff52e9f-83b6-44db-b7a4-01da7e77aeaa,Fazenda Campo Alegre - Luiz Alexandre de Avelar,Fazenda Campo Alegre,LR05853,GISELE MARIA CARVALHO MAGALHAES,Produção/mão de obra total,litros/trabalhador/dia,20,Jan/24-Dec/24,Aug/25-Jul/26,Acima,1189.729359,430.772683,253.406321,-76.039959,937.585326,2.995019,76.0
2,89d1885f-e48a-4d0d-95a5-99bd8dce5fe3,Fazenda Santa Cruz - Adilson Antonio Coelho,Fazenda Santa Cruz,LR05258,THUANY LANCA PEREIRA SILVA,Produção/área destinada à atividade,litros/hectare/ano,20,Jan/24-Dec/24,Aug/25-Jul/26,Acima,60110.245780,14642.392672,12245.994162,-9849.595652,39134.380996,3.712876,76.0
3,e6c94a46-6725-4930-8acd-840de8402efd,Fazenda Arco Íris - Luiz Alexandre de Avelar,Fazenda Arco Íris,LR05878,GISELE MARIA CARVALHO MAGALHAES,Produção/área destinada à atividade,litros/hectare/ano,20,Jan/24-Dec/24,Aug/25-Jul/26,Acima,923102.295918,14642.392672,12245.994162,-9849.595652,39134.380996,74.184251,76.0
4,fff52e9f-83b6-44db-b7a4-01da7e77aeaa,Fazenda Campo Alegre - Luiz Alexandre de Avelar,Fazenda Campo Alegre,LR05853,GISELE MARIA CARVALHO MAGALHAES,Produção/área destinada à atividade,litros/hectare/ano,20,Jan/24-Dec/24,Aug/25-Jul/26,Acima,46803.319842,14642.392672,12245.994162,-9849.595652,39134.380996,2.626241,76.0


,Consultor,"Fazendas atendidas [ATENÇÃO: cada vínculo conta - fazenda com mais de um consultor aparece para todos eles, logo a soma desta coluna EXCEDE o total de fazendas distintas]",Fazendas marcadas,% de fazendas marcadas,Preço médio do concentrado,Margem líquida unitária,Produção/mão de obra total,Produção/área destinada à atividade,Produção/área destinada à atividade considerando reserva,Renda do leite/renda atividade
0,(Sem consultor),35,14,40.000000,1,13,2,3,3,2
1,MATHEUS VARGAS DE CARVALHO,24,10,41.666667,6,2,1,2,2,0
2,ANDRESSA CRISTINA AUGUSTO PEREIRA,21,10,47.619048,1,2,3,4,4,1
3,VANESSA MARTINS FELIPPE DE FREITAS,8,8,100.000000,3,7,1,1,1,1
4,GISELE MARIA CARVALHO MAGALHAES,20,7,35.000000,0,0,5,5,5,1


### 4.4.1 Verificação da Análise de Outliers com Amostra Sintética

Exercita os casos de fronteira do método contra a função pura `calcular_outliers_anuais`,
com uma amostra sintética pequena — sem banco, sem SharePoint e sem dado real de cliente.

In [53]:
# ==============================================================================
# 4.4.1 VERIFICAÇÃO DA ANÁLISE DE OUTLIERS COM AMOSTRA SINTÉTICA
# ==============================================================================
# Exercita os casos de fronteira do método contra a função pura, sem banco, sem
# SharePoint e sem dado real de cliente (AGENTS.md: "prefira amostras sintéticas
# e pequenas"). Verifica comportamento externo observável - os números que saem
# da função -, não como as colunas foram montadas internamente.
# ==============================================================================

def _amostra_sintetica_outliers():
    """Monta um DataFrame anual sintético cobrindo os casos de fronteira do spec."""
    PERIODOS = {                       # período -> fim da janela (ordena "mais recente")
        "jan/21-dez/21": pd.Timestamp("2021-12-01"),   # 1 fazenda  -> DP indefinido
        "jan/22-dez/22": pd.Timestamp("2022-12-01"),   # 29 fazendas -> abaixo do mínimo
        "jan/23-dez/23": pd.Timestamp("2023-12-01"),   # 30 fazendas -> exatamente no mínimo
        "jan/24-dez/24": pd.Timestamp("2024-12-01"),   # 30 fazendas -> exatamente no mínimo
    }
    # Valor do indicador para a fazenda de índice i: base + i * passo. A dispersão
    # linear garante que nenhuma fazenda da base caia fora de média ± 2 DP
    # (o z máximo de 0..29 é 1,65), então todo caso marcado abaixo é intencional.
    ESCALA = {
        "concentrate_mineral_price_annual":         (1.00, 0.10),
        "net_margin_liter_annual":                  (0.10, 0.01),
        "milk_total_labor_day_annual":              (100.0, 10.0),
        "milk_hectare_activity_annual":             (5000.0, 500.0),
        "milk_hectare_activity_com_reserva_annual": (4000.0, 400.0),
        "rbl_rba_percentage_annual":                (70.0, 0.5),
    }

    def valores_da_fazenda(i):
        return {col: base + i * passo for col, (base, passo) in ESCALA.items()}

    linhas = []

    def adicionar(id_property, periodo, consultores, consistencia, valores, nome=None):
        for ordem, consultor in enumerate(consultores, start=1):
            registro = {
                "id_property": id_property,
                "property_name": nome or id_property,
                "property_entrepreneur_label": f"{nome or id_property} - Produtor {id_property}",
                "labor_rural_code": f"LR-{id_property}",
                "annual_period": periodo,
                "annual_period_end": PERIODOS[periodo],
                "annual_consistency_id": consistencia,
                "consultant_name": consultor,
                "filter_1": ordem,
            }
            registro.update(valores)
            linhas.append(registro)

    # --- Base: 30 fazendas em jan/23 e jan/24 (exatamente no mínimo) ----------
    # FAZ-00 tem DOIS consultores: precisa entrar UMA vez só na base (n = 30, não 31)
    # e aparecer para os dois no resumo por consultor.
    for periodo in ("jan/23-dez/23", "jan/24-dez/24"):
        for i in range(30):
            consultores = ["Consultor A", "Consultor B"] if i == 0 else ["Consultor A"]
            adicionar(f"FAZ-{i:02d}", periodo, consultores, 0, valores_da_fazenda(i))

    # --- Período abaixo do mínimo (29 fazendas) e período de uma fazenda só ---
    for i in range(29):
        adicionar(f"FAZ-{i:02d}", "jan/22-dez/22", ["Consultor A"], 0, valores_da_fazenda(i))
    adicionar("FAZ-00", "jan/21-dez/21", ["Consultor A"], 0, valores_da_fazenda(0))

    # --- Sondas: annual_consistency_id = 1 para NÃO entrarem na base de parâmetros,
    #     mas ainda assim serem classificadas (nenhuma fazenda fica sem diagnóstico).
    serie_base = pd.Series([1.00 + i * 0.10 for i in range(30)])
    media_preco = serie_base.mean()
    desvio_preco = serie_base.std(ddof=1)
    limite_superior_preco = media_preco + 2 * desvio_preco

    # Exatamente no limite: NÃO marca.
    v = valores_da_fazenda(10)
    v["concentrate_mineral_price_annual"] = limite_superior_preco
    adicionar("SONDA-LIMITE", "jan/24-dez/24", ["Consultor C"], 1, v)

    # Imediatamente fora do limite: marca.
    v = valores_da_fazenda(10)
    v["concentrate_mineral_price_annual"] = np.nextafter(limite_superior_preco, np.inf)
    adicionar("SONDA-FORA", "jan/24-dez/24", ["Consultor C"], 1, v)

    # Indicador sem valor: "Não avaliado", sem contar como violação.
    v = valores_da_fazenda(10)
    v["concentrate_mineral_price_annual"] = np.nan
    adicionar("SONDA-AUSENTE", "jan/24-dez/24", ["Consultor C"], 1, v)

    # Fazenda com DOIS consultores marcada em DUAS janelas, com valores diferentes:
    # colapsa em uma linha por indicador, com os números vindos da janela mais recente.
    v = valores_da_fazenda(10)
    v["concentrate_mineral_price_annual"] = 9.00
    adicionar("SONDA-DUPLA", "jan/23-dez/23", ["Consultor B", "Consultor C"], 1, v)
    v = valores_da_fazenda(10)
    v["concentrate_mineral_price_annual"] = 12.00
    adicionar("SONDA-DUPLA", "jan/24-dez/24", ["Consultor B", "Consultor C"], 1, v)

    return pd.DataFrame(linhas), media_preco, desvio_preco, limite_superior_preco


def verificar_outliers_anuais_com_amostra_sintetica():
    """Roda a função pura contra a amostra sintética e confere os casos de fronteira."""
    df_sintetico, media_preco, desvio_preco, limite_superior_preco = _amostra_sintetica_outliers()

    df_ok, par_ok, cons_ok, consultor_ok = calcular_outliers_anuais(df_sintetico)

    falhas = []

    def conferir(descricao, condicao, obtido=""):
        if condicao:
            print(f"  OK   {descricao}")
        else:
            falhas.append(f"{descricao} -> {obtido}")
            print(f"  FALHA {descricao} -> {obtido}")

    # 1. N mínimo: exatamente no mínimo gera parâmetro; logo abaixo, não.
    periodos_com_parametro = set(par_ok["Período"])
    conferir("período com n exatamente no mínimo (30) gera parâmetro",
             {"jan/23-dez/23", "jan/24-dez/24"} <= periodos_com_parametro,
             sorted(periodos_com_parametro))
    conferir("período logo abaixo do mínimo (29) NÃO gera parâmetro",
             "jan/22-dez/22" not in periodos_com_parametro)

    # 2. Período de uma única fazenda (DP indefinido) sai como Amostra insuficiente.
    uma_fazenda = df_ok.loc[df_ok["annual_period"].eq("jan/21-dez/21")]
    conferir("período com uma única fazenda -> Amostra insuficiente (sem quebrar)",
             uma_fazenda["outlier_status_annual"].eq("Amostra insuficiente").all(),
             uma_fazenda["outlier_status_annual"].tolist())
    abaixo_minimo = df_ok.loc[df_ok["annual_period"].eq("jan/22-dez/22")]
    conferir("período abaixo do mínimo -> Amostra insuficiente em todas as linhas",
             abaixo_minimo["outlier_status_annual"].eq("Amostra insuficiente").all())

    # 3. Fazenda com dois consultores entra UMA vez só na base de parâmetros.
    n_jan24 = par_ok.loc[
        par_ok["Período"].eq("jan/24-dez/24")
        & par_ok["Coluna técnica"].eq("concentrate_mineral_price_annual"),
        "n (fazendas)",
    ].iloc[0]
    conferir("fazenda com dois consultores conta uma vez na base (n = 30)",
             int(n_jan24) == 30, int(n_jan24))

    # 3b. Média e DP conferem com a conta feita à mão sobre os mesmos 30 valores.
    media_calculada = par_ok.loc[
        par_ok["Período"].eq("jan/24-dez/24")
        & par_ok["Coluna técnica"].eq("concentrate_mineral_price_annual"), "Média"].iloc[0]
    desvio_calculado = par_ok.loc[
        par_ok["Período"].eq("jan/24-dez/24")
        & par_ok["Coluna técnica"].eq("concentrate_mineral_price_annual"), "Desvio padrão"].iloc[0]
    conferir("média e DP amostral (ddof=1) batem com a conta refeita à mão",
             np.isclose(media_calculada, media_preco) and np.isclose(desvio_calculado, desvio_preco),
             f"{media_calculada} / {desvio_calculado}")

    # 4. Valor exatamente no limite NÃO marca; imediatamente fora dele marca.
    linha_limite = df_ok.loc[df_ok["id_property"].eq("SONDA-LIMITE")].iloc[0]
    conferir("valor exatamente no limite superior NÃO marca",
             linha_limite["is_outlier_concentrate_mineral_price_annual"] == "Normal",
             linha_limite["is_outlier_concentrate_mineral_price_annual"])
    linha_fora = df_ok.loc[df_ok["id_property"].eq("SONDA-FORA")].iloc[0]
    conferir("valor imediatamente acima do limite marca como Acima",
             linha_fora["is_outlier_concentrate_mineral_price_annual"] == "Acima",
             linha_fora["is_outlier_concentrate_mineral_price_annual"])

    # 5. Indicador sem valor -> Não avaliado, sem contar como violação.
    linha_ausente = df_ok.loc[df_ok["id_property"].eq("SONDA-AUSENTE")].iloc[0]
    conferir("indicador sem valor -> Não avaliado",
             linha_ausente["is_outlier_concentrate_mineral_price_annual"] == "Não avaliado",
             linha_ausente["is_outlier_concentrate_mineral_price_annual"])
    conferir("indicador sem valor não conta como violação (contagem = 0, status Normal)",
             int(linha_ausente["outlier_count_annual"]) == 0
             and linha_ausente["outlier_status_annual"] == "Normal",
             f"{linha_ausente['outlier_count_annual']} / {linha_ausente['outlier_status_annual']}")

    # 6. Fazenda marcada em várias janelas colapsa em uma linha por indicador.
    linhas_dupla = cons_ok.loc[
        cons_ok["IDFazenda"].eq("SONDA-DUPLA")
        & cons_ok["Indicador"].eq("Preço médio do concentrado")
    ]
    conferir("fazenda marcada em 2 janelas vira UMA linha por indicador",
             len(linhas_dupla) == 1, len(linhas_dupla))
    if len(linhas_dupla) == 1:
        r = linhas_dupla.iloc[0]
        conferir("contagem de janelas marcadas = 2", int(r["Janelas marcadas"]) == 2,
                 r["Janelas marcadas"])
        conferir("primeira e última janela marcadas corretas",
                 r["Primeira janela marcada"] == "jan/23-dez/23"
                 and r["Última janela marcada"] == "jan/24-dez/24",
                 f"{r['Primeira janela marcada']} .. {r['Última janela marcada']}")
        conferir("valor/limites vêm da janela MAIS RECENTE (12,00 e não 9,00)",
                 np.isclose(r["Valor (janela mais recente)"], 12.00)
                 and np.isclose(r["Limite superior"], limite_superior_preco),
                 r["Valor (janela mais recente)"])
        conferir("os dois consultores da fazenda aparecem no consolidado",
                 r["Consultor"] == "Consultor B; Consultor C", r["Consultor"])

    # 7. Resumo por consultor: cada vínculo conta.
    coluna_atendidas = [c for c in consultor_ok.columns if c.startswith("Fazendas atendidas")][0]
    resumo = consultor_ok.set_index("Consultor")
    conferir("fazenda com dois consultores aparece nos DOIS (FAZ-00 em A e B)",
             int(resumo.loc["Consultor A", coluna_atendidas]) == 30
             and int(resumo.loc["Consultor B", coluna_atendidas]) == 2,
             {c: int(resumo.loc[c, coluna_atendidas]) for c in resumo.index})
    # A única fazenda marcada do Consultor B é a SONDA-DUPLA (FAZ-00 é Normal);
    # o Consultor C soma SONDA-DUPLA + SONDA-FORA. Se a SONDA-DUPLA fosse contada
    # para um consultor só, um dos dois cairia para 0 e 1.
    conferir("SONDA-DUPLA marcada aparece marcada para os DOIS consultores",
             int(resumo.loc["Consultor B", "Fazendas marcadas"]) == 1
             and int(resumo.loc["Consultor C", "Fazendas marcadas"]) == 2,
             {c: int(resumo.loc[c, "Fazendas marcadas"]) for c in resumo.index})
    fazendas_distintas = df_ok["id_property"].nunique()
    conferir("a soma das linhas do resumo EXCEDE o total de fazendas distintas",
             int(resumo[coluna_atendidas].sum()) > fazendas_distintas,
             f"{int(resumo[coluna_atendidas].sum())} vs {fazendas_distintas}")
    conferir("o aviso está no cabeçalho da coluna de fazendas atendidas",
             "EXCEDE o total de fazendas distintas" in coluna_atendidas, coluna_atendidas)

    # 8. A função é pura: não altera o DataFrame de entrada.
    conferir("a função não altera o DataFrame de entrada (sem colunas novas nele)",
             not any(c.startswith("is_outlier_") or c.startswith("outlier_")
                     for c in df_sintetico.columns))

    if falhas:
        raise AssertionError(
            f"{len(falhas)} verificação(ões) da amostra sintética falharam:\n  - "
            + "\n  - ".join(falhas)
        )
    print(f"\nAmostra sintética: {len(df_sintetico):,} linhas, "
          f"{df_sintetico['id_property'].nunique()} fazendas, "
          f"{df_sintetico['annual_period'].nunique()} períodos - todas as verificações passaram.")


verificar_outliers_anuais_com_amostra_sintetica()

  OK   período com n exatamente no mínimo (30) gera parâmetro
  OK   período logo abaixo do mínimo (29) NÃO gera parâmetro
  OK   período com uma única fazenda -> Amostra insuficiente (sem quebrar)
  OK   período abaixo do mínimo -> Amostra insuficiente em todas as linhas
  OK   fazenda com dois consultores conta uma vez na base (n = 30)
  OK   média e DP amostral (ddof=1) batem com a conta refeita à mão
  OK   valor exatamente no limite superior NÃO marca
  OK   valor imediatamente acima do limite marca como Acima
  OK   indicador sem valor -> Não avaliado
  OK   indicador sem valor não conta como violação (contagem = 0, status Normal)
  OK   fazenda marcada em 2 janelas vira UMA linha por indicador
  OK   contagem de janelas marcadas = 2
  OK   primeira e última janela marcadas corretas
  OK   valor/limites vêm da janela MAIS RECENTE (12,00 e não 9,00)
  OK   os dois consultores da fazenda aparecem no consolidado
  OK   fazenda com dois consultores aparece nos DOIS (FAZ-00 em A e B)


## 4.5 Exportação dos Arquivos Excel Anuais (Multi-Abas)

In [54]:
# # ==============================================================================
# # 4.6 EXPORTAÇÃO DOS ARQUIVOS EXCEL ANUAIS (MULTI-ABAS)
# # ==============================================================================
# # Garante que a raiz do projeto seja a pasta pai caso o notebook rode dentro da pasta /app
RAIZ_PROJETO = Path.cwd().parent if Path.cwd().name == 'app' else Path.cwd()

DATA_EXPORTACAO = datetime.now().strftime("%Y_%m_%d_%H%M%S")
PASTA_SAIDA = RAIZ_PROJETO / 'data' / 'outputs' / 'annual'
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
CAMINHO_ANUAIS = PASTA_SAIDA / f"{DATA_EXPORTACAO}_indicadores_anuais.xlsx"

df_exportacao_anual = preparar_dataframe_para_excel(preencher_numericos_vazios_com_zero(df_indicadores_anuais))
df_exportacao_anual = df_exportacao_anual.sort_values(["id_property", "annual_period_end"]).reset_index(drop=True)

# # Aba extra com os mesmos nomes/ordem de colunas do antigo calculo_medias.xlsx.
df_exportacao_calculo_medias = preparar_dataframe_para_excel(preencher_numericos_vazios_com_zero(df_calculo_medias))

# # Aba extra com o relatório de cobertura de lançamento de dados (separado dos indicadores).
df_exportacao_cobertura = preparar_dataframe_para_excel(preencher_numericos_vazios_com_zero(df_cobertura_anual))

# # Aba extra com o relatório de auditoria de consistência mensal e anual.
COLUNAS_CONSISTENCIA_EXPORT = [
    c for c in [
        "id_property", "property_name", "labor_rural_code", "annual_period",
        "annual_consistency_status", "inconsistent_months_annual",
        "annual_monthly_inconsistency_details", "annual_violated_consistency_criteria",
        "annual_violated_consistency_details"
    ] if c in df_indicadores_anuais.columns
]
df_exportacao_consistencia = preparar_dataframe_para_excel(preencher_numericos_vazios_com_zero(df_indicadores_anuais[COLUNAS_CONSISTENCIA_EXPORT]))

# # Abas novas da análise de outliers (4.4). Passam por preparar_dataframe_para_excel
# # mas NAO por preencher_numericos_vazios_com_zero: um z ausente virando 0 leria
# # como "exatamente na média", e "não avaliado" viraria "zero".
df_exportacao_outliers = preparar_dataframe_para_excel(df_outliers)
df_exportacao_parametros_outlier = preparar_dataframe_para_excel(df_parametros_outlier)
df_exportacao_outliers_consultor = preparar_dataframe_para_excel(df_outliers_por_consultor)

exportar_varias_abas_xlsx(
    abas={
        "Indicadores Anuais": df_exportacao_anual,
        "Cálculo de Médias": df_exportacao_calculo_medias,
        "Cobertura de Dados": df_exportacao_cobertura,
        "Consistência de Dados": df_exportacao_consistencia,
        "Outliers": df_exportacao_outliers,
        "Parâmetros de Outlier": df_exportacao_parametros_outlier,
        "Outliers por Consultor": df_exportacao_outliers_consultor,
    },
    caminho_saida=CAMINHO_ANUAIS,
    fonte="Aptos",
)
print(f"Linhas anuais exportadas: {len(df_exportacao_anual):,}")
print(f"Linhas na aba Outliers: {len(df_exportacao_outliers):,}")
print(f"Linhas na aba Parâmetros de Outlier: {len(df_exportacao_parametros_outlier):,}")
print(f"Linhas na aba Outliers por Consultor: {len(df_exportacao_outliers_consultor):,}")
print(f"Arquivo: {CAMINHO_ANUAIS}")

Linhas anuais exportadas: 7,527
Linhas na aba Outliers: 366
Linhas na aba Parâmetros de Outlier: 120
Linhas na aba Outliers por Consultor: 58
Arquivo: C:\Users\analy\LABOR RURAL\Analytics - Departamento Analytics\TEMP\GUILHERME\SCRIPTS\projetcs\elabore-views\data\outputs\annual\2026_08_19_120703_indicadores_anuais.xlsx
